# Genex Interview + Activity Brain — v22

Claude-ready notebook built from v21 with final first-bridge, concern-support, activity-family guardrails, and Week 1/Week 2 scheduling.

## Recommended run order for v22

1. Keep `cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx` in the same folder as this notebook.
2. Restart the kernel.
3. Run all cells through **Genex Brain v22 final patch** before running sample cases.
4. Confirm the output says `Loaded Genex Brain v22 patch`.
5. Run sample cases. The final schedule should say `V22 BRIDGE-STEP-1 WEEKLY SCHEDULE`.

Important: older v17/v20 cells remain only as internal legacy base layers. The final planner/display path is v22.

## v22 cleaned source guarantee

The final planner should load:
- file: `cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx`
- sheet: `all_with_bridge_family`
- legend: `activity_family_legend`

Older table filenames remain only as last-resort fallbacks.

In [1]:
# Install once per notebook kernel if needed
%pip install -U pandas openpyxl python-dotenv openai google-genai reportlab numbers-parser



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -U google-genai


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ------------------------------------------------------------
# API key visibility check (safe: reads environment only)
# ------------------------------------------------------------
import os
print("OPENAI_API_KEY visible:", bool(os.environ.get("OPENAI_API_KEY")))
print("GEMINI_API_KEY visible:", bool(os.environ.get("GEMINI_API_KEY")))


OPENAI_API_KEY visible: True
GEMINI_API_KEY visible: True


In [4]:
# ------------------------------------------------------------
# Imports + environment setup
# ------------------------------------------------------------
import os
import re
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

try:
    from google import genai
except ImportError:
    genai = None

from reportlab.lib.pagesizes import landscape, LETTER
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas
from reportlab.pdfbase.pdfmetrics import stringWidth

try:
    load_dotenv()
except AssertionError:
    # Some non-interactive notebook execution contexts can make python-dotenv
    # fail while searching parent frames. Environment variables are still read below.
    pass

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "genex_interview_activity_v17"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

openai_client = OpenAI(api_key=OPENAI_API_KEY) if (OpenAI is not None and OPENAI_API_KEY) else None
gemini_client = genai.Client(api_key=GEMINI_API_KEY) if (genai is not None and GEMINI_API_KEY) else None

print("OpenAI available:", openai_client is not None)
print("Gemini available:", gemini_client is not None)

# Optional temporary setup if needed for debugging.
# Do NOT save real keys inside the notebook file.
#
# import os
# os.environ["OPENAI_API_KEY"] = "your_real_openai_key_here"
# os.environ["GEMINI_API_KEY"] = "your_real_gemini_key_here"
# OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
# GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
# openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
# gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None


OpenAI available: True
Gemini available: True


In [5]:
# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
DOMAIN_CONFIG = {
    "movement_and_physical": {
        "display": "Movement / Physical",
        "short": "motor",
    },
    "social_and_emotional": {
        "display": "Social / Emotional",
        "short": "social_emotional",
    },
    "language_and_communication": {
        "display": "Language / Communication",
        "short": "language_communication",
    },
    "cognitive": {
        "display": "Cognitive / Adaptive",
        "short": "cognitive",
    },
}

ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "physical": "movement_and_physical",
    "motor": "movement_and_physical",
    "gross motor": "movement_and_physical",

    "social and emotional": "social_and_emotional",
    "social and emotial": "social_and_emotional",
    "social_emotional": "social_and_emotional",
    "social": "social_and_emotional",

    "language and communication": "language_and_communication",
    "language": "language_and_communication",
    "speech": "language_and_communication",
    "speech and language": "language_and_communication",

    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.4,
    "with_help": 0.2,
    "no": 0.0,
    "not_sure": 0.1,
}

VALID_ANSWERS = set(ANSWER_SCORES.keys())
VALID_PARENT_ANSWERS = {"yes", "sometimes", "with_help", "no", "not_sure"}


# In v9, all domains use a 3-stage band idea:
# confirmed / emerging / not_demonstrated.
#
# Emerging is allowed everywhere, but it is weighted more strongly in motor/postural
# subdomains because motor skills are often acquired gradually and inconsistently before
# stable mastery.
MOTOR_EMERGING_SUBDOMAINS = {
    "postural_control_and_transitions",
    "gross_motor_mobility_and_coordination",
    "fine_motor_hand_use",
}

GENERAL_EMERGING_PARTIAL_WEIGHT = 0.45
GENERAL_EMERGING_NO_PENALTY = 0.40

MOTOR_EMERGING_PARTIAL_WEIGHT = 0.70
MOTOR_EMERGING_NO_PENALTY = 0.25


In [6]:
# ------------------------------------------------------------
# Milestone source loading
# ------------------------------------------------------------
# v17 has two related sources:
#   1) the CDC milestone backbone used for onboarding questions and gap scoring
#   2) the bridge milestone map used after scoring to create step-by-step activities
#
# If Sara's advisor bridge file is present, this cell now derives the CDC backbone
# by deduplicating the CDC rows inside that bridge file. That prevents the notebook
# from silently using an older CDC-only file when the newer bridge table is available.
# The bridge map itself is still loaded later in the v17 Bridge Engine cell.

PREFERRED_BRIDGE_SOURCE_FILENAMES = [
    "cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx",
    "cdc_milestones_with_bridges_family_cleaned.xlsx",
    "cdc_milestones_with_bridges_family.xlsx",
    "cdc_milestones_with_bridges_previous(1).xlsx",
    "cdc_milestones_with_bridges_previous.xlsx",
    "cdc_milestones_with_bridges.xlsx",
    "cdc_milestones_with_bridges.numbers",
]

PREFERRED_CDC_FILENAMES = [
    "cdc_milestones.xlsx",
    "cdc_milestones(1).xlsx",
    "milestone-cdc-table-improved-subdomains-advisor.xlsx",
    "milestone-cdc-table-improved-subdomains-advisor(2).xlsx",
    "milestone-cdc-table-improved-subdomains.xlsx",
    "milestone-cdc-table-improved-subdomains(2).xlsx",
    "milestone-cdc-table.xlsx",
    "milestone-cdc-table(2).xlsx",
    "milestone-cdc-table(3).xlsx",
]
ADVISOR_SUPPLEMENTAL_SHEET_NAME = "advisor_supplemental_review"


def _milestone_search_roots() -> List[Path]:
    raw_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, PROJECT_ROOT, Path("/mnt/data")]
    search_roots = []
    for root in raw_roots:
        try:
            resolved = root.resolve()
        except Exception:
            resolved = root
        if str(resolved) == "/":
            continue
        if resolved not in search_roots:
            search_roots.append(resolved)
    return search_roots


def _find_named_file(names: List[str]) -> Optional[Path]:
    for name in names:
        for root in _milestone_search_roots():
            candidate = root / name
            if candidate.exists():
                return candidate.resolve()
    return None


def find_cdc_file(path: Optional[str] = None) -> Path:
    """Find the milestone source.

    Priority:
      1. Sara/advisor bridge table, if present
      2. CDC-only milestone table fallback

    The CDC-only table is still valid for the interview backbone, but the bridge
    table is preferred because it is the new single source you are editing for v17.
    """
    if path:
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Provided milestone file path does not exist: {path}")
        return path.resolve()

    bridge_candidate = _find_named_file(PREFERRED_BRIDGE_SOURCE_FILENAMES)
    if bridge_candidate is not None:
        return bridge_candidate

    cdc_candidate = _find_named_file(PREFERRED_CDC_FILENAMES)
    if cdc_candidate is not None:
        return cdc_candidate

    candidate_paths = []
    for root in _milestone_search_roots():
        if root.exists():
            candidate_paths.extend(root.rglob("cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx"))
            candidate_paths.extend(root.rglob("cdc_milestones_with_bridges_family_cleaned.xlsx"))
            candidate_paths.extend(root.rglob("cdc_milestones_with_bridges_family.xlsx"))
            candidate_paths.extend(root.rglob("cdc_milestones_with_bridges.numbers"))
            candidate_paths.extend(root.rglob("cdc_milestones_with_bridges.xlsx"))
            candidate_paths.extend(root.rglob("*family*cleaned*final*app*ready*.xlsx"))
            candidate_paths.extend(root.rglob("*family*cleaned*.xlsx"))
            candidate_paths.extend(root.rglob("*family*bridge*.xlsx"))
            candidate_paths.extend(root.rglob("*with*bridges*.numbers"))
            candidate_paths.extend(root.rglob("*with*bridges*.xlsx"))
            candidate_paths.extend(root.rglob("*milestone*cdc*table*.xlsx"))
            candidate_paths.extend(root.rglob("*milestone*subdomain*.xlsx"))

    candidate_paths = list({p.resolve() for p in candidate_paths if p.exists()})
    if candidate_paths:
        def rank_path(p: Path):
            name = p.name.lower()
            if "family_cleaned_final_app_ready" in name:
                return (0, len(name))
            if "family_cleaned" in name:
                return (1, len(name))
            if "family" in name and "bridge" in name:
                return (2, len(name))
            if "previous" in name:
                return (4, len(name))
            if "with" in name and "bridge" in name:
                return (5, len(name))
            if "subdomain" in name or "improved" in name:
                return (6, len(name))
            return (7, len(name))
        candidate_paths = sorted(candidate_paths, key=rank_path)
        return candidate_paths[0]

    raise FileNotFoundError(
        "Could not find a milestone spreadsheet. Put cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx "
        "or the CDC-only milestone table in the repo root, notebooks parent folder, or /mnt/data."
    )


def _clean_source_column_name(col: Any, position: int) -> str:
    raw = str(col or "").strip().lower()
    raw = re.sub(r"\s+", " ", raw)
    if raw in {"", "none", "nan"}:
        return f"col_{position}"
    return raw.replace(" ", "_")


def _read_numbers_first_usable_table(path: Path) -> Tuple[pd.DataFrame, str]:
    """Read the first usable table from an Apple Numbers file.

    Requires numbers-parser. If this fails locally, the easiest fix is to export
    cdc_milestones_with_bridges.numbers to cdc_milestones_with_bridges.xlsx.
    """
    try:
        from numbers_parser import Document
    except Exception as exc:
        raise ImportError(
            "Apple Numbers files require the 'numbers-parser' package. Run `%pip install numbers-parser` "
            "or export cdc_milestones_with_bridges.numbers to cdc_milestones_with_bridges.xlsx."
        ) from exc

    doc = Document(str(path))
    chosen = None
    sheet_name = "Sheet"
    table_name = "Table"
    for sheet in doc.sheets:
        for table in getattr(sheet, "tables", []):
            if getattr(table, "num_rows", 0) >= 2 and getattr(table, "num_cols", 0) >= 4:
                chosen = table
                sheet_name = getattr(sheet, "name", "Sheet")
                table_name = getattr(table, "name", "Table")
                break
        if chosen is not None:
            break
    if chosen is None:
        raise ValueError(f"No usable table found in Numbers file: {path}")

    rows = []
    for r in range(chosen.num_rows):
        row = []
        for c in range(chosen.num_cols):
            try:
                row.append(chosen.cell(r, c).value)
            except Exception:
                row.append(None)
        rows.append(row)

    header = rows[0]
    columns = [_clean_source_column_name(header[i], i) for i in range(len(header))]
    df = pd.DataFrame(rows[1:], columns=columns)
    return df, f"{sheet_name}/{table_name}"


def _read_candidate_milestone_tables(path: Path) -> List[Tuple[pd.DataFrame, str]]:
    suffix = path.suffix.lower()
    if suffix == ".numbers":
        df, name = _read_numbers_first_usable_table(path)
        return [(df, name)]

    xls = pd.ExcelFile(path)
    candidates = []
    # Prefer likely all/bridge sheets, then scan the rest if needed.
    preferred_names = []
    for sheet_name in xls.sheet_names:
        low = sheet_name.lower()
        if low == "all_with_bridge_family":
            preferred_names.insert(0, sheet_name)
        elif low in {"all_with_bridge", "all", "bridge_milestone_map", "bridge_map", "bridge_map_36_60"} or "bridge" in low:
            preferred_names.append(sheet_name)
    ordered = preferred_names + [s for s in xls.sheet_names if s not in preferred_names]
    for sheet_name in ordered:
        try:
            candidates.append((pd.read_excel(xls, sheet_name=sheet_name), sheet_name))
        except Exception:
            continue
    return candidates


def _normalize_cdc_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [_clean_source_column_name(c, i) for i, c in enumerate(df.columns)]

    rename_map = {
        "category_": "category",
        "milestone_": "milestone",
        "cdc_milestone": "milestone",
        "cdc_milestone_text": "milestone",
        "milestone_text": "milestone",
        "main_category": "category",
        "target_subdomain": "subdomain",
    }
    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    if "subdomain" not in df.columns:
        df["subdomain"] = "unspecified"

    required = {"months", "category", "milestone"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Table is missing required milestone columns: {sorted(missing)}")

    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["subdomain"] = df["subdomain"].fillna("unspecified").astype(str).str.strip().str.lower()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")

    df = df.dropna(subset=["months", "category", "milestone"]).copy()
    df = df[df["milestone"].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA}).notna()].copy()
    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))

    # If the source is the bridge table, there are multiple rows per CDC milestone.
    # Deduplicate here so onboarding still asks each CDC milestone only once.
    df = df.drop_duplicates(subset=["months", "category_key", "milestone", "subdomain"]).copy()
    df = df.sort_values(["months", "category_key", "milestone"]).reset_index(drop=True)
    df["question_id"] = [
        f"{row.category_key}_{row.months}_{i}"
        for i, row in enumerate(df.itertuples(), start=1)
    ]
    return df


def _normalize_supplemental_sheet(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [_clean_source_column_name(c, i) for i, c in enumerate(df.columns)]

    rename_map = {
        "main_category": "category",
        "target_subdomain": "subdomain",
        "proposed_supplemental_milestone_or_followup": "milestone",
        "include_in_model_after_review": "include_in_model",
    }
    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    for col in ["category", "subdomain", "milestone", "include_in_model"]:
        if col not in df.columns:
            df[col] = None
    if "months" not in df.columns:
        df["months"] = None

    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["subdomain"] = df["subdomain"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["include_in_model"] = df["include_in_model"].astype(str).str.strip().str.lower()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")

    include_values = {"yes", "approved", "include", "1", "true", "y"}
    df = df[
        df["include_in_model"].isin(include_values)
        & df["months"].notna()
        & df["category"].astype(bool)
        & df["milestone"].astype(bool)
        & df["subdomain"].astype(bool)
    ].copy()

    if df.empty:
        return df

    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    df["question_id"] = [
        f"{row.category_key}_{row.months}_supp_{i}"
        for i, row in enumerate(df.itertuples(), start=1)
    ]
    df["source"] = "advisor_supplemental"
    return df[["months", "category", "milestone", "subdomain", "category_key", "question_id", "source"]]


def load_cdc_table(path: Optional[str] = None):
    """Load the milestone bank used for interview questions and gap estimation.

    If the advisor-authored bridge table is present, derive the CDC question bank
    from it by deduplicating its CDC milestone rows. The bridge rows themselves
    are loaded later by the v17 Bridge Milestone Engine.
    """
    path = find_cdc_file(path)
    candidates = _read_candidate_milestone_tables(path)

    last_error = None
    base_df = None
    selected_sheet = None
    source_has_bridge_columns = False
    raw_selected_columns = []

    for raw_df, sheet_name in candidates:
        raw_cols = [_clean_source_column_name(c, i) for i, c in enumerate(raw_df.columns)]
        try:
            normalized = _normalize_cdc_dataframe(raw_df)
        except Exception as exc:
            last_error = exc
            continue
        if not normalized.empty:
            base_df = normalized
            selected_sheet = sheet_name
            raw_selected_columns = raw_cols
            source_has_bridge_columns = any("bridge" in c for c in raw_cols) or "with_bridges" in path.name.lower()
            break

    if base_df is None:
        raise ValueError(f"Could not load a valid milestone table from {path}. Last error: {last_error}")

    base_df["source"] = "cdc_from_bridge_table" if source_has_bridge_columns else "cdc_backbone"
    approved_supplemental_df = pd.DataFrame(columns=base_df.columns)

    # Preserve the old advisor supplemental hook for CDC-only Excel sources.
    if path.suffix.lower() == ".xlsx" and not source_has_bridge_columns:
        try:
            xls = pd.ExcelFile(path)
            if ADVISOR_SUPPLEMENTAL_SHEET_NAME in xls.sheet_names:
                supplemental_raw = pd.read_excel(xls, sheet_name=ADVISOR_SUPPLEMENTAL_SHEET_NAME)
                approved_supplemental_df = _normalize_supplemental_sheet(supplemental_raw)
                if not approved_supplemental_df.empty:
                    missing_cols = [c for c in base_df.columns if c not in approved_supplemental_df.columns]
                    for c in missing_cols:
                        approved_supplemental_df[c] = None
                    approved_supplemental_df = approved_supplemental_df[base_df.columns]
        except Exception as exc:
            print(f"Warning: could not read advisor supplemental sheet: {exc}")

    combined_df = pd.concat([base_df, approved_supplemental_df], ignore_index=True, sort=False)
    combined_df = combined_df.sort_values(["months", "category", "milestone"]).reset_index(drop=True)

    return combined_df, path, approved_supplemental_df, selected_sheet, source_has_bridge_columns


cdc_df, cdc_path, approved_supplemental_df, cdc_sheet_name, cdc_source_is_bridge = load_cdc_table()
CDC_AGES = sorted(cdc_df["months"].dropna().unique().tolist())

print("Loaded milestone source file:", cdc_path)
print("Loaded milestone source sheet/table:", cdc_sheet_name)
print("Source type:", "CDC rows deduped from bridge table" if cdc_source_is_bridge else "CDC-only milestone backbone")
print("CDC ages:", CDC_AGES)
print("Rows in combined milestone bank:", len(cdc_df))
print("Rows from CDC backbone:", int((cdc_df.get("source", "") == "cdc_backbone").sum()))
print("Rows deduped from bridge table:", int((cdc_df.get("source", "") == "cdc_from_bridge_table").sum()))
print("Approved supplemental rows loaded:", len(approved_supplemental_df))
print("Has subdomain column:", "subdomain" in cdc_df.columns)
print("Unique subdomains:", cdc_df["subdomain"].nunique())
if not cdc_source_is_bridge:
    print("Note: Bridge milestones will still load later in the v17 Bridge Engine cell. To use your table as the primary source here, place cdc_milestones_with_bridges.numbers or .xlsx next to the notebook or repo root.")


def get_category_questions(category_key: str, min_months: int, max_months: int) -> pd.DataFrame:
    subset = cdc_df[
        (cdc_df["category_key"] == category_key)
        & (cdc_df["months"] >= min_months)
        & (cdc_df["months"] <= max_months)
    ].sort_values(["months", "milestone"])
    return subset.copy()


display(cdc_df.head())


Loaded milestone source file: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx
Loaded milestone source sheet/table: all_with_bridge_family
Source type: CDC rows deduped from bridge table
CDC ages: [2, 4, 6, 9, 12, 15, 18, 24, 30, 36, 48, 60]
Rows in combined milestone bank: 163
Rows from CDC backbone: 0
Rows deduped from bridge table: 163
Approved supplemental rows loaded: 0
Has subdomain column: True
Unique subdomains: 24


,months,category,subdomain,milestone,parent_explanation,bridge_step_number,bridge_step,activity_family,previous_bridge_step,previous_anchor_age,category_key,question_id,source
0,2,cognitive,attention_and_processing,looks at a toy for several seconds,Your baby can focus their eyes on a nearby toy...,1,Notices a high-contrast or nearby object.,visual_attention_tracking,NaN,NaN,cognitive,cognitive_2_1,cdc_from_bridge_table
1,2,cognitive,attention_and_processing,watches you as you move,Your baby can follow you with their eyes as yo...,1,Notices a nearby face or movement.,visual_attention_tracking,Notices a high-contrast or nearby object.,2.0,cognitive,cognitive_2_2,cdc_from_bridge_table
2,2,language and communication,early_vocalization_and_babbling,makes sounds other than crying,"Your baby makes little cooing, gurgling, or vo...",1,Makes sounds other than crying.,early_vocalization_sound_play,NaN,NaN,language_and_communication,language_and_communication_2_3,cdc_from_bridge_table
3,2,language and communication,receptive_language,reacts to loud sounds,"Your baby shows a response to a loud sound, su...",1,Reacts to loud sounds.,sound_response_orientation,NaN,NaN,language_and_communication,language_and_communication_2_4,cdc_from_bridge_table
4,2,movement and physical,postural_control_and_transitions,holds head up when on tummy,"During awake, supervised tummy time, your baby...",1,Tolerates brief tummy time while awake and sup...,head_control_tummy_time,NaN,NaN,movement_and_physical,movement_and_physical_2_5,cdc_from_bridge_table


In [7]:
# ------------------------------------------------------------
# Core helpers + state + concern router
# ------------------------------------------------------------
def normalize_answer(answer_text: str) -> str:
    """Normalize a raw parent answer into the allowed answer set."""
    t = str(answer_text).strip().lower().replace(" ", "_")
    if t in VALID_ANSWERS:
        return t
    if t in {"notsure", "unsure", "maybe"}:
        return "not_sure"
    return "not_sure"

In [8]:
FOLLOWUP_SCHEMAS = {
    "sometimes": {
        "prompt": "Why only sometimes? Choose the best fit:",
        "choices": [
            ("not_consistent_yet", "can do it, but not consistently yet"),
            ("distracted", "can do it, but gets distracted"),
            ("upset_or_refuses", "can do it, but gets upset or refuses"),
            ("only_some_situations", "can do it, but only in some situations"),
            ("not_sure", "not sure"),
        ],
    },
    "with_help": {
        "prompt": "What kind of help is usually needed?",
        "choices": [
            ("physical_help", "physical help / hands-on support"),
            ("reminders_prompting", "reminders or step-by-step prompting"),
            ("emotional_support", "encouragement or emotional support"),
            ("showing_first", "showing first / demonstrating"),
            ("not_sure", "not sure"),
        ],
    },
    "no": {
        "prompt": "Can you tell us a little more about the 'no'? Choose the best fit:",
        "choices": [
            ("not_able_yet", "not able yet"),
            ("does_not_do_even_when_we_try", "does not do it even when we try"),
            ("upset_or_refuses", "gets upset or refuses"),
            ("distracted_before_doing", "gets distracted before doing it"),
            ("not_sure_why", "not sure why"),
        ],
    },
}

In [9]:
FOLLOWUP_LABEL_TO_KEY = {
    answer_norm: {label.lower(): key for key, label in schema["choices"]}
    for answer_norm, schema in FOLLOWUP_SCHEMAS.items()
}

In [10]:
PERFORMANCE_BARRIER_SCORING = {
    ("yes", ""): {
        "skill_ability": "yes",
        "performance_barrier": "none",
        "scoring_norm_answer": "yes",
    },
    ("not_sure", ""): {
        "skill_ability": "unclear",
        "performance_barrier": "unclear",
        "scoring_norm_answer": "not_sure",
    },
    ("sometimes", "not_consistent_yet"): {
        "skill_ability": "emerging",
        "performance_barrier": "skill_emerging",
        "scoring_norm_answer": "sometimes",
    },
    ("sometimes", "distracted"): {
        "skill_ability": "yes",
        "performance_barrier": "distractibility",
        "scoring_norm_answer": "yes",
    },
    ("sometimes", "upset_or_refuses"): {
        "skill_ability": "yes",
        "performance_barrier": "emotional_dysregulation_refusal",
        "scoring_norm_answer": "yes",
    },
    ("sometimes", "only_some_situations"): {
        "skill_ability": "emerging",
        "performance_barrier": "situational_inconsistency",
        "scoring_norm_answer": "sometimes",
    },
    ("sometimes", "not_sure"): {
        "skill_ability": "unclear",
        "performance_barrier": "unclear",
        "scoring_norm_answer": "sometimes",
    },
    ("with_help", "physical_help"): {
        "skill_ability": "emerging",
        "performance_barrier": "physical_help",
        "scoring_norm_answer": "with_help",
    },
    ("with_help", "reminders_prompting"): {
        "skill_ability": "yes",
        "performance_barrier": "needs_prompting",
        "scoring_norm_answer": "yes",
    },
    ("with_help", "emotional_support"): {
        "skill_ability": "yes",
        "performance_barrier": "emotional_support",
        "scoring_norm_answer": "yes",
    },
    ("with_help", "showing_first"): {
        "skill_ability": "yes",
        "performance_barrier": "needs_demonstration",
        "scoring_norm_answer": "yes",
    },
    ("with_help", "not_sure"): {
        "skill_ability": "unclear",
        "performance_barrier": "unclear",
        "scoring_norm_answer": "with_help",
    },
    ("no", "not_able_yet"): {
        "skill_ability": "no",
        "performance_barrier": "not_able_yet",
        "scoring_norm_answer": "no",
    },
    ("no", "does_not_do_even_when_we_try"): {
        "skill_ability": "no",
        "performance_barrier": "persistent_failure",
        "scoring_norm_answer": "no",
    },
    ("no", "upset_or_refuses"): {
        "skill_ability": "yes",
        "performance_barrier": "emotional_dysregulation_refusal",
        "scoring_norm_answer": "yes",
    },
    ("no", "distracted_before_doing"): {
        "skill_ability": "yes",
        "performance_barrier": "distractibility",
        "scoring_norm_answer": "yes",
    },
    ("no", "not_sure_why"): {
        "skill_ability": "unclear",
        "performance_barrier": "unclear",
        "scoring_norm_answer": "not_sure",
    },
}

In [11]:
BEHAVIORAL_PERFORMANCE_BARRIERS = {
    "distractibility",
    "needs_prompting",
    "needs_demonstration",
    "emotional_support",
    "emotional_dysregulation_refusal",
    "situational_inconsistency",
}

In [12]:
def get_followup_schema(answer_norm: str) -> Optional[Dict[str, Any]]:
    return FOLLOWUP_SCHEMAS.get(str(answer_norm).strip())


In [13]:
def followup_label_from_key(answer_norm: str, followup_key: str) -> str:
    schema = get_followup_schema(answer_norm)
    if not schema or not followup_key:
        return ""
    for key, label in schema["choices"]:
        if key == followup_key:
            return label
    return ""


In [14]:
def normalize_followup_answer(answer_norm: str, raw_text: str) -> str:
    answer_norm = str(answer_norm).strip()
    schema = get_followup_schema(answer_norm)
    if not schema:
        return ""

    raw = str(raw_text).strip().lower()
    if not raw:
        return ""

    # numbered selection support for quick live entry
    if raw.isdigit():
        idx = int(raw) - 1
        if 0 <= idx < len(schema["choices"]):
            return schema["choices"][idx][0]

    raw_norm = raw.replace(" ", "_").replace("/", "_").replace("-", "_")
    choice_keys = {k for k, _ in schema["choices"]}
    if raw_norm in choice_keys:
        return raw_norm

    # exact label match
    if raw in FOLLOWUP_LABEL_TO_KEY[answer_norm]:
        return FOLLOWUP_LABEL_TO_KEY[answer_norm][raw]

    # fuzzy keyword matching
    if answer_norm == "sometimes":
        if "distract" in raw:
            return "distracted"
        if "refus" in raw or "upset" in raw or "tantrum" in raw:
            return "upset_or_refuses"
        if "situation" in raw or "setting" in raw or "sometimes at home" in raw:
            return "only_some_situations"
        if "not sure" in raw or "unsure" in raw:
            return "not_sure"
        return "not_consistent_yet"

    if answer_norm == "with_help":
        if "physical" in raw or "hand" in raw or "hands-on" in raw:
            return "physical_help"
        if "reminder" in raw or "prompt" in raw or "step" in raw:
            return "reminders_prompting"
        if "emotion" in raw or "encourag" in raw or "comfort" in raw:
            return "emotional_support"
        if "show" in raw or "demonstrat" in raw or "model" in raw:
            return "showing_first"
        return "not_sure"

    if answer_norm == "no":
        if "not able" in raw or "cannot yet" in raw or "can't yet" in raw:
            return "not_able_yet"
        if "refus" in raw or "upset" in raw or "tantrum" in raw:
            return "upset_or_refuses"
        if "distract" in raw:
            return "distracted_before_doing"
        if "not sure" in raw or "unsure" in raw:
            return "not_sure_why"
        return "does_not_do_even_when_we_try"

    return ""

In [15]:
def ask_live_followup(answer_norm: str) -> Tuple[str, str]:
    schema = get_followup_schema(answer_norm)
    if not schema:
        return "", ""
    print(schema["prompt"])
    for idx, (_, label) in enumerate(schema["choices"], start=1):
        print(f"  {idx}. {label}")
    raw = input("Type the number or phrase that fits best: ").strip()
    followup_key = normalize_followup_answer(answer_norm, raw)
    return followup_key, followup_label_from_key(answer_norm, followup_key)

In [16]:
def derive_performance_interpretation(answer_norm: str, followup_key: str = "") -> Dict[str, Any]:
    answer_norm = normalize_answer(answer_norm)
    followup_key = str(followup_key or "").strip()
    mapping = PERFORMANCE_BARRIER_SCORING.get((answer_norm, followup_key))

    if mapping is None:
        mapping = PERFORMANCE_BARRIER_SCORING.get((answer_norm, ""))

    if mapping is None:
        mapping = {
            "skill_ability": "unclear",
            "performance_barrier": "unclear",
            "scoring_norm_answer": answer_norm if answer_norm in VALID_PARENT_ANSWERS else "not_sure",
        }

    scoring_norm = mapping["scoring_norm_answer"]
    followup_label = followup_label_from_key(answer_norm, followup_key)
    return {
        "followup_key": followup_key,
        "followup_label": followup_label,
        "skill_ability": mapping["skill_ability"],
        "performance_barrier": mapping["performance_barrier"],
        "scoring_norm_answer": scoring_norm,
        "scoring_score": ANSWER_SCORES.get(scoring_norm, ANSWER_SCORES["not_sure"]),
    }

    return "not_sure"

In [17]:
def print_banner(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

In [18]:
def extract_json_block(text: str) -> dict:
    """Parse strict JSON or recover a simple answer from loosely formatted model text."""
    text = str(text).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    m = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    lowered = text.lower()
    for label in ["with_help", "not_sure", "sometimes", "yes", "no"]:
        if label in lowered:
            return {"answer": label, "reason": text[:200]}

    if "with help" in lowered:
        return {"answer": "with_help", "reason": text[:200]}
    if "not sure" in lowered or "unsure" in lowered or "maybe" in lowered:
        return {"answer": "not_sure", "reason": text[:200]}

    raise ValueError(f"Could not parse JSON from model output: {text[:300]}")


In [19]:
def new_state() -> Dict[str, Any]:
    """Initialize the working state dictionary for one case."""
    return {
        "child": {},
        "concern_profile": {},
        "delay_estimates": {},
        "qna": {},
        "dev_age": {},
        "activity_banks": {},
        "weekly_slot_allocation": {},
        "weekly_schedule": {},
        "safety_profile": {},
    }

In [20]:
SUBDOMAIN_TO_CATEGORY = {}
for subdomain, group in cdc_df.groupby("subdomain"):
    cats = [c for c in group["category_key"].dropna().astype(str).unique().tolist() if c]
    if cats:
        SUBDOMAIN_TO_CATEGORY[subdomain] = cats[0]

In [21]:
CATEGORY_TO_SUBDOMAINS = {
    category_key: sorted(
        cdc_df.loc[cdc_df["category_key"] == category_key, "subdomain"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    for category_key in DOMAIN_CONFIG
}

In [22]:
SUBDOMAIN_KEYWORD_MAP = {
    "speech_intelligibility": [
        r"\bapraxia\b",
        r"\bchildhood apraxia of speech\b",
        r"\bcas\b",
        r"hard to understand",
        r"unclear speech",
        r"inconsistent sounds",
        r"speech output",
        r"speech hard to understand",
        r"articulation",
        r"motor speech",
        r"intelligib",
    ],
    "expressive_language": [
        r"no words",
        r"few words",
        r"limited speech",
        r"limited words",
        r"only\s*~?\d+\s*words",
        r"very limited speech",
        r"speech delay",
        r"late talk",
        r"two[- ]word",
        r"phrases",
        r"naming",
    ],
    "receptive_language": [
        r"understands well",
        r"good comprehension",
        r"comprehension",
        r"doesn't understand",
        r"does not understand",
        r"follows? simple directions",
        r"follows? directions",
        r"commands",
        r"receptive",
    ],
    "gestural_communication": [
        r"uses gestures",
        r"gesture",
        r"pointing",
        r"\bpoints?\b",
        r"signs?",
        r"communicat",
    ],
    "early_vocalization_and_babbling": [
        r"limited babbling",
        r"\bbabbl",
        r"\bcoo",
        r"vocal",
        r"raspberr",
        r"squeal",
        r"makes sounds",
    ],
    "conversation_narrative": [
        r"conversation",
        r"answers questions",
        r"tell(s)? (a )?story",
        r"narrative",
    ],
    "social_engagement_and_joint_attention": [
        r"eye contact",
        r"limited eye contact",
        r"joint attention",
        r"shared attention",
        r"responds? to name",
        r"responds? to her name",
        r"responds? to his name",
        r"socially engaged",
        r"good eye contact",
    ],
    "peer_interaction_and_social_rules": [
        r"\bpeer",
        r"friends?",
        r"play with children",
        r"plays with other children",
        r"takes? turns?",
        r"interrupts",
        r"social but rigid",
        r"social rules",
    ],
    "play_and_symbolic_social_play": [
        r"pretend play",
        r"limited pretend play",
        r"symbolic play",
        r"repetitive play",
        r"lines up toys",
    ],
    "emotional_regulation": [
        r"tantrum",
        r"meltdown",
        r"big emotional reactions",
        r"gets frustrated",
        r"frustrat",
        r"emotional regulation",
        r"self-regulation",
        r"impulsive",
        r"very active",
        r"hyperactive",
        r"adhd",
    ],
    "attachment_and_separation": [
        r"clingy",
        r"separation",
        r"drop off",
        r"leave(s|) the room",
        r"when you leave",
    ],
    "empathy_and_prosocial_behavior": [
        r"empathy",
        r"notices when others are hurt",
        r"kind to others",
        r"hurt or upset",
    ],
    "gross_motor_mobility_and_coordination": [
        r"not walking",
        r"walking",
        r"frequent falls",
        r"stairs",
        r"balance",
        r"clumsy gait",
        r"\bgait\b",
        r"run",
        r"jump",
        r"walker",
        r"mobility",
    ],
    "postural_control_and_transitions": [
        r"not sitting",
        r"sitting independently",
        r"sit independently",
        r"head control",
        r"hypotonia",
        r"posture",
        r"rolling",
        r"get to sitting",
        r"pushes up",
    ],
    "fine_motor_hand_use": [
        r"fine motor",
        r"grasp",
        r"beads?",
        r"string",
        r"\bfork\b",
        r"crayon",
        r"pencil",
        r"hand use",
    ],
    "self_help_motor_skills": [
        r"self-care",
        r"dress",
        r"clothes",
        r"buttons?",
        r"zippers?",
        r"utensil",
        r"\bspoon\b",
    ],
    "adaptive_feeding_cues": [
        r"slow feeding",
        r"feeding",
        r"picky eating",
        r"oral motor",
        r"open mouth",
        r"close lips",
    ],
    "attention_and_processing": [
        r"attention",
        r"short attention span",
        r"\bfocus\b",
        r"processing",
    ],
    "concepts_and_following_directions": [
        r"follow(s|) directions",
        r"one-step",
        r"two-step",
        r"concepts?",
        r"colors?",
        r"letters?",
        r"numbers?",
    ],
    "exploration_and_object_use": [
        r"explore",
        r"object use",
        r"toy use",
        r"puts things in (his|her|their) mouth",
    ],
    "imitation_and_play_skills": [
        r"imitat",
        r"\bcopy",
        r"copies",
        r"play skills",
    ],
    "object_permanence_and_problem_solving": [
        r"problem solving",
        r"finds?",
        r"hides?",
        r"object permanence",
    ],
    "pre_academic_skills": [
        r"letters?",
        r"count(ing)?",
        r"colors?",
        r"pre-academic",
    ],
    "safety_awareness": [
        r"safety",
        r"danger",
        r"unsafe",
    ],
}


In [23]:
def _apply_concern_propagation(subdomain_weights: Dict[str, float]) -> Dict[str, float]:
    """Lightly propagate strong signals into related subdomains.

    This keeps the concern router systematic while avoiding brittle one-off hacks.
    """
    weights = dict(subdomain_weights)

    if weights.get("speech_intelligibility", 0) >= 0.60:
        weights["expressive_language"] = max(weights.get("expressive_language", 0), 0.70)

    if weights.get("expressive_language", 0) >= 0.60 and weights.get("gestural_communication", 0) >= 0.40:
        weights["early_vocalization_and_babbling"] = max(weights.get("early_vocalization_and_babbling", 0), 0.40)

    if weights.get("emotional_regulation", 0) >= 0.60:
        weights["concepts_and_following_directions"] = max(weights.get("concepts_and_following_directions", 0), 0.40)

    if weights.get("gross_motor_mobility_and_coordination", 0) >= 0.60:
        weights["postural_control_and_transitions"] = max(weights.get("postural_control_and_transitions", 0), 0.40)

    if weights.get("adaptive_feeding_cues", 0) >= 0.60:
        weights["self_help_motor_skills"] = max(weights.get("self_help_motor_skills", 0), 0.30)

    return weights

In [24]:
POSITIVE_ROUTING_HINTS = [
    "good eye contact",
    "good comprehension",
    "understands well",
    "good cognition",
    "socially engaged",
    "strong language skills",
    "strong language",
    "very verbal",
]

In [25]:
def _pattern_match_weight(pattern_text: str) -> float:
    pattern_text = str(pattern_text).lower()
    if any(hint in pattern_text for hint in POSITIVE_ROUTING_HINTS):
        return 0.18
    return 0.35

In [26]:
def concern_router(child: Dict[str, Any]) -> Dict[str, Any]:
    """Convert diagnosis + concern text into structured subdomain and domain weights.

    v6 keeps this deterministic and inspectable. The goal is not to diagnose from the concern,
    but to create a weak, transparent routing signal that can help question selection and
    borderline support decisions.
    """
    diagnosis = str(child.get("diagnosis", "") or "")
    concern = str(child.get("concern", "") or "")
    combined_text = f"{diagnosis} | {concern}".lower()

    subdomain_weights = {s: 0.0 for s in SUBDOMAIN_TO_CATEGORY.keys()}
    matched_patterns = {s: [] for s in SUBDOMAIN_TO_CATEGORY.keys()}

    for subdomain, patterns in SUBDOMAIN_KEYWORD_MAP.items():
        matches = []
        for pat in patterns:
            if re.search(pat, combined_text):
                matches.append(pat)

        if matches:
            weight = min(1.0, sum(_pattern_match_weight(pat) for pat in matches))
            subdomain_weights[subdomain] = max(subdomain_weights.get(subdomain, 0.0), weight)
            matched_patterns[subdomain] = matches

    subdomain_weights = _apply_concern_propagation(subdomain_weights)

    domain_weights = {k: 0.0 for k in DOMAIN_CONFIG.keys()}
    for subdomain, weight in subdomain_weights.items():
        category_key = SUBDOMAIN_TO_CATEGORY.get(subdomain)
        if category_key in domain_weights:
            domain_weights[category_key] = max(domain_weights[category_key], float(weight))

    top_subdomains = [
        {"subdomain": s, "weight": round(w, 2)}
        for s, w in sorted(subdomain_weights.items(), key=lambda kv: kv[1], reverse=True)
        if w > 0
    ][:8]

    return {
        "combined_text": combined_text,
        "subdomain_weights": subdomain_weights,
        "domain_weights": domain_weights,
        "matched_patterns": matched_patterns,
        "top_subdomains": top_subdomains,
    }


# ------------------------------------------------------------
# v15 focus-domain triage helpers
# ------------------------------------------------------------

V14_MAX_ONBOARDING_DOMAINS = 2
V14_TARGET_QUESTIONS_PER_BAND = 3
V14_MAX_BANDS = 3
V14_MAX_QUESTIONS_TOTAL = 9

def rank_focus_domains_v14(state: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Rank likely focus domains for shorter onboarding.

    v15 uses a hybrid signal:
    - concern-router domain + subdomain weights
    - rough delay estimate by domain

    The goal is not to diagnose. It is to choose the best 1–2 domains to review first.
    """
    concern_profile = ensure_concern_profile(state)
    child = state.get("child", {})
    chrono = max(2, min(int(child.get("chronological_months", 0) or 0), 60))

    ranked = []
    for category_key in DOMAIN_CONFIG:
        domain_weight = float(concern_profile.get("domain_weights", {}).get(category_key, 0.0))
        sub_peak = float(get_category_concern_peak(state, category_key))
        delay_months = float(state.get("delay_estimates", {}).get(category_key, {}).get("delay_months", 0.0))

        concern_signal = max(domain_weight, sub_peak * 0.85)
        delay_signal = min(1.0, delay_months / max(6.0, round(0.20 * chrono)))
        triage_score = round(0.65 * concern_signal + 0.35 * delay_signal, 3)

        ranked.append({
            "category_key": category_key,
            "display": DOMAIN_CONFIG[category_key]["display"],
            "triage_score": triage_score,
            "concern_signal": round(concern_signal, 3),
            "delay_signal": round(delay_signal, 3),
        })

    ranked = sorted(ranked, key=lambda x: (x["triage_score"], x["concern_signal"], x["delay_signal"]), reverse=True)
    return ranked

def choose_default_focus_domains_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    ranked = rank_focus_domains_v14(state)
    if not ranked:
        return list(DOMAIN_CONFIG.keys())[:max_domains]

    top = ranked[0]
    second = ranked[1] if len(ranked) > 1 else None

    use_n = 1
    if second and second["triage_score"] >= 0.35:
        use_n = 2
    if top["triage_score"] < 0.20:
        use_n = 2

    return [r["category_key"] for r in ranked[:min(use_n, max_domains)]]

def apply_focus_domain_selection_v14(
    state: Dict[str, Any],
    selected_categories: List[str],
    source: str = "auto",
) -> List[str]:
    selected = [k for k in selected_categories if k in DOMAIN_CONFIG]
    if not selected:
        selected = choose_default_focus_domains_v14(state)

    state["selected_categories"] = selected
    state["focus_domain_triage"] = {
        "selected_categories": selected,
        "selected_displays": [DOMAIN_CONFIG[k]["display"] for k in selected],
        "source": source,
        "ranked_domains": rank_focus_domains_v14(state),
    }
    return selected

def select_focus_domains_live_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    ranked = rank_focus_domains_v14(state)
    suggested = choose_default_focus_domains_v14(state, max_domains=max_domains)

    print("\n" + "=" * 100)
    print("FOCUS DOMAIN TRIAGE — v15")
    print("=" * 100)
    print("To keep onboarding shorter, Genex recommends starting with only 1–2 domains.")
    print("Suggested focus areas:")
    for idx, item in enumerate(ranked[:4], start=1):
        marker = " [suggested]" if item["category_key"] in suggested else ""
        print(f"  {idx}. {item['display']} | triage_score={item['triage_score']}{marker}")

    raw = input(
        "Press Enter to accept the suggested focus areas, or type up to two numbers separated by commas: "
    ).strip()

    if not raw:
        selected = suggested
        source = "live_confirm_default"
    else:
        picked = []
        parts = [p.strip() for p in raw.split(",") if p.strip()]
        for p in parts[:max_domains]:
            if p.isdigit():
                idx = int(p) - 1
                if 0 <= idx < len(ranked):
                    picked.append(ranked[idx]["category_key"])
        selected = picked or suggested
        source = "live_manual_override"

    selected = apply_focus_domain_selection_v14(state, selected, source=source)
    print("Selected focus areas:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in selected))
    return selected

def select_focus_domains_gemini_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    selected = apply_focus_domain_selection_v14(
        state,
        choose_default_focus_domains_v14(state, max_domains=max_domains),
        source="synthetic_auto",
    )
    print("\n" + "=" * 100)
    print("FOCUS DOMAIN TRIAGE — v15")
    print("=" * 100)
    print("Selected focus areas for this onboarding run:")
    for k in selected:
        print(f"- {DOMAIN_CONFIG[k]['display']}")
    return selected


In [27]:
def ensure_concern_profile(state: Dict[str, Any]) -> Dict[str, Any]:
    """Compute and cache the concern profile if needed."""
    if not state.get("concern_profile"):
        state["concern_profile"] = concern_router(state["child"])
    state["safety_profile"] = build_safety_profile(state["child"])
    return state["concern_profile"]

In [28]:
def print_concern_profile(state: Dict[str, Any]) -> None:
    """Print a compact, human-readable concern-routing summary."""
    profile = ensure_concern_profile(state)

    print("\n" + "=" * 100)
    print("CONCERN ROUTER PROFILE")
    print("=" * 100)

    domain_weights = profile.get("domain_weights", {})
    top_domains = [
        (DOMAIN_CONFIG[k]["display"], round(v, 2))
        for k, v in sorted(domain_weights.items(), key=lambda kv: kv[1], reverse=True)
        if v > 0
    ]
    if top_domains:
        print("Domain weights:")
        for name, weight in top_domains:
            print(f"- {name}: {weight}")
    else:
        print("Domain weights: no strong routed concern signals detected.")

    top_subdomains = profile.get("top_subdomains", [])
    if top_subdomains:
        print("Top subdomains:")
        for item in top_subdomains[:6]:
            print(f"- {item['subdomain']}: {item['weight']}")

In [29]:
def print_child_reference(state: Dict[str, Any]) -> None:
    child = state["child"]
    print("\n" + "=" * 100)
    print("CHILD REFERENCE CHECK")
    print("=" * 100)
    print(f"Name: {child.get('name', '')}")
    print(f"Chronological age: {child.get('chronological_months', '')} months")
    print(f"Diagnosis / condition: {child.get('diagnosis', '')}")
    print(f"Parent concern: {child.get('concern', '')}")
    print(f"Daily time available: {child.get('daily_time_min', '')} minutes")

In [30]:
def intake_agent_live(state: Dict[str, Any]) -> Dict[str, Any]:
    """Collect a live child profile from user input."""
    print_banner("INTAKE AGENT")

    name = input("What is your child's name? ").strip()
    chronological_months = int(
        input("How old is your child in months? (for example: 18, 24, 36, 48, 60) ").strip()
    )
    diagnosis = input("Any diagnosis / condition? (type 'none' if none) ").strip()
    concern = input("What are your main concerns right now? ").strip()
    daily_time_min = int(
        input("About how many minutes per day can you usually spend helping or playing with your child? ").strip()
    )

    state["child"] = {
        "name": name,
        "chronological_months": chronological_months,
        "diagnosis": diagnosis,
        "concern": concern,
        "daily_time_min": daily_time_min,
    }
    state["concern_profile"] = concern_router(state["child"])
    state["safety_profile"] = build_safety_profile(state["child"])
    return state["child"]

In [31]:
def init_state_from_case(case: Dict[str, Any], daily_time_min: int = 10) -> Dict[str, Any]:
    """Initialize a state object from a prefilled synthetic case."""
    state = new_state()
    state["child"] = {
        "name": case["child_name"],
        "chronological_months": int(case["age_months"]),
        "diagnosis": case["diagnosis"],
        "concern": case["concerns"],
        "daily_time_min": int(daily_time_min),
    }
    state["concern_profile"] = concern_router(state["child"])
    state["safety_profile"] = build_safety_profile(state["child"])
    return state

In [32]:
SAFETY_KEYWORD_MAP = {
    "falls_balance_gait": [
        r"frequent falls",
        r"fall",
        r"difficulty with stairs",
        r"stairs",
        r"clumsy gait",
        r"balance",
        r"walker",
        r"unsteady",
        r"cerebral palsy",
        r"ataxia",
    ],
    "postural_low_tone_fatigue": [
        r"hypotonia",
        r"low muscle tone",
        r"low tone",
        r"tires quickly",
        r"fatigue",
        r"weak",
        r"not sitting independently",
        r"not walking",
    ],
    "fine_motor_or_coordination": [
        r"fine motor",
        r"grip",
        r"coordination",
        r"clumsy hand",
        r"uses a fork",
        r"beads",
        r"crayon",
        r"pencil",
    ],
    "feeding_or_oral_motor": [
        r"feeding",
        r"slow feeding",
        r"chew",
        r"swallow",
        r"chok",
        r"drool",
        r"oral motor",
        r"oral-motor",
    ],
    "regulation_frustration": [
        r"tantrum",
        r"frustrat",
        r"difficulty with transitions",
        r"transitions",
        r"rigid",
        r"big emotional reactions",
        r"impuls",
        r"short attention",
        r"attention",
        r"adhd",
    ],
    "sensory_sensitivity": [
        r"sensory",
        r"sensitive",
        r"picky eating",
        r"texture",
        r"noise",
        r"overwhelm",
    ],
    "mobility_equipment_support": [
        r"walker",
        r"wheelchair",
        r"braces",
        r"orthotic",
        r"uses support",
    ],
    "seizure_or_medical_monitoring": [
        r"seizure",
        r"epilep",
        r"drop attack",
        r"medical frag",
    ],
    "high_activity_open_space_risk": [
        r"very active",
        r"bolts",
        r"runs off",
        r"elopes",
        r"unsafe climbing",
    ],
}


In [33]:
SAFETY_CONSTRAINT_TEMPLATES = {
    "falls_balance_gait": "Keep activities ground-level and closely supervised. Avoid high surfaces, jumping from heights, or tasks that assume stable balance without support.",
    "postural_low_tone_fatigue": "Prefer short bouts with rest breaks, supported positioning, and lower-endurance tasks. Avoid long sustained postures beyond the child's tolerance.",
    "fine_motor_or_coordination": "Adapt hand demands with larger tools/items, stabilizing support, and slower pacing rather than precision-heavy expectations.",
    "feeding_or_oral_motor": "Keep feeding or oral-motor tasks upright and closely supervised. Avoid choking-risk foods or unsafe oral-motor suggestions.",
    "regulation_frustration": "Keep tasks short, predictable, and easy to start. Build in transitions, choices, and stop before escalation or overload.",
    "sensory_sensitivity": "Use low-clutter, lower-noise materials when possible and introduce textures/sounds gradually.",
    "mobility_equipment_support": "Adapt tasks for walker/braces/other supports and do not assume unsupported stairs, standing, or walking.",
    "seizure_or_medical_monitoring": "Avoid activities that would be unsafe if sudden loss of awareness or control occurred. Keep supervision explicit and setups low risk.",
    "high_activity_open_space_risk": "Use contained spaces, clear physical boundaries, and active supervision. Avoid tasks that depend on open unsafe spaces or long waiting.",
}

In [34]:
def build_safety_profile(child: Dict[str, Any]) -> Dict[str, Any]:
    """Build a broad safety/practical constraint profile from diagnosis + concern text.

    v9 keeps the broad, reusable risk profile from v7 but now adds:
    - explicit hard-avoid lists for clearly risky activity types
    - preferred adaptation notes that can be passed directly into prompts
    - a deterministic post-generation safety pass downstream
    """
    diagnosis = str(child.get("diagnosis", "") or "")
    concern = str(child.get("concern", "") or "")
    combined_text = f"{diagnosis} | {concern}".lower()

    risk_scores = {k: 0.0 for k in SAFETY_KEYWORD_MAP.keys()}
    matched_patterns = {k: [] for k in SAFETY_KEYWORD_MAP.keys()}

    for risk_name, patterns in SAFETY_KEYWORD_MAP.items():
        matches = []
        for pat in patterns:
            if re.search(pat, combined_text):
                matches.append(pat)
        if matches:
            risk_scores[risk_name] = min(1.0, 0.35 * len(matches))
            matched_patterns[risk_name] = matches

    top_risks = [
        {"risk": k, "weight": round(v, 2)}
        for k, v in sorted(risk_scores.items(), key=lambda kv: kv[1], reverse=True)
        if v > 0
    ][:6]

    constraints = [
        SAFETY_CONSTRAINT_TEMPLATES[risk_name]
        for risk_name, score in sorted(risk_scores.items(), key=lambda kv: kv[1], reverse=True)
        if score >= 0.35 and risk_name in SAFETY_CONSTRAINT_TEMPLATES
    ]

    hard_avoid = []
    preferred_adaptations = []

    high_fall_or_mobility = (
        risk_scores.get("falls_balance_gait", 0.0) >= 0.35
        or risk_scores.get("mobility_equipment_support", 0.0) >= 0.35
    )
    if high_fall_or_mobility:
        hard_avoid.extend([
            "jumping from heights or unsupported jumping drills",
            "trampoline-style activities",
            "playground climbing or high unstable surfaces",
            "unsupported balance challenges on unstable surfaces",
        ])
        preferred_adaptations.extend([
            "prefer ground-level or seated versions when possible",
            "use hand support, wall support, or equipment support as needed",
            "favor supported reaching, weight shifting, step-up, and transfer practice over high-risk movement tasks",
        ])

    if risk_scores.get("postural_low_tone_fatigue", 0.0) >= 0.35:
        hard_avoid.extend([
            "long sustained postures without rest",
            "high-endurance tasks that assume good postural endurance",
        ])
        preferred_adaptations.extend([
            "short bouts with rest breaks",
            "supported positioning and lower-endurance versions",
        ])

    if risk_scores.get("feeding_or_oral_motor", 0.0) >= 0.35:
        hard_avoid.extend([
            "unsafe oral-motor suggestions",
            "choking-risk foods or unsupervised feeding tasks",
        ])
        preferred_adaptations.extend([
            "keep feeding upright and closely supervised",
            "use safer textures and stop if coughing, gagging, or distress occurs",
        ])

    if risk_scores.get("regulation_frustration", 0.0) >= 0.35:
        hard_avoid.extend([
            "long multi-step activities without breaks",
            "activities that depend on long waiting or repeated failure without support",
        ])
        preferred_adaptations.extend([
            "keep tasks short, predictable, and easy to start",
            "offer choices and stop before escalation",
        ])

    if risk_scores.get("sensory_sensitivity", 0.0) >= 0.35:
        preferred_adaptations.extend([
            "use lower-noise and lower-clutter materials when possible",
            "introduce textures and sounds gradually",
        ])

    # de-duplicate while preserving order
    def _dedupe(items):
        seen = set()
        out = []
        for x in items:
            if x not in seen:
                seen.add(x)
                out.append(x)
        return out

    constraints = _dedupe(constraints)
    hard_avoid = _dedupe(hard_avoid)
    preferred_adaptations = _dedupe(preferred_adaptations)

    if not constraints:
        constraints = ["No major safety constraints were inferred from the diagnosis/concern text. Keep activities parent-supervised and home-safe."]

    return {
        "combined_text": combined_text,
        "risk_scores": risk_scores,
        "matched_patterns": matched_patterns,
        "top_risks": top_risks,
        "constraints": constraints,
        "hard_avoid": hard_avoid,
        "preferred_adaptations": preferred_adaptations,
    }


In [35]:
def ensure_safety_profile(state: Dict[str, Any]) -> Dict[str, Any]:
    if not state.get("safety_profile"):
        state["safety_profile"] = build_safety_profile(state["child"])
    return state["safety_profile"]


In [36]:
def format_safety_constraints_for_prompt(profile: Dict[str, Any]) -> str:
    lines = []
    constraints = profile.get("constraints", [])
    hard_avoid = profile.get("hard_avoid", [])
    adaptations = profile.get("preferred_adaptations", [])

    if constraints:
        lines.append("Planning constraints:")
        lines.extend([f"- {c}" for c in constraints])

    if hard_avoid:
        lines.append("Hard avoidances:")
        lines.extend([f"- {c}" for c in hard_avoid])

    if adaptations:
        lines.append("Preferred adaptations:")
        lines.extend([f"- {c}" for c in adaptations])

    if not lines:
        lines = ["- No special safety constraints flagged. Keep activities parent-supervised and home-safe."]

    return "\n".join(lines)

In [37]:
def print_safety_profile(state: Dict[str, Any]) -> None:
    profile = ensure_safety_profile(state)
    print("\n" + "=" * 100)
    print("SAFETY / PRACTICAL CONSTRAINT PROFILE")
    print("=" * 100)
    top_risks = profile.get("top_risks", [])
    if not top_risks:
        print("No elevated safety/practical flags detected from diagnosis/concern text.")
        return
    print("Top inferred risks:")
    for item in top_risks:
        print(f"- {item['risk']}: {item['weight']}")
    print("Planning constraints:")
    for c in profile.get("constraints", []):
        print(f"- {c}")
    if profile.get("hard_avoid"):
        print("Hard avoidances:")
        for c in profile.get("hard_avoid", []):
            print(f"- {c}")
    if profile.get("preferred_adaptations"):
        print("Preferred adaptations:")
        for c in profile.get("preferred_adaptations", []):
            print(f"- {c}")

In [38]:
def print_safety_profile(state: Dict[str, Any]) -> None:
    profile = ensure_safety_profile(state)
    print("\n" + "=" * 100)
    print("SAFETY / PRACTICAL CONSTRAINT PROFILE")
    print("=" * 100)
    top_risks = profile.get("top_risks", [])
    if not top_risks:
        print("No elevated safety/practical flags detected from diagnosis/concern text.")
        return
    print("Top inferred risks:")
    for item in top_risks:
        print(f"- {item['risk']}: {item['weight']}")
    print("Planning constraints:")
    for c in profile.get("constraints", []):
        print(f"- {c}")

In [39]:
# ------------------------------------------------------------
# Delay estimation
# ------------------------------------------------------------
def estimate_delay_gpt(
    diagnosis: str,
    concern: str,
    chronological_months: int,
    category_key: str,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    """Estimate a rough starting delay in months for one domain.

    This is only a starting anchor for question selection, not the final developmental age.
    """
    category_display = DOMAIN_CONFIG[category_key]["display"]
    concern_l = (concern or "").lower()

    fallback_by_category = {
        "movement_and_physical": 3,
        "language_and_communication": 3,
        "social_and_emotional": 6,
        "cognitive": 6,
    }

    domain_keywords = {
        "movement_and_physical": [
            "motor", "movement", "walk", "run", "jump", "balance", "coordination",
            "fine motor", "gross motor", "grasp", "hand", "writing", "stairs", "falls",
            "hypotonia", "sitting", "rolling", "crawling"
        ],
        "language_and_communication": [
            "speech", "language", "talk", "communication", "words", "sentence",
            "understand", "expressive", "receptive", "verbal", "babbling", "no words"
        ],
        "social_and_emotional": [
            "social", "peer", "friend", "play", "emotion", "emotional",
            "behavior", "anger", "meltdown", "interaction", "turn taking",
            "regulation", "eye contact", "transitions", "bullied", "bullying"
        ],
        "cognitive": [
            "attention", "focus", "concentration", "school", "learning", "routine",
            "executive", "task", "independent", "adaptive", "toilet", "dressing",
            "self-care", "directions", "paying attention"
        ],
    }

    has_domain_signal = any(kw in concern_l for kw in domain_keywords.get(category_key, []))

    if openai_client is None:
        delay_months = fallback_by_category.get(category_key, 6)
        if not has_domain_signal:
            delay_months = min(delay_months, 6)
        return {
            "delay_months": delay_months,
            "reason": f"Fallback delay estimate used for {category_display} because the OpenAI client is not available.",
            "source": "fallback",
        }

    prompt = f"""
You are a pediatric developmental delay estimator agent for children ages 0 to 5 years.

Your job is to estimate a SINGLE STARTING DELAY in months for one developmental domain only.

This is NOT a diagnosis.
This is NOT a final developmental age.
This is ONLY a rough starting anchor for question selection.

Definition:
delay_months = chronological age in months - estimated functional developmental age in this specific domain

Child information:
- Chronological age in months: {chronological_months}
- Diagnosis / condition: {diagnosis}
- Parent concern: {concern}
- Domain to estimate: {category_display}

Instructions:
1. Think only about THIS domain, not overall development.
2. If the diagnosis or concern does NOT meaningfully affect this domain, return 0 to 6 months.
3. If this domain IS affected, estimate the child's functional developmental level in this domain, then convert it into delay_months.
4. Be conservative but realistic.
5. Never exceed the child's chronological age.
6. Return only one integer number of months.
7. Return strict JSON only.

Required JSON:
{{
  "delay_months": <integer>,
  "reason": "<one short sentence>"
}}
""".strip()

    try:
        resp = openai_client.chat.completions.create(
            model=model,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only."},
                {"role": "user", "content": prompt},
            ],
        )

        parsed = json.loads(resp.choices[0].message.content)
        delay_months = int(parsed.get("delay_months", fallback_by_category.get(category_key, 6)))
        delay_months = max(0, min(delay_months, chronological_months))

        if not has_domain_signal and delay_months > 6:
            delay_months = 6

        return {
            "delay_months": delay_months,
            "reason": parsed.get("reason", ""),
            "source": "openai",
        }

    except Exception as e:
        return {
            "delay_months": fallback_by_category.get(category_key, 6),
            "reason": f"Fallback delay estimate used because the OpenAI call failed: {e}",
            "source": "fallback",
        }

def delay_agent_all_categories(state: Dict[str, Any], categories: Optional[List[str]] = None) -> Dict[str, Any]:
    """Estimate a starting delay for each category."""
    print_banner("DELAY ESTIMATOR AGENT")
    if not state.get("child"):
        raise ValueError("Child profile missing. Run intake first.")

    categories = categories or list(DOMAIN_CONFIG.keys())
    child = state["child"]

    for category_key in categories:
        est = estimate_delay_gpt(
            diagnosis=child["diagnosis"],
            concern=child["concern"],
            chronological_months=child["chronological_months"],
            category_key=category_key,
        )
        state["delay_estimates"][category_key] = est
        print(f"{DOMAIN_CONFIG[category_key]['display']}: {est['delay_months']} month starting delay | {est['reason']}")

    return state["delay_estimates"]


In [40]:

# ------------------------------------------------------------
# Core milestone engine
# ------------------------------------------------------------
def _age_proximity_score(month: int, approx_dev_months: int, window_months: int) -> float:
    """Score how close a month band is to the current estimated developmental age."""
    denom = max(window_months / 2, 1)
    score = 1.0 - abs(month - approx_dev_months) / denom
    return float(max(0.0, min(1.0, score)))



def build_parent_question_explanation(
    milestone: str,
    category_key: str,
    subdomain: str = "",
    max_words: int = 24,
) -> str:
    """Return one short parent-friendly clarification line for a milestone question.

    This is intentionally concise. It clarifies the question without adding extra
    yes/sometimes/no instructions or making onboarding feel heavy.
    """
    text = str(milestone).strip()
    low = text.lower()
    sub = str(subdomain or "").lower()

    explanation = "This means you usually see this at home, even if it is not perfect every time."

    if re.search(r"\b(name|first name)\b", low):
        explanation = "This counts if familiar adults can usually understand the name, even if pronunciation is not perfect."
    elif re.search(r"\b(two words|2 words|word.*together|more milk)\b", low):
        explanation = "This means using two meaningful words together, not just repeating one word twice."
    elif re.search(r"\b(sentence|sentences|story|stories|conversation)\b", low):
        explanation = "This means using words to share an idea, even if grammar or pronunciation is still developing."
    elif re.search(r"\bfollow.*two|two[- ]step|2[- ]step\b", low):
        explanation = "This means doing both parts of a simple direction without needing each step repeated."
    elif re.search(r"\bpoint|points|show\b", low):
        explanation = "This can be pointing, touching, reaching toward, or clearly showing the item or picture."
    elif re.search(r"\bjump|jumps|hops?\b", low):
        explanation = "This means both feet leave the floor together, even for a very small jump."
    elif re.search(r"\bstand.*one foot|one foot|balance\b", low):
        explanation = "This means briefly lifting one foot while staying mostly steady, with safe support nearby."
    elif re.search(r"\bthrow|throws|catch|catches|kick|kicks\b", low):
        explanation = "This counts when the movement is purposeful, even if aim and coordination are still developing."
    elif re.search(r"\bstair|stairs|climb|climbs\b", low):
        explanation = "This means doing the movement safely with the usual amount of support for that age."
    elif re.search(r"\bdraw|copy|copies|circle|line|person|shape\b", low):
        explanation = "This counts when the attempt is recognizable, even if it is not neat or perfectly formed."
    elif re.search(r"\bstring|bead|beads|scissor|scissors|button|buttons\b", low):
        explanation = "This means using hands together with some control, not necessarily doing it perfectly or quickly."
    elif re.search(r"\bplay|plays|turn|turns|share|shares|pretend\b", low):
        explanation = "This means a short natural play moment, with adult support if needed."
    elif re.search(r"\bcalm|calms|upset|tantrum|transition\b", low) or "emotional" in sub:
        explanation = "This means the child begins to settle with support; they do not need to calm instantly."
    elif re.search(r"\bfeed|feeds|spoon|fork|dress|dresses|shirt|pants|toilet|wash\b", low):
        explanation = "This counts when the child does meaningful parts of the routine, even with some adult help."
    elif category_key == "language_and_communication":
        explanation = "This can include clear attempts, gestures, signs, sounds, or words depending on the milestone."
    elif category_key == "movement_and_physical":
        explanation = "This counts when the movement is purposeful and safe, even if coordination is still emerging."
    elif category_key == "social_and_emotional":
        explanation = "This means a short real-life interaction, not a perfect or long performance."
    elif category_key == "cognitive_and_adaptive":
        explanation = "This means the child understands or participates in the skill during everyday routines."

    words = explanation.split()
    if len(words) > max_words:
        explanation = " ".join(words[:max_words]).rstrip(".,;") + "."
    return explanation

def build_milestone_questions(
    state: Dict[str, Any],
    category_key: str,
    window_months: int = 24,
    target_questions_per_band: int = 3,
    max_bands: int = 3,
    max_questions_total: int = 9,
) -> List[Dict[str, Any]]:
    """Build a focused set of milestone questions.

    v6 still centers the interview around the estimated developmental range, but it now uses
    the concern router as a *bias* within that range rather than asking the same generic
    questions for every case in the same broad category.
    """
    child = state["child"]
    concern_profile = ensure_concern_profile(state)

    chrono_months = min(child["chronological_months"], 60)
    delay_months = state["delay_estimates"].get(category_key, {}).get("delay_months", 12)

    approx_dev_months = max(2, chrono_months - delay_months)
    min_months = max(2, approx_dev_months - window_months // 2)
    max_months = min(60, approx_dev_months + window_months // 2)

    subset = get_category_questions(category_key, min_months=min_months, max_months=max_months)
    if subset.empty:
        subset = get_category_questions(category_key, min_months=min(CDC_AGES), max_months=max(CDC_AGES))

    if subset.empty:
        return []

    subset = subset.copy()
    if "subdomain" not in subset.columns:
        subset["subdomain"] = "unspecified"

    category_concern_weight = concern_profile["domain_weights"].get(category_key, 0.0)

    subset["age_score"] = subset["months"].map(lambda m: _age_proximity_score(int(m), approx_dev_months, window_months))
    subset["subdomain_weight"] = subset["subdomain"].map(
        lambda s: concern_profile["subdomain_weights"].get(str(s), 0.0)
    )
    subset["row_score"] = (
        0.60 * subset["age_score"] +
        0.30 * subset["subdomain_weight"] +
        0.10 * float(category_concern_weight)
    )

    available_months = sorted(subset["months"].unique().tolist())
    base_months = sorted(
        available_months,
        key=lambda m: (abs(m - approx_dev_months), m)
    )[:max_bands]

    selected_months = list(base_months)

    # Concern-aware month injection:
    # if the concern strongly points to a subdomain that lives in a slightly different month band,
    # we allow one targeted band to replace the weakest/farthest band.
    strong_rows = subset[subset["subdomain_weight"] >= 0.55].sort_values(
        ["subdomain_weight", "row_score", "months"], ascending=[False, False, True]
    )
    if category_concern_weight >= 0.50 and not strong_rows.empty:
        concern_month = int(strong_rows.iloc[0]["months"])
        if concern_month not in selected_months and selected_months:
            farthest_base = max(selected_months, key=lambda m: (abs(m - approx_dev_months), m))
            selected_months.remove(farthest_base)
            selected_months.append(concern_month)

    selected_months = sorted(set(selected_months))

    questions = []
    for month in selected_months:
        month_rows = subset[subset["months"] == month].sort_values(
            ["row_score", "milestone"], ascending=[False, True]
        )
        month_rows = month_rows.head(target_questions_per_band)

        for _, row in month_rows.iterrows():
            parent_explanation = build_parent_question_explanation(
                milestone=row["milestone"],
                category_key=category_key,
                subdomain=str(row.get("subdomain", "unspecified")),
            )
            question_text = (
                f"Can {child['name']} {row['milestone']} right now?\n"
                f"Parent note: {parent_explanation}\n"
                f"Answer: yes / sometimes / with_help / no / not_sure"
            )
            questions.append({
                "question_id": row["question_id"],
                "months": int(row["months"]),
                "milestone": row["milestone"],
                "subdomain": str(row.get("subdomain", "unspecified")),
                "parent_explanation": parent_explanation,
                "selection_score": round(float(row.get("row_score", 0.0)), 3),
                "question_text": question_text,
            })

    questions = sorted(questions, key=lambda q: (q["months"], q["selection_score"] * -1))
    return questions[:max_questions_total]


def _band_has_motor_emphasis(items: List[Dict[str, Any]]) -> bool:
    if not items:
        return False
    subdomains = [str(x.get("subdomain", "unspecified")).strip().lower() for x in items]
    motor_count = sum(1 for s in subdomains if s in MOTOR_EMERGING_SUBDOMAINS)
    return motor_count / max(len(subdomains), 1) >= 0.50


def classify_band_stage(
    *,
    total: int,
    yes_count: int,
    partial_count: int,
    no_count: int,
    motor_emphasis: bool,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> Dict[str, Any]:
    """Classify a month band into confirmed / emerging / not_demonstrated.

    Version 9 keeps one shared 3-stage framework across all domains, but allows
    emerging performance to count more strongly in motor/postural subdomains because
    those skills often appear gradually and inconsistently before stable mastery.
    """
    yes_ratio = yes_count / total if total else 0.0
    positive_ratio = (yes_count + partial_count) / total if total else 0.0

    if yes_count >= min_yes_confirm and yes_ratio >= yes_ratio_confirm:
        return {
            "stage": "confirmed",
            "motor_emphasis": motor_emphasis,
            "evidence_score": round(2.0 + yes_ratio, 3),
            "yes_ratio": round(yes_ratio, 2),
            "positive_ratio": round(positive_ratio, 2),
        }

    partial_weight = MOTOR_EMERGING_PARTIAL_WEIGHT if motor_emphasis else GENERAL_EMERGING_PARTIAL_WEIGHT
    no_penalty = MOTOR_EMERGING_NO_PENALTY if motor_emphasis else GENERAL_EMERGING_NO_PENALTY

    evidence_score = (
        float(yes_count) * 1.0 +
        float(partial_count) * partial_weight -
        float(no_count) * no_penalty
    )

    emerging = False

    # Shared emerging logic: visible partial or mixed evidence can indicate developmental level.
    if positive_ratio >= 0.67 and (yes_count > 0 or partial_count >= 2):
        emerging = True

    # Stronger emerging interpretation for motor/postural bands.
    if motor_emphasis:
        if partial_count >= 2 and no_count <= 1:
            emerging = True
        if partial_count == total and total >= 2:
            emerging = True

    if emerging:
        return {
            "stage": "emerging",
            "motor_emphasis": motor_emphasis,
            "evidence_score": round(float(evidence_score), 3),
            "yes_ratio": round(yes_ratio, 2),
            "positive_ratio": round(positive_ratio, 2),
        }

    return {
        "stage": "not_demonstrated",
        "motor_emphasis": motor_emphasis,
        "evidence_score": round(float(evidence_score), 3),
        "yes_ratio": round(yes_ratio, 2),
        "positive_ratio": round(positive_ratio, 2),
    }


def summarize_answers_by_band(
    answers: List[Dict[str, Any]],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> Dict[int, Dict[str, Any]]:
    """Summarize answers by month band for debugging and advisor review.

    Version 12 still uses the shared 3-stage framework, but now allows a lightweight
    performance-barrier layer to modify how an answer contributes to developmental scoring.
    If an item has `scoring_norm_answer`, that answer is used for band summarization;
    otherwise the raw normalized answer is used.
    """
    band_summary = {}

    for a in answers:
        month = int(a["months"])
        norm = a.get("scoring_norm_answer", a["norm_answer"])

        if month not in band_summary:
            band_summary[month] = {
                "total": 0,
                "yes": 0,
                "partial": 0,
                "no": 0,
                "items": [],
            }

        band_summary[month]["total"] += 1
        band_summary[month]["items"].append({
            "milestone": a["milestone"],
            "subdomain": a.get("subdomain", "unspecified"),
            "answer": norm,
            "raw_answer": a.get("norm_answer", norm),
            "performance_barrier": a.get("performance_barrier", "none"),
        })

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    for month in band_summary:
        total = band_summary[month]["total"]
        yes_count = band_summary[month]["yes"]
        partial_count = band_summary[month]["partial"]
        no_count = band_summary[month]["no"]

        motor_emphasis = _band_has_motor_emphasis(band_summary[month]["items"])
        stage_info = classify_band_stage(
            total=total,
            yes_count=yes_count,
            partial_count=partial_count,
            no_count=no_count,
            motor_emphasis=motor_emphasis,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
        )

        band_summary[month]["yes_ratio"] = round(yes_count / total, 2) if total else 0.0
        band_summary[month]["stage"] = stage_info["stage"]
        band_summary[month]["motor_emphasis"] = motor_emphasis
        band_summary[month]["evidence_score"] = stage_info["evidence_score"]
        band_summary[month]["positive_ratio"] = stage_info["positive_ratio"]

    return dict(sorted(band_summary.items()))

def compute_dev_age_from_answers(
    answers: List[Dict[str, Any]],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> int:
    """Estimate developmental age using confirmed / emerging / not_demonstrated band stages.

    Version 12 keeps the v9/v11 3-stage band framework but lets a small amount of
    structured performance-barrier information change how an answer is scored when
    the child's underlying ability appears stronger than day-to-day performance.
    """
    if not answers:
        return 6

    band_summary = summarize_answers_by_band(
        answers,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    answered_months = sorted(band_summary.keys())
    confirmed_months = [m for m, info in band_summary.items() if info["stage"] == "confirmed"]

    if confirmed_months:
        return int(max(confirmed_months))

    emerging_months = [m for m, info in band_summary.items() if info["stage"] == "emerging"]
    if emerging_months:
        best_month = sorted(
            emerging_months,
            key=lambda m: (band_summary[m]["evidence_score"], m),
            reverse=True,
        )[0]
        return int(best_month)

    return int(min(answered_months))



In [41]:

# ------------------------------------------------------------
# Q&A mode A — live parent
# ------------------------------------------------------------


def should_stop_domain_interview_v14(
    asked: List[Dict[str, Any]],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> bool:
    """Stop once the domain estimate is stable enough for a first plan.

    Version 15 is intentionally a little deeper than v14:
    - it still keeps onboarding focused to 1–2 domains
    - it does not stop after only confirming lower bands
    - it requires at least one confirmed band and an attempt at the next higher band
      before early stopping
    """
    if len(asked) < max(5, V14_TARGET_QUESTIONS_PER_BAND * 2):
        return False

    band_summary = summarize_answers_by_band(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )
    months_answered = sorted(band_summary.keys())
    if len(months_answered) < 2:
        return False

    confirmed_bands = [
        m for m in months_answered
        if band_summary.get(m, {}).get("stage") == "confirmed"
    ]
    if not confirmed_bands:
        return False

    highest_confirmed = max(confirmed_bands)
    higher_answered = [m for m in months_answered if m > highest_confirmed]
    if not higher_answered:
        return False

    next_higher = min(higher_answered)
    next_stage = band_summary.get(next_higher, {}).get("stage")

    return next_stage in {"emerging", "not_demonstrated"}

    band_summary = summarize_answers_by_band(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )
    months_answered = sorted(band_summary.keys())
    if len(months_answered) < 2:
        return False

    provisional_dev_age = compute_dev_age_from_answers(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )
    stage = band_summary.get(provisional_dev_age, {}).get("stage")

    if stage not in {"confirmed", "emerging"}:
        return False

    higher_answered = [m for m in months_answered if m > provisional_dev_age]
    if higher_answered:
        next_higher = min(higher_answered)
        if band_summary.get(next_higher, {}).get("stage") == "not_demonstrated":
            return True

    if provisional_dev_age == max(months_answered) and len(months_answered) >= 2:
        return True

    return False


def qna_agent_live(
    state: Dict[str, Any],
    category_key: str,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> Dict[str, Any]:
    """Run one focused domain interview in live mode using manual answers.

    Version 15 keeps the answer format simple:
    - yes
    - sometimes
    - with_help
    - no
    - not_sure

    No extra follow-up questions are asked. The interview is shorter and uses
    early stopping once the developmental estimate is stable enough.
    """
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    concern_profile = ensure_concern_profile(state)
    top_subdomains = [
        item["subdomain"]
        for item in concern_profile.get("top_subdomains", [])
        if SUBDOMAIN_TO_CATEGORY.get(item["subdomain"]) == category_key
    ]
    if top_subdomains:
        print("Concern-router emphasis in this domain:", ", ".join(top_subdomains[:3]))

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=18,
        target_questions_per_band=V14_TARGET_QUESTIONS_PER_BAND,
        max_bands=V14_MAX_BANDS,
        max_questions_total=V14_MAX_QUESTIONS_TOTAL,
    )

    if not questions:
        raise ValueError(
            f"No milestone questions found for category_key={category_key}. "
            f"Check category mapping and CDC normalization."
        )

    asked = []
    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    stop_now = False
    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")
        for q in questions_by_month[month][:V14_TARGET_QUESTIONS_PER_BAND]:
            print(f"Subdomain: {q.get('subdomain', 'unspecified')} | selection_score: {q.get('selection_score', '')}")
            ans = input(q["question_text"] + "\n").strip()
            norm = normalize_answer(ans)
            perf = derive_performance_interpretation(norm, "")

            asked.append({
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "subdomain": q.get("subdomain", "unspecified"),
                "raw_answer": ans,
                "norm_answer": norm,
                "followup_key": "",
                "followup_label": "",
                "skill_ability": perf["skill_ability"],
                "performance_barrier": "none",
                "scoring_norm_answer": norm,
                "score": ANSWER_SCORES.get(norm, ANSWER_SCORES["not_sure"]),
                "answer_status": "ok",
            })

        if should_stop_domain_interview_v14(
            asked,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
        ):
            print("Early stop: enough evidence collected for a useful first estimate in this domain.")
            stop_now = True

        if stop_now:
            break

    dev_age = compute_dev_age_from_answers(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        print(f"- [{item['months']}m | {item.get('subdomain', 'unspecified')}] {item['milestone']} -> {item['norm_answer']}")

    band_summary = summarize_answers_by_band(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']} | "
            f"stage={info['stage']} | evidence={info['evidence_score']}"
            f"{' | motor_emphasis' if info.get('motor_emphasis') else ''}"
        )

    if category_key == "language_and_communication":
        lang_profile = compute_language_scoring_profile(state)
        effective_age = lang_profile.get('effective_dev_age_months', dev_age)
        print("\nLanguage scoring profile:")
        print(f"  raw language dev age: {lang_profile.get('raw_dev_age_months')}")
        print(f"  expressive/speech track: {lang_profile.get('track_dev_ages', {}).get('expressive_speech')}")
        print(f"  receptive track: {lang_profile.get('track_dev_ages', {}).get('receptive')}")
        print(f"  gesture track: {lang_profile.get('track_dev_ages', {}).get('gesture')}")
        print(f"  effective language dev age used for support planning: {effective_age}")
        print(
            f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
            f"raw={dev_age} months | effective_planning_age={effective_age} months "
            f"(chronological age {chrono} months)"
        )
    else:
        print(
            f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
            f"{dev_age} months (chronological age {chrono} months)"
        )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }


In [42]:

# ------------------------------------------------------------
# Q&A mode B — Gemini synthetic parent
# ------------------------------------------------------------

def gemini_parent_answer_one_question(
    child: Dict[str, Any],
    category_display: str,
    question_text: str,
    prior_answers: List[Dict[str, Any]],
    model: str = "gemini-2.5-flash",
    max_retries: int = 3,
) -> Dict[str, Any]:
    """Use Gemini to simulate one parent answer.

    v15 keeps the parent-facing answer format simple and does not collect
    any structured follow-up answer.
    """
    if gemini_client is None:
        raise ValueError(
            "Gemini client is not available. Set GEMINI_API_KEY and rerun the environment setup cell."
        )

    if prior_answers:
        prior_text = "\n".join([f"- {x['question']} -> {x['answer']}" for x in prior_answers[-8:]])
    else:
        prior_text = "None yet."

    prompt = f"""
You are simulating a REAL parent answering developmental milestone questions about their child.

Rules:
- Answer ONLY from the child profile below.
- Be internally consistent across questions.
- Do NOT answer like a clinician or therapist.
- Answer like a parent who knows the child in daily life.
- Choose ONLY one answer:
  yes
  sometimes
  with_help
  no
  not_sure
- Do not add any extra keys or explanations outside the JSON.
- Keep the reason short and parent-like.

Child profile:
- Name: {child.get('name')}
- Age: {child.get('chronological_months')} months
- Diagnosis / condition: {child.get('diagnosis')}
- Main concern: {child.get('concern')}

Current developmental category:
- {category_display}

Previous answers in this same interview:
{prior_text}

Question:
{question_text}

Return strict JSON only:
{{
  "answer": "yes|sometimes|with_help|no|not_sure",
  "reason": "one short parent-style reason"
}}
""".strip()

    last_error = None

    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
            )

            raw_text = response.text if hasattr(response, "text") else str(response)
            parsed = extract_json_block(raw_text)

            answer = str(parsed.get("answer", "")).strip().lower().replace(" ", "_")
            if answer not in VALID_PARENT_ANSWERS:
                answer = "not_sure"

            return {
                "answer": answer,
                "followup_answer": "",
                "reason": str(parsed.get("reason", "")).strip() or raw_text[:200],
                "raw_text": raw_text,
                "status": "ok",
            }

        except Exception as e:
            last_error = e
            print(f"Gemini warning on question: {question_text}")
            print(f"Attempt {attempt+1}/{max_retries} failed: {e}")

            if attempt < max_retries - 1:
                time.sleep(10 * (attempt + 1))

    return {
        "answer": "not_sure",
        "followup_answer": "",
        "reason": f"Fallback after Gemini failure: {last_error}",
        "raw_text": "",
        "status": "api_error",
    }


def qna_agent_gemini_parent(
    state: Dict[str, Any],
    category_key: str,
    ask_limit_per_band: int = V14_TARGET_QUESTIONS_PER_BAND,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
    gemini_model: str = "gemini-2.5-flash",
    delay_between_questions_sec: int = 5,
) -> Dict[str, Any]:
    """Run one focused domain interview in synthetic-parent mode.

    Version 15 keeps onboarding shorter:
    - no follow-up questions
    - fewer questions per band
    - early stopping once the estimate is clear enough
    """
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    concern_profile = ensure_concern_profile(state)
    top_subdomains = [
        item["subdomain"]
        for item in concern_profile.get("top_subdomains", [])
        if SUBDOMAIN_TO_CATEGORY.get(item["subdomain"]) == category_key
    ]
    if top_subdomains:
        print("Concern-router emphasis in this domain:", ", ".join(top_subdomains[:3]))

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=18,
        target_questions_per_band=V14_TARGET_QUESTIONS_PER_BAND,
        max_bands=V14_MAX_BANDS,
        max_questions_total=V14_MAX_QUESTIONS_TOTAL,
    )

    if not questions:
        raise ValueError(
            f"No milestone questions found for category_key={category_key}. "
            f"Check category mapping and CDC normalization."
        )

    asked = []
    prior_answers_for_gemini = []

    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    stop_now = False
    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")

        for q in questions_by_month[month][:ask_limit_per_band]:
            print(f"Subdomain: {q.get('subdomain', 'unspecified')} | selection_score: {q.get('selection_score', '')}")
            sim = gemini_parent_answer_one_question(
                child=state["child"],
                category_display=DOMAIN_CONFIG[category_key]["display"],
                question_text=q["question_text"],
                prior_answers=prior_answers_for_gemini,
                model=gemini_model,
                max_retries=3,
            )

            norm = normalize_answer(sim["answer"])

            asked_item = {
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "subdomain": q.get("subdomain", "unspecified"),
                "raw_answer": sim["answer"],
                "norm_answer": norm,
                "followup_key": "",
                "followup_label": "",
                "skill_ability": norm,
                "performance_barrier": "none",
                "scoring_norm_answer": norm,
                "score": ANSWER_SCORES.get(norm, ANSWER_SCORES["not_sure"]),
                "parent_reason": sim["reason"],
                "answer_status": sim.get("status", "ok"),
            }
            asked.append(asked_item)

            prior_answers_for_gemini.append({
                "question": q["question_text"],
                "answer": sim["answer"],
            })

            print(
                f"Q: {q['question_text']}\n"
                f"A (Gemini-parent): {sim['answer']} | "
                f"status: {sim.get('status', 'ok')} | "
                f"reason: {sim['reason']}"
            )

            time.sleep(delay_between_questions_sec)

        usable_answers = [a for a in asked if a.get("answer_status") != "api_error"]
        if should_stop_domain_interview_v14(
            usable_answers,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
        ):
            print("Early stop: enough evidence collected for a useful first estimate in this domain.")
            stop_now = True

        if stop_now:
            break

    usable_answers = [a for a in asked if a.get("answer_status") != "api_error"]

    dev_age = compute_dev_age_from_answers(
        usable_answers,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        print(
            f"- [{item['months']}m | {item.get('subdomain', 'unspecified')}] {item['milestone']} -> {item['norm_answer']} "
            f"| status: {item.get('answer_status', 'ok')} "
            f"| reason: {item.get('parent_reason', '')}"
        )

    band_summary = summarize_answers_by_band(
        usable_answers,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']} | "
            f"stage={info['stage']} | evidence={info['evidence_score']}"
            f"{' | motor_emphasis' if info.get('motor_emphasis') else ''}"
        )

    if category_key == "language_and_communication":
        lang_profile = compute_language_scoring_profile(state)
        effective_age = lang_profile.get('effective_dev_age_months', dev_age)
        print("\nLanguage scoring profile:")
        print(f"  raw language dev age: {lang_profile.get('raw_dev_age_months')}")
        print(f"  expressive/speech track: {lang_profile.get('track_dev_ages', {}).get('expressive_speech')}")
        print(f"  receptive track: {lang_profile.get('track_dev_ages', {}).get('receptive')}")
        print(f"  gesture track: {lang_profile.get('track_dev_ages', {}).get('gesture')}")
        print(f"  effective language dev age used for support planning: {effective_age}")
        print(
            f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
            f"raw={dev_age} months | effective_planning_age={effective_age} months "
            f"(chronological age {chrono} months)"
        )
    else:
        print(
            f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
            f"{dev_age} months (chronological age {chrono} months)"
        )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }


In [43]:

# ------------------------------------------------------------
# Support tier + shared planning engine
# ------------------------------------------------------------


def get_category_concern_peak(state: Dict[str, Any], category_key: str) -> float:
    concern_profile = ensure_concern_profile(state)
    subdomains = CATEGORY_TO_SUBDOMAINS.get(category_key, [])
    if not subdomains:
        return 0.0
    return max(float(concern_profile["subdomain_weights"].get(s, 0.0)) for s in subdomains)

LANGUAGE_SCORING_TRACKS = {
    "expressive_speech": {
        "subdomains": {
            "expressive_language",
            "speech_intelligibility",
            "early_vocalization_and_babbling",
            "conversation_narrative",
        }
    },
    "receptive": {
        "subdomains": {"receptive_language"}
    },
    "gesture": {
        "subdomains": {"gestural_communication"}
    },
}

def _answers_for_subdomains(
    answers: List[Dict[str, Any]],
    subdomains: set,
) -> List[Dict[str, Any]]:
    return [
        a for a in answers
        if str(a.get("subdomain", "unspecified")) in subdomains
    ]

def compute_language_scoring_profile(
    state: Dict[str, Any],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> Dict[str, Any]:
    """Compute an internal split scoring profile for Language / Communication.

    v9 keeps the public domain label the same, but internally separates:
    - expressive/speech-related items
    - receptive language items
    - gestural communication items

    The goal is to reduce a common failure mode where strong gestures or strong
    comprehension mask a more significant expressive/speech weakness. Version 9
    keeps this split and combines it with the new confirmed / emerging / not_demonstrated
    framework used elsewhere in the notebook.
    """
    raw_dev_age = state.get("dev_age", {}).get("language_and_communication")
    qna_answers = [
        a for a in state.get("qna", {}).get("language_and_communication", [])
        if a.get("answer_status", "ok") != "api_error"
    ]

    if raw_dev_age is None or not qna_answers:
        return {
            "raw_dev_age_months": raw_dev_age,
            "effective_dev_age_months": raw_dev_age,
            "track_dev_ages": {},
            "track_weights": {},
            "track_counts": {},
        }

    track_dev_ages = {}
    track_counts = {}
    for track_name, cfg in LANGUAGE_SCORING_TRACKS.items():
        subset = _answers_for_subdomains(qna_answers, cfg["subdomains"])
        track_counts[track_name] = len(subset)
        if subset:
            track_dev_ages[track_name] = compute_dev_age_from_answers(
                subset,
                min_yes_confirm=min_yes_confirm,
                yes_ratio_confirm=yes_ratio_confirm,
            )

    concern_profile = ensure_concern_profile(state)
    sub_w = concern_profile.get("subdomain_weights", {})

    expressive_pressure = max(
        float(sub_w.get("expressive_language", 0.0)),
        float(sub_w.get("speech_intelligibility", 0.0)),
        float(sub_w.get("early_vocalization_and_babbling", 0.0)),
        float(sub_w.get("conversation_narrative", 0.0)),
    )
    receptive_pressure = float(sub_w.get("receptive_language", 0.0))
    gesture_pressure = float(sub_w.get("gestural_communication", 0.0))

    # Gesture is useful communication information, but v9 caps its influence so it does not
    # fully mask expressive/speech weakness.
    track_weights = {}
    if track_counts.get("expressive_speech", 0) > 0:
        track_weights["expressive_speech"] = max(0.35, expressive_pressure)
    if track_counts.get("receptive", 0) > 0:
        track_weights["receptive"] = max(0.15, receptive_pressure)
    if track_counts.get("gesture", 0) > 0:
        track_weights["gesture"] = min(0.35, max(0.05, gesture_pressure * 0.5))

    if not track_weights:
        effective_age = raw_dev_age
    else:
        weighted_total = 0.0
        weight_sum = 0.0
        for track_name, weight in track_weights.items():
            age = track_dev_ages.get(track_name, raw_dev_age)
            weighted_total += float(age) * float(weight)
            weight_sum += float(weight)

        weighted_age = round(weighted_total / max(weight_sum, 1e-9))
        effective_age = min(int(raw_dev_age), int(weighted_age))

        # When expressive/speech concern is clearly dominant, do not let strong gesture or
        # receptive performance pull the language age above the expressive/speech track.
        if expressive_pressure >= 0.50 and "expressive_speech" in track_dev_ages:
            effective_age = min(effective_age, int(track_dev_ages["expressive_speech"]))

    return {
        "raw_dev_age_months": int(raw_dev_age),
        "effective_dev_age_months": int(effective_age),
        "track_dev_ages": track_dev_ages,
        "track_weights": {k: round(v, 2) for k, v in track_weights.items()},
        "track_counts": track_counts,
    }

def get_effective_dev_age(state: Dict[str, Any], category_key: str) -> Optional[int]:
    raw = state.get("dev_age", {}).get(category_key)
    if raw is None:
        return None
    if category_key != "language_and_communication":
        return int(raw)

    profile = compute_language_scoring_profile(state)
    eff = profile.get("effective_dev_age_months", raw)
    return int(eff) if eff is not None else int(raw)


SPILLOVER_BEHAVIORAL_PATTERN = re.compile(
    r"(distract|attention|adhd|impuls|talkative|blurt|interrupt|runs? off|very active|tantrum|transition|rigid|group activ|turn[- ]taking|wait (his|her|their)? turn|outburst)",
    flags=re.IGNORECASE,
)

DIRECT_LANGUAGE_DELAY_PATTERN = re.compile(
    r"(speech|language|no words|few words|limited speech|late talk|hard to understand|unclear speech|apraxia|articulation|babbl|gesture|pointing|comprehension|understands? well|receptive|expressive)",
    flags=re.IGNORECASE,
)

DIRECT_MOVEMENT_DELAY_PATTERN = re.compile(
    r"(fall|stairs|balance|walker|clumsy|hypotonia|low tone|fine motor|gross motor|coordination|gait|not sitting|not walking|self-care delay|dressing|fork|spoon|cerebral palsy|prematurity)",
    flags=re.IGNORECASE,
)

def _combined_child_text_for_guardrail(state: Dict[str, Any]) -> str:
    child = state.get("child", {})
    return " | ".join([
        str(child.get("diagnosis", "")),
        str(child.get("concern", "")),
    ])

def _category_has_direct_delay_signal(state: Dict[str, Any], category_key: str) -> bool:
    text = _combined_child_text_for_guardrail(state)
    if category_key == "language_and_communication":
        return bool(DIRECT_LANGUAGE_DELAY_PATTERN.search(text))
    if category_key == "movement_and_physical":
        return bool(DIRECT_MOVEMENT_DELAY_PATTERN.search(text))
    return True

def _category_has_behavioral_spillover_signal(state: Dict[str, Any], category_key: str) -> bool:
    text = _combined_child_text_for_guardrail(state)
    qna_answers = state.get("qna", {}).get(category_key, [])
    barrier_hits = sum(
        1 for a in qna_answers
        if a.get("performance_barrier") in BEHAVIORAL_PERFORMANCE_BARRIERS
        and a.get("skill_ability") == "yes"
    )
    return bool(SPILLOVER_BEHAVIORAL_PATTERN.search(text)) or barrier_hits >= 2

def _milestone_evidence_otherwise_strong(state: Dict[str, Any], category_key: str, effective_dev_age: Optional[int]) -> bool:
    if effective_dev_age is None:
        return False
    chrono = min(state.get("child", {}).get("chronological_months", 0), 60)
    if chrono <= 0:
        return False

    gap = max(0, chrono - int(effective_dev_age))
    if gap <= max(3, round(0.10 * chrono)):
        return True

    answers = [a for a in state.get("qna", {}).get(category_key, []) if a.get("answer_status") != "api_error"]
    if not answers:
        return False

    band_summary = summarize_answers_by_band(answers)
    confirmed_months = [m for m, info in band_summary.items() if info.get("stage") == "confirmed"]
    if confirmed_months and max(confirmed_months) >= chrono - max(4, round(0.10 * chrono)):
        return True
    return False

def should_apply_domain_spillover_guardrail(
    state: Dict[str, Any],
    category_key: str,
    effective_dev_age: Optional[int],
) -> bool:
    """v15 disables the v12 spillover guardrail and returns to the simpler backbone."""
    return False

def compute_support_metrics(state: Dict[str, Any], category_key: str) -> Dict[str, Any]:
    """Compute the continuous support score used for support-tier assignment.

    Version 12 keeps the deterministic support score but adds a domain spillover
    guardrail for Language / Communication and Movement / Physical:
    if milestone evidence in that domain is otherwise strong, the system should
    not elevate support just because distractibility, impulsivity, talkativeness,
    or similar behavioral barriers are bleeding into the concern text.
    """
    child = state["child"]
    concern_profile = ensure_concern_profile(state)

    chrono = min(child["chronological_months"], 60)
    raw_dev_age = state["dev_age"].get(category_key, chrono)
    effective_dev_age = get_effective_dev_age(state, category_key)
    if effective_dev_age is None:
        effective_dev_age = raw_dev_age

    delay_est = state["delay_estimates"].get(category_key, {}).get("delay_months", 0)
    concern_domain_weight = float(concern_profile["domain_weights"].get(category_key, 0.0))
    concern_subdomain_peak = float(get_category_concern_peak(state, category_key))

    gap = max(0, chrono - int(effective_dev_age))

    light_gap_threshold = max(2, round(0.10 * chrono))
    primary_gap_threshold = max(5, round(0.20 * chrono))

    light_delay_threshold = max(4, round(0.10 * chrono))
    primary_delay_threshold = max(6, round(0.20 * chrono))

    gap_component = min(1.5, gap / max(primary_gap_threshold, 1))
    raw_delay_component = min(1.5, delay_est / max(primary_delay_threshold, 1))
    raw_concern_component = max(concern_domain_weight, concern_subdomain_peak * 0.85)

    delay_component = raw_delay_component
    concern_component = raw_concern_component

    spillover_guardrail_applied = False
    spillover_guardrail_reason = ""

    if should_apply_domain_spillover_guardrail(state, category_key, effective_dev_age):
        delay_component *= 0.25
        concern_component *= 0.10
        spillover_guardrail_applied = True
        spillover_guardrail_reason = (
            "Strong milestone evidence suggests this domain is broadly intact; "
            "behavioral performance barriers were prevented from inflating support in this domain."
        )

    support_score = (
        0.55 * gap_component +
        0.25 * delay_component +
        0.20 * concern_component
    )

    if gap >= primary_gap_threshold or support_score >= 0.95:
        tier = "needs_special_support"
    elif (
        support_score >= 0.42
        or (gap >= light_gap_threshold and concern_component >= 0.35)
        or (delay_est >= light_delay_threshold and concern_component >= 0.35 and not spillover_guardrail_applied)
    ):
        tier = "monitor_and_enrich"
    else:
        tier = "no_special_support"

    result = {
        "chronological_months": chrono,
        "raw_dev_age_months": int(raw_dev_age) if raw_dev_age is not None else None,
        "effective_dev_age_months": int(effective_dev_age) if effective_dev_age is not None else None,
        "estimated_dev_age_months": int(effective_dev_age) if effective_dev_age is not None else None,
        "milestone_gap_months": gap,
        "estimated_delay_months": delay_est,
        "concern_domain_weight": concern_domain_weight,
        "concern_subdomain_peak": concern_subdomain_peak,
        "light_gap_threshold": light_gap_threshold,
        "primary_gap_threshold": primary_gap_threshold,
        "light_delay_threshold": light_delay_threshold,
        "primary_delay_threshold": primary_delay_threshold,
        "gap_component": round(float(gap_component), 3),
        "raw_delay_component": round(float(raw_delay_component), 3),
        "raw_concern_component": round(float(raw_concern_component), 3),
        "delay_component": round(float(delay_component), 3),
        "concern_component": round(float(concern_component), 3),
        "support_score": round(float(support_score), 3),
        "tier": tier,
        "spillover_guardrail_applied": spillover_guardrail_applied,
        "spillover_guardrail_reason": spillover_guardrail_reason,
    }

    if category_key == "language_and_communication":
        result["language_scoring_profile"] = compute_language_scoring_profile(state)

    return result

def get_support_tier(state: Dict[str, Any], category_key: str) -> str:
    return compute_support_metrics(state, category_key)["tier"]

def no_special_support_needed(state: Dict[str, Any], category_key: str) -> bool:
    return get_support_tier(state, category_key) == "no_special_support"


def determine_family_guidance_floor(state: Dict[str, Any]) -> Dict[str, Any]:
    """Create a soft planning floor when all categories are technically no special support.

    This does NOT change the underlying support tier. It only ensures the family still
    receives short age-appropriate enrichment-and-observation activities in the category
    most aligned with the expressed concern.
    """
    concern_profile = ensure_concern_profile(state)
    child = state["child"]

    supported = [k for k in DOMAIN_CONFIG if get_support_tier(state, k) != "no_special_support"]
    if supported:
        info = {
            "enabled": False,
            "mode": None,
            "category_key": None,
            "planning_tier": None,
            "target_weekly_minutes": 0,
            "summary": "",
        }
        state["family_guidance_floor"] = info
        return info

    ranked = sorted(
        DOMAIN_CONFIG.keys(),
        key=lambda k: (
            float(concern_profile.get("domain_weights", {}).get(k, 0.0)),
            float(get_category_concern_peak(state, k)),
            float(state.get("delay_estimates", {}).get(k, {}).get("delay_months", 0)),
        ),
        reverse=True,
    )
    category_key = ranked[0] if ranked else "language_and_communication"
    category_display = DOMAIN_CONFIG[category_key]["display"]
    daily_time_min = int(child.get("daily_time_min", 10))
    target_weekly_minutes = min(max(15, daily_time_min * 3), daily_time_min * 5)

    summary = (
        f"Based on the milestone interview, {child['name']} does not currently appear to need scheduled special support. "
        f"Because the family expressed concern about {category_display}, the system will still provide short age-appropriate "
        f"enrich-and-observe activities in this category."
    )

    info = {
        "enabled": True,
        "mode": "enrich_and_observe",
        "category_key": category_key,
        "category_display": category_display,
        "planning_tier": "enrich_and_observe",
        "target_weekly_minutes": int(target_weekly_minutes),
        "summary": summary,
    }
    state["family_guidance_floor"] = info
    return info


def is_family_guidance_category(state: Dict[str, Any], category_key: str) -> bool:
    floor = state.get("family_guidance_floor", {})
    return bool(floor.get("enabled") and floor.get("category_key") == category_key)

def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = 6) -> Dict[str, Any]:
    """Select milestones either for support planning or for soft enrich-and-observe guidance."""
    child = state["child"]
    dev_age = get_effective_dev_age(state, category_key)
    soft_floor_active = is_family_guidance_category(state, category_key)

    if dev_age is None and not soft_floor_active:
        raise ValueError(f"No developmental age found for {category_key}. Run Q&A first.")

    if no_special_support_needed(state, category_key) and not soft_floor_active:
        return {
            "status": "no_special_support",
            "message": f"We do not think {child['name']} has a meaningful delay in {DOMAIN_CONFIG[category_key]['display']} and may not need special support in this category right now.",
            "milestones": [],
        }

    chrono = min(child["chronological_months"], 60)
    if soft_floor_active:
        min_m = max(2, chrono - 3)
        max_m = min(60, chrono + 3)
    else:
        min_m = max(2, dev_age)
        max_m = min(60, dev_age + 12)

    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)
    if subset.empty:
        return {"status": "success", "milestones": [], "mode": "soft_floor" if soft_floor_active else "support"}

    subset = subset.sort_values(["months", "milestone"]).copy()
    milestones = []
    seen = set()

    for _, row in subset.iterrows():
        key = (int(row["months"]), str(row["milestone"]).strip())
        if key in seen:
            continue
        seen.add(key)
        milestones.append({
            "months": int(row["months"]),
            "milestone": row["milestone"],
            "subdomain": str(row.get("subdomain", "unspecified")),
        })

    return {
        "status": "success",
        "milestones": milestones[:max_milestones],
        "mode": "soft_floor" if soft_floor_active else "support",
    }

def get_category_activity_guardrails(category_key: str) -> str:
    """Return domain-specific activity guardrails to reduce cross-domain drift."""
    if category_key == "movement_and_physical":
        return """
Category-specific rules for Movement / Physical:
- Focus on posture, strength, balance, coordination, reaching, grasping, sitting, crawling, standing, walking, or fine-motor use.
- The main goal should be physical skill practice.
- Do NOT make this mainly about speech, naming, imitation of sounds, or social-emotional labeling.
"""

    if category_key == "language_and_communication":
        return """
Category-specific rules for Language / Communication:
- Focus on sounds, babbling, turn-taking vocalization, following simple verbal directions, gestures for communication, word use, imitation of sounds, naming, requesting, commenting, comprehension, or speech clarity.
- The main goal should be communication.
- Do NOT make this mainly about gross motor practice, reaching, balance, climbing, or general physical exercise.
- If an object is used, it should support communication, not be the main task.
"""

    if category_key == "social_and_emotional":
        return """
Category-specific rules for Social / Emotional:
- Focus on eye contact, shared attention, social reciprocity, imitation of facial expressions, joint play, turn-taking, response to name, emotional connection, emotional regulation, separation, or interaction with caregiver or peers.
- The main goal should be social engagement or emotional regulation.
- Do NOT make this mainly about motor strengthening or naming/labeling language tasks unless the social interaction is clearly primary.
"""

    if category_key == "cognitive":
        return """
Category-specific rules for Cognitive / Adaptive:
- Focus on attention, problem-solving, object permanence, simple cause-and-effect, routines, imitation of actions, functional play, early self-help, adaptive understanding, or following directions.
- The main goal should be cognitive/adaptive skill building.
- Do NOT make this mainly about speech production or gross motor strengthening unless the cognitive/adaptive task is clearly primary.
"""

    return ""


# ------------------------------------------------------------
# v16 language activity readiness guardrails
# ------------------------------------------------------------
#
# Purpose:
# Prevent activity generation from asking a child to combine words or make
# sentences before the expressive single-word foundation is ready.
#
# This is deliberately placed in the activity-planning layer, not the question
# layer, so onboarding stays short and we do not add parent follow-up questions.

LANGUAGE_LIMITED_EXPRESSIVE_PATTERN = re.compile(
    r"(no words|few words|limited words|not talking|doesn'?t talk|does not talk|"
    r"hardly talks|hardly says|cannot say|can'?t say|less than\s+\d+\s+words|"
    r"under\s+\d+\s+words|not saying\s+\d+\s+words|not say\s+\d+\s+words|"
    r"cannot tell\s+\d+\s+words|can'?t tell\s+\d+\s+words|"
    r"speech delay|expressive delay|late talker|late talking)",
    flags=re.IGNORECASE,
)

LANGUAGE_SINGLE_WORD_FOUNDATION_PATTERN = re.compile(
    r"(mama|dada|specific name|first words?|three or more words|3 or more words|"
    r"says?\s+\d+\s+words?|names? familiar|names? pictures?|labels? objects?|"
    r"uses? words?|tries? to say|imitates? words?|copies? words?)",
    flags=re.IGNORECASE,
)

LANGUAGE_50_WORD_PATTERN = re.compile(
    r"(50\s+words|fifty\s+words|about\s+50|at least\s+50)",
    flags=re.IGNORECASE,
)

LANGUAGE_WORD_COMBINATION_MILESTONE_PATTERN = re.compile(
    r"(two\s+words\s+together|two[- ]word|2[- ]word|two or more words|"
    r"combine[s]?\s+words|combining\s+words|word combination|"
    r"more milk|doggie run|sentences?|four or more words|tells? a story|storytelling)",
    flags=re.IGNORECASE,
)

LANGUAGE_ACTIVITY_FORBIDDEN_PHRASE_PATTERN = re.compile(
    r"(two\s+words\s+together|two[- ]word|2[- ]word|two or more words|"
    r"combine\s+words|combining\s+words|word combinations?|"
    r"make\s+(a\s+)?sentence|say\s+(a\s+)?sentence|sentences?|"
    r"short\s+phrase|phrases?|more milk|more juice|want\s+\w+|"
    r"tell\s+(a\s+)?story|storytelling|four or more words)",
    flags=re.IGNORECASE,
)

LANGUAGE_ACTIVITY_DIRECT_VERBAL_DEMAND_PATTERN = re.compile(
    r"(must say|required to say|ask the child to say\s+['\"]?\w+\s+\w+|"
    r"have the child say\s+['\"]?\w+\s+\w+|child says\s+['\"]?\w+\s+\w+)",
    flags=re.IGNORECASE,
)


def _language_activity_answer_norm(answer: Dict[str, Any]) -> str:
    return str(
        answer.get("scoring_norm_answer", answer.get("norm_answer", ""))
    ).strip().lower()


def _language_activity_is_positive_or_emerging(answer: Dict[str, Any]) -> bool:
    return _language_activity_answer_norm(answer) in {"yes", "sometimes", "with_help"}


def _language_activity_is_reliable_yes(answer: Dict[str, Any]) -> bool:
    return _language_activity_answer_norm(answer) in {"yes", "sometimes"}


def _language_activity_is_no_or_unknown(answer: Dict[str, Any]) -> bool:
    return _language_activity_answer_norm(answer) in {"no", "not_sure", "unknown", ""}


def _language_activity_text_for_answer(answer: Dict[str, Any]) -> str:
    return " ".join(
        str(answer.get(k, ""))
        for k in ("milestone", "question", "subdomain", "raw_answer", "norm_answer", "scoring_norm_answer")
    )


def compute_language_activity_readiness(state: Dict[str, Any]) -> Dict[str, Any]:
    """Compute expressive-language readiness for activity generation.

    This is separate from developmental-age scoring. Its purpose is to stop the
    activity planner from turning a higher milestone into an activity before the
    child has the prerequisite foundation.
    """
    child = state.get("child", {})
    concern_text = " | ".join(
        str(child.get(k, ""))
        for k in ("diagnosis", "condition", "concern", "parent_concern")
    )

    answers = [
        a for a in state.get("qna", {}).get("language_and_communication", [])
        if a.get("answer_status", "ok") != "api_error"
    ]

    expressive_subdomains = {
        "expressive_language",
        "speech_intelligibility",
        "early_vocalization_and_babbling",
        "conversation_narrative",
    }

    expressive_answers = [
        a for a in answers
        if str(a.get("subdomain", "")).strip().lower() in expressive_subdomains
    ]

    # If subdomains are missing or inconsistent, fall back to all language answers.
    if not expressive_answers:
        expressive_answers = answers

    has_emerging_single_words = any(
        LANGUAGE_SINGLE_WORD_FOUNDATION_PATTERN.search(_language_activity_text_for_answer(a))
        and _language_activity_is_positive_or_emerging(a)
        for a in expressive_answers
    )

    explicit_no_50_words = any(
        LANGUAGE_50_WORD_PATTERN.search(_language_activity_text_for_answer(a))
        and _language_activity_is_no_or_unknown(a)
        for a in expressive_answers
    )

    confirmed_50_words = any(
        LANGUAGE_50_WORD_PATTERN.search(_language_activity_text_for_answer(a))
        and _language_activity_is_reliable_yes(a)
        for a in expressive_answers
    )

    has_reliable_word_combination = any(
        LANGUAGE_WORD_COMBINATION_MILESTONE_PATTERN.search(_language_activity_text_for_answer(a))
        and _language_activity_is_reliable_yes(a)
        for a in expressive_answers
    )

    word_combination_not_established = any(
        LANGUAGE_WORD_COMBINATION_MILESTONE_PATTERN.search(_language_activity_text_for_answer(a))
        and _language_activity_is_no_or_unknown(a)
        for a in expressive_answers
    )

    concern_suggests_limited_words = bool(
        LANGUAGE_LIMITED_EXPRESSIVE_PATTERN.search(concern_text)
    )

    # Conservative gating:
    # - If parent concern or answers indicate very limited words, stay at pre/single-word foundations.
    # - If 50 words is explicitly not established, do not require phrase production.
    # - Only allow phrase-building when word combinations are already reliable, or when 50 words are reliable
    #   and word combination was not explicitly marked as absent.
    if concern_suggests_limited_words and not has_emerging_single_words:
        level = "pre_or_early_single_words"
    elif explicit_no_50_words:
        level = "single_word_building"
    elif has_reliable_word_combination:
        level = "phrase_ready"
    elif confirmed_50_words and not word_combination_not_established:
        level = "early_phrase_modeling_ok"
    elif has_emerging_single_words:
        level = "single_word_building"
    else:
        level = "pre_or_early_single_words"

    return {
        "level": level,
        "has_emerging_single_words": has_emerging_single_words,
        "explicit_no_50_words": explicit_no_50_words,
        "confirmed_50_words": confirmed_50_words,
        "has_reliable_word_combination": has_reliable_word_combination,
        "word_combination_not_established": word_combination_not_established,
        "concern_suggests_limited_words": concern_suggests_limited_words,
    }


def build_language_activity_readiness_block(
    state: Dict[str, Any],
    category_key: str,
) -> str:
    """Create a prompt block that tells the LLM what language level is allowed."""
    if category_key != "language_and_communication":
        return "No special language-readiness constraints for this category."

    readiness = compute_language_activity_readiness(state)
    level = readiness["level"]

    if level == "pre_or_early_single_words":
        return """
Language readiness rule:
- The child does not yet show a stable expressive single-word foundation.
- Do NOT create activities that require two-word phrases, combining words, sentences, storytelling, or open-ended verbal answers.
- Parent may model simple words and very short phrases, but the child success goal should be pointing, choosing, gesturing, imitating sounds, imitating one word, finding objects in books, naming familiar items, or making animal/environmental sounds.
- Good examples: find the bear in a book, point to the ball, imitate "ba" for ball, say or gesture "more", choose between two picture cards, or copy animal sounds.
""".strip()

    if level == "single_word_building":
        return """
Language readiness rule:
- The child has some single-word or emerging-word ability, but phrase production is not yet a reliable target.
- Do NOT require the child to produce two-word phrases or sentences.
- Parent may model two-word expansions, such as child says "milk" and parent says "more milk", but the child success goal should remain one sound, one word, pointing, choosing, or imitation.
- Activities should build vocabulary, pronunciation attempts, imitation, receptive understanding, and confident single-word use.
""".strip()

    if level == "early_phrase_modeling_ok":
        return """
Language readiness rule:
- The child appears to have a stronger single-word vocabulary, but phrase production should still be gentle.
- Parent may model short two-word expansions.
- Do not make two-word phrases the only success criterion; accept one word, gesture, pointing, or imitation attempts.
""".strip()

    return """
Language readiness rule:
- The child appears ready for early phrase-building practice.
- Two-word phrase modeling and gentle phrase imitation may be included.
- Keep activities playful and do not pressure the child to perform.
""".strip()


def _language_readiness_safe_activity(
    activity_id: str,
    category_display: str,
    planning_tier: str,
    level: str,
    index: int,
) -> Dict[str, Any]:
    """Return a safe replacement activity when generated content is too advanced."""
    pre_word_templates = [
        {
            "title": "Picture Book Find-and-Try",
            "instructions": (
                "Read a familiar picture book together. Ask the child to find one item, such as "
                "\"Where is the bear?\" The child can point, look, gesture, make a sound, or try one word. "
                "Parent models the word clearly, such as \"bear\", and repeats it warmly without requiring a phrase."
            ),
            "materials": "picture book with familiar animals or objects",
        },
        {
            "title": "Animal Sound Copy",
            "instructions": (
                "Show one animal toy or picture at a time. Parent says the animal name and sound, such as "
                "\"cow, moo.\" The child can point, copy the sound, copy the first sound, or try the single word."
            ),
            "materials": "animal toys, animal pictures, or simple flashcards",
        },
        {
            "title": "Two-Choice Point and Word",
            "instructions": (
                "Offer two familiar choices, such as ball or car. Ask \"Which one?\" The child can point, reach, "
                "gesture, or try one word. Parent repeats the chosen word clearly and celebrates any attempt."
            ),
            "materials": "two familiar toys or picture cards",
        },
        {
            "title": "First-Sound Flashcards",
            "instructions": (
                "Use 3 to 5 simple picture cards. Parent names each picture slowly and emphasizes the first sound, "
                "such as \"b-b-ball.\" The child can point, copy the sound, or try the single word."
            ),
            "materials": "simple flashcards or printed pictures",
        },
    ]

    single_word_templates = [
        {
            "title": "Single-Word Request Practice",
            "instructions": (
                "Place a favorite toy or snack container where the child can see it. Pause and encourage one sound, "
                "gesture, sign, or one word such as \"more\", \"open\", or the item name. Parent can model a short "
                "phrase, but the child only needs one word or a clear communication attempt."
            ),
            "materials": "favorite toy, closed container, or safe snack container",
        },
        {
            "title": "Book Naming and Pointing",
            "instructions": (
                "Look at a picture book and name one object per page. Ask the child to point to the object or try "
                "one word. Parent repeats the single word and may add a short model after the child tries."
            ),
            "materials": "picture book",
        },
        {
            "title": "Object Basket Words",
            "instructions": (
                "Put 4 to 6 familiar objects in a basket. Pull out one object at a time, name it clearly, and wait. "
                "The child can point, hold it up, copy a sound, or try one word. Keep the goal at one word or one sound."
            ),
            "materials": "basket with familiar household objects or toys",
        },
        {
            "title": "Copy My Sound Game",
            "instructions": (
                "Take turns making easy sounds from familiar words, such as \"ba\", \"ma\", \"moo\", or \"pop\". "
                "Accept any attempt, gesture, or smile as participation, and repeat the sound clearly."
            ),
            "materials": "bubbles, animal toys, or favorite small toys",
        },
    ]

    templates = pre_word_templates if level == "pre_or_early_single_words" else single_word_templates
    template = templates[(index - 1) % len(templates)]

    return {
        "activity_id": activity_id,
        "title": template["title"],
        "instructions": template["instructions"],
        "duration_min": 5,
        "materials": template["materials"],
        "level": "current_or_next",
        "goal": planning_tier,
        "category": category_display,
        "is_extended_activity": False,
        "extended_reason": "",
    }


def _language_activity_violates_readiness(activity: Dict[str, Any], level: str) -> bool:
    """Detect activity text that asks for phrase/sentence production too early."""
    if level in {"phrase_ready"}:
        return False

    text = " ".join(
        str(activity.get(k, ""))
        for k in ("title", "instructions", "materials", "level", "extended_reason")
    )

    if LANGUAGE_ACTIVITY_FORBIDDEN_PHRASE_PATTERN.search(text):
        return True

    if level == "pre_or_early_single_words" and LANGUAGE_ACTIVITY_DIRECT_VERBAL_DEMAND_PATTERN.search(text):
        return True

    return False


def apply_language_readiness_constraints_to_activities(
    state: Dict[str, Any],
    category_key: str,
    activities: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Deterministically replace language activities that are too advanced.

    This is the backup safety net in case the LLM ignores the prompt-level
    readiness instruction, or in case fallback milestone activities point to a
    higher expressive milestone like word combinations.
    """
    if category_key != "language_and_communication":
        return activities

    readiness = compute_language_activity_readiness(state)
    level = readiness["level"]

    if level == "phrase_ready":
        return activities

    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", "Language / Communication")
    planning_tier = "current_or_next"
    if activities:
        planning_tier = str(activities[0].get("goal", planning_tier))

    corrected = []
    replacements_made = 0

    for idx, activity in enumerate(activities, start=1):
        a = dict(activity)

        if _language_activity_violates_readiness(a, level):
            replacements_made += 1
            replacement = _language_readiness_safe_activity(
                activity_id=str(a.get("activity_id", f"{category_key}_{idx}")),
                category_display=category_display,
                planning_tier=str(a.get("goal", planning_tier)),
                level=level,
                index=idx,
            )
            replacement["readiness_adjustment"] = (
                "Replaced because the generated activity required phrase/sentence production "
                "before expressive-language prerequisites were established."
            )
            corrected.append(replacement)
        else:
            # Add a gentle success-criteria reminder for early language plans.
            instructions = str(a.get("instructions", ""))
            reminder = (
                " Success can be pointing, gesturing, copying a sound, or trying one word; "
                "do not require two-word phrases."
            )
            if (
                level in {"pre_or_early_single_words", "single_word_building"}
                and "do not require two-word phrases" not in instructions.lower()
            ):
                a["instructions"] = instructions.rstrip() + reminder
            corrected.append(a)

    # Store a small trace for debugging/advisor review without changing public UX.
    state.setdefault("planner_debug", {})
    state["planner_debug"]["language_activity_readiness"] = {
        **readiness,
        "replacements_made": replacements_made,
    }

    return corrected

def generate_category_activity_bank(
    state: Dict[str, Any],
    category_key: str,
    model: str = "gpt-4o-mini",
    activities_per_category: int = 6,
) -> Dict[str, Any]:
    """Generate one activity bank per category."""
    child = state["child"]
    category_display = DOMAIN_CONFIG[category_key]["display"]
    support_tier = get_support_tier(state, category_key)
    soft_floor_active = is_family_guidance_category(state, category_key)
    planning_tier = "enrich_and_observe" if soft_floor_active else support_tier
    support_metrics = compute_support_metrics(state, category_key)
    safety_profile = ensure_safety_profile(state)
    safety_constraints_block = format_safety_constraints_for_prompt(safety_profile)
    next_steps = select_next_milestones(state, category_key)

    if next_steps["status"] == "no_special_support":
        state["activity_banks"][category_key] = {
            "status": "no_special_support",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "summary": next_steps["message"],
            "activities": [],
        }
        return state["activity_banks"][category_key]

    chrono_months = min(child["chronological_months"], 60)
    dev_age = chrono_months if soft_floor_active else state["dev_age"][category_key]
    milestone_gap = max(0, chrono_months - dev_age)
    category_guardrails = get_category_activity_guardrails(category_key)
    language_readiness_block = build_language_activity_readiness_block(state, category_key)
    soft_floor_block = ""
    if soft_floor_active:
        soft_floor_block = (
            "This category is being generated under the family guidance floor. "
            "Use age-appropriate, low-intensity enrichment and observation activities. "
            "Do not imply therapy-level intensity or a significant delay. "
            "The goal is support, confidence-building, and structured observation."
        )

    milestone_lines = "\n".join(
        [f"- ({m['months']} months | {m.get('subdomain', 'unspecified')}) {m['milestone']}" for m in next_steps["milestones"]]
    )
    if not milestone_lines:
        milestone_lines = "- No specific milestone items available in this range."

    if openai_client is None:
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "goal": planning_tier,
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "goal": planning_tier,
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        fallback_activities = apply_language_readiness_constraints_to_activities(state, category_key, fallback_activities)
        fallback_activities = apply_safety_constraints_to_activities(state, category_key, fallback_activities)
        state["activity_banks"][category_key] = {
            "status": "fallback",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "summary": f"Fallback activity bank used for {category_display}.",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]

    prompt = f"""
You are a pediatric home-support planning agent helping a parent at home.

This is NOT a diagnosis and NOT a formal treatment plan.
Create a CATEGORY ACTIVITY BANK, not a day-by-day schedule.

Child:
- Name: {child['name']}
- Chronological age: {child['chronological_months']} months
- Diagnosis / condition: {child['diagnosis']}
- Parent concern: {child['concern']}
- Category: {category_display}
- Support tier for this category: {planning_tier}
- Estimated developmental age in this category: {dev_age} months
- Estimated milestone gap in this category: {milestone_gap} months\n- Continuous support score for this category: {support_metrics['support_score']}\n
Relevant milestone targets:
{milestone_lines}

Category-specific guardrails:\n{category_guardrails}\n\nDevelopmental readiness constraints:\n{language_readiness_block}\n\nSafety / practical constraints inferred from diagnosis + concern:\n{safety_constraints_block}\n\nThese safety items are real planning constraints, not optional suggestions. Do not propose activities that violate the hard-avoid list. Prefer the safer adaptations when relevant.\n\nTask:\nCreate {activities_per_category} realistic home activities for this category.

Instructions:
1. Activities must fit the child's chronological age and estimated developmental level.
2. Activities should be practical for home.
3. Keep language parent-friendly.
4. Include a mix of current-level practice and near next-step practice.
5. Most activities should be short, repeatable, and realistic for home: usually 3, 5, 7, or 10 minutes.
6. The CORE weekly schedule is for short at-home activities only.
7. Context-dependent activities such as playdates, park meetups, playground peer practice, group social activities, or community participation must NEVER be written as normal daily home activities.
8. If you include a playdate-, park-, playground-, meetup-, or group-type activity, mark it as an extended activity intended for OPTIONAL WEEKLY BONUS use only.
9. Any context-dependent bonus activity should be 30, 35, 40, or 45 minutes, not 5 or 10 minutes.
10. Do not make more than 1 context-dependent bonus activity in this category.
11. Each activity must clearly belong to THIS category first and foremost.
12. Avoid cross-domain drift. The main goal of the activity should be obvious from the category.
13. If an activity could fit multiple domains, choose the version that best matches this category's core skill.
14. Set "goal" to exactly the planning tier for this category: {planning_tier}
15. Obey the developmental readiness constraints. If a milestone target seems too advanced for the readiness level, create a prerequisite-building activity instead.
16. Return strict JSON only.

Required JSON:
{{
  "summary": "...",
  "activities": [
    {{
      "activity_id": "1",
      "title": "...",
      "instructions": "...",
      "duration_min": 5,
      "materials": "...",
      "level": "current_or_next",
      "goal": "{planning_tier}",
      "category": "<category name>",
      "is_extended_activity": false,
      "extended_reason": ""
    }}
  ]
}}
""".strip()

    try:
        resp = openai_client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only and stay non-diagnostic, practical, and parent-friendly."},
                {"role": "user", "content": prompt},
            ],
        )

        bank = json.loads(resp.choices[0].message.content)

        if "activities" not in bank or not isinstance(bank["activities"], list):
            bank["activities"] = []

        for idx, activity in enumerate(bank["activities"], start=1):
            activity["activity_id"] = activity.get("activity_id", f"{category_key}_{idx}")
            activity["category"] = activity.get("category", category_display)
            activity["duration_min"] = activity.get("duration_min", 5)
            activity["materials"] = activity.get("materials", "common household items")
            activity["level"] = activity.get("level", "current_or_next")
            activity["goal"] = planning_tier
            activity["is_extended_activity"] = activity.get("is_extended_activity", False)
            activity["extended_reason"] = activity.get("extended_reason", "")

        bank["activities"] = apply_language_readiness_constraints_to_activities(state, category_key, bank["activities"][:activities_per_category])
        bank["activities"] = apply_safety_constraints_to_activities(state, category_key, bank["activities"])

        state["activity_banks"][category_key] = {
            "status": "success",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "summary": bank.get("summary", f"Created activity bank for {category_display}."),
            "activities": bank["activities"],
        }
        return state["activity_banks"][category_key]

    except Exception as e:
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "goal": planning_tier,
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "goal": planning_tier,
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        fallback_activities = apply_language_readiness_constraints_to_activities(state, category_key, fallback_activities)
        fallback_activities = apply_safety_constraints_to_activities(state, category_key, fallback_activities)
        state["activity_banks"][category_key] = {
            "status": "fallback",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "summary": f"Fallback activity bank used for {category_display} because OpenAI failed: {e}",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]


CONTEXT_DEPENDENT_BONUS_PATTERN = re.compile(
    r"\b(playdate|park meetup|playground peer|group social|group activity|peer practice|community class|outing|meetup)\b",
    flags=re.IGNORECASE,
)

def is_context_dependent_bonus_activity(activity: Dict[str, Any]) -> bool:
    text = " ".join(
        [
            str(activity.get("title", "")),
            str(activity.get("instructions", "")),
            str(activity.get("materials", "")),
            str(activity.get("extended_reason", "")),
        ]
    )
    return bool(CONTEXT_DEPENDENT_BONUS_PATTERN.search(text))

def apply_safety_constraints_to_activities(
    state: Dict[str, Any],
    category_key: str,
    activities: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Deterministically adapt or sanitize activities based on the broad safety profile.

    This is intentionally broad and risk-based rather than diagnosis-name-specific.
    """
    profile = ensure_safety_profile(state)
    risk_scores = profile.get("risk_scores", {})

    high_fall_or_mobility = (
        risk_scores.get("falls_balance_gait", 0.0) >= 0.35
        or risk_scores.get("mobility_equipment_support", 0.0) >= 0.35
    )
    low_tone_or_fatigue = risk_scores.get("postural_low_tone_fatigue", 0.0) >= 0.35
    feeding_risk = risk_scores.get("feeding_or_oral_motor", 0.0) >= 0.35
    regulation_risk = risk_scores.get("regulation_frustration", 0.0) >= 0.35
    sensory_risk = risk_scores.get("sensory_sensitivity", 0.0) >= 0.35

    safe_activities = []

    for activity in activities:
        a = dict(activity)
        title = str(a.get("title", ""))
        instructions = str(a.get("instructions", ""))
        materials = str(a.get("materials", "common household items"))
        duration_min = int(a.get("duration_min", 5))
        lower_text = f"{title} {instructions}".lower()

        if high_fall_or_mobility and re.search(r'\b(jump|jumping|trampoline|hop|hopping|climb|climbing|playground)\b', lower_text):
            a["title"] = "Supported Balance and Reach Practice"
            a["instructions"] = (
                "Use a ground-level activity with close adult support. Practice supported standing, "
                "seated or supported reaching, and safe weight shifts while keeping both feet on a stable surface. "
                "Avoid jumping, climbing, or unstable surfaces."
            )
            a["materials"] = "stable chair or couch, wall or caregiver support, favorite toys"
            a["duration_min"] = min(duration_min, 7)
            title = a["title"]
            instructions = a["instructions"]
            materials = a["materials"]
            duration_min = a["duration_min"]
            lower_text = f"{title} {instructions}".lower()

        if high_fall_or_mobility and category_key == "movement_and_physical":
            if "close adult support" not in instructions.lower() and "stable support" not in instructions.lower():
                a["instructions"] = instructions.rstrip() + " Use stable support and close adult supervision throughout."
                instructions = a["instructions"]

        if low_tone_or_fatigue:
            a["duration_min"] = min(int(a.get("duration_min", duration_min)), 7)
            if "rest break" not in instructions.lower():
                a["instructions"] = instructions.rstrip() + " Keep the activity in short bouts and pause for rest if the child tires."
                instructions = a["instructions"]

        if regulation_risk:
            a["duration_min"] = min(int(a.get("duration_min", duration_min)), 5)
            if "predictable" not in instructions.lower():
                a["instructions"] = instructions.rstrip() + " Keep it short and predictable, offer a simple choice, and stop before frustration escalates."
                instructions = a["instructions"]

        if sensory_risk and "lower-noise" not in instructions.lower():
            a["instructions"] = instructions.rstrip() + " Use lower-noise, lower-clutter materials when possible."
            instructions = a["instructions"]

        if feeding_risk and re.search(r'\b(food|snack|chew|mouth|eat|drink|feeding)\b', lower_text):
            if "upright" not in instructions.lower():
                a["instructions"] = instructions.rstrip() + " Keep the child upright and closely supervised. Avoid choking-risk foods and stop if coughing, gagging, or distress occurs."
                instructions = a["instructions"]

        if is_context_dependent_bonus_activity(a):
            a["is_extended_activity"] = True
            normalized_duration = int(a.get("duration_min", duration_min))
            if normalized_duration < 30:
                normalized_duration = 30
            if normalized_duration > 45:
                normalized_duration = 45
            a["duration_min"] = normalized_duration
            if "optional weekly bonus" not in str(a.get("extended_reason", "")).lower():
                a["extended_reason"] = (
                    "Longer, context-dependent social/community activity best treated as an optional weekly bonus rather than a short daily at-home task."
                )

        safe_activities.append(a)

    return safe_activities

def allocate_weekly_slots(state: Dict[str, Any]) -> Dict[str, Any]:
    child = state["child"]

    if "daily_time_min" not in state["child"]:
        raise ValueError(
            "Missing daily_time_min in state['child']. "
            "Check intake_agent_live() and make sure it stores daily_time_min."
        )

    chrono = min(child["chronological_months"], 60)
    daily_time_min = int(child["daily_time_min"])
    weekly_minutes = daily_time_min * 5

    candidate_categories = state.get("planned_categories") or list(DOMAIN_CONFIG.keys())

    supported_categories = []
    gap_by_category = {}
    weight_by_category = {}

    for category_key in candidate_categories:
        tier = get_support_tier(state, category_key)
        if tier == "no_special_support":
            continue

        supported_categories.append(category_key)

        dev_age = get_effective_dev_age(state, category_key)
        if dev_age is None:
            dev_age = state["dev_age"].get(category_key, chrono)
        gap = max(0, chrono - dev_age)
        gap_by_category[category_key] = gap

        if tier == "needs_special_support":
            weight_by_category[category_key] = max(1, gap)
        else:  # monitor_and_enrich
            weight_by_category[category_key] = max(1, gap) * 0.5

    soft_floor = determine_family_guidance_floor(state)
    if not supported_categories and soft_floor.get("enabled"):
        category_key = soft_floor["category_key"]
        target_minutes = min(int(soft_floor.get("target_weekly_minutes", 20)), weekly_minutes)
        allocation = {
            "daily_time_min": daily_time_min,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [category_key],
            "gap_by_category": {category_key: 0},
            "target_minutes_by_category": {category_key: target_minutes},
            "planning_mode": "family_guidance_floor",
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    if not supported_categories:
        allocation = {
            "daily_time_min": daily_time_min,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [],
            "gap_by_category": {},
            "target_minutes_by_category": {},
            "planning_mode": "none",
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    base_minutes_per_category = max(5, daily_time_min // 2)
    target_minutes_by_category = {k: base_minutes_per_category for k in supported_categories}

    used_minutes = base_minutes_per_category * len(supported_categories)
    remaining_minutes = max(0, weekly_minutes - used_minutes)

    weights = weight_by_category.copy()
    total_weight = sum(weights.values())

    if total_weight > 0 and remaining_minutes > 0:
        for k in supported_categories:
            extra = round(remaining_minutes * (weights[k] / total_weight))
            target_minutes_by_category[k] += extra

    total_target = sum(target_minutes_by_category.values())
    while total_target > weekly_minutes:
        biggest = max(target_minutes_by_category, key=target_minutes_by_category.get)
        if target_minutes_by_category[biggest] > 5:
            target_minutes_by_category[biggest] -= 1
            total_target -= 1
        else:
            break

    allocation = {
        "daily_time_min": daily_time_min,
        "weekly_minutes": weekly_minutes,
        "supported_categories": supported_categories,
        "gap_by_category": gap_by_category,
        "target_minutes_by_category": target_minutes_by_category,
        "planning_mode": "tiered_support",
    }

    state["weekly_slot_allocation"] = allocation
    return allocation


def pick_activity_that_fits(
    activities: List[Dict[str, Any]],
    used_indices: set,
    remaining_minutes: int,
) -> Optional[Dict[str, Any]]:
    candidates = []

    for idx, activity in enumerate(activities):
        if idx in used_indices:
            continue
        if activity.get("is_extended_activity", False):
            continue
        if is_context_dependent_bonus_activity(activity):
            continue
        duration = int(activity.get("duration_min", 5))
        if duration <= remaining_minutes:
            candidates.append((duration, idx, activity))

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda x: x[0])
    duration, idx, activity = candidates[0]
    used_indices.add(idx)
    return activity

def pick_weekly_bonus_extended_activity(
    state: Dict[str, Any],
    extended_in_schedule_threshold: int = 15,
    bonus_extended_min_min: int = 30,
    bonus_extended_cap_min: int = 45,
) -> Optional[Dict[str, Any]]:
    daily_time_min = int(state["child"]["daily_time_min"])

    if daily_time_min >= extended_in_schedule_threshold:
        return None

    allocation = state.get("weekly_slot_allocation", {})
    gap_by_category = allocation.get("gap_by_category", {})

    candidate_categories = sorted(
        gap_by_category.keys(),
        key=lambda k: gap_by_category[k],
        reverse=True,
    )

    for category_key in candidate_categories:
        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])

        for activity in activities:
            if not activity.get("is_extended_activity", False):
                continue

            duration = int(activity.get("duration_min", 5))
            if is_context_dependent_bonus_activity(activity):
                duration = max(duration, bonus_extended_min_min)
            if duration < bonus_extended_min_min and is_context_dependent_bonus_activity(activity):
                continue
            if duration > bonus_extended_cap_min:
                continue

            return {
                "category_key": category_key,
                "category": DOMAIN_CONFIG[category_key]["display"],
                "title": activity.get("title"),
                "instructions": activity.get("instructions"),
                "duration_min": duration,
                "materials": activity.get("materials", "common household items"),
                "level": activity.get("level", "current_or_next"),
                "goal": activity.get("goal", get_support_tier(state, category_key)),
                "extended_reason": activity.get("extended_reason", ""),
            }

    return None

def ensure_minimum_presence_for_monitor_categories(state: Dict[str, Any], days: Dict[str, Any]) -> Dict[str, Any]:
    """Repair pass so monitor/enrich categories with a target do not disappear."""
    allocation = state.get("weekly_slot_allocation", {})
    target_minutes_by_category = allocation.get("target_minutes_by_category", {})
    child = state["child"]
    daily_time_min = int(child["daily_time_min"])

    assigned_minutes = {k: 0 for k in target_minutes_by_category.keys()}
    for day_name, day_info in days.items():
        for item in day_info.get("items", []):
            k = item.get("category_key")
            if k in assigned_minutes:
                assigned_minutes[k] += int(item.get("duration_min", 0))

    # First try: place one short activity into any day with room.
    for category_key in target_minutes_by_category.keys():
        tier = get_support_tier(state, category_key)
        if tier != "monitor_and_enrich":
            continue
        if assigned_minutes.get(category_key, 0) > 0:
            continue

        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])
        if not activities:
            continue

        short_candidates = [a for a in activities if int(a.get("duration_min", 5)) <= 5]
        chosen = short_candidates[0] if short_candidates else min(activities, key=lambda a: int(a.get("duration_min", 5)))
        chosen_duration = int(chosen.get("duration_min", 5))

        placed = False
        for day_name, day_info in days.items():
            remaining = daily_time_min - int(day_info.get("total_minutes", 0))
            if remaining >= chosen_duration:
                day_info["items"].append({
                    "category_key": category_key,
                    "category": DOMAIN_CONFIG[category_key]["display"],
                    "title": chosen.get("title"),
                    "instructions": chosen.get("instructions"),
                    "duration_min": chosen_duration,
                    "materials": chosen.get("materials", "common household items"),
                    "level": chosen.get("level", "current_or_next"),
                    "goal": chosen.get("goal", "monitor_and_enrich"),
                })
                day_info["total_minutes"] += chosen_duration
                assigned_minutes[category_key] += chosen_duration
                placed = True
                break

    # Second try: if still missing, swap in one short activity by replacing an activity
    # from a category currently over target.
    for category_key in target_minutes_by_category.keys():
        tier = get_support_tier(state, category_key)
        if tier != "monitor_and_enrich":
            continue
        if assigned_minutes.get(category_key, 0) > 0:
            continue

        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])
        if not activities:
            continue

        short_candidates = [a for a in activities if int(a.get("duration_min", 5)) <= 5]
        chosen = short_candidates[0] if short_candidates else min(activities, key=lambda a: int(a.get("duration_min", 5)))
        chosen_duration = int(chosen.get("duration_min", 5))

        current_assigned = {}
        for k in target_minutes_by_category.keys():
            current_assigned[k] = 0
        for day_name, day_info in days.items():
            for item in day_info.get("items", []):
                k = item.get("category_key")
                if k in current_assigned:
                    current_assigned[k] += int(item.get("duration_min", 0))

        over_target_categories = [
            k for k in current_assigned.keys()
            if current_assigned[k] > target_minutes_by_category.get(k, 0)
        ]

        swapped = False
        for day_name, day_info in days.items():
            day_items = day_info.get("items", [])
            for idx, item in enumerate(day_items):
                existing_key = item.get("category_key")
                existing_duration = int(item.get("duration_min", 0))
                if existing_key not in over_target_categories:
                    continue
                if existing_duration < chosen_duration:
                    continue

                day_items[idx] = {
                    "category_key": category_key,
                    "category": DOMAIN_CONFIG[category_key]["display"],
                    "title": chosen.get("title"),
                    "instructions": chosen.get("instructions"),
                    "duration_min": chosen_duration,
                    "materials": chosen.get("materials", "common household items"),
                    "level": chosen.get("level", "current_or_next"),
                    "goal": chosen.get("goal", "monitor_and_enrich"),
                }
                day_info["total_minutes"] = sum(int(x.get("duration_min", 0)) for x in day_items)
                swapped = True
                break
            if swapped:
                break

    return days

def build_weekly_schedule(state: Dict[str, Any]) -> Dict[str, Any]:
    if "weekly_slot_allocation" not in state:
        allocate_weekly_slots(state)

    allocation = state["weekly_slot_allocation"]
    daily_time_min = allocation["daily_time_min"]
    target_minutes_by_category = allocation["target_minutes_by_category"]

    days = {
        "day_1": {"items": [], "total_minutes": 0},
        "day_2": {"items": [], "total_minutes": 0},
        "day_3": {"items": [], "total_minutes": 0},
        "day_4": {"items": [], "total_minutes": 0},
        "day_5": {"items": [], "total_minutes": 0},
    }

    soft_floor = state.get("family_guidance_floor", {})

    if not target_minutes_by_category:
        state["weekly_schedule"] = {
            "status": "no_special_support",
            "summary": "No categories need a scheduled weekly activity plan right now.",
            "days": days,
        }
        return state["weekly_schedule"]

    assigned_minutes_by_category = {k: 0 for k in target_minutes_by_category.keys()}
    used_activity_indices = {k: set() for k in target_minutes_by_category.keys()}
    day_names = list(days.keys())

    progress_made = True
    while progress_made:
        progress_made = False

        categories_in_priority_order = sorted(
            target_minutes_by_category.keys(),
            key=lambda k: target_minutes_by_category[k] - assigned_minutes_by_category[k],
            reverse=True,
        )

        for day_name in day_names:
            remaining_day_minutes = daily_time_min - days[day_name]["total_minutes"]
            if remaining_day_minutes <= 0:
                continue

            for category_key in categories_in_priority_order:
                remaining_cat_minutes = (
                    target_minutes_by_category[category_key] - assigned_minutes_by_category[category_key]
                )
                if remaining_cat_minutes <= 0:
                    continue

                bank = state["activity_banks"].get(category_key, {})
                activities = bank.get("activities", [])
                if not activities:
                    continue

                activity = pick_activity_that_fits(
                    activities=activities,
                    used_indices=used_activity_indices[category_key],
                    remaining_minutes=remaining_day_minutes,
                )

                if activity is None:
                    continue

                duration = int(activity.get("duration_min", 5))

                days[day_name]["items"].append({
                    "category_key": category_key,
                    "category": DOMAIN_CONFIG[category_key]["display"],
                    "title": activity.get("title"),
                    "instructions": activity.get("instructions"),
                    "duration_min": duration,
                    "materials": activity.get("materials", "common household items"),
                    "level": activity.get("level", "current_or_next"),
                    "goal": activity.get("goal", get_support_tier(state, category_key)),
                })

                days[day_name]["total_minutes"] += duration
                assigned_minutes_by_category[category_key] += duration
                progress_made = True
                break

    days = ensure_minimum_presence_for_monitor_categories(state, days)

    # recompute assigned minutes after repair pass
    assigned_minutes_by_category = {k: 0 for k in target_minutes_by_category.keys()}
    for day_name, day_info in days.items():
        for item in day_info.get("items", []):
            k = item.get("category_key")
            if k in assigned_minutes_by_category:
                assigned_minutes_by_category[k] += int(item.get("duration_min", 0))

    bonus_activity = pick_weekly_bonus_extended_activity(
        state,
        extended_in_schedule_threshold=15,
        bonus_extended_cap_min=20,
    )

    triage_info = state.get("focus_domain_triage", {})
    selected_keys = state.get("selected_categories", []) or triage_info.get("selected_categories", [])
    selected_displays = [DOMAIN_CONFIG[k]["display"] for k in selected_keys if k in DOMAIN_CONFIG]
    unassessed_displays = [
        DOMAIN_CONFIG[k]["display"]
        for k in DOMAIN_CONFIG
        if k not in selected_keys
    ]

    focus_prefix = ""
    if selected_displays:
        focus_prefix = f"This first onboarding plan focused on {', '.join(selected_displays)}. "
        if unassessed_displays:
            focus_prefix += "Other areas were not fully assessed yet and can be reviewed later. "

    summary_text = focus_prefix + "Weekly schedule built from category activity banks and true daily minute limits."
    status_text = "success"
    if soft_floor.get("enabled"):
        status_text = "family_guidance_floor"
        summary_text = focus_prefix + soft_floor.get("summary", "") + " Weekly schedule built as low-intensity enrich-and-observe guidance within the family's time budget."

    schedule = {
        "status": status_text,
        "summary": summary_text,
        "daily_time_min": daily_time_min,
        "target_minutes_by_category": target_minutes_by_category,
        "assigned_minutes_by_category": assigned_minutes_by_category,
        "days": days,
        "weekly_bonus_activity": bonus_activity,
        "selected_focus_areas": selected_displays,
        "not_yet_assessed_areas": unassessed_displays,
    }

    state["weekly_schedule"] = schedule
    return schedule

def summarizer_agent(state: Dict[str, Any]) -> pd.DataFrame:
    child = state["child"]
    rows = []

    allocation = state.get("weekly_slot_allocation", {}).get("target_minutes_by_category", {})
    concern_profile = ensure_concern_profile(state)
    soft_floor = state.get("family_guidance_floor", {})

    for category_key in DOMAIN_CONFIG:
        delay_info = state["delay_estimates"].get(category_key, {})
        raw_dev_age = state["dev_age"].get(category_key)
        effective_dev_age = get_effective_dev_age(state, category_key)
        metrics = compute_support_metrics(state, category_key)

        chrono_for_gap = min(child.get("chronological_months", 0), 60)
        milestone_gap = None if effective_dev_age is None else max(0, chrono_for_gap - effective_dev_age)

        bank = state.get("activity_banks", {}).get(category_key, {})
        support_tier = metrics["tier"]
        planning_tier = support_tier
        family_guidance_floor_active = False
        family_guidance_summary = ""
        if soft_floor.get("enabled") and soft_floor.get("category_key") == category_key:
            planning_tier = "enrich_and_observe"
            family_guidance_floor_active = True
            family_guidance_summary = soft_floor.get("summary", "")

        rows.append({
            "child_name": child.get("name"),
            "chronological_age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "concern": child.get("concern"),
            "daily_time_min": child.get("daily_time_min"),
            "category": DOMAIN_CONFIG[category_key]["display"],
            "estimated_delay_months": delay_info.get("delay_months"),
            "delay_reason": delay_info.get("reason", ""),
            "raw_dev_age_months": raw_dev_age,
            "estimated_dev_age_months": effective_dev_age,
            "effective_dev_age_months": effective_dev_age,
            "milestone_gap_months": milestone_gap,
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "needs_special_support": support_tier == "needs_special_support",
            "concern_domain_weight": round(concern_profile["domain_weights"].get(category_key, 0.0), 2),
            "concern_subdomain_peak": round(get_category_concern_peak(state, category_key), 2),
            "support_score": metrics["support_score"],
            "weekly_target_minutes_for_category": allocation.get(category_key, 0),
            "activity_bank_status": bank.get("status", "missing"),
            "activity_bank_summary": bank.get("summary", ""),
            "selected_for_onboarding": category_key in state.get("selected_categories", []),
            "assessed_in_this_run": raw_dev_age is not None,
        })

    return pd.DataFrame(rows)

def display_weekly_schedule(state: Dict[str, Any]) -> None:
    schedule = state.get("weekly_schedule", {})

    print_banner("WEEKLY SCHEDULE")

    if schedule.get("status") == "no_special_support":
        print(schedule.get("summary", "No weekly schedule available."))
        return

    print("Summary:", schedule.get("summary", ""))
    print("Daily time budget:", schedule.get("daily_time_min"))
    print("Target minutes by category:", schedule.get("target_minutes_by_category", {}))
    print("Assigned minutes by category:", schedule.get("assigned_minutes_by_category", {}))

    for day_name, day_info in schedule.get("days", {}).items():
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min")
        print("-" * 100)

        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue

        for item in items:
            print(f"- [{item['category']}] {item['title']} ({item['duration_min']} min)")
            print(f"  Goal: {item.get('goal', '')}")
            print(f"  Instructions: {item['instructions']}")
            print(f"  Materials: {item['materials']}")

    bonus = schedule.get("weekly_bonus_activity")
    if bonus:
        print("\n" + "=" * 100)
        print("OPTIONAL WEEKLY BONUS ACTIVITY")
        print("=" * 100)
        print(f"- [{bonus['category']}] {bonus['title']} ({bonus['duration_min']} min)")
        print(f"  Goal: {bonus.get('goal', '')}")
        print(f"  Instructions: {bonus['instructions']}")
        print(f"  Materials: {bonus['materials']}")
        print(f"  Why extended: {bonus.get('extended_reason', '')}")

def run_downstream_planning(
    state: Dict[str, Any],
    display_schedule: bool = True,
    selected_categories: Optional[List[str]] = None,
) -> pd.DataFrame:
    determine_family_guidance_floor(state)

    dev_age_map = state.get("dev_age", {})

    categories_to_plan = []
    for category_key in list(selected_categories or []):
        if dev_age_map.get(category_key) is not None:
            categories_to_plan.append(category_key)
        else:
            print(
                f"Skipping {DOMAIN_CONFIG[category_key]['display']} in planning "
                f"because no developmental age was estimated."
            )

    floor_category = state.get("family_guidance_floor", {}).get("category_key")
    if floor_category and floor_category not in categories_to_plan:
        if dev_age_map.get(floor_category) is not None:
            categories_to_plan.append(floor_category)

    if not categories_to_plan:
        for category_key in DOMAIN_CONFIG:
            if dev_age_map.get(category_key) is not None:
                categories_to_plan.append(category_key)

    state["planned_categories"] = categories_to_plan

    for category_key in categories_to_plan:
        generate_category_activity_bank(state, category_key)

    allocate_weekly_slots(state)
    build_weekly_schedule(state)

    summary_df = summarizer_agent(state)

    if display_schedule:
        display_weekly_schedule(state)

    return summary_df



## Legacy base layer used by v21: Bridge Milestone Engine + Activity Bank + Mastery Loop

This older v17 base layer is still kept because later v18/v19/v21 cells override and reuse its helper functions. In this cleaned v21 notebook, its file loaders have been patched to prefer the final `all_with_bridge_family` table, so running all cells should not silently load the old `previous` table.


In [44]:

# ------------------------------------------------------------
# Legacy v17 base layer — Bridge Milestone Engine + Activity Bank + Mastery Loop
# ------------------------------------------------------------
# This cell intentionally overrides selected v16 planning functions while
# preserving the v16 focused onboarding, support-tier logic, safety logic, and
# language-readiness guardrails.

PREFERRED_BRIDGE_FILENAMES = [
    "cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx",
    "cdc_milestones_with_bridges_family_cleaned.xlsx",
    "cdc_milestones_with_bridges_family.xlsx",
    "cdc_milestones_with_bridges_previous(1).xlsx",
    "cdc_milestones_with_bridges_previous.xlsx",
    "cdc_milestones_with_bridges.xlsx",
    "cdc_milestones_with_bridges.numbers",
    # Older draft bridge tables kept only as fallbacks.
    "genex_bridge_milestone_reference_v17_final.xlsx",
    "bridge_milestone_reference.xlsx",
    "bridge_milestone_map.xlsx",
    "genex_bridge_milestone_reference_v17_prerequisite_style.xlsx",
    "genex_bridge_milestone_reference_v17_36_60_continuation.xlsx",
    "genex_bridge_milestone_reference_v17_line_by_line.xlsx",
    "genex_bridge_milestone_reference_v17_specific_draft.xlsx",
]

BRIDGE_SHEET_CANDIDATES = [
    "all_with_bridge_family",
    "all_with_bridge",
    "Bridge_Milestone_Map",
    "Bridge_Map_36_60",
    "bridge_milestone_map",
    "bridge_map",
]

ACTIVITY_QUALITY_GUARDRAILS = """
Every activity must be playful, low-pressure, and success-oriented.
Do not write therapy-style drills.
Use games, songs, books, toys, pretend play, movement, choices, or routines.
The child should be able to succeed even if they cannot fully do the target skill yet.
Avoid making repetition feel like testing; make it feel like play.
""".strip()

DEFAULT_PLAY_THEMES = [
    "book game",
    "animal game",
    "music/song",
    "pretend play",
    "toy rescue",
    "sticker game",
    "treasure hunt",
    "snack routine",
    "bath time",
    "sibling helper game",
    "cleanup game",
    "movement game",
]

ACTIVITY_FEEDBACK_OPTIONS = {
    "difficulty": ["too_hard", "just_right", "too_easy"],
    "performance": ["done_independently", "done_with_help", "couldnt_do_it"],
    "engagement": ["enjoyed_it", "resisted_it", "didnt_like_it"],
}

# Snapshot of the currently uploaded advisor-authored Numbers bridge table.
# The notebook will prefer an external file named cdc_milestones_with_bridges.numbers/.xlsx
# when available, but this embedded fallback lets v17 run even if the file is not next to
# the notebook. Replace the external bridge file later with the corrected advisor-reviewed table.
EMBEDDED_USER_BRIDGE_ROWS = [{'months': 2, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'calms down when spoken to or picked up', 'subdomain': 'emotional_regulation', 'bridge_step_title': 'Responds to caregiver voice, touch, or holding and shows brief calming support.', 'bridge_step_description': 'Responds to caregiver voice, touch, or holding and shows brief calming support.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'calms down when spoken to or picked up', 'subdomain': 'emotional_regulation', 'bridge_step_title': 'Calms down when spoken to or picked up', 'bridge_step_description': 'Calms down when spoken to or picked up'}, {'months': 2, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'looks at your face', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Opens eyes and briefly attends to nearby faces.', 'bridge_step_description': 'Opens eyes and briefly attends to nearby faces.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'looks at your face', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Looks at your face.', 'bridge_step_description': 'Looks at your face.'}, {'months': 2, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'seems happy to see you when you walk up to her', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Notices caregiver’s face, voice, or movement and shows body or facial changes.', 'bridge_step_description': 'Notices caregiver’s face, voice, or movement and shows body or facial changes.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'seems happy to see you when you walk up to her', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Seems happy to see you when you walk up to her.', 'bridge_step_description': 'Seems happy to see you when you walk up to her.'}, {'months': 2, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'smiles when you talk to or smile at her', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Watches caregiver’s face during calm interaction.', 'bridge_step_description': 'Watches caregiver’s face during calm interaction.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'smiles when you talk to or smile at her', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Shows early facial response to voice or smile.', 'bridge_step_description': 'Shows early facial response to voice or smile.'}, {'months': 2, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'smiles when you talk to or smile at her', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'smiles when you talk to or smile at her', 'bridge_step_description': 'smiles when you talk to or smile at her'}, {'months': 2, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'makes sounds other than crying', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Makes small throat, vowel, or comfort sounds.', 'bridge_step_description': 'Makes small throat, vowel, or comfort sounds.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'makes sounds other than crying', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Makes sounds other than crying.', 'bridge_step_description': 'Makes sounds other than crying.'}, {'months': 2, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'reacts to loud sounds', 'subdomain': 'receptive_language', 'bridge_step_title': 'Startles or stills to sudden sound.', 'bridge_step_description': 'Startles or stills to sudden sound.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'reacts to loud sounds', 'subdomain': 'receptive_language', 'bridge_step_title': 'Reacts to loud sounds.', 'bridge_step_description': 'Reacts to loud sounds.'}, {'months': 2, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'watches you as you move', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Notices a nearby face or movement.', 'bridge_step_description': 'Notices a nearby face or movement.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'watches you as you move', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Follows slow movement briefly with eyes.', 'bridge_step_description': 'Follows slow movement briefly with eyes.'}, {'months': 2, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'watches you as you move', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Watches you as you move.', 'bridge_step_description': 'Watches you as you move.'}, {'months': 2, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'looks at a toy for sevedral seconds', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Notices a high-contrast or nearby object.', 'bridge_step_description': 'Notices a high-contrast or nearby object.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'looks at a toy for sevedral seconds', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Looks at a toy for several seconds.', 'bridge_step_description': 'Looks at a toy for several seconds.'}, {'months': 2, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'holds head up when on tummy', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Tolerates brief tummy time while awake and supervised.', 'bridge_step_description': 'Tolerates brief tummy time while awake and supervised.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'holds head up when on tummy', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Turns head side to side while on tummy.', 'bridge_step_description': 'Turns head side to side while on tummy.'}, {'months': 2, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'holds head up when on tummy', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Holds head up when on tummy.', 'bridge_step_description': 'Holds head up when on tummy.'}, {'months': 2, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'moves both arms and both legs', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Shows spontaneous arm or leg movement.', 'bridge_step_description': 'Shows spontaneous arm or leg movement.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'moves both arms and both legs', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Moves both arms and both legs.', 'bridge_step_description': 'Moves both arms and both legs.'}, {'months': 2, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'open hands briefly', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Hands are sometimes loosely closed instead of tightly fisted.', 'bridge_step_description': 'Hands are sometimes loosely closed instead of tightly fisted.'}, {'months': 2, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'open hands briefly', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'open hands briefly', 'bridge_step_description': 'open hands briefly'}, {'months': 4, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'smiles on his own to get your attention', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Watches caregiver’s face with interest.', 'bridge_step_description': 'Watches caregiver’s face with interest.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'smiles on his own to get your attention', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Smiles in response to caregiver interaction.', 'bridge_step_description': 'Smiles in response to caregiver interaction.'}, {'months': 4, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'smiles on his own to get your attention', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Smiles on his own to get your attention.', 'bridge_step_description': 'Smiles on his own to get your attention.'}, {'months': 4, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'chuckles when you try to make him laugh but not yet a full laugh', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Shows excitement during playful interaction.', 'bridge_step_description': 'Shows excitement during playful interaction.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'chuckles when you try to make him laugh but not yet a full laugh', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Chuckles when you try to make him laugh.', 'bridge_step_description': 'Chuckles when you try to make him laugh.'}, {'months': 4, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'looks at you moves or makes sounds to get or keep you attention', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Looks toward caregiver during interaction.', 'bridge_step_description': 'Looks toward caregiver during interaction.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'looks at you moves or makes sounds to get or keep you attention', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Uses body movement or sounds in response to attention.', 'bridge_step_description': 'Uses body movement or sounds in response to attention.'}, {'months': 4, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'looks at you moves or makes sounds to get or keep you attention', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'looks at you moves or makes sounds to get or keep you attention', 'bridge_step_description': 'looks at you moves or makes sounds to get or keep you attention'}, {'months': 4, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'makes cooing sounds like ooooo or aahhh', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Makes soft vowel-like sounds.', 'bridge_step_description': 'Makes soft vowel-like sounds.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'makes cooing sounds like ooooo or aahhh', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Vocalizes during calm face-to-face interaction.', 'bridge_step_description': 'Vocalizes during calm face-to-face interaction.'}, {'months': 4, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'makes cooing sounds like ooooo or aahhh', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'makes cooing sounds like ooooo or aahhh', 'bridge_step_description': 'makes cooing sounds like ooooo or aahhh'}, {'months': 4, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'makes sounds back when you talk to him', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Notices caregiver voice during interaction.', 'bridge_step_description': 'Notices caregiver voice during interaction.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'makes sounds back when you talk to him', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Makes sounds other than crying.', 'bridge_step_description': 'Makes sounds other than crying.'}, {'months': 4, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'makes sounds back when you talk to him', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Makes sounds back when you talk to him.', 'bridge_step_description': 'Makes sounds back when you talk to him.'}, {'months': 4, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'turns head towards the sound of your voice', 'subdomain': 'receptive_language', 'bridge_step_title': 'Notices familiar caregiver voice.', 'bridge_step_description': 'Notices familiar caregiver voice.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'turns head towards the sound of your voice', 'subdomain': 'receptive_language', 'bridge_step_title': 'Moves eyes or head toward the sound.', 'bridge_step_description': 'Moves eyes or head toward the sound.'}, {'months': 4, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'turns head towards the sound of your voice', 'subdomain': 'receptive_language', 'bridge_step_title': 'turns head towards the sound of your voice', 'bridge_step_description': 'turns head towards the sound of your voice'}, {'months': 4, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'if hungry open mouth when she sees breast or bottle', 'subdomain': 'adaptive_feeding_cues', 'bridge_step_title': 'Shows hunger cues such as rooting, sucking, or becoming alert.', 'bridge_step_description': 'Shows hunger cues such as rooting, sucking, or becoming alert.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'if hungry open mouth when she sees breast or bottle', 'subdomain': 'adaptive_feeding_cues', 'bridge_step_title': 'Recognizes feeding routine or caregiver position.', 'bridge_step_description': 'Recognizes feeding routine or caregiver position.'}, {'months': 4, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'if hungry open mouth when she sees breast or bottle', 'subdomain': 'adaptive_feeding_cues', 'bridge_step_title': 'Opens mouth when she sees breast or bottle.', 'bridge_step_description': 'Opens mouth when she sees breast or bottle.'}, {'months': 4, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'looks at her hands with interest', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Brings hands into visual range.', 'bridge_step_description': 'Brings hands into visual range.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'looks at her hands with interest', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Looks at his hands with interest.', 'bridge_step_description': 'Looks at his hands with interest.'}, {'months': 4, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'holds head steady without support when you are holding him', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Holds head briefly upright with caregiver support.', 'bridge_step_description': 'Holds head briefly upright with caregiver support.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'holds head steady without support when you are holding him', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Controls head with less bobbing while held.', 'bridge_step_description': 'Controls head with less bobbing while held.'}, {'months': 4, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'holds head steady without support when you are holding him', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Holds head steady without support when held.', 'bridge_step_description': 'Holds head steady without support when held.'}, {'months': 4, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'holds a toy when you put it in his hand', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Opens hand when an object touches the palm.', 'bridge_step_description': 'Opens hand when an object touches the palm.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'holds a toy when you put it in his hand', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Holds a toy when you put it in his hand.', 'bridge_step_description': 'Holds a toy when you put it in his hand.'}, {'months': 4, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'uses his arm to swing at toys', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Notices a nearby hanging toy.', 'bridge_step_description': 'Notices a nearby hanging toy.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'uses his arm to swing at toys', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Moves arm toward the toy with increasing control.', 'bridge_step_description': 'Moves arm toward the toy with increasing control.'}, {'months': 4, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'uses his arm to swing at toys', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'uses his arm to swing at toys', 'bridge_step_description': 'uses his arm to swing at toys'}, {'months': 4, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'bring hands to mouth', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Can move both hands', 'bridge_step_description': 'Can move both hands'}, {'months': 4, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'bring hands to mouth', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'uses his arm to swing at toys', 'bridge_step_description': 'uses his arm to swing at toys'}, {'months': 4, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'pushes up onto elbows or forearms when on tummy', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Tolerates supervised tummy time while awake.', 'bridge_step_description': 'Tolerates supervised tummy time while awake.'}, {'months': 4, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'pushes up onto elbows or forearms when on tummy', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Lifts head and upper chest while on tummy.', 'bridge_step_description': 'Lifts head and upper chest while on tummy.'}, {'months': 4, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'pushes up onto elbows or forearms when on tummy', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Pushes up onto elbows or forearms when on tummy.', 'bridge_step_description': 'Pushes up onto elbows or forearms when on tummy.'}, {'months': 6, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'knows familiar people', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'Notices and looks toward familiar voices.', 'bridge_step_description': 'Notices and looks toward familiar voices.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'knows familiar people', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'Watches familiar faces with interest.', 'bridge_step_description': 'Watches familiar faces with interest.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'knows familiar people', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'Responds differently to familiar and unfamiliar people.', 'bridge_step_description': 'Responds differently to familiar and unfamiliar people.'}, {'months': 6, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'knows familiar people', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'Knows familiar people.', 'bridge_step_description': 'Knows familiar people.'}, {'months': 6, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'likes to look at self in a mirror', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Looks at faces with interest.', 'bridge_step_description': 'Looks at faces with interest.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'likes to look at self in a mirror', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Likes to look at self in a mirror.', 'bridge_step_description': 'Likes to look at self in a mirror.'}, {'months': 6, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'laughs', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Shows excitement with sounds or body movement.', 'bridge_step_description': 'Shows excitement with sounds or body movement.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'laughs', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Responds to playful social interaction with bigger joy.', 'bridge_step_description': 'Responds to playful social interaction with bigger joy.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'laughs', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'laughs', 'bridge_step_description': 'laughs'}, {'months': 6, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'takes turns making sounds with you', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Makes sounds or coos during calm interaction.', 'bridge_step_description': 'Makes sounds or coos during calm interaction.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'takes turns making sounds with you', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Vocalizes back after hearing a sound.', 'bridge_step_description': 'Vocalizes back after hearing a sound.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'takes turns making sounds with you', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Takes turns making sounds with you.', 'bridge_step_description': 'Takes turns making sounds with you.'}, {'months': 6, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'blows raspberries or sticks tongue out and blows', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Moves lips and tongue during vocal play.', 'bridge_step_description': 'Moves lips and tongue during vocal play.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'blows raspberries or sticks tongue out and blows', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Pushes air out while moving lips or tongue.', 'bridge_step_description': 'Pushes air out while moving lips or tongue.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'blows raspberries or sticks tongue out and blows', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Blows raspberries or sticks tongue out and blows.', 'bridge_step_description': 'Blows raspberries or sticks tongue out and blows.'}, {'months': 6, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'makes squealing noises', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Makes vowel-like sounds.', 'bridge_step_description': 'Makes vowel-like sounds.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'makes squealing noises', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Uses voice to show excitement or get attention.', 'bridge_step_description': 'Uses voice to show excitement or get attention.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'makes squealing noises', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'makes squealing noises', 'bridge_step_description': 'makes squealing noises'}, {'months': 6, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'puts things in her mouth to explore them', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Reaches for and touches safe objects.', 'bridge_step_description': 'Reaches for and touches safe objects.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'puts things in her mouth to explore them', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Grasps and holds a safe object.', 'bridge_step_description': 'Grasps and holds a safe object.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'puts things in her mouth to explore them', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'puts things in her mouth to explore them', 'bridge_step_description': 'puts things in her mouth to explore them'}, {'months': 6, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'reaches to grab a toy she wants', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Looks at a toy with interest.', 'bridge_step_description': 'Looks at a toy with interest.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'reaches to grab a toy she wants', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Moves arm toward a nearby toy.', 'bridge_step_description': 'Moves arm toward a nearby toy.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'reaches to grab a toy she wants', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'reaches to grab a toy she wants', 'bridge_step_description': 'reaches to grab a toy she wants'}, {'months': 6, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': "closes lips to show she doesn't want more food", 'subdomain': 'adaptive_feeding_cues', 'bridge_step_title': 'Shows different responses during feeding.', 'bridge_step_description': 'Shows different responses during feeding.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': "closes lips to show she doesn't want more food", 'subdomain': 'adaptive_feeding_cues', 'bridge_step_title': 'Pauses or turns away when full or uninterested.', 'bridge_step_description': 'Pauses or turns away when full or uninterested.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': "closes lips to show she doesn't want more food", 'subdomain': 'adaptive_feeding_cues', 'bridge_step_title': "Closes lips to show she doesn't want more food.", 'bridge_step_description': "Closes lips to show she doesn't want more food."}, {'months': 6, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'rolls from tummy to back', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Lifts head and chest during tummy time.', 'bridge_step_description': 'Lifts head and chest during tummy time.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'rolls from tummy to back', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Turns body from tummy onto side.', 'bridge_step_description': 'Turns body from tummy onto side.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'rolls from tummy to back', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Rolls from tummy to back.', 'bridge_step_description': 'Rolls from tummy to back.'}, {'months': 6, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'pushes up with straight arms when on tummy', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Tolerates tummy time for short periods.', 'bridge_step_description': 'Tolerates tummy time for short periods.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'pushes up with straight arms when on tummy', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Pushes up on forearms.', 'bridge_step_description': 'Pushes up on forearms.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'pushes up with straight arms when on tummy', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Pushes up with straight arms when on tummy.', 'bridge_step_description': 'Pushes up with straight arms when on tummy.'}, {'months': 6, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'leans on hands to support herself when sitting', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Holds head and trunk steadier in supported sitting.', 'bridge_step_description': 'Holds head and trunk steadier in supported sitting.'}, {'months': 6, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'leans on hands to support herself when sitting', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Sits with support and brings hands forward.', 'bridge_step_description': 'Sits with support and brings hands forward.'}, {'months': 6, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'leans on hands to support herself when sitting', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'leans on hands to support herself when sitting', 'bridge_step_description': 'leans on hands to support herself when sitting'}, {'months': 9, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'is shy or clingy or fearful around strangers', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'Recognizes familiar caregivers.', 'bridge_step_description': 'Recognizes familiar caregivers.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'is shy or clingy or fearful around strangers', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'Shows comfort with familiar people.', 'bridge_step_description': 'Shows comfort with familiar people.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'is shy or clingy or fearful around strangers', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'Is shy, clingy, or fearful around strangers.', 'bridge_step_description': 'Is shy, clingy, or fearful around strangers.'}, {'months': 9, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'shows several facial expressions like happy or sad or angry or surprised', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Watches familiar faces closely.', 'bridge_step_description': 'Watches familiar faces closely.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'shows several facial expressions like happy or sad or angry or surprised', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Responds to another person’s facial expression.', 'bridge_step_description': 'Responds to another person’s facial expression.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'shows several facial expressions like happy or sad or angry or surprised', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Shows several facial expressions.', 'bridge_step_description': 'Shows several facial expressions.'}, {'months': 9, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'looks when you call her name', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Turns toward sounds nearby.', 'bridge_step_description': 'Turns toward sounds nearby.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'looks when you call her name', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Responds to familiar voices.', 'bridge_step_description': 'Responds to familiar voices.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'looks when you call her name', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'looks when you call her name', 'bridge_step_description': 'looks when you call her name'}, {'months': 9, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'reacts when you leave', 'subdomain': 'emotional_regulation', 'bridge_step_title': 'Recognizes familiar caregivers.', 'bridge_step_description': 'Recognizes familiar caregivers.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'reacts when you leave', 'subdomain': 'emotional_regulation', 'bridge_step_title': 'Reacts when you leave.', 'bridge_step_description': 'Reacts when you leave.'}, {'months': 9, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'smiles or laughs when you play peek a boo', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Watches caregiver’s face during play.', 'bridge_step_description': 'Watches caregiver’s face during play.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'smiles or laughs when you play peek a boo', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Responds to playful voice or facial expressions.', 'bridge_step_description': 'Responds to playful voice or facial expressions.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'smiles or laughs when you play peek a boo', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Smiles or laughs when you play peek-a-boo.', 'bridge_step_description': 'Smiles or laughs when you play peek-a-boo.'}, {'months': 9, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'makes a lot of different sounds like mamamamama and bababababa', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Makes vowel-like sounds.', 'bridge_step_description': 'Makes vowel-like sounds.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'makes a lot of different sounds like mamamamama and bababababa', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Makes sounds in response to people.', 'bridge_step_description': 'Makes sounds in response to people.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'makes a lot of different sounds like mamamamama and bababababa', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Repeats simple sounds.', 'bridge_step_description': 'Repeats simple sounds.'}, {'months': 9, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'makes a lot of different sounds like mamamamama and bababababa', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'Combines consonant and vowel sounds.', 'bridge_step_description': 'Combines consonant and vowel sounds.'}, {'months': 9, 'bridge_step_number': 5, 'category': 'language and communication', 'cdc_milestone': 'makes a lot of different sounds like mamamamama and bababababa', 'subdomain': 'early_vocalization_and_babbling', 'bridge_step_title': 'makes a lot of different sounds like mamamamama and bababababa', 'bridge_step_description': 'makes a lot of different sounds like mamamamama and bababababa'}, {'months': 9, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'lifts arms up to be picked up', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Looks toward caregiver to request attention.', 'bridge_step_description': 'Looks toward caregiver to request attention.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'lifts arms up to be picked up', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Reaches toward people or objects.', 'bridge_step_description': 'Reaches toward people or objects.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'lifts arms up to be picked up', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Lifts arms up to be picked up.', 'bridge_step_description': 'Lifts arms up to be picked up.'}, {'months': 9, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'looks for objects when dropped out of sight', 'subdomain': 'object_permanence_and_problem_solving', 'bridge_step_title': 'Watches moving objects.', 'bridge_step_description': 'Watches moving objects.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'looks for objects when dropped out of sight', 'subdomain': 'object_permanence_and_problem_solving', 'bridge_step_title': 'looks for objects when dropped out of sight', 'bridge_step_description': 'looks for objects when dropped out of sight'}, {'months': 9, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'bangs to things together', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Grasps one safe object.', 'bridge_step_description': 'Grasps one safe object.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'bangs to things together', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Holds objects in both hands.', 'bridge_step_description': 'Holds objects in both hands.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'bangs to things together', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Bangs two things together.', 'bridge_step_description': 'Bangs two things together.'}, {'months': 9, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'gets to sitting position by herself', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Holds head and trunk steady with support.', 'bridge_step_description': 'Holds head and trunk steady with support.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'gets to sitting position by herself', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Rolls or shifts weight during floor play.', 'bridge_step_description': 'Rolls or shifts weight during floor play.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'gets to sitting position by herself', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Pushes up through arms while on tummy.', 'bridge_step_description': 'Pushes up through arms while on tummy.'}, {'months': 9, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'gets to sitting position by herself', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Moves from side-lying or hands-and-knees toward sitting.', 'bridge_step_description': 'Moves from side-lying or hands-and-knees toward sitting.'}, {'months': 9, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'gets to sitting position by herself', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Gets to a sitting position by herself.', 'bridge_step_description': 'Gets to a sitting position by herself.'}, {'months': 9, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'moves things from one hand to her other hand', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Reaches for objects with either hand.', 'bridge_step_description': 'Reaches for objects with either hand.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'moves things from one hand to her other hand', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Holds objects in both hands.', 'bridge_step_description': 'Holds objects in both hands.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'moves things from one hand to her other hand', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Moves things from one hand to her other hand.', 'bridge_step_description': 'Moves things from one hand to her other hand.'}, {'months': 9, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'uses fingers to rake food towards himself', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Reaches toward safe soft food pieces.', 'bridge_step_description': 'Reaches toward safe soft food pieces.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'uses fingers to rake food towards himself', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Uses whole hand to touch or gather food.', 'bridge_step_description': 'Uses whole hand to touch or gather food.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'uses fingers to rake food towards himself', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'uses fingers to rake food towards himself', 'bridge_step_description': 'uses fingers to rake food towards himself'}, {'months': 9, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'sits without support', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Holds head steady.', 'bridge_step_description': 'Holds head steady.'}, {'months': 9, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'sits without support', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Sits with trunk support.', 'bridge_step_description': 'Sits with trunk support.'}, {'months': 9, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'sits without support', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Props with hands while sitting.', 'bridge_step_description': 'Props with hands while sitting.'}, {'months': 9, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'sits without support', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Sits briefly without using hands for support.', 'bridge_step_description': 'Sits briefly without using hands for support.'}, {'months': 9, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'sits without support', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'sits without support', 'bridge_step_description': 'sits without support'}, {'months': 12, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'play games with you like pat a cake', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Watches caregiver’s face and hands.', 'bridge_step_description': 'Watches caregiver’s face and hands.'}, {'months': 12, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'play games with you like pat a cake', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Responds to playful voice or gestures.', 'bridge_step_description': 'Responds to playful voice or gestures.'}, {'months': 12, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'play games with you like pat a cake', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Imitates simple hand or body movements.', 'bridge_step_description': 'Imitates simple hand or body movements.'}, {'months': 12, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'play games with you like pat a cake', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Plays games with you, like pat-a-cake.', 'bridge_step_description': 'Plays games with you, like pat-a-cake.'}, {'months': 12, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'waves byebye', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Watches caregiver’s face and hands.', 'bridge_step_description': 'Watches caregiver’s face and hands.'}, {'months': 12, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'waves byebye', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Waves with a cue or model.', 'bridge_step_description': 'Waves with a cue or model.'}, {'months': 12, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'waves byebye', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Waves bye-bye.', 'bridge_step_description': 'Waves bye-bye.'}, {'months': 12, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'calls parents mama or dada or another special name', 'subdomain': 'expressive_language', 'bridge_step_title': 'Make different sounds', 'bridge_step_description': 'Make different sounds'}, {'months': 12, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'calls parents mama or dada or another special name', 'subdomain': 'expressive_language', 'bridge_step_title': 'Make constant vowel combination', 'bridge_step_description': 'Make constant vowel combination'}, {'months': 12, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'calls parents mama or dada or another special name', 'subdomain': 'expressive_language', 'bridge_step_title': 'Makes repeated sounds like “ma,” “ba,” or “da.”', 'bridge_step_description': 'Makes repeated sounds like “ma,” “ba,” or “da.”'}, {'months': 12, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'calls parents mama or dada or another special name', 'subdomain': 'expressive_language', 'bridge_step_title': 'Imitates familiar sound patterns.', 'bridge_step_description': 'Imitates familiar sound patterns.'}, {'months': 12, 'bridge_step_number': 5, 'category': 'language and communication', 'cdc_milestone': 'calls parents mama or dada or another special name', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses a consistent sound or words.', 'bridge_step_description': 'Uses a consistent sound or words.'}, {'months': 12, 'bridge_step_number': 6, 'category': 'language and communication', 'cdc_milestone': 'calls parents mama or dada or another special name', 'subdomain': 'expressive_language', 'bridge_step_title': 'Calls parent “mama,” “dada,” or another special name.', 'bridge_step_description': 'Calls parent “mama,” “dada,” or another special name.'}, {'months': 12, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'understands no so pauses briefly or stops when you say it', 'subdomain': 'receptive_language', 'bridge_step_title': 'Responds to caregiver’s voice tone.', 'bridge_step_description': 'Responds to caregiver’s voice tone.'}, {'months': 12, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'understands no so pauses briefly or stops when you say it', 'subdomain': 'receptive_language', 'bridge_step_title': 'Looks toward caregiver when corrected.', 'bridge_step_description': 'Looks toward caregiver when corrected.'}, {'months': 12, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'understands no so pauses briefly or stops when you say it', 'subdomain': 'receptive_language', 'bridge_step_title': 'Stops or slows an action briefly with a physical or a verbal cue.', 'bridge_step_description': 'Stops or slows an action briefly with a physical or a verbal cue.'}, {'months': 12, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'understands no so pauses briefly or stops when you say it', 'subdomain': 'receptive_language', 'bridge_step_title': 'understands no so pauses briefly or stops when you say it', 'bridge_step_description': 'understands no so pauses briefly or stops when you say it'}, {'months': 12, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'puts something in a container like a block in a cap', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Holds and explores safe objects.', 'bridge_step_description': 'Holds and explores safe objects.'}, {'months': 12, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'puts something in a container like a block in a cap', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Releases objects intentionally.', 'bridge_step_description': 'Releases objects intentionally.'}, {'months': 12, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'puts something in a container like a block in a cap', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Places an object into a large opening.', 'bridge_step_description': 'Places an object into a large opening.'}, {'months': 12, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'puts something in a container like a block in a cap', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'puts something in a container like a block in a cap', 'bridge_step_description': 'puts something in a container like a block in a cap'}, {'months': 12, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'looks for things he sees you hide like a toy under a blanket', 'subdomain': 'object_permanence_and_problem_solving', 'bridge_step_title': 'Watches a toy or object closely.', 'bridge_step_description': 'Watches a toy or object closely.'}, {'months': 12, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'looks for things he sees you hide like a toy under a blanket', 'subdomain': 'object_permanence_and_problem_solving', 'bridge_step_title': 'Follows an object as it moves.', 'bridge_step_description': 'Follows an object as it moves.'}, {'months': 12, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'looks for things he sees you hide like a toy under a blanket', 'subdomain': 'object_permanence_and_problem_solving', 'bridge_step_title': 'Reaches toward or uncovers a partly hidden object.', 'bridge_step_description': 'Reaches toward or uncovers a partly hidden object.'}, {'months': 12, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'looks for things he sees you hide like a toy under a blanket', 'subdomain': 'object_permanence_and_problem_solving', 'bridge_step_title': 'Looks for things he sees you hide.', 'bridge_step_description': 'Looks for things he sees you hide.'}, {'months': 12, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'pulls up to stand', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Sits with steady trunk control.', 'bridge_step_description': 'Sits with steady trunk control.'}, {'months': 12, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'pulls up to stand', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Moves from sitting toward kneeling.', 'bridge_step_description': 'Moves from sitting toward kneeling.'}, {'months': 12, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'pulls up to stand', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Pulls on a stable surface to rise.', 'bridge_step_description': 'Pulls on a stable surface to rise.'}, {'months': 12, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'pulls up to stand', 'subdomain': 'postural_control_and_transitions', 'bridge_step_title': 'Pulls up to stand.', 'bridge_step_description': 'Pulls up to stand.'}, {'months': 12, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'walks holding on to furniture', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Stands while holding a stable surface.', 'bridge_step_description': 'Stands while holding a stable surface.'}, {'months': 12, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'walks holding on to furniture', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Takes side steps while holding support.', 'bridge_step_description': 'Takes side steps while holding support.'}, {'months': 12, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'walks holding on to furniture', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Moves along furniture with balance and control.', 'bridge_step_description': 'Moves along furniture with balance and control.'}, {'months': 12, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'walks holding on to furniture', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Walks holding on to furniture.', 'bridge_step_description': 'Walks holding on to furniture.'}, {'months': 12, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'drinks from a cup without a lid as you hold it', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Holds head and trunk steady while sitting.', 'bridge_step_description': 'Holds head and trunk steady while sitting.'}, {'months': 12, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'drinks from a cup without a lid as you hold it', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Closes lips around the cup edge when prompted', 'bridge_step_description': 'Closes lips around the cup edge when prompted'}, {'months': 12, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'drinks from a cup without a lid as you hold it', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Drinks from a cup without a lid as you hold it', 'bridge_step_description': 'Drinks from a cup without a lid as you hold it'}, {'months': 12, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'picks things up between thumb and pointer finger like small bits of food', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Reaches for small safe food pieces.', 'bridge_step_description': 'Reaches for small safe food pieces.'}, {'months': 12, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'picks things up between thumb and pointer finger like small bits of food', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Uses whole hand to grasp small items.', 'bridge_step_description': 'Uses whole hand to grasp small items.'}, {'months': 12, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'picks things up between thumb and pointer finger like small bits of food', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Uses fingers to rake items closer.', 'bridge_step_description': 'Uses fingers to rake items closer.'}, {'months': 12, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'picks things up between thumb and pointer finger like small bits of food', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Picks things up between thumb and pointer finger.', 'bridge_step_description': 'Picks things up between thumb and pointer finger.'}, {'months': 15, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'copies other children while playing like taking toys out of a container when another child does', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Notices another child playing nearby.', 'bridge_step_description': 'Notices another child playing nearby.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'copies other children while playing like taking toys out of a container when another child does', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Imitates a simple action from an adult.', 'bridge_step_description': 'Imitates a simple action from an adult.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'copies other children while playing like taking toys out of a container when another child does', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Can play with other children or near them with adult invitation and supervision.', 'bridge_step_description': 'Can play with other children or near them with adult invitation and supervision.'}, {'months': 15, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'copies other children while playing like taking toys out of a container when another child does', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Copies other children while playing, like taking toys out of a container', 'bridge_step_description': 'Copies other children while playing, like taking toys out of a container'}, {'months': 15, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'shows you an object she likes', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Holds or explores a preferred object.', 'bridge_step_description': 'Holds or explores a preferred object.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'shows you an object she likes', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Pay attention to a caregiver nearby', 'bridge_step_description': 'Pay attention to a caregiver nearby'}, {'months': 15, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'shows you an object she likes', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Shows you an object she likes.', 'bridge_step_description': 'Shows you an object she likes.'}, {'months': 15, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'claps when excited', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Shows excitement with face, voice, or body movement.', 'bridge_step_description': 'Shows excitement with face, voice, or body movement.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'claps when excited', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Imitates clapping with support or modeling.', 'bridge_step_description': 'Imitates clapping with support or modeling.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'claps when excited', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Claps when excited.', 'bridge_step_description': 'Claps when excited.'}, {'months': 15, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'hugs stuffed doll or other toy', 'subdomain': 'play_and_symbolic_social_play', 'bridge_step_title': 'Shows interest in a doll, stuffed animal, or soft toy.', 'bridge_step_description': 'Shows interest in a doll, stuffed animal, or soft toy.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'hugs stuffed doll or other toy', 'subdomain': 'play_and_symbolic_social_play', 'bridge_step_title': 'Holds or carries the toy close.', 'bridge_step_description': 'Holds or carries the toy close.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'hugs stuffed doll or other toy', 'subdomain': 'play_and_symbolic_social_play', 'bridge_step_title': 'Copy and imitate adult hugging a toy', 'bridge_step_description': 'Copy and imitate adult hugging a toy'}, {'months': 15, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'hugs stuffed doll or other toy', 'subdomain': 'play_and_symbolic_social_play', 'bridge_step_title': 'hugs stuffed doll or other toy', 'bridge_step_description': 'hugs stuffed doll or other toy'}, {'months': 15, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'shows you affection like hugs or cuddles or kisses you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Seeks closeness with a familiar caregiver.', 'bridge_step_description': 'Seeks closeness with a familiar caregiver.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'shows you affection like hugs or cuddles or kisses you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Responds positively to gentle affection.', 'bridge_step_description': 'Responds positively to gentle affection.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'shows you affection like hugs or cuddles or kisses you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Initiates simple affectionate contact.', 'bridge_step_description': 'Initiates simple affectionate contact.'}, {'months': 15, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'shows you affection like hugs or cuddles or kisses you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'shows you affection like hugs or cuddles or kisses you', 'bridge_step_description': 'shows you affection like hugs or cuddles or kisses you'}, {'months': 15, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'tries to say or or two words besides mama or dada like ba for ball or da for dog', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses sounds or gestures to communicate wants.', 'bridge_step_description': 'Uses sounds or gestures to communicate wants.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'tries to say or or two words besides mama or dada like ba for ball or da for dog', 'subdomain': 'expressive_language', 'bridge_step_title': 'Imitates simple sounds or word-like sounds.', 'bridge_step_description': 'Imitates simple sounds or word-like sounds.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'tries to say or or two words besides mama or dada like ba for ball or da for dog', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses a consistent sound for a familiar object or person.', 'bridge_step_description': 'Uses a consistent sound for a familiar object or person.'}, {'months': 15, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'tries to say or or two words besides mama or dada like ba for ball or da for dog', 'subdomain': 'expressive_language', 'bridge_step_title': 'Make constant vowel combination', 'bridge_step_description': 'Make constant vowel combination'}, {'months': 15, 'bridge_step_number': 5, 'category': 'language and communication', 'cdc_milestone': 'tries to say or or two words besides mama or dada like ba for ball or da for dog', 'subdomain': 'expressive_language', 'bridge_step_title': 'Tries to say one or two words besides “mama” or “dada.”', 'bridge_step_description': 'Tries to say one or two words besides “mama” or “dada.”'}, {'months': 15, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'looks at a ffamiliar object when you name it', 'subdomain': 'receptive_language', 'bridge_step_title': 'Notices familiar objects during routines.', 'bridge_step_description': 'Notices familiar objects during routines.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'looks at a ffamiliar object when you name it', 'subdomain': 'receptive_language', 'bridge_step_title': 'Looks at a familiar object when you name it.', 'bridge_step_description': 'Looks at a familiar object when you name it.'}, {'months': 15, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'follows directions given with both a gesture and words for example she gives you a toy when you hold out your hand and say give me the toy', 'subdomain': 'receptive_language', 'bridge_step_title': 'Responds to caregiver’s voice and attention cues.', 'bridge_step_description': 'Responds to caregiver’s voice and attention cues.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'follows directions given with both a gesture and words for example she gives you a toy when you hold out your hand and say give me the toy', 'subdomain': 'receptive_language', 'bridge_step_title': 'Follows a simple direction with physical help.', 'bridge_step_description': 'Follows a simple direction with physical help.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'follows directions given with both a gesture and words for example she gives you a toy when you hold out your hand and say give me the toy', 'subdomain': 'receptive_language', 'bridge_step_title': 'Follows directions given with both a gesture and words.', 'bridge_step_description': 'Follows directions given with both a gesture and words.'}, {'months': 15, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'points to ask for something or to get help', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Shows interest in a wanted object or help.', 'bridge_step_description': 'Shows interest in a wanted object or help.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'points to ask for something or to get help', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Reaches toward what she wants.', 'bridge_step_description': 'Reaches toward what she wants.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'points to ask for something or to get help', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Uses a finger or hand gesture or sounds and eyes to direct attention.', 'bridge_step_description': 'Uses a finger or hand gesture or sounds and eyes to direct attention.'}, {'months': 15, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'points to ask for something or to get help', 'subdomain': 'gestural_communication', 'bridge_step_title': 'points to ask for something or to get help', 'bridge_step_description': 'points to ask for something or to get help'}, {'months': 15, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'tries to use things the right way like a phone or a cup or a book', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Explores objects with hands, eyes, and mouth safely.', 'bridge_step_description': 'Explores objects with hands, eyes, and mouth safely.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'tries to use things the right way like a phone or a cup or a book', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Watches how adults use familiar objects.', 'bridge_step_description': 'Watches how adults use familiar objects.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'tries to use things the right way like a phone or a cup or a book', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Can hold items in her hands', 'bridge_step_description': 'Can hold items in her hands'}, {'months': 15, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'tries to use things the right way like a phone or a cup or a book', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Imitates simple actions with real objects.', 'bridge_step_description': 'Imitates simple actions with real objects.'}, {'months': 15, 'bridge_step_number': 5, 'category': 'cognitive', 'cdc_milestone': 'tries to use things the right way like a phone or a cup or a book', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Tries to use things the right way.', 'bridge_step_description': 'Tries to use things the right way.'}, {'months': 15, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'stacks at least two small objects like blocks', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Grasps and releases objects with control.', 'bridge_step_description': 'Grasps and releases objects with control.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'stacks at least two small objects like blocks', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Places one object onto a surface intentionally.', 'bridge_step_description': 'Places one object onto a surface intentionally.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'stacks at least two small objects like blocks', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Places one object on top of another with support.', 'bridge_step_description': 'Places one object on top of another with support.'}, {'months': 15, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'stacks at least two small objects like blocks', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'stacks at least two small objects like blocks', 'bridge_step_description': 'stacks at least two small objects like blocks'}, {'months': 15, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'takes a few steps on her own', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Stands while holding a stable surface.', 'bridge_step_description': 'Stands while holding a stable surface.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'takes a few steps on her own', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Stands briefly without support.', 'bridge_step_description': 'Stands briefly without support.'}, {'months': 15, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'takes a few steps on her own', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Cruises along furniture.', 'bridge_step_description': 'Cruises along furniture.'}, {'months': 15, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'takes a few steps on her own', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Walks holding on to furniture.', 'bridge_step_description': 'Walks holding on to furniture.'}, {'months': 15, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'takes a few steps on her own', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Take a few steps on her own.', 'bridge_step_description': 'Take a few steps on her own.'}, {'months': 15, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'uses fingers to feed herself some food', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Reaches toward safe food pieces.', 'bridge_step_description': 'Reaches toward safe food pieces.'}, {'months': 15, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'uses fingers to feed herself some food', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Use fist to get some food', 'bridge_step_description': 'Use fist to get some food'}, {'months': 15, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'uses fingers to feed herself some food', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'uses fingers to feed herself some food', 'bridge_step_description': 'uses fingers to feed herself some food'}, {'months': 18, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'moves away from you but looks to make sure you are close by', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'Stays calm while exploring near a familiar adult.', 'bridge_step_description': 'Stays calm while exploring near a familiar adult.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'moves away from you but looks to make sure you are close by', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'Moves a short distance away during play.', 'bridge_step_description': 'Moves a short distance away during play.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'moves away from you but looks to make sure you are close by', 'subdomain': 'attachment_and_separation', 'bridge_step_title': 'moves away from you but looks to make sure you are close by', 'bridge_step_description': 'moves away from you but looks to make sure you are close by'}, {'months': 18, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'points to show you something interesting', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Notices interesting objects or events.', 'bridge_step_description': 'Notices interesting objects or events.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'points to show you something interesting', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Reaches toward or touches the object.', 'bridge_step_description': 'Reaches toward or touches the object.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'points to show you something interesting', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Points to show you something interesting.', 'bridge_step_description': 'Points to show you something interesting.'}, {'months': 18, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'puts hands out for you to wash them', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Tolerates handwashing with help.', 'bridge_step_description': 'Tolerates handwashing with help.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'puts hands out for you to wash them', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Responds to a simple cue during the routine.', 'bridge_step_description': 'Responds to a simple cue during the routine.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'puts hands out for you to wash them', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Puts hands out for you to wash them.', 'bridge_step_description': 'Puts hands out for you to wash them.'}, {'months': 18, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'looks at a few pages in a book with you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Sits briefly with an adult for a shared activity.', 'bridge_step_description': 'Sits briefly with an adult for a shared activity.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'looks at a few pages in a book with you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Looks at a book or pictures for a short time.', 'bridge_step_description': 'Looks at a book or pictures for a short time.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'looks at a few pages in a book with you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'looks at a few pages in a book with you', 'bridge_step_description': 'looks at a few pages in a book with you'}, {'months': 18, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'helps you dress him by pushing the arm through the sleeve or lifting up the foot', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Tolerates dressing routines.', 'bridge_step_description': 'Tolerates dressing routines.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'helps you dress him by pushing the arm through the sleeve or lifting up the foot', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Moves an arm or leg during dressing with help.', 'bridge_step_description': 'Moves an arm or leg during dressing with help.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'helps you dress him by pushing the arm through the sleeve or lifting up the foot', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Pushes a body part forward when prompted.', 'bridge_step_description': 'Pushes a body part forward when prompted.'}, {'months': 18, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'helps you dress him by pushing the arm through the sleeve or lifting up the foot', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Helps you dress him by pushing the arm through the sleeve or lifting up a foot.', 'bridge_step_description': 'Helps you dress him by pushing the arm through the sleeve or lifting up a foot.'}, {'months': 18, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'tries to say three or more words besides mama or dada', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses sounds or vocalizations to communicate.', 'bridge_step_description': 'Uses sounds or vocalizations to communicate.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'tries to say three or more words besides mama or dada', 'subdomain': 'expressive_language', 'bridge_step_title': 'imitate simple sounds.', 'bridge_step_description': 'imitate simple sounds.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'tries to say three or more words besides mama or dada', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses one meaningful word consistently.', 'bridge_step_description': 'Uses one meaningful word consistently.'}, {'months': 18, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'tries to say three or more words besides mama or dada', 'subdomain': 'expressive_language', 'bridge_step_title': 'Tries to say familiar words like mama or dada.', 'bridge_step_description': 'Tries to say familiar words like mama or dada.'}, {'months': 18, 'bridge_step_number': 5, 'category': 'language and communication', 'cdc_milestone': 'tries to say three or more words besides mama or dada', 'subdomain': 'expressive_language', 'bridge_step_title': 'Tries to say three or more words besides “mama” or “dada.”', 'bridge_step_description': 'Tries to say three or more words besides “mama” or “dada.”'}, {'months': 18, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'follows one step directions without any gestures like giving you the toy when you say give it to me', 'subdomain': 'receptive_language', 'bridge_step_title': 'Understands and responds to familiar words and name.', 'bridge_step_description': 'Understands and responds to familiar words and name.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'follows one step directions without any gestures like giving you the toy when you say give it to me', 'subdomain': 'receptive_language', 'bridge_step_title': 'Follows a one-step direction with a gesture cue or help.', 'bridge_step_description': 'Follows a one-step direction with a gesture cue or help.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'follows one step directions without any gestures like giving you the toy when you say give it to me', 'subdomain': 'receptive_language', 'bridge_step_title': 'Follows one-step directions without any gestures.', 'bridge_step_description': 'Follows one-step directions without any gestures.'}, {'months': 18, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'copies you doing chores like sweeping with a broom', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Watches an adult perform simple actions.', 'bridge_step_description': 'Watches an adult perform simple actions.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'copies you doing chores like sweeping with a broom', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Copies simple play actions with objects when adult showing', 'bridge_step_description': 'Copies simple play actions with objects when adult showing'}, {'months': 18, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'copies you doing chores like sweeping with a broom', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Uses some familiar objects in a functional way.', 'bridge_step_description': 'Uses some familiar objects in a functional way.'}, {'months': 18, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'copies you doing chores like sweeping with a broom', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Copies you doing chores, like sweeping with a broom.', 'bridge_step_description': 'Copies you doing chores, like sweeping with a broom.'}, {'months': 18, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'plays with toys in a simple way like pushing a toy car', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Reaches for and explores toys.', 'bridge_step_description': 'Reaches for and explores toys.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'plays with toys in a simple way like pushing a toy car', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Holds and manipulates a toy with purpose.', 'bridge_step_description': 'Holds and manipulates a toy with purpose.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'plays with toys in a simple way like pushing a toy car', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Watches and copy an adult playing with a toy.', 'bridge_step_description': 'Watches and copy an adult playing with a toy.'}, {'months': 18, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'plays with toys in a simple way like pushing a toy car', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Plays with toys in a simple way, like pushing a toy car.', 'bridge_step_description': 'Plays with toys in a simple way, like pushing a toy car.'}, {'months': 18, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'walks without holding on to anyone or anything', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Pulls to stand and stands with support.', 'bridge_step_description': 'Pulls to stand and stands with support.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'walks without holding on to anyone or anything', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Cruises along furniture.', 'bridge_step_description': 'Cruises along furniture.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'walks without holding on to anyone or anything', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Stands briefly without support.', 'bridge_step_description': 'Stands briefly without support.'}, {'months': 18, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'walks without holding on to anyone or anything', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Takes a few independent steps.', 'bridge_step_description': 'Takes a few independent steps.'}, {'months': 18, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'walks without holding on to anyone or anything', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Walks without holding on to anyone or anything.', 'bridge_step_description': 'Walks without holding on to anyone or anything.'}, {'months': 18, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'scribbles', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Grasps a crayon or marker.', 'bridge_step_description': 'Grasps a crayon or marker.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'scribbles', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Makes accidental marks on paper.', 'bridge_step_description': 'Makes accidental marks on paper.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'scribbles', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'scribbles', 'bridge_step_description': 'scribbles'}, {'months': 18, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'drinks from a cup without a lid and may spill sometimes', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Holds a bottle or handled cup during drinking.', 'bridge_step_description': 'Holds a bottle or handled cup during drinking.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'drinks from a cup without a lid and may spill sometimes', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Brings an open cup to the mouth with some control and assistance.', 'bridge_step_description': 'Brings an open cup to the mouth with some control and assistance.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'drinks from a cup without a lid and may spill sometimes', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Drinks from a cup without a lid and may spill sometimes.', 'bridge_step_description': 'Drinks from a cup without a lid and may spill sometimes.'}, {'months': 18, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'feeds himself with his fingers', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Reaches for food pieces', 'bridge_step_description': 'Reaches for food pieces'}, {'months': 18, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'feeds himself with his fingers', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Pick up food mostly with fingers', 'bridge_step_description': 'Pick up food mostly with fingers'}, {'months': 18, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'feeds himself with his fingers', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'feeds himself with his fingers', 'bridge_step_description': 'feeds himself with his fingers'}, {'months': 18, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'tries to use a spoon', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'feeds himself with his hands', 'bridge_step_description': 'feeds himself with his hands'}, {'months': 18, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'tries to use a spoon', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Holds a spoon during meals', 'bridge_step_description': 'Holds a spoon during meals'}, {'months': 18, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'tries to use a spoon', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Scoop or dips food or other small objects from one container to another', 'bridge_step_description': 'Scoop or dips food or other small objects from one container to another'}, {'months': 18, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'tries to use a spoon', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Tries to use a spoon.', 'bridge_step_description': 'Tries to use a spoon.'}, {'months': 18, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'climbs on and off a couch or chair without help', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Pulls to stand on stable surfaces.', 'bridge_step_description': 'Pulls to stand on stable surfaces.'}, {'months': 18, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'climbs on and off a couch or chair without help', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Shifts weight to climb onto a low surface.', 'bridge_step_description': 'Shifts weight to climb onto a low surface.'}, {'months': 18, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'climbs on and off a couch or chair without help', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Climbs on and off a couch or chair without help.', 'bridge_step_description': 'Climbs on and off a couch or chair without help.'}, {'months': 24, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'notices when others are hurt or upset like pausing or looking sad when someone is crying', 'subdomain': 'empathy_and_prosocial_behavior', 'bridge_step_title': 'Looks toward someone who is crying or hurt.', 'bridge_step_description': 'Looks toward someone who is crying or hurt.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'notices when others are hurt or upset like pausing or looking sad when someone is crying', 'subdomain': 'empathy_and_prosocial_behavior', 'bridge_step_title': 'Pauses activities when someone shows distress or crying.', 'bridge_step_description': 'Pauses activities when someone shows distress or crying.'}, {'months': 24, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'looks at your face to see how to react in a new situation', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Looks toward familiar adult during play or routines.', 'bridge_step_description': 'Looks toward familiar adult during play or routines.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'looks at your face to see how to react in a new situation', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Shares attention between an object and an adult', 'bridge_step_description': 'Shares attention between an object and an adult'}, {'months': 24, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'looks at your face to see how to react in a new situation', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Checks an adults face when unsure', 'bridge_step_description': 'Checks an adults face when unsure'}, {'months': 24, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'looks at your face to see how to react in a new situation', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'looks at your face to see how to react in a new situation', 'bridge_step_description': 'looks at your face to see how to react in a new situation'}, {'months': 24, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'points to things in a book when you ask like where is the bear', 'subdomain': 'receptive_language', 'bridge_step_title': 'Looks at pictures in a book with attention.', 'bridge_step_description': 'Looks at pictures in a book with attention.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'points to things in a book when you ask like where is the bear', 'subdomain': 'receptive_language', 'bridge_step_title': 'Recognizes familiar objects or animals in pictures.', 'bridge_step_description': 'Recognizes familiar objects or animals in pictures.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'points to things in a book when you ask like where is the bear', 'subdomain': 'receptive_language', 'bridge_step_title': 'Points to a named picture with support.', 'bridge_step_description': 'Points to a named picture with support.'}, {'months': 24, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'points to things in a book when you ask like where is the bear', 'subdomain': 'receptive_language', 'bridge_step_title': 'Points to things in a book when you ask.', 'bridge_step_description': 'Points to things in a book when you ask.'}, {'months': 24, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'says at least two words together like more milk', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses at least 50 words for people, objects, actions, or needs.', 'bridge_step_description': 'Uses at least 50 words for people, objects, actions, or needs.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'says at least two words together like more milk', 'subdomain': 'expressive_language', 'bridge_step_title': 'Combines a gesture with a word.', 'bridge_step_description': 'Combines a gesture with a word.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'says at least two words together like more milk', 'subdomain': 'expressive_language', 'bridge_step_title': 'Imitates simple two-word phrases.', 'bridge_step_description': 'Imitates simple two-word phrases.'}, {'months': 24, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'says at least two words together like more milk', 'subdomain': 'expressive_language', 'bridge_step_title': 'says at least two words together like more milk', 'bridge_step_description': 'says at least two words together like more milk'}, {'months': 24, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'points at least two body parts when you ask her to show you', 'subdomain': 'receptive_language', 'bridge_step_title': 'Attends when a body part is named.', 'bridge_step_description': 'Attends when a body part is named.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'points at least two body parts when you ask her to show you', 'subdomain': 'receptive_language', 'bridge_step_title': 'Points to one body part with support or copy when shown.', 'bridge_step_description': 'Points to one body part with support or copy when shown.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'points at least two body parts when you ask her to show you', 'subdomain': 'receptive_language', 'bridge_step_title': 'Points to at least two body parts when asked.', 'bridge_step_description': 'Points to at least two body parts when asked.'}, {'months': 24, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'uses more gestures than just waving or pointing like blowing a kiss or nodding yes', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Uses eye contact, reaching, or body movement to communicate.', 'bridge_step_description': 'Uses eye contact, reaching, or body movement to communicate.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'uses more gestures than just waving or pointing like blowing a kiss or nodding yes', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Uses basic gestures like waving or pointing.', 'bridge_step_description': 'Uses basic gestures like waving or pointing.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'uses more gestures than just waving or pointing like blowing a kiss or nodding yes', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Copies simple gestures from others.', 'bridge_step_description': 'Copies simple gestures from others.'}, {'months': 24, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'uses more gestures than just waving or pointing like blowing a kiss or nodding yes', 'subdomain': 'gestural_communication', 'bridge_step_title': 'Uses more gestures than just waving and pointing.', 'bridge_step_description': 'Uses more gestures than just waving and pointing.'}, {'months': 24, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'holds something in one hand while using the other hand for example holding a container and taking the lid off', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Uses both hands together during play.', 'bridge_step_description': 'Uses both hands together during play.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'holds something in one hand while using the other hand for example holding a container and taking the lid off', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Transfers objects from one hand to the other.', 'bridge_step_description': 'Transfers objects from one hand to the other.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'holds something in one hand while using the other hand for example holding a container and taking the lid off', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Holds something in one hand while using the other hand.', 'bridge_step_description': 'Holds something in one hand while using the other hand.'}, {'months': 24, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'tries to use switches or knobs or buttons on a toy', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Explores toys with hands and fingers.', 'bridge_step_description': 'Explores toys with hands and fingers.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'tries to use switches or knobs or buttons on a toy', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Pushes large buttons or surfaces.', 'bridge_step_description': 'Pushes large buttons or surfaces.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'tries to use switches or knobs or buttons on a toy', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Uses finger strength to press or pull parts of a toy.', 'bridge_step_description': 'Uses finger strength to press or pull parts of a toy.'}, {'months': 24, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'tries to use switches or knobs or buttons on a toy', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'Turns or moves simple toy parts with purpose.', 'bridge_step_description': 'Turns or moves simple toy parts with purpose.'}, {'months': 24, 'bridge_step_number': 5, 'category': 'cognitive', 'cdc_milestone': 'tries to use switches or knobs or buttons on a toy', 'subdomain': 'exploration_and_object_use', 'bridge_step_title': 'tries to use switches or knobs or buttons on a toy', 'bridge_step_description': 'tries to use switches or knobs or buttons on a toy'}, {'months': 24, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'plays with more than one toy at the same time like putting toy food on a toy plate', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Uses a toy in a simple functional way.', 'bridge_step_description': 'Uses a toy in a simple functional way.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'plays with more than one toy at the same time like putting toy food on a toy plate', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Copies simple play actions.', 'bridge_step_description': 'Copies simple play actions.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'plays with more than one toy at the same time like putting toy food on a toy plate', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Combines two related toys in play with help.', 'bridge_step_description': 'Combines two related toys in play with help.'}, {'months': 24, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'plays with more than one toy at the same time like putting toy food on a toy plate', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Plays with more than one toy at the same time.', 'bridge_step_description': 'Plays with more than one toy at the same time.'}, {'months': 24, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'kicks a ball', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Stands with balance while both feet are on the ground.', 'bridge_step_description': 'Stands with balance while both feet are on the ground.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'kicks a ball', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Lifts one foot briefly without falling.', 'bridge_step_description': 'Lifts one foot briefly without falling.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'kicks a ball', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Kicks a ball.', 'bridge_step_description': 'Kicks a ball.'}, {'months': 24, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'runs', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Walks steadily without frequent falling.', 'bridge_step_description': 'Walks steadily without frequent falling.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'runs', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Changes speed without loosing control.', 'bridge_step_description': 'Changes speed without loosing control.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'runs', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Uses coordinated arm and leg movement.', 'bridge_step_description': 'Uses coordinated arm and leg movement.'}, {'months': 24, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'runs', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Runs', 'bridge_step_description': 'Runs'}, {'months': 24, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'walks not climbs up a few stairs wih or without help', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Stands and walks with steady balance.', 'bridge_step_description': 'Stands and walks with steady balance.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'walks not climbs up a few stairs wih or without help', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Steps over small objects with support.', 'bridge_step_description': 'Steps over small objects with support.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'walks not climbs up a few stairs wih or without help', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Walks up a few stairs with or without help.', 'bridge_step_description': 'Walks up a few stairs with or without help.'}, {'months': 24, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'eats with a spoon', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Feed herself with hands.', 'bridge_step_description': 'Feed herself with hands.'}, {'months': 24, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'eats with a spoon', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Holds a spoon with the whole hand.', 'bridge_step_description': 'Holds a spoon with the whole hand.'}, {'months': 24, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'eats with a spoon', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Scoops food or other small items from one container to another.', 'bridge_step_description': 'Scoops food or other small items from one container to another.'}, {'months': 24, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'eats with a spoon', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Keep food on spoon and bring to mouth with help.', 'bridge_step_description': 'Keep food on spoon and bring to mouth with help.'}, {'months': 24, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'eats with a spoon', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'eats with a spoon', 'bridge_step_description': 'eats with a spoon'}, {'months': 30, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'plays next to other children and sometimes plays with them', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Plays near other children with separate toys.', 'bridge_step_description': 'Plays near other children with separate toys.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'plays next to other children and sometimes plays with them', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Share space without disrupting others play.', 'bridge_step_description': 'Share space without disrupting others play.'}, {'months': 30, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'plays next to other children and sometimes plays with them', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Plays next to other children and sometimes plays with them.', 'bridge_step_description': 'Plays next to other children and sometimes plays with them.'}, {'months': 30, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'shows you what she can do by saying look at me', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Uses a gesture or sound to get attention.', 'bridge_step_description': 'Uses a gesture or sound to get attention.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'shows you what she can do by saying look at me', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Uses words to get attention.', 'bridge_step_description': 'Uses words to get attention.'}, {'months': 30, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'shows you what she can do by saying look at me', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'shows you what she can do by saying look at me', 'bridge_step_description': 'shows you what she can do by saying look at me'}, {'months': 30, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'follows simple routines when told like helping to pick up toys when you say it is clean up time', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Completes one routine step with help.', 'bridge_step_description': 'Completes one routine step with help.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'follows simple routines when told like helping to pick up toys when you say it is clean up time', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Follows simple routines when told, like helping to pick up toys.', 'bridge_step_description': 'Follows simple routines when told, like helping to pick up toys.'}, {'months': 30, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'says about 50 words', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses a few meaningful words.', 'bridge_step_description': 'Uses a few meaningful words.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'says about 50 words', 'subdomain': 'expressive_language', 'bridge_step_title': 'Names familiar people or objects.', 'bridge_step_description': 'Names familiar people or objects.'}, {'months': 30, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'says about 50 words', 'subdomain': 'expressive_language', 'bridge_step_title': 'Learns and uses new words regularly.', 'bridge_step_description': 'Learns and uses new words regularly.'}, {'months': 30, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'says about 50 words', 'subdomain': 'expressive_language', 'bridge_step_title': 'says about 50 words', 'bridge_step_description': 'says about 50 words'}, {'months': 30, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'say two or more words toghether with one action word like doggie run', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses more than 50 single words to name objects or people.', 'bridge_step_description': 'Uses more than 50 single words to name objects or people.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'say two or more words toghether with one action word like doggie run', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses action words like go, eat, run, sleep.', 'bridge_step_description': 'Uses action words like go, eat, run, sleep.'}, {'months': 30, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'say two or more words toghether with one action word like doggie run', 'subdomain': 'expressive_language', 'bridge_step_title': 'Combines two words together.', 'bridge_step_description': 'Combines two words together.'}, {'months': 30, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'say two or more words toghether with one action word like doggie run', 'subdomain': 'expressive_language', 'bridge_step_title': 'Says two or more words together, with one action word.', 'bridge_step_description': 'Says two or more words together, with one action word.'}, {'months': 30, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'name things in a book when you point and ask what is this', 'subdomain': 'expressive_language', 'bridge_step_title': 'Point to familiar pictures.', 'bridge_step_description': 'Point to familiar pictures.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'name things in a book when you point and ask what is this', 'subdomain': 'expressive_language', 'bridge_step_title': 'Match pictures to real objects.', 'bridge_step_description': 'Match pictures to real objects.'}, {'months': 30, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'name things in a book when you point and ask what is this', 'subdomain': 'expressive_language', 'bridge_step_title': 'name things in a book when you point and ask what is this', 'bridge_step_description': 'name things in a book when you point and ask what is this'}, {'months': 30, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'says words like I me or we', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses own name to refer to self.', 'bridge_step_description': 'Uses own name to refer to self.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'says words like I me or we', 'subdomain': 'expressive_language', 'bridge_step_title': 'Imitates pronouns in short phrases.', 'bridge_step_description': 'Imitates pronouns in short phrases.'}, {'months': 30, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'says words like I me or we', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses me or I in simple speech', 'bridge_step_description': 'Uses me or I in simple speech'}, {'months': 30, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'says words like I me or we', 'subdomain': 'expressive_language', 'bridge_step_title': 'says words like I me or we', 'bridge_step_description': 'says words like I me or we'}, {'months': 30, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'uses things to pretend like feeding a block to a doll as if it were food', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'Copies simple actions with real objects.', 'bridge_step_description': 'Copies simple actions with real objects.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'uses things to pretend like feeding a block to a doll as if it were food', 'subdomain': 'imitation_and_play_skills', 'bridge_step_title': 'uses things to pretend like feeding a block to a doll as if it were food', 'bridge_step_description': 'uses things to pretend like feeding a block to a doll as if it were food'}, {'months': 30, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'shows simple problem solving skills like standing on a small stool to reach something', 'subdomain': 'object_permanence_and_problem_solving', 'bridge_step_title': 'Uses an adult’s help to solve a simple problem.', 'bridge_step_description': 'Uses an adult’s help to solve a simple problem.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'shows simple problem solving skills like standing on a small stool to reach something', 'subdomain': 'object_permanence_and_problem_solving', 'bridge_step_title': 'Moves around barriers to reach something', 'bridge_step_description': 'Moves around barriers to reach something'}, {'months': 30, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'shows simple problem solving skills like standing on a small stool to reach something', 'subdomain': 'object_permanence_and_problem_solving', 'bridge_step_title': 'shows simple problem solving skills like standing on a small stool to reach something', 'bridge_step_description': 'shows simple problem solving skills like standing on a small stool to reach something'}, {'months': 30, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'follows two step instructions like put the toy down and close the door', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Follows one simple instruction with gesture', 'bridge_step_description': 'Follows one simple instruction with gesture'}, {'months': 30, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'follows two step instructions like put the toy down and close the door', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Follows one instruction without gesture', 'bridge_step_description': 'Follows one instruction without gesture'}, {'months': 30, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'follows two step instructions like put the toy down and close the door', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Completes two familiar actions with support', 'bridge_step_description': 'Completes two familiar actions with support'}, {'months': 30, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'follows two step instructions like put the toy down and close the door', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Completes two familiar actions with reminder', 'bridge_step_description': 'Completes two familiar actions with reminder'}, {'months': 30, 'bridge_step_number': 5, 'category': 'cognitive', 'cdc_milestone': 'follows two step instructions like put the toy down and close the door', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'follows two step instructions like put the toy down and close the door', 'bridge_step_description': 'follows two step instructions like put the toy down and close the door'}, {'months': 30, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'shows she knows at least one color like pointing to a red crayon when you ask which one is red', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Matches two items by color', 'bridge_step_description': 'Matches two items by color'}, {'months': 30, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'shows she knows at least one color like pointing to a red crayon when you ask which one is red', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Sorts items by color with help', 'bridge_step_description': 'Sorts items by color with help'}, {'months': 30, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'shows she knows at least one color like pointing to a red crayon when you ask which one is red', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'shows she knows at least one color like pointing to a red crayon when you ask which one is red', 'bridge_step_description': 'shows she knows at least one color like pointing to a red crayon when you ask which one is red'}, {'months': 30, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'uses hands to twist things like turning doorknobs or unscrewing lids', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Hold objects with both hands.', 'bridge_step_description': 'Hold objects with both hands.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'uses hands to twist things like turning doorknobs or unscrewing lids', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Uses both hands for simple actions.', 'bridge_step_description': 'Uses both hands for simple actions.'}, {'months': 30, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'uses hands to twist things like turning doorknobs or unscrewing lids', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Turns large knobs or round objects with help.', 'bridge_step_description': 'Turns large knobs or round objects with help.'}, {'months': 30, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'uses hands to twist things like turning doorknobs or unscrewing lids', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'uses hands to twist things like turning doorknobs or unscrewing lids', 'bridge_step_description': 'uses hands to twist things like turning doorknobs or unscrewing lids'}, {'months': 30, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'takes some clothes off by himself like loose pants or an open jacket', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Help when clothing by pushing and pulling arms and legs.', 'bridge_step_description': 'Help when clothing by pushing and pulling arms and legs.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'takes some clothes off by himself like loose pants or an open jacket', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Pushes loose pants down', 'bridge_step_description': 'Pushes loose pants down'}, {'months': 30, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'takes some clothes off by himself like loose pants or an open jacket', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'takes some clothes off by himself like loose pants or an open jacket', 'bridge_step_description': 'takes some clothes off by himself like loose pants or an open jacket'}, {'months': 30, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'jumps off the ground with both feet', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Walks and runs with steady balance.', 'bridge_step_description': 'Walks and runs with steady balance.'}, {'months': 30, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'jumps off the ground with both feet', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Walks up and down low steps with support.', 'bridge_step_description': 'Walks up and down low steps with support.'}, {'months': 30, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'jumps off the ground with both feet', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Bends knees and squats without falling.', 'bridge_step_description': 'Bends knees and squats without falling.'}, {'months': 30, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'jumps off the ground with both feet', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Rises onto toes and pushes through both feet.', 'bridge_step_description': 'Rises onto toes and pushes through both feet.'}, {'months': 30, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'jumps off the ground with both feet', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Makes small two-footed bounces with support.', 'bridge_step_description': 'Makes small two-footed bounces with support.'}, {'months': 30, 'bridge_step_number': 6, 'category': 'movement and physical', 'cdc_milestone': 'jumps off the ground with both feet', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Jumps off the ground with both feet.', 'bridge_step_description': 'Jumps off the ground with both feet.'}, {'months': 30, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'turns book pages one at a time when you read to her', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Holds or explores a book', 'bridge_step_description': 'Holds or explores a book'}, {'months': 30, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'turns book pages one at a time when you read to her', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Turn several pages at once', 'bridge_step_description': 'Turn several pages at once'}, {'months': 30, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'turns book pages one at a time when you read to her', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Can pick up small items by two fingers', 'bridge_step_description': 'Can pick up small items by two fingers'}, {'months': 30, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'turns book pages one at a time when you read to her', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'turns book pages one at a time when you read to her', 'bridge_step_description': 'turns book pages one at a time when you read to her'}, {'months': 36, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'calms down within 10 minutes after you leave her like at childcare drop off', 'subdomain': 'emotional_regulation', 'bridge_step_title': 'Separate briefly from a similar adult with support', 'bridge_step_description': 'Separate briefly from a similar adult with support'}, {'months': 36, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'calms down within 10 minutes after you leave her like at childcare drop off', 'subdomain': 'emotional_regulation', 'bridge_step_title': 'Accepts comfort from another trusted adult', 'bridge_step_description': 'Accepts comfort from another trusted adult'}, {'months': 36, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'calms down within 10 minutes after you leave her like at childcare drop off', 'subdomain': 'emotional_regulation', 'bridge_step_title': 'Uses familiar comfort items or routine.', 'bridge_step_description': 'Uses familiar comfort items or routine.'}, {'months': 36, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'calms down within 10 minutes after you leave her like at childcare drop off', 'subdomain': 'emotional_regulation', 'bridge_step_title': 'calms down within 10 minutes after you leave her like at childcare drop off', 'bridge_step_description': 'calms down within 10 minutes after you leave her like at childcare drop off'}, {'months': 36, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'notices other children and joins them to play', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Plays near other children.', 'bridge_step_description': 'Plays near other children.'}, {'months': 36, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'notices other children and joins them to play', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Shows interest in what other children are doing.', 'bridge_step_description': 'Shows interest in what other children are doing.'}, {'months': 36, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'notices other children and joins them to play', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Can play with other children or near them with adult invitation and supervision.', 'bridge_step_description': 'Can play with other children or near them with adult invitation and supervision.'}, {'months': 36, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'notices other children and joins them to play', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Notices other children and joins them to play.', 'bridge_step_description': 'Notices other children and joins them to play.'}, {'months': 36, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'talks with you in conversation using at least two back and forth exchangess', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Respond to simple questions', 'bridge_step_description': 'Respond to simple questions'}, {'months': 36, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'talks with you in conversation using at least two back and forth exchangess', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Add a related word or phrase when someone talks to her.', 'bridge_step_description': 'Add a related word or phrase when someone talks to her.'}, {'months': 36, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'talks with you in conversation using at least two back and forth exchangess', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Uses two back and forth exchange with support and encouragement.', 'bridge_step_description': 'Uses two back and forth exchange with support and encouragement.'}, {'months': 36, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'talks with you in conversation using at least two back and forth exchangess', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'talks with you in conversation using at least two back and forth exchangess', 'bridge_step_description': 'talks with you in conversation using at least two back and forth exchangess'}, {'months': 36, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'ask who or what or where or why questions like where is mommy or where is daddy', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses rising tone or gesture to ask for help.', 'bridge_step_description': 'Uses rising tone or gesture to ask for help.'}, {'months': 36, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'ask who or what or where or why questions like where is mommy or where is daddy', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses at least 50 words.', 'bridge_step_description': 'Uses at least 50 words.'}, {'months': 36, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'ask who or what or where or why questions like where is mommy or where is daddy', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses simple request words like give, please, come', 'bridge_step_description': 'Uses simple request words like give, please, come'}, {'months': 36, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'ask who or what or where or why questions like where is mommy or where is daddy', 'subdomain': 'expressive_language', 'bridge_step_title': 'Answers to simple questions with what, where, who', 'bridge_step_description': 'Answers to simple questions with what, where, who'}, {'months': 36, 'bridge_step_number': 5, 'category': 'language and communication', 'cdc_milestone': 'ask who or what or where or why questions like where is mommy or where is daddy', 'subdomain': 'expressive_language', 'bridge_step_title': 'ask who or what or where or why questions like where is mommy or where is daddy', 'bridge_step_description': 'ask who or what or where or why questions like where is mommy or where is daddy'}, {'months': 36, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'says what action is happening in a picture or book when asked like running or eating or playing', 'subdomain': 'expressive_language', 'bridge_step_title': 'Looks at picture in a book with an adult', 'bridge_step_description': 'Looks at picture in a book with an adult'}, {'months': 36, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'says what action is happening in a picture or book when asked like running or eating or playing', 'subdomain': 'expressive_language', 'bridge_step_title': 'Points to people or objects in the picture book when asked', 'bridge_step_description': 'Points to people or objects in the picture book when asked'}, {'months': 36, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'says what action is happening in a picture or book when asked like running or eating or playing', 'subdomain': 'expressive_language', 'bridge_step_title': 'Name familiar items in real life', 'bridge_step_description': 'Name familiar items in real life'}, {'months': 36, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'says what action is happening in a picture or book when asked like running or eating or playing', 'subdomain': 'expressive_language', 'bridge_step_title': 'Name familiar actions in real life like run, go, eat', 'bridge_step_description': 'Name familiar actions in real life like run, go, eat'}, {'months': 36, 'bridge_step_number': 5, 'category': 'language and communication', 'cdc_milestone': 'says what action is happening in a picture or book when asked like running or eating or playing', 'subdomain': 'expressive_language', 'bridge_step_title': 'says what action is happening in a picture or book when asked like running or eating or playing', 'bridge_step_description': 'says what action is happening in a picture or book when asked like running or eating or playing'}, {'months': 36, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'says first name when asked', 'subdomain': 'expressive_language', 'bridge_step_title': 'Respondsto own name', 'bridge_step_description': 'Respondsto own name'}, {'months': 36, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'says first name when asked', 'subdomain': 'expressive_language', 'bridge_step_title': 'Points to self when named', 'bridge_step_description': 'Points to self when named'}, {'months': 36, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'says first name when asked', 'subdomain': 'expressive_language', 'bridge_step_title': 'Imitate own name after an adult', 'bridge_step_description': 'Imitate own name after an adult'}, {'months': 36, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'says first name when asked', 'subdomain': 'expressive_language', 'bridge_step_title': 'says first name when asked', 'bridge_step_description': 'says first name when asked'}, {'months': 36, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'talks well enough for others to understand most of the time', 'subdomain': 'speech_intelligibility', 'bridge_step_title': 'Uses sounds and words to communicate clearly to a familiar adult', 'bridge_step_description': 'Uses sounds and words to communicate clearly to a familiar adult'}, {'months': 36, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'talks well enough for others to understand most of the time', 'subdomain': 'speech_intelligibility', 'bridge_step_title': 'Says familiar words more clearly', 'bridge_step_description': 'Says familiar words more clearly'}, {'months': 36, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'talks well enough for others to understand most of the time', 'subdomain': 'speech_intelligibility', 'bridge_step_title': 'Uses short phrases with understandable words', 'bridge_step_description': 'Uses short phrases with understandable words'}, {'months': 36, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'talks well enough for others to understand most of the time', 'subdomain': 'speech_intelligibility', 'bridge_step_title': 'Speaks clearly enough for familiar people to understand often', 'bridge_step_description': 'Speaks clearly enough for familiar people to understand often'}, {'months': 36, 'bridge_step_number': 5, 'category': 'language and communication', 'cdc_milestone': 'talks well enough for others to understand most of the time', 'subdomain': 'speech_intelligibility', 'bridge_step_title': 'talks well enough for others to understand most of the time', 'bridge_step_description': 'talks well enough for others to understand most of the time'}, {'months': 36, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'draws a circle when you show her how', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Hold a crayon or marker with control and scribe', 'bridge_step_description': 'Hold a crayon or marker with control and scribe'}, {'months': 36, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'draws a circle when you show her how', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Makes lines and round scribbles', 'bridge_step_description': 'Makes lines and round scribbles'}, {'months': 36, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'draws a circle when you show her how', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Copies a circle with help and support', 'bridge_step_description': 'Copies a circle with help and support'}, {'months': 36, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'draws a circle when you show her how', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'draws a circle when you show her how', 'bridge_step_description': 'draws a circle when you show her how'}, {'months': 36, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'avoids touching hot objects like the stove when you warn her', 'subdomain': 'safety_awareness', 'bridge_step_title': 'Stops briefly when an adult says “stop.”', 'bridge_step_description': 'Stops briefly when an adult says “stop.”'}, {'months': 36, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'avoids touching hot objects like the stove when you warn her', 'subdomain': 'safety_awareness', 'bridge_step_title': 'recognize simple warning words like hot or no', 'bridge_step_description': 'recognize simple warning words like hot or no'}, {'months': 36, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'avoids touching hot objects like the stove when you warn her', 'subdomain': 'safety_awareness', 'bridge_step_title': 'avoids touching hot objects like the stove when you warn her', 'bridge_step_description': 'avoids touching hot objects like the stove when you warn her'}, {'months': 36, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'strings items together like large beads or macaroni', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Uses both hands for simple actions and plays', 'bridge_step_description': 'Uses both hands for simple actions and plays'}, {'months': 36, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'strings items together like large beads or macaroni', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Picks up small objects with fingers', 'bridge_step_description': 'Picks up small objects with fingers'}, {'months': 36, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'strings items together like large beads or macaroni', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Places objects onto a peg or stick', 'bridge_step_description': 'Places objects onto a peg or stick'}, {'months': 36, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'strings items together like large beads or macaroni', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Pushes a string through a large opening', 'bridge_step_description': 'Pushes a string through a large opening'}, {'months': 36, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'strings items together like large beads or macaroni', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Strings one or two large items with support', 'bridge_step_description': 'Strings one or two large items with support'}, {'months': 36, 'bridge_step_number': 6, 'category': 'movement and physical', 'cdc_milestone': 'strings items together like large beads or macaroni', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'strings items together like large beads or macaroni', 'bridge_step_description': 'strings items together like large beads or macaroni'}, {'months': 36, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'puts on some clothes by himself like loose pants or a jacket', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Helps during dressing by pushing arms or legs through.', 'bridge_step_description': 'Helps during dressing by pushing arms or legs through.'}, {'months': 36, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'puts on some clothes by himself like loose pants or a jacket', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Pulls clothing up or down with help.', 'bridge_step_description': 'Pulls clothing up or down with help.'}, {'months': 36, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'puts on some clothes by himself like loose pants or a jacket', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Puts on some cloth with support', 'bridge_step_description': 'Puts on some cloth with support'}, {'months': 36, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'puts on some clothes by himself like loose pants or a jacket', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'puts on some clothes by himself like loose pants or a jacket', 'bridge_step_description': 'puts on some clothes by himself like loose pants or a jacket'}, {'months': 36, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'uses a fork', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Can feed herself with hands', 'bridge_step_description': 'Can feed herself with hands'}, {'months': 36, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'uses a fork', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Holds a ford with assistance', 'bridge_step_description': 'Holds a ford with assistance'}, {'months': 36, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'uses a fork', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Stabs the food with fork and feed with help', 'bridge_step_description': 'Stabs the food with fork and feed with help'}, {'months': 36, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'uses a fork', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Uses a fork for some bites independently.', 'bridge_step_description': 'Uses a fork for some bites independently.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'pretends to be something else during play like teacher or superhero or a dog', 'subdomain': 'play_and_symbolic_social_play', 'bridge_step_title': 'Copies simple pretend actions from an adult.', 'bridge_step_description': 'Copies simple pretend actions from an adult.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'pretends to be something else during play like teacher or superhero or a dog', 'subdomain': 'play_and_symbolic_social_play', 'bridge_step_title': 'Plays a pretended game with an adult.', 'bridge_step_description': 'Plays a pretended game with an adult.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'pretends to be something else during play like teacher or superhero or a dog', 'subdomain': 'play_and_symbolic_social_play', 'bridge_step_title': 'Pretends to be something else during play, like a teacher or superhero.', 'bridge_step_description': 'Pretends to be something else during play, like a teacher or superhero.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'asks to go play with children if none are around like can I play with Alex', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Joins simple play when invited.', 'bridge_step_description': 'Joins simple play when invited.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'asks to go play with children if none are around like can I play with Alex', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Asks to join or joins the play when children are around.', 'bridge_step_description': 'Asks to join or joins the play when children are around.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'asks to go play with children if none are around like can I play with Alex', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Asks to go play with children if none are around.', 'bridge_step_description': 'Asks to go play with children if none are around.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'comforts others who are hurt or sad like hugging a crying friend', 'subdomain': 'empathy_and_prosocial_behavior', 'bridge_step_title': 'Names or recognizes simple feelings.', 'bridge_step_description': 'Names or recognizes simple feelings.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'comforts others who are hurt or sad like hugging a crying friend', 'subdomain': 'empathy_and_prosocial_behavior', 'bridge_step_title': 'Looks concerned or pauses when another person is upset.', 'bridge_step_description': 'Looks concerned or pauses when another person is upset.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'comforts others who are hurt or sad like hugging a crying friend', 'subdomain': 'empathy_and_prosocial_behavior', 'bridge_step_title': 'Offers help or comfort when prompted for example when someone is sad or hurt.', 'bridge_step_description': 'Offers help or comfort when prompted for example when someone is sad or hurt.'}, {'months': 48, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'comforts others who are hurt or sad like hugging a crying friend', 'subdomain': 'empathy_and_prosocial_behavior', 'bridge_step_title': 'Comforts others who are hurt or sad.', 'bridge_step_description': 'Comforts others who are hurt or sad.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'avoids danger like not jumping from tall heights at the playgroung', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Stops briefly when an adult says “stop.”', 'bridge_step_description': 'Stops briefly when an adult says “stop.”'}, {'months': 48, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'avoids danger like not jumping from tall heights at the playgroung', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Recognizes simple unsafe actions.', 'bridge_step_description': 'Recognizes simple unsafe actions.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'avoids danger like not jumping from tall heights at the playgroung', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Avoids a known danger with little or no reminder.', 'bridge_step_description': 'Avoids a known danger with little or no reminder.'}, {'months': 48, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'avoids danger like not jumping from tall heights at the playgroung', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Avoids danger, like not jumping from tall heights at the playground.', 'bridge_step_description': 'Avoids danger, like not jumping from tall heights at the playground.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'likes to be a helper', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Helps when directly asked in a small simple helping action.', 'bridge_step_description': 'Helps when directly asked in a small simple helping action.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'likes to be a helper', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Likes to be a helper.', 'bridge_step_description': 'Likes to be a helper.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'changes behavior based on where she is like place of worship or library or playground', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Follows one simple behavior rule with support.', 'bridge_step_description': 'Follows one simple behavior rule with support.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'changes behavior based on where she is like place of worship or library or playground', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Uses quieter or calmer behavior when reminded.', 'bridge_step_description': 'Uses quieter or calmer behavior when reminded.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'changes behavior based on where she is like place of worship or library or playground', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Changes behavior based on where she is, like a place of worship or library.', 'bridge_step_description': 'Changes behavior based on where she is, like a place of worship or library.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'says sentences with four or more words', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses more than 50 single words to communicate needs or ideas.', 'bridge_step_description': 'Uses more than 50 single words to communicate needs or ideas.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'says sentences with four or more words', 'subdomain': 'expressive_language', 'bridge_step_title': 'Combines two words together.', 'bridge_step_description': 'Combines two words together.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'says sentences with four or more words', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses short three-word phrases.', 'bridge_step_description': 'Uses short three-word phrases.'}, {'months': 48, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'says sentences with four or more words', 'subdomain': 'expressive_language', 'bridge_step_title': 'Adds describing or action words to sentences.', 'bridge_step_description': 'Adds describing or action words to sentences.'}, {'months': 48, 'bridge_step_number': 5, 'category': 'language and communication', 'cdc_milestone': 'says sentences with four or more words', 'subdomain': 'expressive_language', 'bridge_step_title': 'Says sentences with four or more words.', 'bridge_step_description': 'Says sentences with four or more words.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'says some words from a song or story or nursery rhyme', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Listens to familiar songs, stories, or rhymes.', 'bridge_step_description': 'Listens to familiar songs, stories, or rhymes.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'says some words from a song or story or nursery rhyme', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Joins in with sounds or gestures in familiar songs, stories, or rhymes.', 'bridge_step_description': 'Joins in with sounds or gestures in familiar songs, stories, or rhymes.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'says some words from a song or story or nursery rhyme', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Repeats short familiar phrases.', 'bridge_step_description': 'Repeats short familiar phrases.'}, {'months': 48, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'says some words from a song or story or nursery rhyme', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Says some words from a song, story, or nursery rhyme.', 'bridge_step_description': 'Says some words from a song, story, or nursery rhyme.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'talks about at least one thing that happended during her day like I played soccer', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Answers simple questions about what happened in a book or an event.', 'bridge_step_description': 'Answers simple questions about what happened in a book or an event.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'talks about at least one thing that happended during her day like I played soccer', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Uses action words to describe an event.', 'bridge_step_description': 'Uses action words to describe an event.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'talks about at least one thing that happended during her day like I played soccer', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Says one short sentence about a recent activity.', 'bridge_step_description': 'Says one short sentence about a recent activity.'}, {'months': 48, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'talks about at least one thing that happended during her day like I played soccer', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Talks about at least one thing that happened during her day.', 'bridge_step_description': 'Talks about at least one thing that happened during her day.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'answers simple questions like what is a coat for or what is crayon for', 'subdomain': 'expressive_language', 'bridge_step_title': 'Recognize and name common objects.', 'bridge_step_description': 'Recognize and name common objects.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'answers simple questions like what is a coat for or what is crayon for', 'subdomain': 'expressive_language', 'bridge_step_title': 'Shows how familiar objects are used.', 'bridge_step_description': 'Shows how familiar objects are used.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'answers simple questions like what is a coat for or what is crayon for', 'subdomain': 'expressive_language', 'bridge_step_title': 'Answers simple “what” questions about objects in books or real life.', 'bridge_step_description': 'Answers simple “what” questions about objects in books or real life.'}, {'months': 48, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'answers simple questions like what is a coat for or what is crayon for', 'subdomain': 'expressive_language', 'bridge_step_title': 'Answers simple questions like “What is a coat for?” or “What is a crayon for?”', 'bridge_step_description': 'Answers simple questions like “What is a coat for?” or “What is a crayon for?”'}, {'months': 48, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'names a few colors or items', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Matches items that are the same color.', 'bridge_step_description': 'Matches items that are the same color.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'names a few colors or items', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Sorts items by color.', 'bridge_step_description': 'Sorts items by color.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'names a few colors or items', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Name a few familiar colors', 'bridge_step_description': 'Name a few familiar colors'}, {'months': 48, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'tell what comes next in a well known story', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Listens to a familiar story.', 'bridge_step_description': 'Listens to a familiar story.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'tell what comes next in a well known story', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Names familiar characters or events.', 'bridge_step_description': 'Names familiar characters or events.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'tell what comes next in a well known story', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Answers simple questions about the story.', 'bridge_step_description': 'Answers simple questions about the story.'}, {'months': 48, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'tell what comes next in a well known story', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Tells what comes next in a well-known story.', 'bridge_step_description': 'Tells what comes next in a well-known story.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'draws a person with three or more body parts', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Copies simple lines and circles while holding the tools correctly.', 'bridge_step_description': 'Copies simple lines and circles while holding the tools correctly.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'draws a person with three or more body parts', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Draws simple objects like ball, car, tree.', 'bridge_step_description': 'Draws simple objects like ball, car, tree.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'draws a person with three or more body parts', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Draws a person with one or two body parts.', 'bridge_step_description': 'Draws a person with one or two body parts.'}, {'months': 48, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'draws a person with three or more body parts', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Draws a person with three or more body parts.', 'bridge_step_description': 'Draws a person with three or more body parts.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'catches a large ball most of the times', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Throughs or tosses a ball.', 'bridge_step_description': 'Throughs or tosses a ball.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'catches a large ball most of the times', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Reaches arms toward a slowly rolled or tossed ball.', 'bridge_step_description': 'Reaches arms toward a slowly rolled or tossed ball.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'catches a large ball most of the times', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Traps a large ball against the body.', 'bridge_step_description': 'Traps a large ball against the body.'}, {'months': 48, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'catches a large ball most of the times', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Catches a large ball with arms and body.', 'bridge_step_description': 'Catches a large ball with arms and body.'}, {'months': 48, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'catches a large ball most of the times', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Catches a large ball most of the time.', 'bridge_step_description': 'Catches a large ball most of the time.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'serves herself food or pours water with adult supervision', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Holds a cup, spoon, or serving tool with control.', 'bridge_step_description': 'Holds a cup, spoon, or serving tool with control.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'serves herself food or pours water with adult supervision', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Transfers food or water from one container to another.', 'bridge_step_description': 'Transfers food or water from one container to another.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'serves herself food or pours water with adult supervision', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Serves herself food or pours water with adult supervision.', 'bridge_step_description': 'Serves herself food or pours water with adult supervision.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'unbuttons some buttons', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Pulls apart large fasteners or simple closures.', 'bridge_step_description': 'Pulls apart large fasteners or simple closures.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'unbuttons some buttons', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Pushes a large button partly through a loose buttonhole.', 'bridge_step_description': 'Pushes a large button partly through a loose buttonhole.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'unbuttons some buttons', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Unbuttons some buttons.', 'bridge_step_description': 'Unbuttons some buttons.'}, {'months': 48, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'holds crayon or pencil between fingers and thumb not fist', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Holds a crayon or pencil in a full-hand grasp.', 'bridge_step_description': 'Holds a crayon or pencil in a full-hand grasp.'}, {'months': 48, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'holds crayon or pencil between fingers and thumb not fist', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Pick up small items with two fingers.', 'bridge_step_description': 'Pick up small items with two fingers.'}, {'months': 48, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'holds crayon or pencil between fingers and thumb not fist', 'subdomain': 'fine_motor_hand_use', 'bridge_step_title': 'Holds crayon or pencil between fingers and thumb, not fist.', 'bridge_step_description': 'Holds crayon or pencil between fingers and thumb, not fist.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'follows rules or takesake turns when playing games with other  children', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Plays simple back-and-forth games with an adult.', 'bridge_step_description': 'Plays simple back-and-forth games with an adult.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'follows rules or takesake turns when playing games with other  children', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Waits briefly for a turn with support.', 'bridge_step_description': 'Waits briefly for a turn with support.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'follows rules or takesake turns when playing games with other  children', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Follows one simple game rule.', 'bridge_step_description': 'Follows one simple game rule.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'social and emotial', 'cdc_milestone': 'follows rules or takesake turns when playing games with other  children', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Takes turns with another child during a simple game.', 'bridge_step_description': 'Takes turns with another child during a simple game.'}, {'months': 60, 'bridge_step_number': 5, 'category': 'social and emotial', 'cdc_milestone': 'follows rules or takesake turns when playing games with other  children', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Follows rules or takes turns when playing games with other children.', 'bridge_step_description': 'Follows rules or takes turns when playing games with other children.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'sings dances or acts for you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Joins in briefly during a familiar song or action.', 'bridge_step_description': 'Joins in briefly during a familiar song or action.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'sings dances or acts for you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Performs a short action when encouraged.', 'bridge_step_description': 'Performs a short action when encouraged.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'social and emotial', 'cdc_milestone': 'sings dances or acts for you', 'subdomain': 'social_engagement_and_joint_attention', 'bridge_step_title': 'Sings, dances, or acts for you.', 'bridge_step_description': 'Sings, dances, or acts for you.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'social and emotial', 'cdc_milestone': 'does simple chores at home like matching socks or cleaning the table after eating', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Completes a short chore with reminders.', 'bridge_step_description': 'Completes a short chore with reminders.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'social and emotial', 'cdc_milestone': 'does simple chores at home like matching socks or cleaning the table after eating', 'subdomain': 'peer_interaction_and_social_rules', 'bridge_step_title': 'Does simple chores at home, like matching socks or clearing the table.', 'bridge_step_description': 'Does simple chores at home, like matching socks or clearing the table.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'tells a storys she heard or made up with at least two events for example cat was stuck in a tree and firefighter saved it', 'subdomain': 'expressive_language', 'bridge_step_title': 'Describes one thing that happened.', 'bridge_step_description': 'Describes one thing that happened.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'tells a storys she heard or made up with at least two events for example cat was stuck in a tree and firefighter saved it', 'subdomain': 'expressive_language', 'bridge_step_title': 'Answers simple questions about what happened.', 'bridge_step_description': 'Answers simple questions about what happened.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'tells a storys she heard or made up with at least two events for example cat was stuck in a tree and firefighter saved it', 'subdomain': 'expressive_language', 'bridge_step_title': 'Tells two related ideas with support.', 'bridge_step_description': 'Tells two related ideas with support.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'tells a storys she heard or made up with at least two events for example cat was stuck in a tree and firefighter saved it', 'subdomain': 'expressive_language', 'bridge_step_title': 'Uses sequence words like “first,” “then,” or “next.”', 'bridge_step_description': 'Uses sequence words like “first,” “then,” or “next.”'}, {'months': 60, 'bridge_step_number': 5, 'category': 'language and communication', 'cdc_milestone': 'tells a storys she heard or made up with at least two events for example cat was stuck in a tree and firefighter saved it', 'subdomain': 'expressive_language', 'bridge_step_title': 'Tells a story she heard or made up with at least two events.', 'bridge_step_description': 'Tells a story she heard or made up with at least two events.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'answers simple questions about a book or a story after you read or tell it to her', 'subdomain': 'receptive_language', 'bridge_step_title': 'Answers simple “who” or “what” questions with support.', 'bridge_step_description': 'Answers simple “who” or “what” questions with support.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'answers simple questions about a book or a story after you read or tell it to her', 'subdomain': 'receptive_language', 'bridge_step_title': 'Answers simple questions about what happened.', 'bridge_step_description': 'Answers simple questions about what happened.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'answers simple questions about a book or a story after you read or tell it to her', 'subdomain': 'receptive_language', 'bridge_step_title': 'Answers simple questions about a book or story after you read or tell it.', 'bridge_step_description': 'Answers simple questions about a book or story after you read or tell it.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'keep a conversation going with more than three back and forth exchanges', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Takes one back-and-forth turn in conversation.', 'bridge_step_description': 'Takes one back-and-forth turn in conversation.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'keep a conversation going with more than three back and forth exchanges', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Takes two back-and-forth turns on the same topic.', 'bridge_step_description': 'Takes two back-and-forth turns on the same topic.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'keep a conversation going with more than three back and forth exchanges', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Adds a related comment or asks a simple question.', 'bridge_step_description': 'Adds a related comment or asks a simple question.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'keep a conversation going with more than three back and forth exchanges', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Keeps a conversation going with more than three back-and-forth exchanges.', 'bridge_step_description': 'Keeps a conversation going with more than three back-and-forth exchanges.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'language and communication', 'cdc_milestone': 'uses or recognizes simple rhymes like bat cat or ball tall', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Notices when two words sound similar.', 'bridge_step_description': 'Notices when two words sound similar.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'language and communication', 'cdc_milestone': 'uses or recognizes simple rhymes like bat cat or ball tall', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Chooses a matching rhyme from two options.', 'bridge_step_description': 'Chooses a matching rhyme from two options.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'language and communication', 'cdc_milestone': 'uses or recognizes simple rhymes like bat cat or ball tall', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Repeats simple rhyming word pairs.', 'bridge_step_description': 'Repeats simple rhyming word pairs.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'language and communication', 'cdc_milestone': 'uses or recognizes simple rhymes like bat cat or ball tall', 'subdomain': 'conversation_narrative', 'bridge_step_title': 'Uses or recognizes simple rhymes, like bat-cat or ball-tall.', 'bridge_step_description': 'Uses or recognizes simple rhymes, like bat-cat or ball-tall.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'counts to 10', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Says or recognizes number words.', 'bridge_step_description': 'Says or recognizes number words.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'counts to 10', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Counts to 3 in order.', 'bridge_step_description': 'Counts to 3 in order.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'counts to 10', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Counts to 5 in order.', 'bridge_step_description': 'Counts to 5 in order.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'counts to 10', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Counts objects one by one with support.', 'bridge_step_description': 'Counts objects one by one with support.'}, {'months': 60, 'bridge_step_number': 5, 'category': 'cognitive', 'cdc_milestone': 'counts to 10', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Counts to 10.', 'bridge_step_description': 'Counts to 10.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'names some numbers between 1 and 5 when you point to them', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Matches the same number to another same number.', 'bridge_step_description': 'Matches the same number to another same number.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'names some numbers between 1 and 5 when you point to them', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Names some numbers between 1 and 5 when you point to them.', 'bridge_step_description': 'Names some numbers between 1 and 5 when you point to them.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'uses words about time like yesterday or tomorrow or morning or night', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Understands simple daily routine words.', 'bridge_step_description': 'Understands simple daily routine words.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'uses words about time like yesterday or tomorrow or morning or night', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Uses basic time words like “now” or “later.”', 'bridge_step_description': 'Uses basic time words like “now” or “later.”'}, {'months': 60, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'uses words about time like yesterday or tomorrow or morning or night', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Understands parts of the day, like morning or night.', 'bridge_step_description': 'Understands parts of the day, like morning or night.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'uses words about time like yesterday or tomorrow or morning or night', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Uses past or future words with support.', 'bridge_step_description': 'Uses past or future words with support.'}, {'months': 60, 'bridge_step_number': 5, 'category': 'cognitive', 'cdc_milestone': 'uses words about time like yesterday or tomorrow or morning or night', 'subdomain': 'concepts_and_following_directions', 'bridge_step_title': 'Uses words about time, like yesterday, tomorrow, morning, or night.', 'bridge_step_description': 'Uses words about time, like yesterday, tomorrow, morning, or night.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'pays attention for 5 to 10 minutes during activities for example during story time or making arts and crafts', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Stays with an activity for 2–3 minutes with support.', 'bridge_step_description': 'Stays with an activity for 2–3 minutes with support.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'pays attention for 5 to 10 minutes during activities for example during story time or making arts and crafts', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Returns to the activity after a brief distraction.', 'bridge_step_description': 'Returns to the activity after a brief distraction.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'pays attention for 5 to 10 minutes during activities for example during story time or making arts and crafts', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Completes a short up to 5 minutes activity from start to finish.', 'bridge_step_description': 'Completes a short up to 5 minutes activity from start to finish.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'pays attention for 5 to 10 minutes during activities for example during story time or making arts and crafts', 'subdomain': 'attention_and_processing', 'bridge_step_title': 'Pays attention for 5 to 10 minutes during activities.', 'bridge_step_description': 'Pays attention for 5 to 10 minutes during activities.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'writes some letters in her name', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Copies simple lines and shapes with correct hold of writing tools.', 'bridge_step_description': 'Copies simple lines and shapes with correct hold of writing tools.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'writes some letters in her name', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Copies letter-like forms.', 'bridge_step_description': 'Copies letter-like forms.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'writes some letters in her name', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Traces letters from her name.', 'bridge_step_description': 'Traces letters from her name.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'cognitive', 'cdc_milestone': 'writes some letters in her name', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Writes some letters in her name.', 'bridge_step_description': 'Writes some letters in her name.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'cognitive', 'cdc_milestone': 'names some letters when you point to them', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Matches the same letter to another same letter.', 'bridge_step_description': 'Matches the same letter to another same letter.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'cognitive', 'cdc_milestone': 'names some letters when you point to them', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Names a familiar letter with a visual cue.', 'bridge_step_description': 'Names a familiar letter with a visual cue.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'cognitive', 'cdc_milestone': 'names some letters when you point to them', 'subdomain': 'pre_academic_skills', 'bridge_step_title': 'Names some letters when you point to them.', 'bridge_step_description': 'Names some letters when you point to them.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'buttons some buttons', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Pushes large objects through small openings.', 'bridge_step_description': 'Pushes large objects through small openings.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'buttons some buttons', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Pulls large beads or shapes through loops.', 'bridge_step_description': 'Pulls large beads or shapes through loops.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'buttons some buttons', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Pushes large buttons through loose buttonholes.', 'bridge_step_description': 'Pushes large buttons through loose buttonholes.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'buttons some buttons', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Buttons one or two medium-size buttons with help.', 'bridge_step_description': 'Buttons one or two medium-size buttons with help.'}, {'months': 60, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'buttons some buttons', 'subdomain': 'self_help_motor_skills', 'bridge_step_title': 'Buttons some buttons independently.', 'bridge_step_description': 'Buttons some buttons independently.'}, {'months': 60, 'bridge_step_number': 1, 'category': 'movement and physical', 'cdc_milestone': 'hops on one foot', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Stands briefly on one foot with support.', 'bridge_step_description': 'Stands briefly on one foot with support.'}, {'months': 60, 'bridge_step_number': 2, 'category': 'movement and physical', 'cdc_milestone': 'hops on one foot', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Stands on one foot without support for a few seconds.', 'bridge_step_description': 'Stands on one foot without support for a few seconds.'}, {'months': 60, 'bridge_step_number': 3, 'category': 'movement and physical', 'cdc_milestone': 'hops on one foot', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Lifts one foot and makes a small bounce while holding support.', 'bridge_step_description': 'Lifts one foot and makes a small bounce while holding support.'}, {'months': 60, 'bridge_step_number': 4, 'category': 'movement and physical', 'cdc_milestone': 'hops on one foot', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'Hops one time on one foot without support.', 'bridge_step_description': 'Hops one time on one foot without support.'}, {'months': 60, 'bridge_step_number': 5, 'category': 'movement and physical', 'cdc_milestone': 'hops on one foot', 'subdomain': 'gross_motor_mobility_and_coordination', 'bridge_step_title': 'hops on one foot', 'bridge_step_description': 'hops on one foot'}]


# Keep aliases to v16 functions for debugging or comparison if needed.
_v16_generate_category_activity_bank = generate_category_activity_bank
_v16_build_weekly_schedule = build_weekly_schedule
_v16_summarizer_agent = summarizer_agent
_v16_display_weekly_schedule = display_weekly_schedule
_v16_allocate_weekly_slots = allocate_weekly_slots


def _norm_text_key(x: Any) -> str:
    """Normalize text for robust milestone matching across spreadsheets."""
    s = str(x or "").strip().lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _category_to_key(category: Any) -> str:
    raw = str(category or "").strip().lower()
    if raw in ALIAS_TO_CATEGORY:
        return ALIAS_TO_CATEGORY[raw]
    raw = raw.replace("/", " ").replace("&", " and ")
    raw = re.sub(r"\s+", " ", raw).strip()
    if raw in ALIAS_TO_CATEGORY:
        return ALIAS_TO_CATEGORY[raw]
    # Handle common variants and typos from the CDC table.
    if "language" in raw or "communication" in raw:
        return "language_and_communication"
    if "social" in raw or "emotional" in raw or "emotial" in raw:
        return "social_and_emotional"
    if "movement" in raw or "physical" in raw or "motor" in raw:
        return "movement_and_physical"
    if "cognitive" in raw or "adaptive" in raw or "learning" in raw:
        return "cognitive_and_adaptive"
    return raw.replace(" ", "_")


def find_bridge_milestone_file(path: Optional[str] = None) -> Optional[Path]:
    """Find a bridge milestone reference spreadsheet.

    The final advisor-reviewed table can replace the draft file later as long as
    it has compatible columns.
    """
    if path:
        p = Path(path)
        if not p.exists():
            raise FileNotFoundError(f"Bridge milestone file not found: {p}")
        return p.resolve()

    raw_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path("/mnt/data")]
    search_roots = []
    for root in raw_roots:
        try:
            resolved = root.resolve()
        except Exception:
            resolved = root
        if str(resolved) == "/":
            continue
        if resolved not in search_roots:
            search_roots.append(resolved)

    for name in PREFERRED_BRIDGE_FILENAMES:
        for root in search_roots:
            candidate = root / name
            if candidate.exists():
                return candidate.resolve()

    candidates = []
    for root in search_roots:
        if root.exists():
            candidates.extend(root.rglob("cdc_milestones_with_bridges.numbers"))
            candidates.extend(root.rglob("cdc_milestones_with_bridges.xlsx"))
            candidates.extend(root.rglob("*with*bridges*.numbers"))
            candidates.extend(root.rglob("*with*bridges*.xlsx"))
            candidates.extend(root.rglob("*bridge*milestone*.xlsx"))
            candidates.extend(root.rglob("*bridge*reference*.xlsx"))

    candidates = sorted({p.resolve() for p in candidates if p.exists()}, key=lambda p: len(p.name))
    return candidates[0] if candidates else None



def _clean_bridge_column_name(col: Any, position: int) -> str:
    """Normalize bridge-table column names, including blank Numbers columns.

    Sara's current Numbers bridge table has the bridge step number in column 2
    and the bridge/prerequisite text in column 6, with some blank headers.
    This function preserves that structure without forcing a heavy Excel schema.
    """
    raw = str(col or "").strip().lower()
    raw = re.sub(r"\s+", " ", raw)
    if raw in {"", "none", "nan"}:
        return f"col_{position}"
    return raw.replace(" ", "_")


def _read_numbers_table_to_dataframe(path: Path) -> Tuple[pd.DataFrame, str]:
    """Read the first table from an Apple Numbers file into a DataFrame.

    Requires numbers-parser. If not installed, run:
        %pip install numbers-parser
    """
    try:
        from numbers_parser import Document
    except Exception as exc:
        raise ImportError(
            "Apple Numbers bridge tables require the 'numbers-parser' package. "
            "Run `%pip install numbers-parser` or export the bridge table to .xlsx."
        ) from exc

    doc = Document(str(path))
    table = None
    sheet_name = ""
    table_name = ""
    for sheet in doc.sheets:
        if not sheet.tables:
            continue
        # Prefer a table with enough columns to look like the bridge map.
        for candidate in sheet.tables:
            if candidate.num_rows >= 2 and candidate.num_cols >= 5:
                table = candidate
                sheet_name = getattr(sheet, "name", "Sheet")
                table_name = getattr(candidate, "name", "Table")
                break
        if table is not None:
            break
    if table is None:
        raise ValueError(f"No usable table found in Numbers file: {path}")

    rows = []
    for r in range(table.num_rows):
        rows.append([table.cell(r, c).value for c in range(table.num_cols)])

    if not rows:
        return pd.DataFrame(), f"{sheet_name}/{table_name}"

    header = rows[0]
    columns = [_clean_bridge_column_name(header[i], i) for i in range(len(header))]
    df = pd.DataFrame(rows[1:], columns=columns)
    return df, f"{sheet_name}/{table_name}"


def _read_bridge_reference_file(bridge_path: Path) -> Tuple[pd.DataFrame, str]:
    """Read .numbers or .xlsx bridge reference files."""
    suffix = bridge_path.suffix.lower()
    if suffix == ".numbers":
        return _read_numbers_table_to_dataframe(bridge_path)
    xls = pd.ExcelFile(bridge_path)
    sheet = _pick_bridge_sheet(xls)
    return pd.read_excel(xls, sheet_name=sheet), sheet

def _pick_bridge_sheet(xls: pd.ExcelFile) -> str:
    lower_map = {s.lower(): s for s in xls.sheet_names}
    for wanted in BRIDGE_SHEET_CANDIDATES:
        if wanted.lower() in lower_map:
            return lower_map[wanted.lower()]
    for s in xls.sheet_names:
        if "bridge" in s.lower():
            return s
    return xls.sheet_names[0]


def load_bridge_milestone_map(path: Optional[str] = None) -> Tuple[pd.DataFrame, Optional[Path]]:
    """Load and normalize the advisor-editable bridge milestone reference table."""
    bridge_path = find_bridge_milestone_file(path)
    if bridge_path is None:
        if EMBEDDED_USER_BRIDGE_ROWS:
            print("No external bridge milestone reference file found. Using embedded cdc_milestones_with_bridges snapshot.")
            df = pd.DataFrame(EMBEDDED_USER_BRIDGE_ROWS)
            sheet = "embedded_user_bridge_rows"
        else:
            print("WARNING: No bridge milestone reference file found. v17 will use conservative fallback bridge steps.")
            return pd.DataFrame(), None
    else:
        df, sheet = _read_bridge_reference_file(bridge_path)
    df = df.copy()
    df.columns = [_clean_bridge_column_name(c, i) for i, c in enumerate(df.columns)]

    # Standard schema mapping. Supports both the richer v17 draft table and
    # Sara's compact Numbers table:
    # months | bridge_step_number | category | milestone | subdomain | bridge_step_text
    rename_map = {
        "cdc_milestone": "cdc_milestone",
        "cdc milestone": "cdc_milestone",
        "cdc_milestone_text": "cdc_milestone",
        "milestone": "cdc_milestone",
        "milestone_text": "cdc_milestone",
        "bridge": "bridge_step_title",
        "bridge_step": "bridge_step_title",
        "bridge_text": "bridge_step_title",
        "bridge_title": "bridge_step_title",
        "bridge_milestone": "bridge_step_title",
        "bridge_milestone_text": "bridge_step_title",
        "bridge_step_description": "bridge_step_description",
        "bridge_description": "bridge_step_description",
        "description": "bridge_step_description",
        "step_number": "bridge_step_number",
        "bridge_number": "bridge_step_number",
        "bridge_step_no": "bridge_step_number",
        "review_status": "advisor_review_status",
    }
    for old_col, new_col in rename_map.items():
        normalized_old = old_col.replace(" ", "_")
        if normalized_old in df.columns and new_col not in df.columns:
            df = df.rename(columns={normalized_old: new_col})

    # Positional compatibility for the current cdc_milestones_with_bridges.numbers table.
    positional_map = {
        "col_1": "bridge_step_number",
        "col_5": "bridge_step_title",
    }
    for old_col, new_col in positional_map.items():
        if old_col in df.columns and new_col not in df.columns:
            df = df.rename(columns={old_col: new_col})

    if "bridge_step_title" in df.columns and "bridge_step_description" not in df.columns:
        df["bridge_step_description"] = df["bridge_step_title"]
    if "bridge_step_description" in df.columns and "bridge_step_title" not in df.columns:
        df["bridge_step_title"] = df["bridge_step_description"]

    required_defaults = {
        "months": None,
        "category": "",
        "subdomain": "unspecified",
        "cdc_milestone": "",
        "bridge_step_number": 1,
        "bridge_step_title": "Bridge step",
        "bridge_step_description": "Build a smaller prerequisite skill before the CDC milestone.",
        "success_criteria": "Child participates in the bridge step with low frustration.",
        "activity_focus": "prerequisite skill building",
        "easier_hint": "Add adult support and accept a smaller attempt.",
        "harder_hint": "Reduce support or move to the next bridge step when easy.",
        "what_to_avoid": "Do not require the final CDC milestone before prerequisites are ready.",
        "play_theme_suggestions": "; ".join(DEFAULT_PLAY_THEMES[:5]),
        "group_play_hint": "Let a sibling or another caregiver model one short turn.",
        "advisor_review_status": "Draft - needs advisor review",
    }
    for col, default in required_defaults.items():
        if col not in df.columns:
            df[col] = default

    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df["bridge_step_number"] = pd.to_numeric(df["bridge_step_number"], errors="coerce").fillna(1).astype(int)
    df["bridge_step_title"] = df["bridge_step_title"].astype(str).str.strip()
    df["bridge_step_description"] = df["bridge_step_description"].astype(str).str.strip()
    # Remove empty/placeholder bridge rows but keep final-step rows if the advisor table includes the CDC milestone as the last bridge.
    df = df[df["bridge_step_title"].replace({"": pd.NA, "None": pd.NA, "nan": pd.NA}).notna()].copy()
    df["category_key"] = df["category"].map(_category_to_key)
    df["milestone_key"] = df["cdc_milestone"].map(_norm_text_key)
    df["subdomain_key"] = df["subdomain"].map(lambda x: str(x or "unspecified").strip().lower())
    df = df[df["cdc_milestone"].astype(str).str.len() > 0].copy()
    df = df.sort_values(["category_key", "months", "milestone_key", "bridge_step_number"]).reset_index(drop=True)

    milestone_count = df.groupby(["months", "category_key", "milestone_key"]).ngroups if not df.empty else 0
    step_distribution = df.groupby(["months", "category_key", "milestone_key"]).size().value_counts().sort_index().to_dict() if not df.empty else {}
    print(f"Loaded bridge milestone file: {bridge_path if bridge_path is not None else '[embedded snapshot]'}")
    print(f"Loaded bridge sheet/table: {sheet}")
    print(f"Bridge rows: {len(df)} | CDC milestones covered: {milestone_count}")
    print(f"Bridge steps per milestone distribution: {step_distribution}")
    return df, bridge_path


BRIDGE_MILESTONE_DF, BRIDGE_MILESTONE_PATH = load_bridge_milestone_map()


def get_bridge_rows_for_target(
    category_key: str,
    months: int,
    cdc_milestone: str,
    subdomain: str = "unspecified",
) -> List[Dict[str, Any]]:
    """Return advisor-table bridge rows for one selected CDC target milestone."""
    if BRIDGE_MILESTONE_DF is None or BRIDGE_MILESTONE_DF.empty:
        return []

    key = _norm_text_key(cdc_milestone)
    sub_key = str(subdomain or "unspecified").strip().lower()
    df = BRIDGE_MILESTONE_DF

    exact = df[(df["category_key"] == category_key) & (df["milestone_key"] == key)]
    if not exact.empty:
        # Prefer same age band, but do not require it because some rows may shift by source version.
        same_age = exact[exact["months"].fillna(-1).astype(int) == int(months)]
        if not same_age.empty:
            exact = same_age
        return exact.sort_values("bridge_step_number").to_dict("records")

    # Softer fallback: same milestone text regardless of category.
    exact_any_category = df[df["milestone_key"] == key]
    if not exact_any_category.empty:
        return exact_any_category.sort_values("bridge_step_number").to_dict("records")

    # Last fallback: same category/month/subdomain if available.
    same_context = df[
        (df["category_key"] == category_key)
        & (df["months"].fillna(-1).astype(int) == int(months))
        & (df["subdomain_key"] == sub_key)
    ]
    if not same_context.empty:
        return same_context.sort_values("bridge_step_number").head(5).to_dict("records")

    return []


def _fallback_bridge_rows_for_target(category_key: str, target: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Conservative fallback used only when the bridge table lacks a CDC milestone.

    The fallback is intentionally marked for advisor review and should be replaced by
    the validated bridge reference table.
    """
    cdc = str(target.get("milestone", "selected CDC milestone"))
    subdomain = str(target.get("subdomain", "unspecified"))
    months = int(target.get("months", 0) or 0)
    return [
        {
            "months": months,
            "category_key": category_key,
            "category": DOMAIN_CONFIG.get(category_key, {}).get("display", category_key),
            "subdomain": subdomain,
            "cdc_milestone": cdc,
            "bridge_step_number": 1,
            "bridge_step_title": "Foundational prerequisite",
            "bridge_step_description": f"Build a simpler prerequisite that prepares for: {cdc}.",
            "success_criteria": "Child participates with support and low frustration.",
            "activity_focus": "foundational prerequisite",
            "easier_hint": "Add adult support, simplify the materials, and accept a smaller attempt.",
            "harder_hint": "Reduce support or move toward the next bridge step when easy.",
            "what_to_avoid": "Do not require the final CDC milestone yet.",
            "play_theme_suggestions": "; ".join(DEFAULT_PLAY_THEMES[:5]),
            "group_play_hint": "Let a sibling or caregiver model one short turn.",
            "advisor_review_status": "Fallback - replace with advisor-reviewed bridge row",
        },
        {
            "months": months,
            "category_key": category_key,
            "category": DOMAIN_CONFIG.get(category_key, {}).get("display", category_key),
            "subdomain": subdomain,
            "cdc_milestone": cdc,
            "bridge_step_number": 2,
            "bridge_step_title": "Supported practice",
            "bridge_step_description": f"Practice a supported version of the skill that comes before: {cdc}.",
            "success_criteria": "Child completes part of the skill with adult help.",
            "activity_focus": "supported bridge practice",
            "easier_hint": "Use hand-over-hand, visual cues, or a shorter turn.",
            "harder_hint": "Fade one cue or add one more turn.",
            "what_to_avoid": "Do not pressure the child to perform the final milestone.",
            "play_theme_suggestions": "; ".join(DEFAULT_PLAY_THEMES[2:7]),
            "group_play_hint": "Have a family member take the first turn, then invite the child.",
            "advisor_review_status": "Fallback - replace with advisor-reviewed bridge row",
        },
        {
            "months": months,
            "category_key": category_key,
            "category": DOMAIN_CONFIG.get(category_key, {}).get("display", category_key),
            "subdomain": subdomain,
            "cdc_milestone": cdc,
            "bridge_step_number": 3,
            "bridge_step_title": "Near-milestone attempt",
            "bridge_step_description": f"Try a low-pressure near-version of the CDC milestone: {cdc}.",
            "success_criteria": "Child makes a clear attempt without distress.",
            "activity_focus": "near-milestone practice",
            "easier_hint": "Return to supported practice if this is frustrating.",
            "harder_hint": "Try the final milestone only when this is easy and enjoyable.",
            "what_to_avoid": "Do not treat refusal or an unclear attempt as failure.",
            "play_theme_suggestions": "; ".join(DEFAULT_PLAY_THEMES[4:9]),
            "group_play_hint": "Turn it into a short family game with cheering, not testing.",
            "advisor_review_status": "Fallback - replace with advisor-reviewed bridge row",
        },
    ]


def build_bridge_plan_for_category(
    state: Dict[str, Any],
    category_key: str,
    next_steps: Dict[str, Any],
    max_cdc_targets: int = 1,
    active_bridge_count: int = 2,
) -> Dict[str, Any]:
    """Create/store a v17 bridge plan for a category.

    The CDC milestone is the current destination. The active bridge milestones are
    the smaller prerequisite goals used for activity generation.
    """
    if "bridge_plans" not in state:
        state["bridge_plans"] = {}

    targets = list(next_steps.get("milestones", []) or [])[:max_cdc_targets]
    cdc_targets = []
    active_bridge_milestones = []

    for t_index, target in enumerate(targets, start=1):
        rows = get_bridge_rows_for_target(
            category_key=category_key,
            months=int(target.get("months", 0) or 0),
            cdc_milestone=str(target.get("milestone", "")),
            subdomain=str(target.get("subdomain", "unspecified")),
        )
        source = "bridge_reference_table"
        if not rows:
            rows = _fallback_bridge_rows_for_target(category_key, target)
            source = "fallback_needs_advisor_review"

        bridge_rows = []
        for row in rows:
            item = {
                "bridge_id": f"{category_key}_{_norm_text_key(target.get('milestone', 'target')).replace(' ', '_')}_b{int(row.get('bridge_step_number', 1))}",
                "cdc_target_index": t_index,
                "cdc_target_milestone": str(target.get("milestone", "")),
                "cdc_target_months": int(target.get("months", 0) or 0),
                "subdomain": str(target.get("subdomain", row.get("subdomain", "unspecified"))),
                "bridge_step_number": int(row.get("bridge_step_number", 1)),
                "bridge_step_title": str(row.get("bridge_step_title", "Bridge step")),
                "bridge_step_description": str(row.get("bridge_step_description", "")),
                "success_criteria": str(row.get("success_criteria", "")),
                "activity_focus": str(row.get("activity_focus", "")),
                "easier_hint": str(row.get("easier_hint", "")),
                "harder_hint": str(row.get("harder_hint", "")),
                "what_to_avoid": str(row.get("what_to_avoid", "")),
                "play_theme_suggestions": str(row.get("play_theme_suggestions", "")),
                "group_play_hint": str(row.get("group_play_hint", "")),
                "advisor_review_status": str(row.get("advisor_review_status", "Draft - needs advisor review")),
                "source": source,
                "status": "locked",
            }
            bridge_rows.append(item)

        bridge_rows = sorted(bridge_rows, key=lambda x: x["bridge_step_number"])
        for b in bridge_rows[:active_bridge_count]:
            b["status"] = "active"
            active_bridge_milestones.append(b)

        cdc_targets.append({
            "target_index": t_index,
            "months": int(target.get("months", 0) or 0),
            "milestone": str(target.get("milestone", "")),
            "subdomain": str(target.get("subdomain", "unspecified")),
            "bridge_source": source,
            "bridge_milestones": bridge_rows,
        })

    plan = {
        "status": "success" if cdc_targets else "no_targets",
        "category_key": category_key,
        "category": DOMAIN_CONFIG.get(category_key, {}).get("display", category_key),
        "bridge_table_path": str(BRIDGE_MILESTONE_PATH) if BRIDGE_MILESTONE_PATH else "not_found",
        "cdc_targets": cdc_targets,
        "active_bridge_milestones": active_bridge_milestones,
        "active_rule": "Work on only the first 1–2 unmastered bridge milestones at a time.",
    }
    state["bridge_plans"][category_key] = plan
    return plan


def _themes_from_bridge(bridge: Dict[str, Any]) -> List[str]:
    raw = str(bridge.get("play_theme_suggestions", "") or "")
    parts = [p.strip() for p in re.split(r"[;,|]", raw) if p.strip()]
    themes = []
    for p in parts + DEFAULT_PLAY_THEMES:
        if p not in themes:
            themes.append(p)
    return themes[:6]


def _activity_type_counts(total: int = 8) -> Dict[str, int]:
    # User-preferred bank: 3–4 core, 1–2 easier, 1–2 harder.
    if total <= 6:
        return {"core": 3, "easier_backup": 1, "harder_stretch": max(1, total - 4)}
    return {"core": 4, "easier_backup": 2, "harder_stretch": 2}


def _activity_type_for_index(idx: int, total: int = 8) -> str:
    counts = _activity_type_counts(total)
    if idx <= counts["core"]:
        return "core"
    if idx <= counts["core"] + counts["easier_backup"]:
        return "easier_backup"
    return "harder_stretch"


def _bridge_for_activity_index(active_bridges: List[Dict[str, Any]], idx: int) -> Dict[str, Any]:
    if not active_bridges:
        return {}
    return active_bridges[(idx - 1) % len(active_bridges)]


def _fallback_v17_activity(
    category_key: str,
    bridge: Dict[str, Any],
    idx: int,
    total: int,
    planning_tier: str,
) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key)
    activity_type = _activity_type_for_index(idx, total)
    themes = _themes_from_bridge(bridge)
    theme = themes[(idx - 1) % len(themes)] if themes else DEFAULT_PLAY_THEMES[(idx - 1) % len(DEFAULT_PLAY_THEMES)]
    bridge_title = str(bridge.get("bridge_step_title", "Bridge step"))
    bridge_desc = str(bridge.get("bridge_step_description", "Practice the current bridge step."))
    cdc_target = str(bridge.get("cdc_target_milestone", "selected CDC milestone"))

    if activity_type == "easier_backup":
        title = f"Easier: {bridge_title}"
        focus = bridge.get("easier_hint") or "Make the step easier with more support."
    elif activity_type == "harder_stretch":
        title = f"Stretch: {bridge_title}"
        focus = bridge.get("harder_hint") or "Make the step a little harder only if the child is succeeding."
    else:
        title = f"{bridge_title} — {theme.title()}"
        focus = bridge_desc

    steps = [
        f"Set up a short {theme} using familiar household items.",
        f"Practice the bridge step: {bridge_title}.",
        "Stop while it is still fun; a small successful attempt counts.",
    ]
    instructions = " ".join(steps)

    return {
        "activity_id": f"{category_key}_v17_{idx}",
        "activity_type": activity_type,
        "title": title,
        "target_skill": str(bridge.get("activity_focus", bridge_title)),
        "cdc_target_milestone": cdc_target,
        "bridge_id": bridge.get("bridge_id", ""),
        "bridge_step_number": int(bridge.get("bridge_step_number", 1) or 1),
        "bridge_step_title": bridge_title,
        "play_theme": theme,
        "theme_variations": themes[:3],
        "why_it_helps": f"This builds a prerequisite for: {cdc_target}.",
        "materials": "common toys or household items",
        "how_to_do_it": steps,
        "instructions": instructions,
        "success_criteria": str(bridge.get("success_criteria", "Child participates with low frustration.")),
        "make_easier": str(bridge.get("easier_hint", "Add adult support and accept a smaller attempt.")),
        "make_harder": str(bridge.get("harder_hint", "Reduce support or try the next bridge step when easy.")),
        "group_play_line": str(bridge.get("group_play_hint", "Let a sibling or caregiver model one short turn.")),
        "repeat_guidance": "Repeat this activity during the 1–2 week cycle until it is easy, enjoyable, or mastered.",
        "mastery_rule": "Move on after 3 independent successes or 2 too-easy flags without major frustration.",
        "duration_min": 5,
        "level": "bridge_step",
        "goal": planning_tier,
        "category": category_display,
        "is_extended_activity": False,
        "extended_reason": "",
        "what_to_avoid": str(bridge.get("what_to_avoid", "Do not require the final milestone yet.")),
    }


def _normalize_v17_activity(
    activity: Dict[str, Any],
    category_key: str,
    bridge: Dict[str, Any],
    idx: int,
    planning_tier: str,
) -> Dict[str, Any]:
    """Make sure an LLM-returned activity has every v17 field plus v16 compatibility fields."""
    a = dict(activity or {})
    fallback = _fallback_v17_activity(category_key, bridge, idx, 8, planning_tier)
    for k, v in fallback.items():
        if k not in a or a.get(k) in [None, "", []]:
            a[k] = v

    # Normalize activity_type.
    at = str(a.get("activity_type", "core")).strip().lower().replace(" ", "_")
    if at not in {"core", "easier_backup", "harder_stretch"}:
        at = "core"
    a["activity_type"] = at

    # Duration guardrail.
    try:
        duration = int(a.get("duration_min", 5))
    except Exception:
        duration = 5
    if not a.get("is_extended_activity", False):
        duration = min(max(duration, 3), 10)
    a["duration_min"] = duration

    # Ensure how_to_do_it and instructions are aligned.
    if isinstance(a.get("how_to_do_it"), str):
        a["how_to_do_it"] = [x.strip() for x in re.split(r"\n+|\s*;\s*", a["how_to_do_it"]) if x.strip()]
    if not isinstance(a.get("how_to_do_it"), list) or not a["how_to_do_it"]:
        a["how_to_do_it"] = fallback["how_to_do_it"]
    if not str(a.get("instructions", "")).strip():
        a["instructions"] = " ".join(str(x) for x in a["how_to_do_it"][:4])

    # Short group line.
    group_line = str(a.get("group_play_line", "")).strip()
    if group_line:
        words = group_line.split()
        if len(words) > 28:
            group_line = " ".join(words[:28]).rstrip(".,;") + "."
    a["group_play_line"] = group_line or fallback["group_play_line"]

    a["goal"] = planning_tier
    a["category"] = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key)
    a["level"] = "bridge_step"
    return a


def generate_category_activity_bank(
    state: Dict[str, Any],
    category_key: str,
    model: str = "gpt-4o-mini",
    activities_per_category: int = 8,
) -> Dict[str, Any]:
    """v17: Generate an activity bank from active bridge milestones, not directly from CDC milestones."""
    child = state["child"]
    category_display = DOMAIN_CONFIG[category_key]["display"]
    support_tier = get_support_tier(state, category_key)
    soft_floor_active = is_family_guidance_category(state, category_key)
    planning_tier = "enrich_and_observe" if soft_floor_active else support_tier
    support_metrics = compute_support_metrics(state, category_key)
    safety_profile = ensure_safety_profile(state)
    safety_constraints_block = format_safety_constraints_for_prompt(safety_profile)
    next_steps = select_next_milestones(state, category_key)

    if next_steps["status"] == "no_special_support":
        state["activity_banks"][category_key] = {
            "status": "no_special_support",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "summary": next_steps["message"],
            "cycle_length_days": 14,
            "activities": [],
        }
        return state["activity_banks"][category_key]

    bridge_plan = build_bridge_plan_for_category(
        state=state,
        category_key=category_key,
        next_steps=next_steps,
        max_cdc_targets=1,
        active_bridge_count=2,
    )
    active_bridges = bridge_plan.get("active_bridge_milestones", [])

    chrono_months = min(child["chronological_months"], 60)
    dev_age = chrono_months if soft_floor_active else state["dev_age"][category_key]
    milestone_gap = max(0, chrono_months - dev_age)
    category_guardrails = get_category_activity_guardrails(category_key)
    language_readiness_block = build_language_activity_readiness_block(state, category_key)

    if not active_bridges:
        active_bridges = [_fallback_bridge_rows_for_target(category_key, {"milestone": "current category goal", "months": dev_age, "subdomain": "unspecified"})[0]]

    cdc_target_lines = []
    bridge_lines = []
    for target in bridge_plan.get("cdc_targets", []):
        cdc_target_lines.append(
            f"- ({target.get('months')} months | {target.get('subdomain')}) {target.get('milestone')}"
        )
    for b in active_bridges:
        bridge_lines.append(
            f"- Bridge {b.get('bridge_step_number')}: {b.get('bridge_step_title')} — "
            f"{b.get('bridge_step_description')} | Success: {b.get('success_criteria')} | "
            f"Avoid: {b.get('what_to_avoid')} | Themes: {b.get('play_theme_suggestions')}"
        )

    cdc_target_block = "\n".join(cdc_target_lines) or "- No CDC target available."
    bridge_block = "\n".join(bridge_lines) or "- No bridge milestones available."

    soft_floor_block = ""
    if soft_floor_active:
        soft_floor_block = (
            "This category is being generated under the family guidance floor. "
            "Use age-appropriate, low-intensity enrichment and observation activities. "
            "Do not imply therapy-level intensity or a significant delay. "
            "The goal is support, confidence-building, and structured observation."
        )

    def fallback_bank(reason: str) -> Dict[str, Any]:
        fallback_activities = []
        for i in range(1, activities_per_category + 1):
            bridge = _bridge_for_activity_index(active_bridges, i)
            fallback_activities.append(_fallback_v17_activity(category_key, bridge, i, activities_per_category, planning_tier))
        fallback_activities = apply_language_readiness_constraints_to_activities(state, category_key, fallback_activities)
        fallback_activities = apply_safety_constraints_to_activities(state, category_key, fallback_activities)
        # Re-normalize after safety/language replacement so v17 fields remain present.
        normalized = []
        for i, a in enumerate(fallback_activities, start=1):
            normalized.append(_normalize_v17_activity(a, category_key, _bridge_for_activity_index(active_bridges, i), i, planning_tier))
        bank = {
            "status": "fallback",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "cycle_length_days": 14,
            "bridge_plan": bridge_plan,
            "summary": f"v17 fallback activity bank used for {category_display}: {reason}",
            "activities": normalized,
            "feedback_options": ACTIVITY_FEEDBACK_OPTIONS,
            "activity_quality_guardrails": ACTIVITY_QUALITY_GUARDRAILS,
        }
        state["activity_banks"][category_key] = bank
        return bank

    if openai_client is None:
        return fallback_bank("OpenAI client is not available")

    prompt = f"""
You are the Genex v17 pediatric home-support activity planner.

This is NOT a diagnosis and NOT a formal treatment plan.
Create a CATEGORY ACTIVITY BANK for a 1–2 week repeat cycle.

Child:
- Name: {child['name']}
- Chronological age: {child['chronological_months']} months
- Diagnosis / condition: {child['diagnosis']}
- Parent concern: {child['concern']}
- Daily time available: {child.get('daily_time_min')} minutes
- Category: {category_display}
- Support tier for this category: {planning_tier}
- Estimated developmental age in this category: {dev_age} months
- Estimated milestone gap in this category: {milestone_gap} months
- Continuous support score for this category: {support_metrics['support_score']}

Current CDC milestone goal(s):
{cdc_target_block}

Active bridge milestones to practice now:
{bridge_block}

Important v17 rule:
- The CDC milestone is the destination, not necessarily the first activity.
- Generate activities for the ACTIVE BRIDGE MILESTONES above.
- Do NOT ask the child to perform the final CDC milestone unless the active bridge step is already that final step.
- Work on only 1–2 bridge steps at a time.

Activity quality guardrails:
{ACTIVITY_QUALITY_GUARDRAILS}

Category-specific guardrails:
{category_guardrails}

Developmental readiness constraints:
{language_readiness_block}

Safety / practical constraints inferred from diagnosis + concern:
{safety_constraints_block}

{soft_floor_block}

Task:
Create exactly {activities_per_category} short home activities for this category.
The bank should include:
- 4 core activities
- 2 easier_backup activities
- 2 harder_stretch activities

Each activity must include:
- a playful title
- the active bridge milestone it practices
- one play_theme and 2–4 theme_variations
- why_it_helps
- materials
- how_to_do_it as 3–5 parent-friendly steps
- success_criteria
- make_easier
- make_harder
- one short group_play_line for sibling/parent participation
- repeat_guidance
- mastery_rule

Parent feedback options this bank will use:
Difficulty: too_hard, just_right, too_easy
Performance: done_independently, done_with_help, couldnt_do_it
Engagement: enjoyed_it, resisted_it, didnt_like_it

Context-dependent activities such as playdates, park meetups, playground peer practice,
group social activities, or community participation must not be normal daily home activities.
If included at all, mark them is_extended_activity=true and extended_reason="optional weekly bonus".

Return strict JSON only.

Required JSON:
{{
  "summary": "...",
  "cycle_length_days": 14,
  "activities": [
    {{
      "activity_id": "1",
      "activity_type": "core|easier_backup|harder_stretch",
      "title": "...",
      "target_skill": "...",
      "cdc_target_milestone": "...",
      "bridge_id": "...",
      "bridge_step_number": 1,
      "bridge_step_title": "...",
      "play_theme": "...",
      "theme_variations": ["...", "..."],
      "why_it_helps": "...",
      "materials": "...",
      "how_to_do_it": ["step 1", "step 2", "step 3"],
      "instructions": "short combined parent-facing instructions",
      "success_criteria": "...",
      "make_easier": "...",
      "make_harder": "...",
      "group_play_line": "...",
      "repeat_guidance": "...",
      "mastery_rule": "Move on after 3 independent successes or 2 too-easy flags without major frustration.",
      "duration_min": 5,
      "level": "bridge_step",
      "goal": "{planning_tier}",
      "category": "{category_display}",
      "is_extended_activity": false,
      "extended_reason": ""
    }}
  ]
}}
""".strip()

    try:
        resp = openai_client.chat.completions.create(
            model=model,
            temperature=0.25,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only. You are non-diagnostic, practical, parent-friendly, playful, and developmentally cautious."},
                {"role": "user", "content": prompt},
            ],
        )
        raw_bank = json.loads(resp.choices[0].message.content)
        raw_activities = raw_bank.get("activities", []) if isinstance(raw_bank, dict) else []
        if not isinstance(raw_activities, list):
            raw_activities = []

        normalized = []
        for i, activity in enumerate(raw_activities[:activities_per_category], start=1):
            bridge = _bridge_for_activity_index(active_bridges, i)
            normalized.append(_normalize_v17_activity(activity, category_key, bridge, i, planning_tier))

        while len(normalized) < activities_per_category:
            i = len(normalized) + 1
            bridge = _bridge_for_activity_index(active_bridges, i)
            normalized.append(_fallback_v17_activity(category_key, bridge, i, activities_per_category, planning_tier))

        normalized = apply_language_readiness_constraints_to_activities(state, category_key, normalized)
        normalized = apply_safety_constraints_to_activities(state, category_key, normalized)
        normalized = [
            _normalize_v17_activity(a, category_key, _bridge_for_activity_index(active_bridges, i), i, planning_tier)
            for i, a in enumerate(normalized[:activities_per_category], start=1)
        ]

        bank = {
            "status": "success",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "cycle_length_days": int(raw_bank.get("cycle_length_days", 14)) if isinstance(raw_bank, dict) else 14,
            "bridge_plan": bridge_plan,
            "summary": raw_bank.get("summary", f"Created v17 bridge-based activity bank for {category_display}.") if isinstance(raw_bank, dict) else f"Created v17 bridge-based activity bank for {category_display}.",
            "activities": normalized,
            "feedback_options": ACTIVITY_FEEDBACK_OPTIONS,
            "activity_quality_guardrails": ACTIVITY_QUALITY_GUARDRAILS,
        }
        state["activity_banks"][category_key] = bank
        return bank

    except Exception as e:
        return fallback_bank(f"OpenAI activity generation failed: {e}")


def allocate_weekly_slots(state: Dict[str, Any], cycle_days: int = 7) -> Dict[str, Any]:
    """v17: allocate visible weekly minutes over a 7-day schedule.

    The bank itself is designed for 1–2 weeks, but the notebook displays the first week.
    """
    child = state["child"]
    if "daily_time_min" not in child:
        raise ValueError("Missing daily_time_min in state['child'].")

    chrono = min(child["chronological_months"], 60)
    daily_time_min = int(child["daily_time_min"])
    weekly_minutes = daily_time_min * cycle_days
    candidate_categories = state.get("planned_categories") or list(DOMAIN_CONFIG.keys())

    supported_categories = []
    gap_by_category = {}
    weight_by_category = {}

    for category_key in candidate_categories:
        tier = get_support_tier(state, category_key)
        if tier == "no_special_support":
            continue
        supported_categories.append(category_key)
        dev_age = get_effective_dev_age(state, category_key)
        if dev_age is None:
            dev_age = state.get("dev_age", {}).get(category_key, chrono)
        gap = max(0, chrono - dev_age)
        gap_by_category[category_key] = gap
        weight_by_category[category_key] = max(1, gap) if tier == "needs_special_support" else max(1, gap) * 0.5

    soft_floor = determine_family_guidance_floor(state)
    if not supported_categories and soft_floor.get("enabled"):
        category_key = soft_floor["category_key"]
        target_minutes = min(int(soft_floor.get("target_weekly_minutes", 20)), weekly_minutes)
        allocation = {
            "daily_time_min": daily_time_min,
            "cycle_days": cycle_days,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [category_key],
            "gap_by_category": {category_key: 0},
            "target_minutes_by_category": {category_key: target_minutes},
            "planning_mode": "family_guidance_floor",
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    if not supported_categories:
        allocation = {
            "daily_time_min": daily_time_min,
            "cycle_days": cycle_days,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [],
            "gap_by_category": {},
            "target_minutes_by_category": {},
            "planning_mode": "none",
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    # Start with one short activity per supported category across the week, then distribute extra time by gap.
    base_minutes_per_category = max(10, min(35, daily_time_min * 2))
    target_minutes_by_category = {k: base_minutes_per_category for k in supported_categories}
    used_minutes = sum(target_minutes_by_category.values())
    remaining_minutes = max(0, weekly_minutes - used_minutes)

    total_weight = sum(weight_by_category.values())
    if total_weight > 0 and remaining_minutes > 0:
        for k in supported_categories:
            extra = round(remaining_minutes * (weight_by_category[k] / total_weight))
            target_minutes_by_category[k] += extra

    while sum(target_minutes_by_category.values()) > weekly_minutes:
        biggest = max(target_minutes_by_category, key=target_minutes_by_category.get)
        target_minutes_by_category[biggest] = max(5, target_minutes_by_category[biggest] - 1)
        if all(v <= 5 for v in target_minutes_by_category.values()):
            break

    allocation = {
        "daily_time_min": daily_time_min,
        "cycle_days": cycle_days,
        "weekly_minutes": weekly_minutes,
        "supported_categories": supported_categories,
        "gap_by_category": gap_by_category,
        "target_minutes_by_category": target_minutes_by_category,
        "planning_mode": "v17_bridge_activity_bank",
    }
    state["weekly_slot_allocation"] = allocation
    return allocation


def _activity_feedback_counts(state: Dict[str, Any], activity_id: str = "", bridge_id: str = "") -> Dict[str, int]:
    counts = {
        "too_hard": 0,
        "just_right": 0,
        "too_easy": 0,
        "done_independently": 0,
        "done_with_help": 0,
        "couldnt_do_it": 0,
        "enjoyed_it": 0,
        "resisted_it": 0,
        "didnt_like_it": 0,
    }
    for item in state.get("activity_feedback_log", []):
        if activity_id and item.get("activity_id") != activity_id:
            continue
        if bridge_id and item.get("bridge_id") != bridge_id:
            continue
        for key in ["difficulty", "performance", "engagement"]:
            value = item.get(key)
            if value in counts:
                counts[value] += 1
    return counts


def _preferred_activity_type_from_feedback(counts: Dict[str, int]) -> str:
    if counts.get("couldnt_do_it", 0) >= 1 or counts.get("too_hard", 0) >= 2:
        return "easier_backup"
    if counts.get("too_easy", 0) >= 1 or counts.get("done_independently", 0) >= 3:
        return "harder_stretch"
    return "core"


def _choose_activity_for_category_day(
    state: Dict[str, Any],
    category_key: str,
    day_index: int,
    remaining_minutes: int,
    used_activity_ids: set,
) -> Optional[Dict[str, Any]]:
    bank = state.get("activity_banks", {}).get(category_key, {})
    activities = [a for a in bank.get("activities", []) if not a.get("is_extended_activity", False)]
    activities = [a for a in activities if not is_context_dependent_bonus_activity(a)]
    activities = [a for a in activities if int(a.get("duration_min", 5)) <= remaining_minutes]
    if not activities:
        return None

    # Prefer activities whose bridge feedback suggests the right difficulty level.
    scored = []
    for idx, a in enumerate(activities):
        activity_id = str(a.get("activity_id", idx))
        bridge_id = str(a.get("bridge_id", ""))
        counts = _activity_feedback_counts(state, bridge_id=bridge_id)
        preferred_type = _preferred_activity_type_from_feedback(counts)
        activity_type = str(a.get("activity_type", "core"))

        score = 0
        if activity_type == preferred_type:
            score += 10
        if activity_type == "core":
            score += 3
        if activity_id in used_activity_ids:
            score -= 8
        if counts.get("enjoyed_it", 0) > counts.get("didnt_like_it", 0):
            score += 2
        if counts.get("didnt_like_it", 0) >= 2 or counts.get("resisted_it", 0) >= 2:
            # Do not eliminate; same skill may still be right, but prefer another activity/theme.
            score -= 3

        # Gentle rotation so all core activities appear across the week.
        score -= abs((idx % max(1, len(activities))) - (day_index % max(1, len(activities)))) * 0.05
        scored.append((score, idx, a))

    scored = sorted(scored, key=lambda x: (x[0], -x[1]), reverse=True)
    return dict(scored[0][2]) if scored else None


def _schedule_item_from_activity(category_key: str, activity: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "category_key": category_key,
        "category": DOMAIN_CONFIG[category_key]["display"],
        "activity_id": activity.get("activity_id"),
        "activity_type": activity.get("activity_type", "core"),
        "title": activity.get("title"),
        "instructions": activity.get("instructions"),
        "duration_min": int(activity.get("duration_min", 5)),
        "materials": activity.get("materials", "common household items"),
        "level": activity.get("level", "bridge_step"),
        "goal": activity.get("goal", get_support_tier(state, category_key) if 'state' in globals() else ""),
        "cdc_target_milestone": activity.get("cdc_target_milestone", ""),
        "bridge_id": activity.get("bridge_id", ""),
        "bridge_step_title": activity.get("bridge_step_title", ""),
        "play_theme": activity.get("play_theme", ""),
        "theme_variations": activity.get("theme_variations", []),
        "why_it_helps": activity.get("why_it_helps", ""),
        "success_criteria": activity.get("success_criteria", ""),
        "make_easier": activity.get("make_easier", ""),
        "make_harder": activity.get("make_harder", ""),
        "group_play_line": activity.get("group_play_line", ""),
        "repeat_guidance": activity.get("repeat_guidance", ""),
        "mastery_rule": activity.get("mastery_rule", ""),
    }


def build_weekly_schedule(state: Dict[str, Any], cycle_days: int = 7) -> Dict[str, Any]:
    """v17: Build a visible daily schedule from the bridge-based activity bank.

    The bank is meant to repeat for 1–2 weeks; this function displays the first week.
    """
    if "weekly_slot_allocation" not in state:
        allocate_weekly_slots(state, cycle_days=cycle_days)

    allocation = state["weekly_slot_allocation"]
    daily_time_min = int(allocation["daily_time_min"])
    target_minutes_by_category = allocation.get("target_minutes_by_category", {})

    days = {
        f"day_{i}": {"items": [], "total_minutes": 0}
        for i in range(1, cycle_days + 1)
    }

    if not target_minutes_by_category:
        state["weekly_schedule"] = {
            "status": "no_special_support",
            "summary": "No categories need a scheduled weekly activity plan right now.",
            "daily_time_min": daily_time_min,
            "days": days,
        }
        return state["weekly_schedule"]

    assigned_minutes_by_category = {k: 0 for k in target_minutes_by_category.keys()}
    used_activity_ids = set()

    categories_in_priority_order = sorted(
        target_minutes_by_category.keys(),
        key=lambda k: target_minutes_by_category[k],
        reverse=True,
    )

    for day_num, day_name in enumerate(days.keys(), start=1):
        # For 2 domains and short daily time, try one per domain. For 1 domain, allow two short activities.
        made_progress = True
        pass_count = 0
        while made_progress and pass_count < 3:
            made_progress = False
            pass_count += 1
            for category_key in categories_in_priority_order:
                remaining_day_minutes = daily_time_min - days[day_name]["total_minutes"]
                if remaining_day_minutes <= 0:
                    continue
                if assigned_minutes_by_category[category_key] >= target_minutes_by_category.get(category_key, 0) and pass_count > 1:
                    continue

                activity = _choose_activity_for_category_day(
                    state=state,
                    category_key=category_key,
                    day_index=day_num + pass_count,
                    remaining_minutes=remaining_day_minutes,
                    used_activity_ids=used_activity_ids,
                )
                if activity is None:
                    continue

                item = _schedule_item_from_activity(category_key, activity)
                days[day_name]["items"].append(item)
                days[day_name]["total_minutes"] += item["duration_min"]
                assigned_minutes_by_category[category_key] += item["duration_min"]
                used_activity_ids.add(str(item.get("activity_id")))
                made_progress = True

                # Avoid crowding: if there are multiple categories, one activity each is usually enough.
                if len(categories_in_priority_order) > 1:
                    break

    weekly_bonus = pick_weekly_bonus_extended_activity(state)

    schedule = {
        "status": "success",
        "summary": (
            "v17 bridge-based weekly schedule. Activities should repeat for 1–2 weeks; "
            "the app varies play themes, uses feedback to simplify or level up, and advances after mastery."
        ),
        "daily_time_min": daily_time_min,
        "cycle_days_displayed": cycle_days,
        "bank_repeat_window_days": 14,
        "target_minutes_by_category": target_minutes_by_category,
        "assigned_minutes_by_category": assigned_minutes_by_category,
        "feedback_options": ACTIVITY_FEEDBACK_OPTIONS,
        "days": days,
        "weekly_bonus_activity": weekly_bonus,
    }
    state["weekly_schedule"] = schedule
    return schedule


def log_activity_feedback(
    state: Dict[str, Any],
    activity_id: str,
    difficulty: str,
    performance: str,
    engagement: str,
    bridge_id: str = "",
    category_key: str = "",
    notes: str = "",
) -> Dict[str, Any]:
    """Record one parent feedback event after an activity."""
    difficulty = str(difficulty).strip().lower()
    performance = str(performance).strip().lower()
    engagement = str(engagement).strip().lower()
    if difficulty not in ACTIVITY_FEEDBACK_OPTIONS["difficulty"]:
        raise ValueError(f"difficulty must be one of {ACTIVITY_FEEDBACK_OPTIONS['difficulty']}")
    if performance not in ACTIVITY_FEEDBACK_OPTIONS["performance"]:
        raise ValueError(f"performance must be one of {ACTIVITY_FEEDBACK_OPTIONS['performance']}")
    if engagement not in ACTIVITY_FEEDBACK_OPTIONS["engagement"]:
        raise ValueError(f"engagement must be one of {ACTIVITY_FEEDBACK_OPTIONS['engagement']}")

    entry = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "activity_id": activity_id,
        "bridge_id": bridge_id,
        "category_key": category_key,
        "difficulty": difficulty,
        "performance": performance,
        "engagement": engagement,
        "notes": notes,
    }
    state.setdefault("activity_feedback_log", []).append(entry)
    update_play_preferences_from_feedback(state)
    update_bridge_progress_from_feedback(state)
    return entry


def update_play_preferences_from_feedback(state: Dict[str, Any]) -> Dict[str, Any]:
    """Update lightweight child play preferences from feedback logs."""
    prefs = state.setdefault("play_preferences", {
        "liked_themes": [],
        "disliked_themes": [],
        "works_well_with_sibling": None,
    })
    activity_by_id = {}
    for bank in state.get("activity_banks", {}).values():
        for a in bank.get("activities", []):
            activity_by_id[str(a.get("activity_id"))] = a

    for fb in state.get("activity_feedback_log", []):
        a = activity_by_id.get(str(fb.get("activity_id")))
        if not a:
            continue
        theme = str(a.get("play_theme", "")).strip()
        if not theme:
            continue
        if fb.get("engagement") == "enjoyed_it" and theme not in prefs["liked_themes"]:
            prefs["liked_themes"].append(theme)
        if fb.get("engagement") in {"resisted_it", "didnt_like_it"} and theme not in prefs["disliked_themes"]:
            prefs["disliked_themes"].append(theme)
    return prefs


def update_bridge_progress_from_feedback(state: Dict[str, Any]) -> Dict[str, Any]:
    """Apply simple mastery/adaptation rules to bridge milestones based on feedback."""
    bridge_status = state.setdefault("bridge_progress", {})
    all_bridge_ids = []
    for plan in state.get("bridge_plans", {}).values():
        for b in plan.get("active_bridge_milestones", []):
            bridge_id = b.get("bridge_id")
            if bridge_id:
                all_bridge_ids.append(bridge_id)
                bridge_status.setdefault(bridge_id, {
                    "status": "active",
                    "mastery_count": 0,
                    "too_hard_count": 0,
                    "too_easy_count": 0,
                    "dislike_count": 0,
                    "recommendation": "repeat_current_bridge_step",
                })

    for bridge_id in all_bridge_ids:
        counts = _activity_feedback_counts(state, bridge_id=bridge_id)
        status = bridge_status.setdefault(bridge_id, {})
        status["mastery_count"] = counts.get("done_independently", 0)
        status["too_hard_count"] = counts.get("too_hard", 0) + counts.get("couldnt_do_it", 0)
        status["too_easy_count"] = counts.get("too_easy", 0)
        status["dislike_count"] = counts.get("didnt_like_it", 0) + counts.get("resisted_it", 0)

        if counts.get("done_independently", 0) >= 3 or counts.get("too_easy", 0) >= 2:
            status["status"] = "mastered_or_ready_to_level_up"
            status["recommendation"] = "activate_next_bridge_step_or_use_harder_stretch"
        elif status["too_hard_count"] >= 2:
            status["status"] = "needs_easier_support"
            status["recommendation"] = "use_easier_backup_or_previous_bridge_step"
        elif status["dislike_count"] >= 2:
            status["status"] = "change_theme_keep_skill"
            status["recommendation"] = "keep_same_bridge_step_but_change_play_theme"
        else:
            status["status"] = "active"
            status["recommendation"] = "repeat_current_bridge_step"

    return bridge_status


def build_activity_hub(state: Dict[str, Any]) -> Dict[str, Any]:
    """Create a simple app-friendly activity hub structure."""
    schedule = state.get("weekly_schedule", {})
    banks = state.get("activity_banks", {})
    by_domain = {}
    sibling_family = []
    repeat_this_cycle = []

    for category_key, bank in banks.items():
        display_name = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key)
        acts = bank.get("activities", [])
        by_domain[display_name] = acts
        repeat_this_cycle.extend(acts)
        sibling_family.extend([a for a in acts if str(a.get("group_play_line", "")).strip()])

    return {
        "today": schedule.get("days", {}).get("day_1", {}).get("items", []),
        "repeat_this_cycle": repeat_this_cycle,
        "by_domain": by_domain,
        "sibling_family_activities": sibling_family,
        "saved_favorites": state.get("saved_favorite_activity_ids", []),
        "feedback_options": ACTIVITY_FEEDBACK_OPTIONS,
    }


def generate_parent_progress_summary(state: Dict[str, Any]) -> str:
    """Generate a lightweight parent-facing progress summary from feedback and bridge progress."""
    child_name = state.get("child", {}).get("name", "Your child")
    bridge_progress = update_bridge_progress_from_feedback(state)
    mastered = [bid for bid, s in bridge_progress.items() if s.get("status") == "mastered_or_ready_to_level_up"]
    needs_easier = [bid for bid, s in bridge_progress.items() if s.get("status") == "needs_easier_support"]
    change_theme = [bid for bid, s in bridge_progress.items() if s.get("status") == "change_theme_keep_skill"]
    prefs = update_play_preferences_from_feedback(state)

    lines = [f"{child_name}'s Genex progress summary:"]
    if mastered:
        lines.append(f"- {len(mastered)} bridge step(s) look ready to level up based on independent success or too-easy feedback.")
    if needs_easier:
        lines.append(f"- {len(needs_easier)} bridge step(s) may need an easier backup or more support.")
    if change_theme:
        lines.append(f"- {len(change_theme)} bridge step(s) may be right, but the play theme should change.")
    if prefs.get("liked_themes"):
        lines.append(f"- Enjoyed themes: {', '.join(prefs['liked_themes'][:5])}.")
    if not state.get("activity_feedback_log"):
        lines.append("- No activity feedback has been logged yet. Start with the first-week schedule and repeat familiar skills.")
    return "\n".join(lines)


def build_doctor_therapist_report_payload(state: Dict[str, Any]) -> Dict[str, Any]:
    """Create a structured payload for future doctor/therapist reports.

    This is parent-reported support information, not a clinical diagnosis.
    """
    return {
        "report_type": "parent_reported_genex_support_summary",
        "child": state.get("child", {}),
        "selected_categories": state.get("selected_categories", []),
        "developmental_estimates": state.get("dev_age", {}),
        "delay_estimates": state.get("delay_estimates", {}),
        "bridge_plans": state.get("bridge_plans", {}),
        "activity_feedback_log": state.get("activity_feedback_log", []),
        "bridge_progress": update_bridge_progress_from_feedback(state),
        "play_preferences": update_play_preferences_from_feedback(state),
        "disclaimer": "Parent-reported developmental support summary. Not a diagnosis or treatment plan.",
    }


def summarizer_agent(state: Dict[str, Any]) -> pd.DataFrame:
    """v17 summary with bridge-plan fields added to the v16 category summary."""
    child = state["child"]
    rows = []
    allocation = state.get("weekly_slot_allocation", {}).get("target_minutes_by_category", {})
    concern_profile = ensure_concern_profile(state)
    soft_floor = state.get("family_guidance_floor", {})

    for category_key in DOMAIN_CONFIG:
        delay_info = state["delay_estimates"].get(category_key, {})
        raw_dev_age = state["dev_age"].get(category_key)
        effective_dev_age = get_effective_dev_age(state, category_key)
        metrics = compute_support_metrics(state, category_key)
        chrono_for_gap = min(child.get("chronological_months", 0), 60)
        milestone_gap = None if effective_dev_age is None else max(0, chrono_for_gap - effective_dev_age)
        bank = state.get("activity_banks", {}).get(category_key, {})
        bridge_plan = state.get("bridge_plans", {}).get(category_key, {})
        active_bridge_titles = [b.get("bridge_step_title", "") for b in bridge_plan.get("active_bridge_milestones", [])]
        cdc_targets = [t.get("milestone", "") for t in bridge_plan.get("cdc_targets", [])]

        support_tier = metrics["tier"]
        planning_tier = support_tier
        family_guidance_floor_active = False
        family_guidance_summary = ""
        if soft_floor.get("enabled") and soft_floor.get("category_key") == category_key:
            planning_tier = "enrich_and_observe"
            family_guidance_floor_active = True
            family_guidance_summary = soft_floor.get("summary", "")

        rows.append({
            "child_name": child.get("name"),
            "chronological_age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "concern": child.get("concern"),
            "daily_time_min": child.get("daily_time_min"),
            "category": DOMAIN_CONFIG[category_key]["display"],
            "estimated_delay_months": delay_info.get("delay_months"),
            "delay_reason": delay_info.get("reason", ""),
            "raw_dev_age_months": raw_dev_age,
            "estimated_dev_age_months": effective_dev_age,
            "effective_dev_age_months": effective_dev_age,
            "milestone_gap_months": milestone_gap,
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "needs_special_support": support_tier == "needs_special_support",
            "concern_domain_weight": round(concern_profile["domain_weights"].get(category_key, 0.0), 2),
            "concern_subdomain_peak": round(get_category_concern_peak(state, category_key), 2),
            "support_score": metrics["support_score"],
            "weekly_target_minutes_for_category": allocation.get(category_key, 0),
            "activity_bank_status": bank.get("status", "missing"),
            "activity_bank_summary": bank.get("summary", ""),
            "v17_cdc_targets": "; ".join(cdc_targets),
            "v17_active_bridge_steps": "; ".join(active_bridge_titles),
            "v17_bridge_table_path": bridge_plan.get("bridge_table_path", ""),
            "selected_for_onboarding": category_key in state.get("selected_categories", []),
            "assessed_in_this_run": raw_dev_age is not None,
            "family_guidance_floor_active": family_guidance_floor_active,
            "family_guidance_summary": family_guidance_summary,
        })

    return pd.DataFrame(rows)


def display_weekly_schedule(state: Dict[str, Any]) -> None:
    """v17 display with bridge step, play theme, group line, and easier/harder fields."""
    schedule = state.get("weekly_schedule", {})
    print_banner("V17 BRIDGE-BASED WEEKLY SCHEDULE")

    if schedule.get("status") == "no_special_support":
        print(schedule.get("summary", "No weekly schedule available."))
        return

    print("Summary:", schedule.get("summary", ""))
    print("Daily time budget:", schedule.get("daily_time_min"))
    print("Bank repeat window:", schedule.get("bank_repeat_window_days", 14), "days")
    print("Feedback options:", schedule.get("feedback_options", ACTIVITY_FEEDBACK_OPTIONS))

    for day_name, day_info in schedule.get("days", {}).items():
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min")
        print("-" * 100)
        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue
        for item in items:
            print(f"- [{item.get('category')}] {item.get('title')} ({item.get('duration_min')} min)")
            print(f"  Type: {item.get('activity_type', '')} | Theme: {item.get('play_theme', '')}")
            print(f"  CDC goal: {item.get('cdc_target_milestone', '')}")
            print(f"  Bridge step: {item.get('bridge_step_title', '')}")
            print(f"  Why: {item.get('why_it_helps', '')}")
            print(f"  Instructions: {item.get('instructions', '')}")
            print(f"  Success: {item.get('success_criteria', '')}")
            print(f"  Easier: {item.get('make_easier', '')}")
            print(f"  Harder: {item.get('make_harder', '')}")
            print(f"  Group play: {item.get('group_play_line', '')}")
            print(f"  Materials: {item.get('materials', '')}")

    bonus = schedule.get("weekly_bonus_activity")
    if bonus:
        print("\n" + "=" * 100)
        print("OPTIONAL WEEKLY BONUS ACTIVITY")
        print("=" * 100)
        print(f"- [{bonus['category']}] {bonus['title']} ({bonus['duration_min']} min)")
        print(f"  Instructions: {bonus['instructions']}")
        print(f"  Why extended: {bonus.get('extended_reason', '')}")


Loaded bridge milestone file: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx
Loaded bridge sheet/table: all_with_bridge_family
Bridge rows: 369 | CDC milestones covered: 163
Bridge steps per milestone distribution: {1: 23, 2: 87, 3: 41, 4: 11, 5: 1}


In [45]:
# ------------------------------------------------------------
# v17.1 Patch — Evidence-targeting, curated question notes, safety, and scheduler fixes
# ------------------------------------------------------------
# This patch is intentionally placed after the v17 Bridge Engine cell so it can
# override only the pieces that needed tightening after a full parent/synthetic run.
# It keeps the v17 architecture the same:
# focused onboarding → CDC target → bridge steps → activity bank → daily selection → feedback/mastery.
#
# What this patch fixes:
# 1) CDC targets now come from actual answered gaps (no/sometimes/with_help), not generic age-band rows.
# 2) Answered-YES milestones are treated as mastered evidence and are not selected as active goals.
# 3) Language targets respect the v16 speech-readiness gate; two-word goals are penalized when single words are not ready.
# 4) Activity cards are forced to stay anchored to the selected bridge step/CDC target.
# 5) Safety constraints are stricter for seizure/Dravet/unstable-walking profiles.
# 6) Weekly scheduling balances selected domains and prevents duplicate activities on the same day.
# 7) Parent question explanations first use table-curated notes, then safer conservative one-line fallbacks.

# Keep references to the immediate pre-patch functions for debugging/comparison.
_v17_prepatch_build_parent_question_explanation = build_parent_question_explanation
_v17_prepatch_select_next_milestones = select_next_milestones
_v17_prepatch_build_safety_profile = build_safety_profile
_v17_prepatch_format_safety_constraints_for_prompt = format_safety_constraints_for_prompt
_v17_prepatch_apply_safety_constraints_to_activities = apply_safety_constraints_to_activities
_v17_prepatch_normalize_v17_activity = _normalize_v17_activity
_v17_prepatch_allocate_weekly_slots = allocate_weekly_slots
_v17_prepatch_build_weekly_schedule = build_weekly_schedule

PARENT_EXPLANATION_COLUMN_CANDIDATES = [
    "parent_explanation",
    "parent_note",
    "question_explanation",
    "question_note",
    "one_line_parent_explanation",
    "parent_friendly_explanation",
    "clarification",
]


def _clean_one_line(text: Any, max_words: int = 24) -> str:
    """Clean a parent-facing line while keeping it short."""
    s = str(text or "").strip()
    s = re.sub(r"\s+", " ", s)
    if not s or s.lower() in {"nan", "none", "null"}:
        return ""
    # Keep only the first sentence-ish unit if it is very long.
    parts = re.split(r"(?<=[.!?])\s+", s)
    if parts and len(s.split()) > max_words:
        s = parts[0]
    words = s.split()
    if len(words) > max_words:
        s = " ".join(words[:max_words]).rstrip(".,;:") + "."
    return s


def _lookup_curated_parent_explanation(milestone: str, category_key: str, subdomain: str = "") -> str:
    """Use a curated one-line explanation from the loaded table when present."""
    try:
        df = cdc_df
    except Exception:
        return ""
    if df is None or getattr(df, "empty", True):
        return ""
    available_cols = [c for c in PARENT_EXPLANATION_COLUMN_CANDIDATES if c in df.columns]
    if not available_cols:
        return ""
    key = _norm_text_key(milestone)
    subset = df[df["milestone"].map(_norm_text_key) == key].copy()
    if "category_key" in subset.columns:
        exact = subset[subset["category_key"] == category_key]
        if not exact.empty:
            subset = exact
    if subdomain and "subdomain" in subset.columns:
        sub = str(subdomain).strip().lower()
        exact_sub = subset[subset["subdomain"].astype(str).str.strip().str.lower() == sub]
        if not exact_sub.empty:
            subset = exact_sub
    if subset.empty:
        return ""
    for _, row in subset.iterrows():
        for col in available_cols:
            line = _clean_one_line(row.get(col, ""), max_words=24)
            if line:
                return line
    return ""


def build_parent_question_explanation(
    milestone: str,
    category_key: str,
    subdomain: str = "",
    max_words: int = 24,
) -> str:
    """v17.1: table-first, conservative one-line parent clarification.

    This avoids the LLM inventing or over-explaining milestone definitions live.
    If Sara/advisors add a parent_explanation column to the bridge/CDC table, that
    curated line is used directly. Otherwise this fallback uses conservative,
    milestone-specific wording.
    """
    curated = _lookup_curated_parent_explanation(milestone, category_key, subdomain)
    if curated:
        return _clean_one_line(curated, max_words=max_words)

    text = str(milestone or "").strip()
    low = text.lower()
    sub = str(subdomain or "").lower()

    explanation = "Count this when you see the skill during normal routines, not only during a perfect test moment."

    # Language and communication — keep definitions precise, especially for unclear speech.
    if re.search(r"\b(first )?name\b", low):
        explanation = "Count this when the child says an understandable version of their name; unclear attempts can be marked sometimes."
    elif re.search(r"\b(three|3|four|4|five|5|\d+)\s+(or more\s+)?words?\b", low):
        explanation = "Words should be used meaningfully and consistently; close approximations can count if familiar adults recognize them."
    elif re.search(r"\b(50|fifty)\s+words?\b", low):
        explanation = "Count words the child uses meaningfully and repeatedly, including clear approximations, signs, or spoken words."
    elif re.search(r"\b(two words|2 words|two-word|2-word|words together|more milk)\b", low):
        explanation = "This means two meaningful words together; repeating one word twice or unclear sounds alone does not count."
    elif re.search(r"\b(sentence|sentences|story|stories|conversation|talks? about)\b", low):
        explanation = "This means using words to share an idea; grammar can be imperfect if the message is understandable."
    elif re.search(r"\bfollows?.*(gesture|words)|directions?.*gesture", low):
        explanation = "This counts when the child responds to both your words and gesture in a familiar everyday situation."
    elif re.search(r"\bfollows?.*one[- ]?step|one[- ]?step direction|give it to me", low):
        explanation = "This means following one simple direction from words alone, without needing you to point or repeat it."
    elif re.search(r"\bfollows?.*two|two[- ]?step|2[- ]?step\b", low):
        explanation = "This means doing both parts of a simple direction without needing each step repeated separately."
    elif re.search(r"\bpoint|points|show|where is|body parts?\b", low):
        explanation = "Pointing, touching, reaching toward, or clearly showing the correct item can count; speech is not required."
    elif re.search(r"\blook|looks|turns head|reacts? to sound|sound\b", low) and category_key == "language_and_communication":
        explanation = "This counts when the child consistently notices or responds to the sound during calm everyday moments."

    # Movement / physical.
    elif re.search(r"\bruns?\b", low):
        explanation = "This means moving faster than walking with a brief lift-off; fast walking alone is not the same."
    elif re.search(r"\bjump|jumps|jumping\b", low):
        explanation = "Both feet should leave the flat floor together, even briefly; supported practice can be marked sometimes."
    elif re.search(r"\bhops?|one foot\b", low):
        explanation = "This means briefly balancing and lifting on one foot safely, with close support nearby if needed."
    elif re.search(r"\bstair|stairs|climb|climbs\b", low):
        explanation = "Count the skill only at the support level described, such as with a rail or adult hand if stated."
    elif re.search(r"\bthrow|throws|catch|catches|kick|kicks\b", low):
        explanation = "The movement should be purposeful, even if aim, strength, or coordination is still developing."
    elif re.search(r"\bturns? book pages?\b", low):
        explanation = "This means turning one page at a time most of the time, not grabbing several pages together."
    elif re.search(r"\bdraw|copy|copies|circle|line|person|shape\b", low):
        explanation = "A recognizable attempt counts; it does not need to be neat, centered, or adult-like."
    elif re.search(r"\bstring|bead|beads|macaroni\b", low):
        explanation = "This requires using both hands together to put large items onto a lace, string, or pipe cleaner."
    elif re.search(r"\bscissor|scissors|button|buttons|zip|zips\b", low):
        explanation = "This counts when the child does meaningful parts of the task, even if it is slow or needs setup."
    elif re.search(r"\bspoon|fork|eat|feeds?\b", low):
        explanation = "This means bringing food toward the mouth with the utensil; spills are expected while learning."
    elif re.search(r"\bdress|clothes|shirt|pants|jacket|wash|toilet|potty\b", low):
        explanation = "This counts when the child completes meaningful parts of the routine, even with normal reminders."

    # Social/emotional and cognitive/adaptive.
    elif re.search(r"\bplay|plays|turn|turns|share|shares|pretend|with other children\b", low):
        explanation = "This means a short natural play moment; adult setup or gentle support is okay."
    elif re.search(r"\bcalm|calms|upset|transition|separates?\b", low) or "emotional" in sub:
        explanation = "This means the child begins to settle with support; they do not need to calm instantly."
    elif re.search(r"\bmatch|sort|same|different|color|shape|number|letter\b", low):
        explanation = "This counts when the child shows the idea through pointing, choosing, placing, or saying it."
    elif re.search(r"\bpretend|make-believe|imaginary\b", low):
        explanation = "This means using objects or actions as if they represent something else during play."
    elif category_key == "language_and_communication":
        explanation = "For communication skills, gestures, signs, sounds, word attempts, or words may count depending on the milestone."
    elif category_key == "movement_and_physical":
        explanation = "Count purposeful, safe movement during normal play, even if coordination is still emerging."
    elif category_key == "social_and_emotional":
        explanation = "This means a real-life interaction or regulation moment, not a perfect or long performance."
    elif category_key == "cognitive":
        explanation = "This means the child understands or participates in the skill during everyday routines or play."

    return _clean_one_line(explanation, max_words=max_words)


# -----------------------------
# Evidence-based target selection
# -----------------------------

def _answer_norm_for_targeting(answer: Dict[str, Any]) -> str:
    return str(answer.get("scoring_norm_answer", answer.get("norm_answer", answer.get("answer", "")))).strip().lower()


def _concern_milestone_relevance_score(state: Dict[str, Any], category_key: str, milestone: str, subdomain: str) -> float:
    """Small deterministic relevance score from concern text to milestone/subdomain."""
    child = state.get("child", {})
    text = f"{child.get('diagnosis', '')} {child.get('concern', '')}".lower()
    m = str(milestone or "").lower()
    s = str(subdomain or "").lower()
    score = 0.0

    if category_key == "movement_and_physical":
        gait_terms = r"(unstable|unsteady|walk|walking|gait|balance|fall|stairs|run|running|jump|gross motor|motor)"
        if re.search(gait_terms, text):
            if "gross_motor" in s or re.search(r"\b(run|runs|walk|walking|stair|stairs|jump|jumps|hop|balance|climb|kick)\b", m):
                score += 24
            if "self_help" in s or re.search(r"\b(spoon|fork|clothes|dress|button|wash|toilet)\b", m):
                score -= 18
            if "fine_motor" in s and not re.search(r"fine motor|hand|grip|crayon|bead|scissor", text):
                score -= 10

    elif category_key == "language_and_communication":
        speech_terms = r"(speech|talk|word|language|communication|regression|not talking|late talk|expressive)"
        if re.search(speech_terms, text):
            if any(x in s for x in ["expressive", "vocal", "speech", "babbling"]):
                score += 24
            if re.search(r"\b(say|says|word|words|sound|sounds|name|talk|tries to say)\b", m):
                score += 20
            if re.search(r"\bfollows?|directions?|point|book|body parts?\b", m):
                score += 6
        if re.search(r"learning|delay|understand|receptive", text) and ("receptive" in s or re.search(r"follow|point|show|where", m)):
            score += 12

    elif category_key == "cognitive":
        if re.search(r"learning|cognitive|adaptive|attention|understand|delay", text):
            score += 18
            if re.search(r"match|sort|solve|imitat|pretend|routine|direction|find|look", m):
                score += 10

    elif category_key == "social_and_emotional":
        if re.search(r"social|emotion|tantrum|regulation|play|peer|transition|attention|autism|adhd", text):
            score += 18
            if re.search(r"play|turn|share|calm|upset|separate|look|smile|respond", m):
                score += 10

    return score


def _language_target_readiness_penalty(state: Dict[str, Any], milestone: str) -> float:
    """Penalize phrase/sentence CDC targets when the child lacks single-word readiness."""
    m = str(milestone or "")
    if not LANGUAGE_WORD_COMBINATION_MILESTONE_PATTERN.search(m):
        return 0.0
    try:
        readiness = compute_language_activity_readiness(state)
        level = readiness.get("level", "")
    except Exception:
        level = ""
    if level in {"pre_or_early_single_words", "single_word_building"}:
        return -35.0
    if level == "early_phrase_modeling_ok":
        return -12.0
    return 0.0


def _target_score_from_answer(state: Dict[str, Any], category_key: str, answer: Dict[str, Any], dev_age: int, chrono: int) -> float:
    norm = _answer_norm_for_targeting(answer)
    if norm == "yes":
        return -9999.0

    month = int(answer.get("months", dev_age) or dev_age)
    milestone = str(answer.get("milestone", ""))
    subdomain = str(answer.get("subdomain", "unspecified"))

    base = {
        "no": 55.0,
        "sometimes": 45.0,
        "with_help": 43.0,
        "not_sure": 25.0,
    }.get(norm, 20.0)

    # Prefer targets around the child's demonstrated/developmental level, not far-future rows.
    age_distance = abs(month - int(dev_age))
    age_score = max(0.0, 18.0 - 1.5 * age_distance)
    if month < int(dev_age) - 9:
        age_score -= 8.0
    if month > int(chrono) + 3:
        age_score -= 25.0

    concern_profile = ensure_concern_profile(state)
    sub_weight = float(concern_profile.get("subdomain_weights", {}).get(subdomain, 0.0))
    q_score = float(answer.get("selection_score", 0.0) or 0.0)

    score = base + age_score + 22.0 * sub_weight + 8.0 * q_score
    score += _concern_milestone_relevance_score(state, category_key, milestone, subdomain)
    if category_key == "language_and_communication":
        score += _language_target_readiness_penalty(state, milestone)
    return float(score)


def _answered_yes_keys(state: Dict[str, Any], category_key: str) -> set:
    yes_keys = set()
    for a in state.get("qna", {}).get(category_key, []) or []:
        if _answer_norm_for_targeting(a) == "yes":
            yes_keys.add((int(a.get("months", 0) or 0), _norm_text_key(a.get("milestone", ""))))
    return yes_keys


def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = 6) -> Dict[str, Any]:
    """v17.1: Select active CDC goals from actual parent answers first.

    Rules:
    - answered YES = mastered evidence, never active target
    - answered NO / sometimes / with_help = strongest candidates
    - concern-router subdomain and parent concern text raise relevance
    - phrase/sentence targets are penalized when v16 speech readiness says not ready
    - only use generic age-band fallback when no answered gap exists
    """
    child = state["child"]
    dev_age = get_effective_dev_age(state, category_key)
    soft_floor_active = is_family_guidance_category(state, category_key)

    if dev_age is None and not soft_floor_active:
        raise ValueError(f"No developmental age found for {category_key}. Run Q&A first.")

    if no_special_support_needed(state, category_key) and not soft_floor_active:
        return {
            "status": "no_special_support",
            "message": f"We do not think {child['name']} has a meaningful delay in {DOMAIN_CONFIG[category_key]['display']} and may not need special support in this category right now.",
            "milestones": [],
        }

    chrono = min(int(child["chronological_months"]), 60)
    dev = int(chrono if soft_floor_active else dev_age)

    # First choice: use actual answered gaps.
    answers = [
        a for a in state.get("qna", {}).get(category_key, []) or []
        if a.get("answer_status", "ok") != "api_error"
    ]
    candidates = []
    seen = set()
    for a in answers:
        norm = _answer_norm_for_targeting(a)
        if norm == "yes":
            continue
        milestone = str(a.get("milestone", "")).strip()
        if not milestone:
            continue
        month = int(a.get("months", dev) or dev)
        key = (month, _norm_text_key(milestone))
        if key in seen:
            continue
        seen.add(key)
        cand = {
            "months": month,
            "milestone": milestone,
            "subdomain": str(a.get("subdomain", "unspecified")),
            "evidence_answer": norm,
            "selection_reason": "answered_gap",
            "target_score": round(_target_score_from_answer(state, category_key, a, dev, chrono), 3),
        }
        candidates.append(cand)

    candidates = [c for c in candidates if c["target_score"] > -1000]
    candidates = sorted(candidates, key=lambda c: (c["target_score"], -abs(c["months"] - dev), -c["months"]), reverse=True)
    if candidates:
        return {
            "status": "success",
            "milestones": candidates[:max_milestones],
            "mode": "evidence_based_answered_gaps",
            "selection_rule": "answered no/sometimes/with_help + concern subdomain + developmental age; answered yes excluded",
        }

    # Fallback: choose not-yet-answered rows near the developmental level, excluding answered-YES milestones.
    yes_keys = _answered_yes_keys(state, category_key)
    if soft_floor_active:
        min_m = max(2, chrono - 3)
        max_m = min(60, chrono + 3)
    else:
        min_m = max(2, dev)
        max_m = min(60, dev + 12)

    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)
    if subset.empty:
        return {"status": "success", "milestones": [], "mode": "fallback_no_candidates"}

    rows = []
    concern_profile = ensure_concern_profile(state)
    for _, row in subset.iterrows():
        month = int(row["months"])
        milestone = str(row["milestone"]).strip()
        key = (month, _norm_text_key(milestone))
        if key in yes_keys:
            continue
        subdomain = str(row.get("subdomain", "unspecified"))
        age_score = max(0.0, 15.0 - abs(month - dev))
        sub_weight = float(concern_profile.get("subdomain_weights", {}).get(subdomain, 0.0))
        score = age_score + 22.0 * sub_weight + _concern_milestone_relevance_score(state, category_key, milestone, subdomain)
        if category_key == "language_and_communication":
            score += _language_target_readiness_penalty(state, milestone)
        rows.append({
            "months": month,
            "milestone": milestone,
            "subdomain": subdomain,
            "selection_reason": "unanswered_fallback_near_dev_age",
            "target_score": round(score, 3),
        })

    rows = sorted(rows, key=lambda c: (c["target_score"], -abs(c["months"] - dev), -c["months"]), reverse=True)
    return {
        "status": "success",
        "milestones": rows[:max_milestones],
        "mode": "unanswered_fallback",
        "selection_rule": "fallback rows near developmental level, excluding answered-yes milestones",
    }


# -----------------------------
# Medical/safety-sensitive patch
# -----------------------------

V17_HIGH_RISK_MOVEMENT_PATTERN = re.compile(
    r"\b(race|relay|obstacle|obstacles|trampoline|jump from|climb|climbing|run fast|fast running|"
    r"balance beam|playground|stairs challenge|high surface|uneven surface|hopscotch)\b",
    flags=re.IGNORECASE,
)


def _has_seizure_or_high_mobility_risk(state_or_child: Dict[str, Any]) -> bool:
    child = state_or_child.get("child", state_or_child)
    text = f"{child.get('diagnosis', '')} {child.get('condition', '')} {child.get('concern', '')}".lower()
    return bool(re.search(r"\b(dravet|seizure|epilep|unstable|unsteady|ataxia|fall|falls|balance|gait)\b", text))


def build_safety_profile(child: Dict[str, Any]) -> Dict[str, Any]:
    """v17.1: extend existing safety profile for seizure/Dravet and unstable gait."""
    profile = _v17_prepatch_build_safety_profile(child)
    text = profile.get("combined_text", f"{child.get('diagnosis', '')} {child.get('concern', '')}".lower())
    seizure_or_dravet = bool(re.search(r"\b(dravet|seizure|epilep|drop attack)\b", text))
    unstable_gait = bool(re.search(r"\b(unstable|unsteady|ataxia|gait|balance|falls?|walking unstable)\b", text))

    def add_unique(key: str, items: List[str]):
        existing = list(profile.get(key, []) or [])
        for item in items:
            if item not in existing:
                existing.append(item)
        profile[key] = existing

    if seizure_or_dravet:
        profile.setdefault("risk_scores", {})["seizure_or_medical_monitoring"] = max(
            float(profile.get("risk_scores", {}).get("seizure_or_medical_monitoring", 0.0)), 0.7
        )
        add_unique("constraints", [
            "Seizure/medical monitoring concern: keep activities calm, closely supervised, low-height, and stop immediately if the child seems unwell.",
        ])
        add_unique("hard_avoid", [
            "racing, obstacle courses, or fast movement games",
            "climbing, heights, stairs-as-play, trampolines, or jumping from surfaces",
            "water play, flashing-light games, or unsupervised activity",
        ])
        add_unique("preferred_adaptations", [
            "use a clear flat floor, seated/ground-level play, and close caregiver support",
            "choose slow supported movement, short turns, and frequent rest breaks",
            "follow the family’s medical/seizure safety plan and therapy guidance",
        ])

    if unstable_gait:
        profile.setdefault("risk_scores", {})["falls_balance_gait"] = max(
            float(profile.get("risk_scores", {}).get("falls_balance_gait", 0.0)), 0.7
        )
        add_unique("hard_avoid", [
            "unsupported balance challenges, obstacle courses, or speed-based games",
        ])
        add_unique("preferred_adaptations", [
            "prefer hand-held, wall-supported, seated, or floor-level versions of movement games",
        ])

    return profile


def format_safety_constraints_for_prompt(profile: Dict[str, Any]) -> str:
    """v17.1: same information as before, plus clearer hard rules for high-risk profiles."""
    base = _v17_prepatch_format_safety_constraints_for_prompt(profile)
    high_medical = float(profile.get("risk_scores", {}).get("seizure_or_medical_monitoring", 0.0)) >= 0.35
    high_gait = float(profile.get("risk_scores", {}).get("falls_balance_gait", 0.0)) >= 0.35
    if high_medical or high_gait:
        extra = """
Extra v17.1 safety rule:
- For seizure/Dravet/unstable-walking profiles, do not use races, relay races, obstacle courses, climbing, heights, fast running, unsupported balance, or jumping from surfaces.
- Prefer flat-floor, seated, hand-held, wall-supported, or caregiver-supported practice.
- Movement activities should be slow, short, closely supervised, and stopped before fatigue or distress.
""".strip()
        return base + "\n" + extra
    return base


def _safe_supported_activity_from_bridge(category_key: str, activity: Dict[str, Any]) -> Dict[str, Any]:
    a = dict(activity)
    bridge_title = str(a.get("bridge_step_title") or a.get("target_skill") or "current bridge step")
    a["title"] = f"Supported {bridge_title} Game"
    a["play_theme"] = "supported movement game"
    a["theme_variations"] = ["floor sticker game", "caregiver hand-support game", "toy path game"]
    a["instructions"] = (
        "Use a clear, flat floor with close adult support. Practice only the current bridge step slowly, such as a small weight shift, supported step, or gentle reach. "
        "Avoid racing, obstacles, climbing, fast movement, jumping from surfaces, or unsupported balance challenges. Stop before fatigue or distress."
    )
    a["how_to_do_it"] = [
        "Set up on a clear, flat floor with a stable wall, couch, or caregiver hand nearby.",
        f"Practice the bridge step slowly: {bridge_title}.",
        "Accept a small safe attempt as success and stop while it is still easy.",
    ]
    a["materials"] = "floor stickers or tape, favorite toy, stable wall/couch or caregiver hand support"
    a["success_criteria"] = a.get("success_criteria") or "Child participates safely with support and low frustration."
    a["make_easier"] = "Use seated play, two-hand support, or a smaller movement."
    a["make_harder"] = "Only reduce support slightly if easy; do not add speed, obstacles, heights, or jumping from surfaces."
    a["group_play_line"] = "A sibling can model one slow turn while an adult stays close to help."
    a["duration_min"] = min(int(a.get("duration_min", 5) or 5), 5)
    a["safety_note"] = "High-safety version for seizure/Dravet/unstable gait concerns."
    return a


def apply_safety_constraints_to_activities(
    state: Dict[str, Any],
    category_key: str,
    activities: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """v17.1: deterministic safety pass with stronger filtering for Dravet/seizure/unstable gait."""
    # First run the existing broad safety pass.
    safe = _v17_prepatch_apply_safety_constraints_to_activities(state, category_key, activities)
    profile = ensure_safety_profile(state)
    high_medical = float(profile.get("risk_scores", {}).get("seizure_or_medical_monitoring", 0.0)) >= 0.35
    high_gait = float(profile.get("risk_scores", {}).get("falls_balance_gait", 0.0)) >= 0.35

    if not (high_medical or high_gait):
        return safe

    out = []
    for activity in safe:
        a = dict(activity)
        text = " ".join([
            str(a.get("title", "")),
            str(a.get("instructions", "")),
            " ".join(a.get("how_to_do_it", [])) if isinstance(a.get("how_to_do_it"), list) else str(a.get("how_to_do_it", "")),
            str(a.get("make_harder", "")),
        ])
        if category_key == "movement_and_physical" and V17_HIGH_RISK_MOVEMENT_PATTERN.search(text):
            a = _safe_supported_activity_from_bridge(category_key, a)
        else:
            if category_key == "movement_and_physical":
                instr = str(a.get("instructions", "")).rstrip()
                if "clear, flat floor" not in instr.lower():
                    instr += " Use a clear, flat floor with close adult support; avoid speed, obstacles, heights, or unsupported balance."
                a["instructions"] = instr
                a["duration_min"] = min(int(a.get("duration_min", 5) or 5), 5)
                harder = str(a.get("make_harder", "")).strip()
                if V17_HIGH_RISK_MOVEMENT_PATTERN.search(harder):
                    a["make_harder"] = "Reduce support only slightly if safe and easy; do not add speed, obstacles, heights, or jumping from surfaces."
            if high_medical:
                a["safety_note"] = "Keep closely supervised and follow the family’s medical/seizure safety plan."
        out.append(a)
    return out


# -----------------------------
# Activity anchoring + normalization patch
# -----------------------------

def _normalize_v17_activity(
    activity: Dict[str, Any],
    category_key: str,
    bridge: Dict[str, Any],
    idx: int,
    planning_tier: str,
) -> Dict[str, Any]:
    """v17.1: normalize LLM activity and force it to stay anchored to the active bridge.

    The LLM may write a nice activity but drift to the wrong CDC target. This function
    treats the bridge table as source of truth.
    """
    a = dict(activity or {})
    fallback = _fallback_v17_activity(category_key, bridge, idx, 8, planning_tier)
    for k, v in fallback.items():
        if k not in a or a.get(k) in [None, "", []]:
            a[k] = v

    # Force bank composition by index: 4 core, 2 easier, 2 harder for an 8-item bank.
    a["activity_type"] = _activity_type_for_index(idx, 8)

    # Force bridge/CDC anchoring from the reference table, not the LLM text.
    a["cdc_target_milestone"] = str(bridge.get("cdc_target_milestone", fallback.get("cdc_target_milestone", "")))
    a["bridge_id"] = str(bridge.get("bridge_id", fallback.get("bridge_id", "")))
    a["bridge_step_number"] = int(bridge.get("bridge_step_number", fallback.get("bridge_step_number", 1)) or 1)
    a["bridge_step_title"] = str(bridge.get("bridge_step_title", fallback.get("bridge_step_title", "Bridge step")))
    a["target_skill"] = str(bridge.get("activity_focus") or bridge.get("bridge_step_title") or a.get("target_skill") or "bridge prerequisite")
    a["what_to_avoid"] = str(bridge.get("what_to_avoid") or a.get("what_to_avoid") or "Do not require the final CDC milestone before bridge steps are ready.")

    # Duration guardrail.
    try:
        duration = int(a.get("duration_min", 5))
    except Exception:
        duration = 5
    if not a.get("is_extended_activity", False):
        duration = min(max(duration, 3), 7)
    a["duration_min"] = duration

    # Ensure how_to_do_it and instructions are aligned.
    if isinstance(a.get("how_to_do_it"), str):
        a["how_to_do_it"] = [x.strip() for x in re.split(r"\n+|\s*;\s*", a["how_to_do_it"]) if x.strip()]
    if not isinstance(a.get("how_to_do_it"), list) or not a["how_to_do_it"]:
        a["how_to_do_it"] = fallback["how_to_do_it"]
    if not str(a.get("instructions", "")).strip():
        a["instructions"] = " ".join(str(x) for x in a["how_to_do_it"][:4])

    # Short group line.
    group_line = _clean_one_line(a.get("group_play_line", ""), max_words=28)
    a["group_play_line"] = group_line or fallback["group_play_line"]

    # Theme cleanup.
    if isinstance(a.get("theme_variations"), str):
        a["theme_variations"] = [x.strip() for x in re.split(r"[,;|]", a["theme_variations"]) if x.strip()]
    if not isinstance(a.get("theme_variations"), list) or not a["theme_variations"]:
        a["theme_variations"] = _themes_from_bridge(bridge)[:3]
    a["play_theme"] = str(a.get("play_theme") or a["theme_variations"][0] or fallback.get("play_theme", "playful home routine"))

    a["goal"] = planning_tier
    a["category"] = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key)
    a["level"] = "bridge_step"
    return a


# -----------------------------
# Balanced allocation and anti-duplicate scheduler
# -----------------------------

def allocate_weekly_slots(state: Dict[str, Any], cycle_days: int = 7) -> Dict[str, Any]:
    """v17.1: balanced allocation for the displayed 7-day schedule.

    For 2 selected focus domains and short daily time, parents expect one activity per
    domain most days, not 5 days of one domain and only 2 days of the other.
    """
    child = state["child"]
    if "daily_time_min" not in child:
        raise ValueError("Missing daily_time_min in state['child'].")

    daily_time_min = int(child["daily_time_min"])
    weekly_minutes = daily_time_min * int(cycle_days)

    candidate_categories = state.get("planned_categories") or state.get("selected_categories") or list(DOMAIN_CONFIG.keys())
    supported_categories = []
    gap_by_category = {}
    support_score_by_category = {}
    chrono = min(int(child.get("chronological_months", 60)), 60)

    for category_key in candidate_categories:
        if category_key not in DOMAIN_CONFIG:
            continue
        tier = get_support_tier(state, category_key)
        if tier == "no_special_support":
            continue
        if get_effective_dev_age(state, category_key) is None and not is_family_guidance_category(state, category_key):
            continue
        supported_categories.append(category_key)
        dev_age = get_effective_dev_age(state, category_key) or chrono
        gap = max(0, chrono - int(dev_age))
        gap_by_category[category_key] = gap
        try:
            support_score_by_category[category_key] = float(compute_support_metrics(state, category_key).get("support_score", gap))
        except Exception:
            support_score_by_category[category_key] = float(gap)

    if not supported_categories:
        allocation = {
            "daily_time_min": daily_time_min,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [],
            "target_minutes_by_category": {},
            "gap_by_category": {},
            "weight_by_category": {},
            "planning_mode": "v17_1_no_special_support",
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    # Balanced allocation, with only a small adjustment for priority.
    n = len(supported_categories)
    base = weekly_minutes // n
    target_minutes = {k: base for k in supported_categories}
    remaining = weekly_minutes - base * n
    priority_order = sorted(supported_categories, key=lambda k: support_score_by_category.get(k, 0.0), reverse=True)
    for k in priority_order[:remaining]:
        target_minutes[k] += 1

    allocation = {
        "daily_time_min": daily_time_min,
        "weekly_minutes": weekly_minutes,
        "supported_categories": supported_categories,
        "target_minutes_by_category": target_minutes,
        "gap_by_category": gap_by_category,
        "weight_by_category": support_score_by_category,
        "planning_mode": "v17_1_balanced_bridge_activity_bank",
    }
    state["weekly_slot_allocation"] = allocation
    return allocation


def _activity_title_key(activity: Dict[str, Any]) -> str:
    return _norm_text_key(activity.get("title", ""))


def _choose_activity_for_category_day_v17_1(
    state: Dict[str, Any],
    category_key: str,
    day_index: int,
    remaining_minutes: int,
    day_used_titles: set,
    weekly_title_counts: Dict[str, int],
    recent_titles_by_category: Dict[str, List[str]],
) -> Optional[Dict[str, Any]]:
    bank = state.get("activity_banks", {}).get(category_key, {})
    raw_activities = [a for a in bank.get("activities", []) if not a.get("is_extended_activity", False)]
    raw_activities = [a for a in raw_activities if not is_context_dependent_bonus_activity(a)]
    raw_activities = [a for a in raw_activities if int(a.get("duration_min", 5)) <= remaining_minutes]
    if not raw_activities:
        return None

    # If there is no feedback, prefer core activities only. Easier/harder are reserved for flags.
    feedback_exists = bool(state.get("activity_feedback_log"))
    core_activities = [a for a in raw_activities if str(a.get("activity_type", "core")) == "core"]
    activities = core_activities if core_activities and not feedback_exists else raw_activities

    # Hard block: no exact title duplicate in the same day if alternatives exist.
    non_duplicate = [a for a in activities if _activity_title_key(a) not in day_used_titles]
    if non_duplicate:
        activities = non_duplicate

    scored = []
    recent_titles = set(recent_titles_by_category.get(category_key, [])[-2:])
    for idx, a in enumerate(activities):
        title_key = _activity_title_key(a)
        bridge_id = str(a.get("bridge_id", ""))
        counts = _activity_feedback_counts(state, bridge_id=bridge_id)
        preferred_type = _preferred_activity_type_from_feedback(counts) if feedback_exists else "core"
        activity_type = str(a.get("activity_type", "core"))

        score = 0.0
        if activity_type == preferred_type:
            score += 30.0
        if activity_type == "core":
            score += 8.0
        if activity_type in {"easier_backup", "harder_stretch"} and not feedback_exists:
            score -= 25.0

        # Rotate titles and bridge steps across the week.
        score -= 12.0 * weekly_title_counts.get(title_key, 0)
        if title_key in recent_titles:
            score -= 18.0
        score -= abs((idx % max(1, len(activities))) - ((day_index - 1) % max(1, len(activities)))) * 0.2

        # Engagement feedback changes theme/activity, not the developmental skill.
        if counts.get("enjoyed_it", 0) > counts.get("didnt_like_it", 0):
            score += 4.0
        if counts.get("didnt_like_it", 0) >= 1 or counts.get("resisted_it", 0) >= 1:
            score -= 8.0

        scored.append((score, idx, a))

    if not scored:
        return None
    scored = sorted(scored, key=lambda x: (x[0], -x[1]), reverse=True)
    return dict(scored[0][2])


def build_weekly_schedule(state: Dict[str, Any], cycle_days: int = 7) -> Dict[str, Any]:
    """v17.1: balanced daily schedule with anti-duplicate and theme-rotation rules."""
    allocate_weekly_slots(state, cycle_days=cycle_days)
    allocation = state["weekly_slot_allocation"]
    daily_time_min = int(allocation["daily_time_min"])
    target_minutes_by_category = allocation.get("target_minutes_by_category", {})

    days = {f"day_{i}": {"items": [], "total_minutes": 0} for i in range(1, cycle_days + 1)}
    if not target_minutes_by_category:
        state["weekly_schedule"] = {
            "status": "no_special_support",
            "summary": "No categories need a scheduled weekly activity plan right now.",
            "daily_time_min": daily_time_min,
            "days": days,
        }
        return state["weekly_schedule"]

    categories = list(target_minutes_by_category.keys())
    assigned_minutes_by_category = {k: 0 for k in categories}
    weekly_title_counts = {}
    recent_titles_by_category = {k: [] for k in categories}

    for day_num, day_name in enumerate(days.keys(), start=1):
        day_used_titles = set()
        # Each day, start with the category most behind its weekly target. This keeps two-domain
        # plans balanced instead of using one domain for the first half of the week.
        cats_today = sorted(
            categories,
            key=lambda k: (target_minutes_by_category.get(k, 0) - assigned_minutes_by_category.get(k, 0), target_minutes_by_category.get(k, 0)),
            reverse=True,
        )

        if len(cats_today) > 1:
            # Short daily plan: one activity per category if time allows.
            for category_key in cats_today:
                remaining = daily_time_min - days[day_name]["total_minutes"]
                if remaining < 3:
                    break
                activity = _choose_activity_for_category_day_v17_1(
                    state=state,
                    category_key=category_key,
                    day_index=day_num,
                    remaining_minutes=remaining,
                    day_used_titles=day_used_titles,
                    weekly_title_counts=weekly_title_counts,
                    recent_titles_by_category=recent_titles_by_category,
                )
                if activity is None:
                    continue
                item = _schedule_item_from_activity(category_key, activity)
                title_key = _activity_title_key(item)
                if title_key in day_used_titles:
                    continue
                days[day_name]["items"].append(item)
                days[day_name]["total_minutes"] += int(item.get("duration_min", 5))
                assigned_minutes_by_category[category_key] += int(item.get("duration_min", 5))
                day_used_titles.add(title_key)
                weekly_title_counts[title_key] = weekly_title_counts.get(title_key, 0) + 1
                recent_titles_by_category.setdefault(category_key, []).append(title_key)
                if days[day_name]["total_minutes"] >= daily_time_min:
                    break
        else:
            # One focus domain: allow more than one short activity, but avoid duplicate titles.
            category_key = cats_today[0]
            while daily_time_min - days[day_name]["total_minutes"] >= 3 and len(days[day_name]["items"]) < 2:
                activity = _choose_activity_for_category_day_v17_1(
                    state=state,
                    category_key=category_key,
                    day_index=day_num + len(days[day_name]["items"]),
                    remaining_minutes=daily_time_min - days[day_name]["total_minutes"],
                    day_used_titles=day_used_titles,
                    weekly_title_counts=weekly_title_counts,
                    recent_titles_by_category=recent_titles_by_category,
                )
                if activity is None:
                    break
                item = _schedule_item_from_activity(category_key, activity)
                title_key = _activity_title_key(item)
                if title_key in day_used_titles:
                    break
                days[day_name]["items"].append(item)
                days[day_name]["total_minutes"] += int(item.get("duration_min", 5))
                assigned_minutes_by_category[category_key] += int(item.get("duration_min", 5))
                day_used_titles.add(title_key)
                weekly_title_counts[title_key] = weekly_title_counts.get(title_key, 0) + 1
                recent_titles_by_category.setdefault(category_key, []).append(title_key)

    weekly_bonus = pick_weekly_bonus_extended_activity(state)
    schedule = {
        "status": "success",
        "summary": (
            "v17.1 bridge-based weekly schedule. Activities repeat for 1–2 weeks, but the daily view balances domains, "
            "avoids duplicate activities on the same day, varies play themes, and uses feedback to simplify or level up."
        ),
        "daily_time_min": daily_time_min,
        "cycle_days_displayed": cycle_days,
        "bank_repeat_window_days": 14,
        "target_minutes_by_category": target_minutes_by_category,
        "assigned_minutes_by_category": assigned_minutes_by_category,
        "feedback_options": ACTIVITY_FEEDBACK_OPTIONS,
        "days": days,
        "weekly_bonus_activity": weekly_bonus,
        "scheduler_rules": [
            "No same activity title twice on the same day.",
            "For two focus domains and 10 minutes/day, try one short activity per domain each day.",
            "Without feedback flags, show core activities first; reserve easier/harder activities for parent flags.",
            "Repeat bridge skills across the 1–2 week cycle, but rotate play themes and titles.",
        ],
    }
    state["weekly_schedule"] = schedule
    return schedule

print("Loaded v17.1 patch: evidence-based targets, curated parent notes, stricter safety, and balanced anti-duplicate scheduler.")


Loaded v17.1 patch: evidence-based targets, curated parent notes, stricter safety, and balanced anti-duplicate scheduler.


In [46]:
# ------------------------------------------------------------
# Legacy v18 stable patch — table-first bridge helpers used by v21
# ------------------------------------------------------------
# This cell replaces the earlier fragile v18 override. It is deliberately split
# into small, boring, table-first helper functions so a syntax issue cannot hide
# the fact that v18 did not load.
#
# Core corrections in this version:
# 1) Uses Sara's final cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx / all_with_bridge_family sheet as
#    the source of truth for CDC rows, parent explanations, and bridge chains.
# 2) Parent explanations come from the table only. No LLM/fallback generic notes.
# 3) Weekly planning creates a multi-milestone coverage set per selected domain.
# 4) Activities are generated for the active bridge step, not directly for the final CDC milestone.
# 5) The prompt and post-processing include allowed themes and what-to-avoid constraints.
# 6) The schedule header and summary now correctly identify v18.

import math
from collections import defaultdict, Counter

V18_VERSION = "v18_table_first_multimilestone_bridge_chain"
V18_BRIDGE_SHEET_NAME = "all_with_bridge_family"
V18_PREFERRED_BRIDGE_FILENAMES = [
    "cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx",
    "cdc_milestones_with_bridges_family_cleaned.xlsx",
    "cdc_milestones_with_bridges_family.xlsx",
    "cdc_milestones_with_bridges_previous(1).xlsx",
    "cdc_milestones_with_bridges_previous.xlsx",
    "cdc_milestones_with_bridges.xlsx",
    "cdc_milestones_with_bridges.numbers",
]
V18_MIN_MILESTONES_PER_DOMAIN = 3
V18_MAX_MILESTONES_PER_DOMAIN = 5
V18_DEFAULT_CYCLE_DAYS = 7

V18_FALLBACK_PARENT_NOTE = "Explanation not yet reviewed."
V18_ACTIVITY_FEEDBACK_OPTIONS = {
    "difficulty": ["too_hard", "just_right", "too_easy"],
    "performance": ["done_independently", "done_with_help", "couldnt_do_it"],
    "engagement": ["enjoyed_it", "resisted_it", "didnt_like_it"],
}


def _v18_norm(x: Any) -> str:
    s = str(x or "").strip().lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def _v18_clean_col(col: Any, position: int = 0) -> str:
    raw = str(col or "").strip().lower()
    raw = raw.replace("/", "_").replace("&", "and")
    raw = re.sub(r"[^a-z0-9]+", "_", raw)
    raw = re.sub(r"_+", "_", raw).strip("_")
    return raw or f"unnamed_{position}"


def _v18_category_to_key(category: Any) -> str:
    raw = str(category or "").strip().lower().replace("/", " ").replace("&", " and ")
    raw = re.sub(r"\s+", " ", raw).strip()
    if "language" in raw or "communication" in raw:
        return "language_and_communication"
    if "movement" in raw or "physical" in raw or "motor" in raw:
        return "movement_and_physical"
    if "social" in raw or "emotional" in raw:
        return "social_and_emotional"
    if "cognitive" in raw or "adaptive" in raw:
        return "cognitive"
    if raw in globals().get("ALIAS_TO_CATEGORY", {}):
        return globals()["ALIAS_TO_CATEGORY"][raw]
    return raw.replace(" ", "_")


def _v18_search_roots() -> List[Path]:
    roots = []
    for candidate in [globals().get("PROJECT_ROOT", None), Path.cwd(), Path.cwd().parent, Path("/mnt/data")]:
        if candidate is None:
            continue
        p = Path(candidate)
        if p not in roots:
            roots.append(p)
    return roots


def _v18_find_bridge_file(path: Optional[str] = None) -> Optional[Path]:
    if path:
        p = Path(path).expanduser()
        if p.exists():
            return p
    for root in _v18_search_roots():
        for name in V18_PREFERRED_BRIDGE_FILENAMES:
            for p in [root / name, root / "notebooks" / name, root / "mvp" / name, root / "mvp" / "notebooks" / name]:
                if p.exists():
                    return p
    return None


def _v18_read_excel_table(path: Path, sheet_name: str = V18_BRIDGE_SHEET_NAME) -> Tuple[pd.DataFrame, str]:
    """Read Sara's Numbers-exported Excel sheet, including cases where the first row is a table title."""
    xls = pd.ExcelFile(path)
    selected_sheet = sheet_name if sheet_name in xls.sheet_names else xls.sheet_names[0]
    raw = pd.read_excel(path, sheet_name=selected_sheet, header=None)
    header_row = 0
    for idx in range(min(len(raw), 10)):
        vals = [_v18_clean_col(v, j) for j, v in enumerate(raw.iloc[idx].tolist())]
        has_months = "months" in vals
        has_milestone = "milestone" in vals or "milestone_" in vals
        has_bridge = "bridge_step" in vals
        if has_months and has_milestone and has_bridge:
            header_row = idx
            break
    headers = [_v18_clean_col(v, j) for j, v in enumerate(raw.iloc[header_row].tolist())]
    df = raw.iloc[header_row + 1:].copy().reset_index(drop=True)
    df.columns = headers
    df = df.loc[:, [c for c in df.columns if not c.startswith("unnamed_")]]
    return df, selected_sheet


def _v18_read_bridge_file(path: Path) -> Tuple[pd.DataFrame, str]:
    suffix = path.suffix.lower()
    if suffix == ".numbers":
        # Prefer numbers-parser if the user keeps the source as a Numbers file.
        try:
            from numbers_parser import Document
            doc = Document(str(path))
            sheets = doc.sheets
            for sheet in sheets:
                tables = sheet.tables
                for table in tables:
                    rows = table.rows(values_only=True)
                    raw = pd.DataFrame(rows)
                    if raw.empty:
                        continue
                    header_row = None
                    for idx in range(min(len(raw), 10)):
                        vals = [_v18_clean_col(v, j) for j, v in enumerate(raw.iloc[idx].tolist())]
                        if "months" in vals and "bridge_step" in vals:
                            header_row = idx
                            break
                    if header_row is None:
                        continue
                    headers = [_v18_clean_col(v, j) for j, v in enumerate(raw.iloc[header_row].tolist())]
                    df = raw.iloc[header_row + 1:].copy().reset_index(drop=True)
                    df.columns = headers
                    df = df.loc[:, [c for c in df.columns if not c.startswith("unnamed_")]]
                    return df, f"{sheet.name}/{table.name}"
        except Exception as exc:
            raise ValueError(f"Could not read Numbers bridge file: {exc}")
    return _v18_read_excel_table(path, V18_BRIDGE_SHEET_NAME)


def _v18_first_existing(row: pd.Series, names: List[str], default: Any = "") -> Any:
    for name in names:
        if name in row.index:
            val = row.get(name)
            if pd.notna(val) and str(val).strip() != "":
                return val
    return default


def _v18_normalize_bridge_df(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()
    df.columns = [_v18_clean_col(c, i) for i, c in enumerate(df.columns)]

    # Normalize common column variants.
    alias_map = {
        "category_": "category",
        "milestone_": "milestone",
        "cdc_milestone": "milestone",
        "target_milestone": "milestone",
        "age_months": "months",
        "month": "months",
        "bridge": "bridge_step",
        "bridge_milestone": "bridge_step",
        "previous_step": "previous_bridge_step",
        "previous_prerequisite": "previous_bridge_step",
        "parent_note": "parent_explanation",
        "question_explanation": "parent_explanation",
        "themes": "allowed_themes",
        "play_themes": "allowed_themes",
        "avoid": "what_to_avoid",
        "avoid_rules": "what_to_avoid",
    }
    df = df.rename(columns={c: alias_map.get(c, c) for c in df.columns})

    required = ["months", "category", "subdomain", "milestone", "parent_explanation", "bridge_step_number", "bridge_step", "previous_bridge_step", "previous_anchor_age"]
    for col in required:
        if col not in df.columns:
            df[col] = ""

    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df = df[df["months"].notna()].copy()
    df["months"] = df["months"].astype(int)
    df["category"] = df["category"].astype(str).str.strip()
    df["category_key"] = df["category"].map(_v18_category_to_key)
    df["subdomain"] = df["subdomain"].fillna("unspecified").astype(str).str.strip().replace("", "unspecified")
    df["milestone"] = df["milestone"].fillna("").astype(str).str.strip()
    df["parent_explanation"] = df["parent_explanation"].fillna("").astype(str).str.strip()
    df["bridge_step"] = df["bridge_step"].fillna("").astype(str).str.strip()
    df["previous_bridge_step"] = df["previous_bridge_step"].fillna("").astype(str).str.strip()
    df["previous_anchor_age"] = df["previous_anchor_age"].fillna("").astype(str).str.strip()
    df["bridge_step_number"] = pd.to_numeric(df["bridge_step_number"], errors="coerce").fillna(1).astype(int)

    if "allowed_themes" not in df.columns:
        df["allowed_themes"] = ""
    if "what_to_avoid" not in df.columns:
        df["what_to_avoid"] = ""
    df["allowed_themes"] = df["allowed_themes"].fillna("").astype(str).str.strip()
    df["what_to_avoid"] = df["what_to_avoid"].fillna("").astype(str).str.strip()

    df = df[df["milestone"].str.len() > 0].copy()
    df["milestone_key"] = df["milestone"].map(_v18_norm)
    df["bridge_key"] = df["bridge_step"].map(_v18_norm)
    df = df.sort_values(["months", "category_key", "subdomain", "milestone_key", "bridge_step_number"]).reset_index(drop=True)
    return df


def _v18_build_cdc_df_from_bridge(bridge_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    grouped = bridge_df.groupby(["months", "category_key", "category", "subdomain", "milestone_key", "milestone"], dropna=False)
    for (months, category_key, category, subdomain, milestone_key, milestone), g in grouped:
        parent_explanations = [str(x).strip() for x in g["parent_explanation"].tolist() if str(x).strip()]
        question_id = f"{category_key}_{int(months)}_{abs(hash((category_key, int(months), milestone_key))) % 100000}"
        rows.append({
            "months": int(months),
            "category": category,
            "category_key": category_key,
            "subdomain": subdomain or "unspecified",
            "milestone": milestone,
            "milestone_key": milestone_key,
            "parent_explanation": parent_explanations[0] if parent_explanations else V18_FALLBACK_PARENT_NOTE,
            "question_id": question_id,
            "source": "v18_all_with_bridge",
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["months", "category_key", "subdomain", "milestone"]).reset_index(drop=True)


def _v18_refresh_tables(path: Optional[str] = None) -> pd.DataFrame:
    """Load Sara's all_with_bridge sheet and override the CDC backbone from it."""
    global BRIDGE_MILESTONE_DF, MILESTONE_DF, cdc_df, CDC_AGES, V18_BRIDGE_SOURCE_PATH, V18_BRIDGE_SOURCE_SHEET
    bridge_path = _v18_find_bridge_file(path)
    if bridge_path is None:
        raise FileNotFoundError(
            "v18 could not find the final app-ready bridge table or any bridge-table fallback. "
            "Place it next to the notebook, in the project root, or in /mnt/data."
        )
    raw, sheet = _v18_read_bridge_file(bridge_path)
    bridge_df = _v18_normalize_bridge_df(raw)
    if bridge_df.empty:
        raise ValueError(f"v18 bridge table was empty after normalization: {bridge_path}")
    BRIDGE_MILESTONE_DF = bridge_df
    MILESTONE_DF = bridge_df
    cdc_df = _v18_build_cdc_df_from_bridge(bridge_df)
    CDC_AGES = sorted(cdc_df["months"].dropna().unique().tolist())
    V18_BRIDGE_SOURCE_PATH = bridge_path
    V18_BRIDGE_SOURCE_SHEET = sheet
    print("Loaded Genex Brain v18 table-first bridge source:", bridge_path)
    print("Loaded bridge sheet/table:", sheet)
    print("Bridge rows:", len(BRIDGE_MILESTONE_DF))
    print("Deduped CDC milestones:", len(cdc_df))
    print("Rows with parent_explanation:", int((BRIDGE_MILESTONE_DF["parent_explanation"].str.len() > 0).sum()))
    print("Rows with previous_bridge_step:", int((BRIDGE_MILESTONE_DF["previous_bridge_step"].str.len() > 0).sum()))
    print("Rows with explicit allowed_themes:", int((BRIDGE_MILESTONE_DF["allowed_themes"].str.len() > 0).sum()))
    print("Rows with explicit what_to_avoid:", int((BRIDGE_MILESTONE_DF["what_to_avoid"].str.len() > 0).sum()))
    return bridge_df


# Load now so all downstream cells use the v18 source.
_v18_refresh_tables()


# -----------------------------
# Question text: table-only parent explanation
# -----------------------------
def _v18_lookup_parent_explanation(milestone: str, category_key: str = "", subdomain: str = "", months: Optional[int] = None) -> str:
    df = globals().get("BRIDGE_MILESTONE_DF", pd.DataFrame())
    if df is None or df.empty:
        return V18_FALLBACK_PARENT_NOTE
    key = _v18_norm(milestone)
    subset = df[df["milestone_key"] == key]
    if category_key:
        cat_subset = subset[subset["category_key"] == category_key]
        if not cat_subset.empty:
            subset = cat_subset
    if subdomain:
        sub_subset = subset[subset["subdomain"].map(_v18_norm) == _v18_norm(subdomain)]
        if not sub_subset.empty:
            subset = sub_subset
    if months is not None:
        month_subset = subset[subset["months"] == int(months)]
        if not month_subset.empty:
            subset = month_subset
    notes = [str(x).strip() for x in subset.get("parent_explanation", pd.Series(dtype=str)).tolist() if str(x).strip()]
    return notes[0] if notes else V18_FALLBACK_PARENT_NOTE


def build_parent_question_explanation(milestone: str, category_key: str, subdomain: str = "", max_words: int = 26) -> str:
    """v18: use the advisor/table parent_explanation only; do not invent notes live."""
    note = _v18_lookup_parent_explanation(milestone, category_key=category_key, subdomain=subdomain)
    if not note or note == V18_FALLBACK_PARENT_NOTE:
        return V18_FALLBACK_PARENT_NOTE
    words = str(note).strip().split()
    if len(words) > max_words:
        return " ".join(words[:max_words]).rstrip(".,;:") + "."
    return str(note).strip()


def get_category_questions(category_key: str, min_months: int, max_months: int) -> pd.DataFrame:
    subset = cdc_df[
        (cdc_df["category_key"] == category_key)
        & (cdc_df["months"] >= int(min_months))
        & (cdc_df["months"] <= int(max_months))
    ].sort_values(["months", "subdomain", "milestone"])
    return subset.copy()


def build_milestone_questions(
    state: Dict[str, Any],
    category_key: str,
    window_months: int = 24,
    target_questions_per_band: int = 3,
    max_bands: int = 3,
    max_questions_total: int = 9,
) -> List[Dict[str, Any]]:
    """v18: same focused onboarding, but question notes come from all_with_bridge only."""
    child = state["child"]
    concern_profile = ensure_concern_profile(state)
    chrono_months = min(int(child["chronological_months"]), 60)
    delay_months = state.get("delay_estimates", {}).get(category_key, {}).get("delay_months", 12)
    approx_dev_months = max(2, chrono_months - int(delay_months))
    min_m = max(2, approx_dev_months - window_months // 2)
    max_m = min(60, approx_dev_months + window_months // 2)
    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)
    if subset.empty:
        subset = get_category_questions(category_key, min_months=min(CDC_AGES), max_months=max(CDC_AGES))
    if subset.empty:
        return []
    subset = subset.copy()
    subset["subdomain"] = subset.get("subdomain", "unspecified").fillna("unspecified")
    category_concern_weight = concern_profile["domain_weights"].get(category_key, 0.0)
    subset["age_score"] = subset["months"].map(lambda m: _age_proximity_score(int(m), approx_dev_months, window_months))
    subset["subdomain_weight"] = subset["subdomain"].map(lambda s: concern_profile["subdomain_weights"].get(str(s), 0.0))
    subset["row_score"] = 0.60 * subset["age_score"] + 0.30 * subset["subdomain_weight"] + 0.10 * float(category_concern_weight)
    available_months = sorted(subset["months"].unique().tolist())
    selected_months = sorted(available_months, key=lambda m: (abs(int(m) - approx_dev_months), int(m)))[:max_bands]
    strong_rows = subset[subset["subdomain_weight"] >= 0.55].sort_values(["subdomain_weight", "row_score", "months"], ascending=[False, False, True])
    if category_concern_weight >= 0.50 and not strong_rows.empty:
        concern_month = int(strong_rows.iloc[0]["months"])
        if concern_month not in selected_months and selected_months:
            farthest = max(selected_months, key=lambda m: (abs(int(m) - approx_dev_months), int(m)))
            selected_months.remove(farthest)
            selected_months.append(concern_month)
    selected_months = sorted(set(int(m) for m in selected_months))
    questions = []
    for month in selected_months:
        month_rows = subset[subset["months"] == month].sort_values(["row_score", "subdomain", "milestone"], ascending=[False, True, True])
        for _, row in month_rows.head(target_questions_per_band).iterrows():
            parent_explanation = _v18_lookup_parent_explanation(
                row["milestone"], category_key=category_key, subdomain=str(row.get("subdomain", "unspecified")), months=int(row["months"])
            )
            question_text = (
                f"Can {child['name']} {row['milestone']}?\n"
                f"Parent note: {parent_explanation}\n"
                "Answer: yes / sometimes / with_help / no / not_sure"
            )
            questions.append({
                "question_id": row.get("question_id", f"{category_key}_{month}_{len(questions)+1}"),
                "months": int(row["months"]),
                "milestone": row["milestone"],
                "subdomain": str(row.get("subdomain", "unspecified")),
                "parent_explanation": parent_explanation,
                "selection_score": round(float(row.get("row_score", 0.0)), 3),
                "question_text": question_text,
            })
    questions = sorted(questions, key=lambda q: (q["months"], -q["selection_score"], q["milestone"]))
    return questions[:max_questions_total]


# -----------------------------
# Theme and avoid rules
# -----------------------------
def _v18_parse_list(raw: Any) -> List[str]:
    if raw is None or (isinstance(raw, float) and math.isnan(raw)):
        return []
    if isinstance(raw, list):
        return [str(x).strip() for x in raw if str(x).strip()]
    text = str(raw).strip()
    if not text:
        return []
    pieces = re.split(r"[,;|/]+", text)
    return [p.strip() for p in pieces if p.strip()]


def _v18_unique(items: List[str], max_n: int = 8) -> List[str]:
    seen = set()
    out = []
    for item in items:
        clean = str(item).strip()
        key = _v18_norm(clean)
        if clean and key not in seen:
            seen.add(key)
            out.append(clean)
        if len(out) >= max_n:
            break
    return out


def infer_allowed_themes(category_key: str, subdomain: str = "", milestone: str = "", bridge_step: str = "") -> List[str]:
    text = _v18_norm(" ".join([str(subdomain), str(milestone), str(bridge_step)]))
    themes = []
    if category_key == "language_and_communication":
        if any(k in text for k in ["sentence", "sentences", "four or more words", "phrase", "conversation"]):
            themes += ["sentence expansion game", "picture description game", "puppet conversation", "story builder", "family turn-taking"]
        if any(k in text for k in ["action", "picture", "book", "running", "eating", "playing"]):
            themes += ["picture hunt", "book game", "action card game", "puppet action game", "sibling acting game"]
        if any(k in text for k in ["name", "first name"]):
            themes += ["mirror game", "hello song", "family photo game", "puppet greeting", "name card game"]
        if any(k in text for k in ["song", "story", "rhyme", "nursery"]):
            themes += ["song game", "rhyme game", "story repeat game", "puppet story", "family singing turn"]
        if any(k in text for k in ["question", "what", "where", "why", "who", "coat", "crayon"]):
            themes += ["object function game", "picture question game", "pretend shopping", "kitchen helper", "toy choice game"]
        # Do not use generic animal/sound themes just because a milestone contains the word "words".
        # Reserve sound/animal themes for sound imitation, early vocalization, or animal-specific rows.
        if any(k in text for k in ["sound", "imitate", "vocal", "animal", "single word", "one word"]):
            themes += ["sound imitation game", "toy choice game", "animal sound game", "snack request game"]
        themes += ["book game", "picture hunt", "toy choice game", "puppet play"]
    elif category_key == "movement_and_physical":
        if any(k in text for k in ["ball", "catch", "throw", "toss", "kick"]):
            themes += ["ball game", "soft toy toss", "laundry basket toss", "sibling roll game", "parent catch game"]
        if any(k in text for k in ["jump", "hop", "balance", "stand", "run", "walk", "step", "stairs"]):
            themes += ["floor sticker game", "animal movement game", "slow movement path", "safe stepping game", "bubble pop game"]
        if any(k in text for k in ["crayon", "pencil", "draw", "copy", "line", "circle"]):
            themes += ["crayon art game", "sticker tracing game", "drawing treasure", "family art turn"]
        if any(k in text for k in ["button", "unbutton", "zip", "cloth", "dress", "shirt", "pants"]):
            themes += ["dress-up game", "doll/teddy dressing", "morning routine", "button/zipper game", "sibling helper"]
        if any(k in text for k in ["spoon", "fork", "food", "pour", "serve", "water"]):
            themes += ["snack routine", "kitchen helper", "pretend restaurant", "pouring station", "family helper"]
        themes += ["fine-motor treasure game", "playdough game", "sticker game"]
    elif category_key == "cognitive":
        themes += ["matching game", "sorting game", "puzzle game", "hide-and-find game", "cleanup game", "simple direction game", "choice-making game"]
        if any(k in text for k in ["color", "shape", "count", "number"]):
            themes += ["shape/color game", "counting game"]
        if any(k in text for k in ["pretend", "doll", "toy"]):
            themes += ["pretend play", "toy rescue", "pretend shopping"]
    else:
        themes += ["turn-taking game", "sibling helper game", "family circle game", "pretend play", "doll/teddy care", "copy-me game"]
        if any(k in text for k in ["emotion", "sad", "hurt", "happy", "feel"]):
            themes += ["emotion faces game", "feelings book", "puppet feelings"]
    return _v18_unique(themes, 8)


def infer_what_to_avoid(category_key: str, subdomain: str = "", milestone: str = "", bridge_step: str = "") -> str:
    text = _v18_norm(" ".join([str(subdomain), str(milestone), str(bridge_step)]))
    avoid = []
    avoid.append("Avoid pressure, testing, repeated correction, or continuing when the child is frustrated.")
    if category_key == "language_and_communication":
        avoid.append("Do not require speech if gestures, pointing, sounds, or approximations are valid for this bridge step.")
        if any(k in text for k in ["two word", "sentence", "sentences", "story", "conversation"]):
            avoid.append("Do not jump to longer phrases if earlier single-word or response steps are not stable.")
    if category_key == "movement_and_physical":
        avoid.append("Avoid unsafe surfaces, fast racing, high furniture, or activities that increase fall risk.")
        if any(k in text for k in ["jump", "hop", "stairs", "balance"]):
            avoid.append("Do not ask for unsupported jumping, hopping, or stairs until the bridge step is ready.")
        if any(k in text for k in ["ball", "throw", "catch", "toss"]):
            avoid.append("Avoid hard balls, aiming at faces, or long-distance throwing.")
        if any(k in text for k in ["button", "dress", "cloth", "zip"]):
            avoid.append("Avoid tight clothing, rushing, or turning dressing into a performance test.")
    if category_key == "social_and_emotional":
        avoid.append("Avoid forcing peer interaction; keep participation short and supported.")
    return " ".join(_v18_unique(avoid, 5))


def _v18_allowed_themes_for_row(row: Dict[str, Any]) -> List[str]:
    explicit = _v18_parse_list(row.get("allowed_themes", ""))
    if explicit:
        return _v18_unique(explicit, 8)
    return infer_allowed_themes(row.get("category_key", ""), row.get("subdomain", ""), row.get("milestone", ""), row.get("bridge_step", ""))


def _v18_avoid_for_row(row: Dict[str, Any]) -> str:
    explicit = str(row.get("what_to_avoid", "") or "").strip()
    if explicit:
        return explicit
    return infer_what_to_avoid(row.get("category_key", ""), row.get("subdomain", ""), row.get("milestone", ""), row.get("bridge_step", ""))


# -----------------------------
# Milestone coverage selection
# -----------------------------
def _v18_answer_norm(answer: Dict[str, Any]) -> str:
    return str(answer.get("scoring_norm_answer", answer.get("norm_answer", "not_sure"))).strip().lower().replace("-", "_")


def _v18_answer_index(state: Dict[str, Any], category_key: str) -> Dict[Tuple[int, str], Dict[str, Any]]:
    out = {}
    for a in state.get("qna", {}).get(category_key, []):
        key = (int(a.get("months", 0) or 0), _v18_norm(a.get("milestone", "")))
        out[key] = a
    return out


def _v18_working_age_band(state: Dict[str, Any], category_key: str) -> int:
    dev = int(get_effective_dev_age(state, category_key) or state.get("dev_age", {}).get(category_key) or state["child"].get("chronological_months", 60))
    chrono = int(min(state["child"].get("chronological_months", 60), 60))
    ages = sorted([int(x) for x in CDC_AGES if int(x) <= 60])
    higher = [a for a in ages if a > dev and a <= max(chrono, dev)]
    if higher:
        return higher[0]
    same_or_lower = [a for a in ages if a <= dev]
    return same_or_lower[-1] if same_or_lower else ages[0]


def _v18_milestone_candidate_score(state: Dict[str, Any], category_key: str, row: Dict[str, Any], working_age: int, answer: Optional[Dict[str, Any]]) -> float:
    score = 0.0
    ans = _v18_answer_norm(answer or {})
    if ans in {"no", "not_sure"}:
        score += 100
    elif ans in {"sometimes", "with_help"}:
        score += 120
    elif ans == "yes":
        score -= 1000
    else:
        score += 20
    score += max(0, 40 - abs(int(row.get("months", working_age)) - working_age) * 4)
    concern_profile = ensure_concern_profile(state)
    score += 30 * float(concern_profile.get("subdomain_weights", {}).get(str(row.get("subdomain", "")), 0.0))
    text = _v18_norm(" ".join([str(row.get("milestone", "")), str(row.get("subdomain", ""))]))
    concern = _v18_norm(state.get("child", {}).get("concern", ""))
    if concern and any(tok in text for tok in concern.split() if len(tok) > 4):
        score += 10
    return score


def _v18_select_diverse(candidates: List[Dict[str, Any]], min_milestones: int = V18_MIN_MILESTONES_PER_DOMAIN, max_milestones: int = V18_MAX_MILESTONES_PER_DOMAIN) -> List[Dict[str, Any]]:
    selected = []
    used_keys = set()
    used_subdomains = Counter()
    for cand in sorted(candidates, key=lambda x: x.get("score", 0), reverse=True):
        mkey = cand.get("milestone_key")
        if mkey in used_keys:
            continue
        sub = cand.get("subdomain", "unspecified")
        # First pass: maximize subdomain diversity.
        if len(selected) < min_milestones and used_subdomains[sub] >= 1 and len({c.get("subdomain") for c in selected}) < min_milestones:
            remaining_new = any((c.get("subdomain") not in used_subdomains and c.get("milestone_key") not in used_keys) for c in candidates)
            if remaining_new:
                continue
        selected.append(cand)
        used_keys.add(mkey)
        used_subdomains[sub] += 1
        if len(selected) >= max_milestones:
            break
    if len(selected) < min_milestones:
        for cand in sorted(candidates, key=lambda x: x.get("score", 0), reverse=True):
            if cand.get("milestone_key") not in used_keys:
                selected.append(cand)
                used_keys.add(cand.get("milestone_key"))
            if len(selected) >= min_milestones:
                break
    return selected[:max_milestones]


def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = V18_MAX_MILESTONES_PER_DOMAIN, min_milestones: int = V18_MIN_MILESTONES_PER_DOMAIN) -> Dict[str, Any]:
    """v18: select a coverage set of several target milestones, not one repeated milestone.

    Priority order:
    1) all answered no/sometimes/with_help milestones from the focused interview,
       because the parent has already told us these are emerging gaps.
    2) if fewer than 3 targets are available, fill from the working age band while
       respecting chronological age and subdomain diversity.
    3) never choose milestones answered yes, and do not jump above the child's
       chronological age band.
    """
    if no_special_support_needed(state, category_key):
        return {"status": "no_special_support", "milestones": [], "message": "No special support needed for this category."}

    working_age = _v18_working_age_band(state, category_key)
    chrono = int(min(state.get("child", {}).get("chronological_months", 60), 60))
    answer_idx = _v18_answer_index(state, category_key)
    domain_df = cdc_df[cdc_df["category_key"] == category_key].copy()
    if domain_df.empty:
        return {"status": "no_milestones", "milestones": [], "message": "No milestone rows found for this category."}

    selected = []
    selected_keys = set()

    # 1) Take answered gaps first. This fixes the old behavior that ignored several
    # sometimes/no answers and over-repeated one milestone.
    answered_candidates = []
    for key, answer in answer_idx.items():
        ans = _v18_answer_norm(answer)
        if ans == "yes":
            continue
        month, mkey = key
        matches = domain_df[(domain_df["months"] == month) & (domain_df["milestone_key"] == mkey)]
        if matches.empty:
            matches = domain_df[domain_df["milestone_key"] == mkey]
        for _, row in matches.iterrows():
            if int(row.get("months", 0)) > chrono:
                continue
            d = row.to_dict()
            d["answer_norm"] = ans
            d["selection_reason"] = f"parent_answer_{ans}"
            d["score"] = _v18_milestone_candidate_score(state, category_key, d, working_age, answer) + 1000
            answered_candidates.append(d)

    for cand in sorted(answered_candidates, key=lambda x: x.get("score", 0), reverse=True):
        key = (int(cand.get("months", 0)), cand.get("milestone_key"))
        if key in selected_keys:
            continue
        selected.append(cand)
        selected_keys.add(key)
        if len(selected) >= max_milestones:
            break

    # 2) Fill only if we have fewer than the target minimum. Do not include future
    # age bands above the child's chronological age.
    if len(selected) < min_milestones:
        nearby_ages = sorted(set([working_age] + [a for a in CDC_AGES if abs(int(a) - working_age) <= 12 and int(a) <= chrono]))
        pool = domain_df[(domain_df["months"].isin(nearby_ages)) & (domain_df["months"] <= chrono)].copy()
        filler = []
        for _, row in pool.iterrows():
            key = (int(row.get("months", 0)), row.get("milestone_key"))
            if key in selected_keys:
                continue
            ans_obj = answer_idx.get(key, {})
            if _v18_answer_norm(ans_obj) == "yes":
                continue
            d = row.to_dict()
            d["answer_norm"] = _v18_answer_norm(ans_obj)
            d["selection_reason"] = "working_age_band_coverage"
            d["score"] = _v18_milestone_candidate_score(state, category_key, d, working_age, ans_obj)
            filler.append(d)
        for cand in _v18_select_diverse(filler, min_milestones=min_milestones-len(selected), max_milestones=max_milestones-len(selected)):
            key = (int(cand.get("months", 0)), cand.get("milestone_key"))
            if key not in selected_keys:
                selected.append(cand)
                selected_keys.add(key)
            if len(selected) >= max_milestones:
                break

    if not selected:
        return {"status": "no_milestones", "milestones": [], "message": "No target milestones selected."}
    return {
        "status": "needs_support",
        "working_age_band": working_age,
        "milestones": selected[:max_milestones],
        "message": f"v18 selected {len(selected[:max_milestones])} target milestones for broad 1–2 week coverage.",
    }


# -----------------------------
# Bridge-chain logic
# -----------------------------
def get_bridge_rows_for_target(category_key: str, months: int, cdc_milestone: str, subdomain: str = "unspecified") -> List[Dict[str, Any]]:
    df = globals().get("BRIDGE_MILESTONE_DF", pd.DataFrame())
    if df is None or df.empty:
        return []
    key = _v18_norm(cdc_milestone)
    subset = df[(df["category_key"] == category_key) & (df["milestone_key"] == key)]
    exact = subset[subset["months"] == int(months)]
    if not exact.empty:
        subset = exact
    if subdomain and subdomain != "unspecified":
        sub_exact = subset[subset["subdomain"].map(_v18_norm) == _v18_norm(subdomain)]
        if not sub_exact.empty:
            subset = sub_exact
    subset = subset.sort_values(["bridge_step_number", "bridge_step"])
    rows = []
    for _, row in subset.iterrows():
        d = row.to_dict()
        d["allowed_themes_list"] = _v18_allowed_themes_for_row(d)
        d["what_to_avoid_text"] = _v18_avoid_for_row(d)
        rows.append(d)
    return rows


def _v18_progress_status_for_bridge(state: Dict[str, Any], bridge_key: str) -> str:
    if not bridge_key:
        return "not_applicable"
    for item in state.get("bridge_progress", []):
        if _v18_norm(item.get("bridge_step", "")) == bridge_key and item.get("status") == "mastered":
            return "mastered"
    return "unknown"


def _v18_select_active_bridge(state: Dict[str, Any], category_key: str, target: Dict[str, Any]) -> Dict[str, Any]:
    rows = get_bridge_rows_for_target(category_key, int(target.get("months", 0)), target.get("milestone", ""), target.get("subdomain", "unspecified"))
    if not rows:
        # No generic foundation labels. Make the missing table issue explicit.
        return {
            "target_milestone": target.get("milestone", ""),
            "cdc_milestone": target.get("milestone", ""),
            "months": int(target.get("months", 0) or 0),
            "subdomain": target.get("subdomain", "unspecified"),
            "bridge_step_number": 1,
            "bridge_step": "Bridge step missing from table — review all_with_bridge.",
            "previous_bridge_step": "",
            "previous_anchor_age": "",
            "previous_bridge_status": "not_available",
            "allowed_themes": infer_allowed_themes(category_key, target.get("subdomain", ""), target.get("milestone", ""), ""),
            "what_to_avoid": infer_what_to_avoid(category_key, target.get("subdomain", ""), target.get("milestone", ""), ""),
            "source": "missing_bridge_table_row",
            "answer_norm": target.get("answer_norm", "not_sure"),
            "selection_reason": target.get("selection_reason", ""),
        }
    ans = str(target.get("answer_norm", "not_sure")).lower()
    n = len(rows)
    if ans in {"sometimes", "with_help"}:
        idx = min(n - 1, max(0, n // 2))
    else:
        idx = 0
    # If previous bridge is known not mastered from progress, step back.
    chosen = rows[idx]
    prev_key = _v18_norm(chosen.get("previous_bridge_step", ""))
    prev_status = _v18_progress_status_for_bridge(state, prev_key)
    if prev_key and prev_status == "unknown" and ans in {"no", "not_sure", ""} and idx > 0:
        chosen = rows[max(0, idx - 1)]
        prev_key = _v18_norm(chosen.get("previous_bridge_step", ""))
        prev_status = _v18_progress_status_for_bridge(state, prev_key)
    chosen = dict(chosen)
    chosen["target_milestone"] = target.get("milestone", "")
    chosen["cdc_milestone"] = target.get("milestone", "")
    chosen["months"] = int(target.get("months", chosen.get("months", 0)) or 0)
    chosen["answer_norm"] = ans
    chosen["selection_reason"] = target.get("selection_reason", "")
    chosen["previous_bridge_status"] = "not_required" if not prev_key else ("likely_emerging_from_parent_answer" if ans in {"sometimes", "with_help"} else prev_status)
    chosen["allowed_themes"] = _v18_allowed_themes_for_row(chosen)
    chosen["what_to_avoid"] = _v18_avoid_for_row(chosen)
    chosen["source"] = "all_with_bridge"
    return chosen


def build_bridge_plan_for_category(state: Dict[str, Any], category_key: str, target_milestones: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    targets = target_milestones or select_next_milestones(state, category_key).get("milestones", [])
    active_bridges = []
    for target in targets:
        active_bridges.append(_v18_select_active_bridge(state, category_key, target))
    plan = {
        "status": "ok" if active_bridges else "no_targets",
        "category_key": category_key,
        "target_milestones": targets,
        "active_bridge_steps": active_bridges,
        "logic": "v18 uses one active bridge step per selected CDC milestone; previous_bridge_step is checked or used as a warm-up/prerequisite when needed.",
    }
    state.setdefault("bridge_plans", {})[category_key] = plan
    return plan


# -----------------------------
# Activity generation
# -----------------------------
def _v18_theme_for_bridge(bridge: Dict[str, Any], offset: int = 0, state: Optional[Dict[str, Any]] = None) -> str:
    allowed = bridge.get("allowed_themes") or bridge.get("allowed_themes_list") or []
    if isinstance(allowed, str):
        allowed = _v18_parse_list(allowed)
    allowed = allowed or infer_allowed_themes("", bridge.get("subdomain", ""), bridge.get("cdc_milestone", ""), bridge.get("bridge_step", ""))
    # Later: weight liked themes from state. For now, deterministic variety.
    return allowed[offset % len(allowed)] if allowed else "playful home routine"


def _v18_group_play_line(bridge: Dict[str, Any], theme: str) -> str:
    cdc = str(bridge.get("cdc_milestone", bridge.get("target_milestone", "the skill"))).lower()
    if "action" in cdc or "picture" in cdc:
        return "2+ people: one person acts or points to a picture, and the child points, gestures, or says the action word."
    if "question" in cdc or "what" in cdc or "where" in cdc or "why" in cdc:
        return "2+ people: one person asks the simple question, another models an answer, and the child answers by pointing, gesturing, or words."
    if "name" in cdc:
        return "2+ people: each person takes a hello turn, and the child can answer with a look, gesture, sound, or name attempt."
    if "ball" in cdc or "catch" in cdc or "throw" in cdc:
        return "2+ people: roll or toss the soft ball in turns while one adult stays close to support safety."
    if "cloth" in cdc or "dress" in cdc or "button" in cdc:
        return "2+ people: one person models on a doll or clothing item, then the child tries one small step with help."
    return "2+ people: one person models the step, another gives a simple cue, and the child joins at their own level."


def _v18_materials_for_theme(theme: str, bridge: Dict[str, Any]) -> str:
    t = _v18_norm(theme)
    if "book" in t or "picture" in t:
        return "picture book, family photos, or simple picture cards"
    if "song" in t or "music" in t or "rhyme" in t:
        return "favorite song or rhyme, optional toy microphone"
    if "ball" in t or "toss" in t:
        return "soft ball, rolled socks, or soft toy"
    if "sticker" in t:
        return "floor stickers or paper targets"
    if "dressing" in t or "dress" in t or "button" in t:
        return "loose clothing, doll/teddy clothing, or large button practice cloth"
    if "kitchen" in t or "snack" in t or "restaurant" in t:
        return "safe snack items, cup, spoon, or pretend food"
    if "puppet" in t:
        return "puppet, stuffed animal, or favorite toy"
    return "common toys or household items"


def _v18_deterministic_activity(category_key: str, bridge: Dict[str, Any], activity_type: str, theme_offset: int = 0, duration_min: int = 5) -> Dict[str, Any]:
    theme = _v18_theme_for_bridge(bridge, theme_offset)
    category_display = globals().get("DOMAIN_CONFIG", {}).get(category_key, {}).get("display", category_key.replace("_", " ").title())
    cdc_goal = bridge.get("cdc_milestone") or bridge.get("target_milestone") or bridge.get("milestone") or "target milestone"
    bridge_step = bridge.get("bridge_step") or "Bridge step missing from table — review all_with_bridge."
    prev = bridge.get("previous_bridge_step", "")
    avoid = bridge.get("what_to_avoid") or _v18_avoid_for_row(bridge)
    title_base = str(bridge_step).strip().rstrip(".")
    if len(title_base.split()) > 7:
        title_base = " ".join(title_base.split()[:7])
    type_label = {"core": "", "easier_backup": "Easy ", "harder_stretch": "Stretch "}.get(activity_type, "")
    title = f"{type_label}{title_base} ({theme})".strip()
    if prev:
        intro = f"First check or warm up with: {prev}. Then practice: {bridge_step}."
    else:
        intro = f"Practice: {bridge_step}."
    if activity_type == "easier_backup":
        success = f"The child participates in an easier version of: {bridge_step}."
        easier = "Add more modeling, reduce choices, shorten the turn, or provide hands-on support."
        harder = f"Return to the core bridge step when this feels comfortable: {bridge_step}."
    elif activity_type == "harder_stretch":
        success = f"The child does the bridge step with less help or more independence: {bridge_step}."
        easier = f"Go back to the core version or prerequisite: {prev or bridge_step}."
        harder = f"Gently connect this step toward the CDC goal: {cdc_goal}."
    else:
        success = f"The child attempts the bridge step with support: {bridge_step}."
        easier = f"Practice the previous bridge step first: {prev}." if prev else "Make the task shorter, slower, or more modeled."
        harder = "Reduce adult help slightly or add one extra turn only if the child is enjoying it."
    return {
        "activity_id": "",
        "title": title,
        "activity_type": activity_type,
        "theme": theme,
        "play_theme": theme,
        "category": category_display,
        "category_key": category_key,
        "duration_min": duration_min,
        "cdc_goal": cdc_goal,
        "target_milestone": cdc_goal,
        "bridge_step": bridge_step,
        "previous_bridge_step": prev,
        "previous_bridge_status": bridge.get("previous_bridge_status", "unknown"),
        "allowed_themes": bridge.get("allowed_themes", []),
        "what_to_avoid": avoid,
        "why_it_helps": f"Builds the prerequisite skill needed before the CDC goal: {cdc_goal}.",
        "instructions": f"Use the {theme} format. {intro} Keep it playful and stop before frustration. Do not practice the final milestone directly unless this bridge step is already easy.",
        "success_criteria": success,
        "success": success,
        "make_easier": easier,
        "make_harder": harder,
        "group_play_line": _v18_group_play_line(bridge, theme),
        "group_play": _v18_group_play_line(bridge, theme),
        "materials": _v18_materials_for_theme(theme, bridge),
        "level": "bridge_step",
        "goal": "bridge_practice",
        "is_extended_activity": False,
        "extended_reason": "",
        "milestone_key": _v18_norm(cdc_goal),
        "bridge_step_number": bridge.get("bridge_step_number", 1),
        "subdomain": bridge.get("subdomain", "unspecified"),
    }


def _v18_activity_contexts_for_prompt(active_bridges: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    contexts = []
    for i, bridge in enumerate(active_bridges, start=1):
        contexts.append({
            "context_id": f"ctx_{i}",
            "cdc_goal": bridge.get("cdc_milestone", bridge.get("target_milestone", "")),
            "target_age_months": bridge.get("months", ""),
            "subdomain": bridge.get("subdomain", "unspecified"),
            "active_bridge_step": bridge.get("bridge_step", ""),
            "previous_bridge_step": bridge.get("previous_bridge_step", ""),
            "previous_bridge_status": bridge.get("previous_bridge_status", "unknown"),
            "allowed_themes": bridge.get("allowed_themes", []),
            "what_to_avoid": bridge.get("what_to_avoid", ""),
            "answer_norm": bridge.get("answer_norm", ""),
        })
    return contexts


def _v18_make_activity_prompt(state: Dict[str, Any], category_key: str, active_bridges: List[Dict[str, Any]], planning_tier: str) -> str:
    child = state.get("child", {})
    category_display = globals().get("DOMAIN_CONFIG", {}).get(category_key, {}).get("display", category_key.replace("_", " ").title())
    contexts_json = json.dumps(_v18_activity_contexts_for_prompt(active_bridges), ensure_ascii=False, indent=2)
    safety_profile = build_safety_profile(state) if "build_safety_profile" in globals() else {}
    safety_constraints = format_safety_constraints_for_prompt(safety_profile) if "format_safety_constraints_for_prompt" in globals() else "Use safe, supervised home activities."
    language_readiness = build_language_activity_readiness_block(state, category_key) if "build_language_activity_readiness_block" in globals() else ""
    return f"""
You are creating a Genex v18 activity bank for a parent.

Non-negotiable rules:
- Use ONLY the CDC goals and active bridge steps provided in Activity Contexts.
- Do NOT invent a different CDC goal.
- Do NOT replace a bridge step with a generic phrase like "foundational prerequisite" or "supported practice".
- Use only one of the allowed_themes listed for that context.
- Follow what_to_avoid exactly.
- Generate activities for the active bridge step, not for the final CDC milestone, unless the bridge step is already the final step.
- If previous_bridge_step is not empty, include it as a quick warm-up/check before the active bridge step.
- Group play must involve 2+ people with specific roles. Do not write vague lines like "Let's all be animals together."
- Activities must be playful, low-pressure, success-oriented, and realistic at home.
- Return strict JSON only.

Child:
- Name: {child.get('name')}
- Age: {child.get('chronological_months')} months
- Diagnosis / condition: {child.get('diagnosis')}
- Parent concern: {child.get('concern')}
- Daily time available: {child.get('daily_time_min')} minutes

Category: {category_display}
Planning tier: {planning_tier}

Language readiness constraints:
{language_readiness}

Safety constraints:
{safety_constraints}

Activity Contexts:
{contexts_json}

Create exactly 3 activities for each context: one core, one easier_backup, and one harder_stretch.

Required JSON shape:
{{
  "summary": "...",
  "activities": [
    {{
      "context_id": "ctx_1",
      "activity_type": "core | easier_backup | harder_stretch",
      "title": "...",
      "theme": "must be one allowed theme from that context",
      "instructions": "...",
      "success_criteria": "...",
      "make_easier": "...",
      "make_harder": "...",
      "group_play_line": "2+ people: ...",
      "materials": "...",
      "duration_min": 5
    }}
  ]
}}
""".strip()


def _v18_normalize_generated_activity(raw: Dict[str, Any], category_key: str, bridge: Dict[str, Any], idx: int, fallback_type: str) -> Dict[str, Any]:
    allowed = bridge.get("allowed_themes", []) or []
    if isinstance(allowed, str):
        allowed = _v18_parse_list(allowed)
    raw_theme = str(raw.get("theme", raw.get("play_theme", ""))).strip()
    theme = raw_theme if _v18_norm(raw_theme) in {_v18_norm(t) for t in allowed} else _v18_theme_for_bridge(bridge, idx)
    act = _v18_deterministic_activity(category_key, bridge, raw.get("activity_type", fallback_type), idx)
    # Keep safe LLM wording, but force all structural fields from the table.
    for key in ["title", "instructions", "success_criteria", "make_easier", "make_harder", "materials"]:
        val = str(raw.get(key, "")).strip()
        if val:
            act[key] = val
    group = str(raw.get("group_play_line", raw.get("group_play", ""))).strip()
    if group and re.search(r"\b(parent|sibling|caregiver|family|person|people|turn|model|hide|act|choose)\b", group.lower()):
        if not group.lower().startswith("2+"):
            group = "2+ people: " + group
        act["group_play_line"] = group
        act["group_play"] = group
    act["theme"] = theme
    act["play_theme"] = theme
    act["activity_type"] = raw.get("activity_type", fallback_type) if raw.get("activity_type", fallback_type) in {"core", "easier_backup", "harder_stretch"} else fallback_type
    act["activity_id"] = f"{category_key}_{_v18_norm(act['cdc_goal'])[:18]}_{act['bridge_step_number']}_{act['activity_type']}_{idx}"
    # Table-anchored fields always win.
    act["cdc_goal"] = bridge.get("cdc_milestone", bridge.get("target_milestone", ""))
    act["target_milestone"] = act["cdc_goal"]
    act["bridge_step"] = bridge.get("bridge_step", "")
    act["previous_bridge_step"] = bridge.get("previous_bridge_step", "")
    act["allowed_themes"] = allowed
    act["what_to_avoid"] = bridge.get("what_to_avoid", "")
    return act


def generate_category_activity_bank(state: Dict[str, Any], category_key: str, model: str = "gpt-4o-mini", activities_per_category: Optional[int] = None) -> Dict[str, Any]:
    child = state["child"]
    category_display = globals().get("DOMAIN_CONFIG", {}).get(category_key, {}).get("display", category_key.replace("_", " ").title())
    support_tier = get_support_tier(state, category_key)
    planning_tier = "enrich_and_observe" if is_family_guidance_category(state, category_key) else support_tier
    next_steps = select_next_milestones(state, category_key, max_milestones=V18_MAX_MILESTONES_PER_DOMAIN, min_milestones=V18_MIN_MILESTONES_PER_DOMAIN)
    if next_steps.get("status") == "no_special_support":
        bank = {"status": "no_special_support", "support_tier": support_tier, "planning_tier": planning_tier, "summary": next_steps.get("message", ""), "activities": []}
        state.setdefault("activity_banks", {})[category_key] = bank
        return bank
    targets = next_steps.get("milestones", [])
    bridge_plan = build_bridge_plan_for_category(state, category_key, targets)
    active_bridges = bridge_plan.get("active_bridge_steps", [])

    activities = []
    used_llm = False
    if openai_client is not None and active_bridges:
        try:
            prompt = _v18_make_activity_prompt(state, category_key, active_bridges, planning_tier)
            resp = openai_client.chat.completions.create(
                model=model,
                temperature=0.1,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "You return strict JSON only. You must obey table-provided CDC goals, bridge steps, allowed themes, and avoid rules."},
                    {"role": "user", "content": prompt},
                ],
            )
            parsed = json.loads(resp.choices[0].message.content)
            by_context = defaultdict(list)
            for raw in parsed.get("activities", []):
                by_context[str(raw.get("context_id", ""))].append(raw)
            for i, bridge in enumerate(active_bridges, start=1):
                context_id = f"ctx_{i}"
                raws = by_context.get(context_id, [])
                type_order = ["core", "easier_backup", "harder_stretch"]
                for j, typ in enumerate(type_order):
                    raw_match = next((r for r in raws if r.get("activity_type") == typ), None)
                    if raw_match is None:
                        raw_match = {}
                    activities.append(_v18_normalize_generated_activity(raw_match, category_key, bridge, i + j, typ))
            used_llm = True
        except Exception as exc:
            print(f"v18 activity LLM fallback for {category_display}: {exc}")
            activities = []

    if not activities:
        for i, bridge in enumerate(active_bridges, start=1):
            activities.append(_v18_deterministic_activity(category_key, bridge, "core", i))
            activities.append(_v18_deterministic_activity(category_key, bridge, "easier_backup", i + 1))
            activities.append(_v18_deterministic_activity(category_key, bridge, "harder_stretch", i + 2))

    # Preserve v16/v17 safety and speech readiness filters after table anchoring.
    activities = apply_language_readiness_constraints_to_activities(state, category_key, activities)
    activities = apply_safety_constraints_to_activities(state, category_key, activities)
    # Re-anchor fields after safety filters in case replacements changed fields.
    bridge_by_key = {_v18_norm(b.get("cdc_milestone", b.get("target_milestone", ""))): b for b in active_bridges}
    for idx, act in enumerate(activities, start=1):
        act.setdefault("activity_id", f"{category_key}_v18_{idx}")
        act.setdefault("activity_type", "core")
        act.setdefault("theme", act.get("play_theme", "playful home routine"))
        act.setdefault("group_play_line", act.get("group_play", "2+ people: one person models and the child joins at their own level."))
        act.setdefault("group_play", act.get("group_play_line"))

    bank = {
        "status": "success" if used_llm else "fallback_table_based",
        "version": V18_VERSION,
        "support_tier": support_tier,
        "planning_tier": planning_tier,
        "working_age_band": next_steps.get("working_age_band"),
        "summary": f"v18 created a multi-milestone bridge activity bank for {category_display}: {len(targets)} target milestones, {len(active_bridges)} active bridge steps, {len(activities)} activities.",
        "target_milestones": targets,
        "active_bridge_steps": active_bridges,
        "activities": activities,
    }
    state.setdefault("activity_banks", {})[category_key] = bank
    return bank


# -----------------------------
# v18 schedule: rotate milestones, not only activity titles
# -----------------------------
def allocate_weekly_slots(state: Dict[str, Any], cycle_days: int = V18_DEFAULT_CYCLE_DAYS) -> Dict[str, Any]:
    categories = state.get("planned_categories") or state.get("selected_categories", [])
    daily_time = int(state.get("child", {}).get("daily_time_min", 10) or 10)
    per_day_slots = 2 if daily_time >= 10 else 1
    state["weekly_slot_plan"] = {"cycle_days": cycle_days, "daily_time_min": daily_time, "per_day_slots": per_day_slots, "categories": categories}
    return state["weekly_slot_plan"]


def _schedule_item_from_activity(category_key: str, activity: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "category_key": category_key,
        "category": globals().get("DOMAIN_CONFIG", {}).get(category_key, {}).get("display", activity.get("category", category_key)),
        "activity_id": activity.get("activity_id"),
        "title": activity.get("title"),
        "duration_min": int(activity.get("duration_min", 5) or 5),
        "activity_type": activity.get("activity_type", "core"),
        "theme": activity.get("theme", activity.get("play_theme", "")),
        "cdc_goal": activity.get("cdc_goal", activity.get("target_milestone", "")),
        "bridge_step": activity.get("bridge_step", ""),
        "previous_bridge_step": activity.get("previous_bridge_step", ""),
        "previous_bridge_status": activity.get("previous_bridge_status", ""),
        "why_it_helps": activity.get("why_it_helps", activity.get("why", "")),
        "instructions": activity.get("instructions", ""),
        "success_criteria": activity.get("success_criteria", activity.get("success", "")),
        "make_easier": activity.get("make_easier", activity.get("easier", "")),
        "make_harder": activity.get("make_harder", activity.get("harder", "")),
        "group_play_line": activity.get("group_play_line", activity.get("group_play", "")),
        "materials": activity.get("materials", ""),
        "what_to_avoid": activity.get("what_to_avoid", ""),
        "subdomain": activity.get("subdomain", "unspecified"),
        "milestone_key": activity.get("milestone_key", _v18_norm(activity.get("cdc_goal", ""))),
    }


def _v18_choose_activity(bank: Dict[str, Any], day_idx: int, slot_idx: int, used_titles: set, used_milestones: set, counts_by_milestone: Counter, counts_by_activity: Counter) -> Optional[Dict[str, Any]]:
    activities = [a for a in bank.get("activities", []) if a.get("activity_type", "core") == "core"]
    if not activities:
        activities = bank.get("activities", [])
    if not activities:
        return None
    def sort_key(a: Dict[str, Any]):
        title_key = _v18_norm(a.get("title", ""))
        mkey = _v18_norm(a.get("cdc_goal", a.get("target_milestone", "")))
        theme_key = _v18_norm(a.get("theme", ""))
        return (
            1 if title_key in used_titles else 0,
            1 if mkey in used_milestones else 0,
            counts_by_milestone[mkey],
            counts_by_activity[_v18_norm(a.get("activity_id", title_key))],
            (day_idx + slot_idx + abs(hash(theme_key))) % 7,
        )
    chosen = sorted(activities, key=sort_key)[0]
    return chosen


def build_weekly_schedule(state: Dict[str, Any], cycle_days: int = V18_DEFAULT_CYCLE_DAYS) -> Dict[str, Any]:
    categories = state.get("planned_categories") or state.get("selected_categories", [])
    daily_time = int(state.get("child", {}).get("daily_time_min", 10) or 10)
    if not categories:
        state["weekly_schedule"] = {"status": "no_categories", "summary": "No categories selected for planning.", "days": {}}
        return state["weekly_schedule"]
    for cat in categories:
        if cat not in state.get("activity_banks", {}):
            generate_category_activity_bank(state, cat)
    per_activity_min = 5
    max_items = max(1, daily_time // per_activity_min)
    days = {}
    counts_by_milestone = Counter()
    counts_by_activity = Counter()
    last_day_titles = set()
    for d in range(1, cycle_days + 1):
        day_key = f"day_{d}"
        items = []
        used_titles = set()
        used_milestones = set()
        # If two or more domains: one slot per domain first. If one domain: multiple milestones from same bank.
        slot_categories = []
        if len(categories) >= 2:
            for i in range(max_items):
                slot_categories.append(categories[(d + i - 1) % len(categories)])
        else:
            slot_categories = [categories[0] for _ in range(max_items)]
        for slot_idx, cat in enumerate(slot_categories):
            bank = state.get("activity_banks", {}).get(cat, {})
            act = _v18_choose_activity(bank, d, slot_idx, used_titles | last_day_titles if slot_idx == 0 else used_titles, used_milestones, counts_by_milestone, counts_by_activity)
            if not act:
                continue
            item = _schedule_item_from_activity(cat, act)
            title_key = _v18_norm(item.get("title", ""))
            mkey = _v18_norm(item.get("cdc_goal", ""))
            used_titles.add(title_key)
            used_milestones.add(mkey)
            counts_by_milestone[mkey] += 1
            counts_by_activity[_v18_norm(item.get("activity_id", title_key))] += 1
            items.append(item)
        last_day_titles = set(used_titles)
        total = sum(int(i.get("duration_min", 5)) for i in items)
        days[day_key] = {"total_minutes": total, "items": items}
    schedule = {
        "status": "ok",
        "version": V18_VERSION,
        "summary": "v18 bridge-chain weekly schedule. The 1–2 week bank covers several milestones per selected domain, uses table bridge steps, varies compatible play themes, and adjusts with parent feedback.",
        "daily_time_min": daily_time,
        "bank_repeat_window_days": 14,
        "feedback_options": V18_ACTIVITY_FEEDBACK_OPTIONS,
        "selection_rules": [
            "Cover several CDC milestones per selected domain, not one repeated milestone.",
            "Generate activities for active bridge steps from all_with_bridge.",
            "Use allowed themes and avoid rules from the table or deterministic inference.",
            "Avoid duplicate activity titles on the same day and avoid same-title back-to-back repetition.",
            "Repeat skills across 1–2 weeks while varying compatible themes.",
        ],
        "days": days,
    }
    state["weekly_schedule"] = schedule
    return schedule


def display_weekly_schedule(state: Dict[str, Any]) -> None:
    schedule = state.get("weekly_schedule", {})
    print_banner("V18 BRIDGE-CHAIN WEEKLY SCHEDULE")
    if schedule.get("status") == "no_special_support":
        print(schedule.get("summary", "No weekly schedule available."))
        return
    print("Summary:", schedule.get("summary", ""))
    print("Daily time budget:", schedule.get("daily_time_min"))
    print("Bank repeat window:", schedule.get("bank_repeat_window_days", 14), "days")
    print("Feedback options:", schedule.get("feedback_options", V18_ACTIVITY_FEEDBACK_OPTIONS))
    print("Source bridge table:", globals().get("V18_BRIDGE_SOURCE_PATH", "not loaded"))
    for cat, bank in state.get("activity_banks", {}).items():
        if bank.get("target_milestones"):
            print(f"\nCoverage set — {globals().get('DOMAIN_CONFIG', {}).get(cat, {}).get('display', cat)}:")
            for idx, t in enumerate(bank.get("target_milestones", []), start=1):
                print(f"  {idx}. {int(t.get('months', 0))}m | {t.get('subdomain', 'unspecified')} | {t.get('milestone')} [{t.get('answer_norm', 'unasked')}]")
    for day_name, day_info in schedule.get("days", {}).items():
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min")
        print("-" * 100)
        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue
        for item in items:
            print(f"- [{item.get('category')}] {item.get('title')} ({item.get('duration_min', 5)} min)")
            print(f"  Type: {item.get('activity_type')} | Theme: {item.get('theme')}")
            print(f"  CDC goal: {item.get('cdc_goal')}")
            print(f"  Previous bridge: {item.get('previous_bridge_step') or 'None / not needed'}")
            print(f"  Active bridge: {item.get('bridge_step')}")
            print(f"  Why: {item.get('why_it_helps')}")
            print(f"  Instructions: {item.get('instructions')}")
            print(f"  Success: {item.get('success_criteria')}")
            print(f"  Easier: {item.get('make_easier')}")
            print(f"  Harder: {item.get('make_harder')}")
            print(f"  Group play: {item.get('group_play_line')}")
            print(f"  Avoid: {item.get('what_to_avoid')}")
            print(f"  Materials: {item.get('materials')}")


def v18_bridge_table_quality_report() -> pd.DataFrame:
    df = globals().get("BRIDGE_MILESTONE_DF", pd.DataFrame())
    if df is None or df.empty:
        return pd.DataFrame()
    grouped = df.groupby(["months", "category_key", "subdomain", "milestone"], dropna=False).agg(
        bridge_steps=("bridge_step", lambda s: sum(1 for x in s if str(x).strip())),
        has_parent_explanation=("parent_explanation", lambda s: any(str(x).strip() for x in s)),
        has_previous_bridge=("previous_bridge_step", lambda s: any(str(x).strip() for x in s)),
        has_allowed_themes=("allowed_themes", lambda s: any(str(x).strip() for x in s)),
        has_avoid_rules=("what_to_avoid", lambda s: any(str(x).strip() for x in s)),
    ).reset_index()
    grouped["needs_review"] = (~grouped["bridge_steps"].between(3, 5)) | (~grouped["has_parent_explanation"])
    return grouped.sort_values(["needs_review", "months", "category_key", "subdomain"], ascending=[False, True, True, True])


# Make output folder version-correct if the setup cell used an older folder name.
try:
    OUTPUT_DIR = (globals().get("PROJECT_ROOT", Path.cwd()) / "outputs" / "genex_interview_activity_v18")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    pass

print("Loaded Genex Brain v18 corrected patch:", V18_VERSION)
print("Run v18_bridge_table_quality_report() to inspect bridge coverage and missing review items.")


Loaded Genex Brain v18 table-first bridge source: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx
Loaded bridge sheet/table: all_with_bridge_family
Bridge rows: 369
Deduped CDC milestones: 163
Rows with parent_explanation: 369
Rows with previous_bridge_step: 359
Rows with explicit allowed_themes: 0
Rows with explicit what_to_avoid: 0
Loaded Genex Brain v18 corrected patch: v18_table_first_multimilestone_bridge_chain
Run v18_bridge_table_quality_report() to inspect bridge coverage and missing review items.


In [47]:
# ------------------------------------------------------------
# Main runners
# ------------------------------------------------------------
def run_all_categories_live():
    """Run one focused live onboarding in v15."""
    state = new_state()
    intake_agent_live(state)
    print_child_reference(state)
    print_concern_profile(state)

    delay_agent_all_categories(state)
    selected_categories = select_focus_domains_live_v14(state)

    for category_key in selected_categories:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=2,
            yes_ratio_confirm=0.60,
        )

    dev_age_map = state.get("dev_age", {})
    selected_categories = [c for c in selected_categories if dev_age_map.get(c) is not None]

    summary_df = run_downstream_planning(
        state,
        display_schedule=True,
        selected_categories=selected_categories,
    )
    return state, summary_df

def run_all_categories_gemini_parent(
    case: Dict[str, Any],
    daily_time_min: int = 10,
    gemini_model: str = "gemini-2.5-flash",
):
    """Run one focused case in synthetic-parent mode using Gemini (v15)."""
    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_child_reference(state)
    print_concern_profile(state)

    delay_agent_all_categories(state)
    selected_categories = select_focus_domains_gemini_v14(state)

    for category_key in selected_categories:
        qna_agent_gemini_parent(
            state,
            category_key,
            ask_limit_per_band=V14_TARGET_QUESTIONS_PER_BAND,
            min_yes_confirm=2,
            yes_ratio_confirm=0.60,
            gemini_model=gemini_model,
            delay_between_questions_sec=5,
        )

    dev_age_map = state.get("dev_age", {})
    selected_categories = [c for c in selected_categories if dev_age_map.get(c) is not None]

    summary_df = run_downstream_planning(
        state,
        display_schedule=True,
        selected_categories=selected_categories,
    )
    return state, summary_df

def print_case_reference(case: dict, daily_time_min: int):
    print("\n" + "=" * 110)
    print(f"STARTING {case['case_id']} | {case['child_name']} | {case['age_label']} | {case['diagnosis']}")
    print("=" * 110)
    print(f"Concerns: {case['concerns']}")
    print(f"Daily time: {daily_time_min} min/day")

def run_case_live_from_prefilled_case(
    case: dict,
    default_daily_time_min: int = 10,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
):
    """Run a prefilled case live with focused v15 onboarding."""
    raw_daily = input(
        f"[{case['case_id']}] Daily minutes available? Press Enter for default {default_daily_time_min}: "
    ).strip()
    daily_time_min = int(raw_daily) if raw_daily else int(default_daily_time_min)

    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_case_reference(case, daily_time_min)
    print_concern_profile(state)

    delay_agent_all_categories(state)
    selected_categories = select_focus_domains_live_v14(state)

    for category_key in selected_categories:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
        )

    dev_age_map = state.get("dev_age", {})
    selected_categories = [c for c in selected_categories if dev_age_map.get(c) is not None]

    summary_df = run_downstream_planning(
        state,
        display_schedule=True,
        selected_categories=selected_categories,
    )
    return state, summary_df


In [48]:
# ------------------------------------------------------------
# Test cases
# ------------------------------------------------------------
cases = [
    {"case_id": "C01", "child_name": "Noah", "age_months": 10, "age_label": "10 months",
     "diagnosis": "Down syndrome",
     "concerns": "Not sitting independently yet, Hypotonia, slow feeding, socially engaged"},

    {"case_id": "C02", "child_name": "Maya", "age_months": 18, "age_label": "18 months",
     "diagnosis": "No formal diagnosis",
     "concerns": "No words yet, uses gestures, understands simple commands, good eye contact"},

    {"case_id": "C03", "child_name": "Leo", "age_months": 26, "age_label": "2y 2m",
     "diagnosis": "Autism spectrum disorder",
     "concerns": "Limited eye contact and few words, Repetitive play, transitions hard, picky eating"},

    {"case_id": "C04", "child_name": "Sofia", "age_months": 37, "age_label": "3y 1m",
     "diagnosis": "Prematurity history",
     "concerns": "Speech hard to understand, Fine motor delay, good comprehension"},

    {"case_id": "C05", "child_name": "Ethan", "age_months": 54, "age_label": "4y 6m",
     "diagnosis": "Cerebral palsy",
     "concerns": "Trouble keeping up in preschool, Self-care delays, frequent falls, difficulty with stairs, uses walker sometimes, good cognition"},

    {"case_id": "C06", "child_name": "Amina", "age_months": 60, "age_label": "5 years",
     "diagnosis": "Global developmental delay",
     "concerns": "Learning and routines are hard, Short attention span, follows only simple directions"},

    {"case_id": "C07", "child_name": "Lucas", "age_months": 14, "age_label": "14 months",
     "diagnosis": "Fragile X syndrome",
     "concerns": "Not walking, limited babbling, Sensory sensitivity, shy socially"},

    {"case_id": "C08", "child_name": "Emma", "age_months": 32, "age_label": "2y 8m",
     "diagnosis": "Suspected childhood apraxia of speech",
     "concerns": "Very limited speech output, Uses gestures, gets frustrated, inconsistent sounds"},

    {"case_id": "C09", "child_name": "Zain", "age_months": 48, "age_label": "4 years",
     "diagnosis": "Attention and self-regulation concerns (possible ADHD)",
     "concerns": "Very active, impulsive, Big emotional reactions, interrupts, difficulty following routines, social interest intact"},

    {"case_id": "C10", "child_name": "Isla", "age_months": 44, "age_label": "3y 8m",
     "diagnosis": "SYNGAP1-related disorder",
     "concerns": "Delayed speech and balance, Attention issues, clumsy gait, limited pretend play"},

    {"case_id": "C11", "child_name": "Aria", "age_months": 20, "age_label": "20 months",
     "diagnosis": "No diagnosis",
     "concerns": "Only ~10 words, very active, good eye contact, understands well"},

    {"case_id": "C12", "child_name": "Mateo", "age_months": 30, "age_label": "2y 6m",
     "diagnosis": "No formal diagnosis",
     "concerns": "Good eye contact but no words, lines up toys, responds to name inconsistently"},

    {"case_id": "C13", "child_name": "Luna", "age_months": 42, "age_label": "3y 6m",
     "diagnosis": "Speech delay",
     "concerns": "Limited speech, understands well, parents have very limited time for activities"},

    {"case_id": "C14", "child_name": "Ryan", "age_months": 50, "age_label": "4y 2m",
     "diagnosis": "No diagnosis",
     "concerns": "Tantrums, difficulty with transitions, strong language skills, social but rigid"},
]

df_cases = pd.DataFrame(cases)
# display(df_cases)  # optional: uncomment to inspect synthetic advisor test cases


In [49]:

# ------------------------------------------------------------
# Advisor-report helpers
# ------------------------------------------------------------
def severity_from_gap(gap_months: float) -> str:
    gap_months = 0 if pd.isna(gap_months) else float(gap_months)
    if gap_months <= 6:
        return "minimal / mild"
    elif gap_months <= 18:
        return "moderate"
    else:
        return "significant"

def split_concerns_to_bullets(concern_text: str, max_items: int = 4):
    items = [x.strip(" -•\n\t") for x in re.split(r"[,\n;]+", str(concern_text)) if x.strip()]
    return items[:max_items]

def _ordered_category_keys_for_pdf(summary_df: pd.DataFrame) -> List[str]:
    if summary_df is None or summary_df.empty:
        return list(DOMAIN_CONFIG.keys())

    work = summary_df.copy()
    work["category_key"] = work["category"].map(
        {v["display"]: k for k, v in DOMAIN_CONFIG.items()}
    )
    tier_col = "planning_tier" if "planning_tier" in work.columns else "support_tier"
    work["tier_rank"] = work[tier_col].map({
        "needs_special_support": 2,
        "monitor_and_enrich": 1,
        "enrich_and_observe": 1,
        "no_special_support": 0,
    }).fillna(0)
    work = work.sort_values(["tier_rank", "milestone_gap_months"], ascending=[False, False])

    ordered = [k for k in work["category_key"].dropna().tolist() if k in DOMAIN_CONFIG]
    seen = set()
    deduped = []
    for k in ordered:
        if k not in seen:
            seen.add(k)
            deduped.append(k)

    for k in DOMAIN_CONFIG.keys():
        if k not in seen:
            deduped.append(k)

    return deduped

def extract_parent_qa_lines(
    state: dict,
    priority_category_keys: Optional[List[str]] = None,
    max_lines_for_pdf: int = 16,
):
    full_lines = []

    ordered_keys = priority_category_keys[:] if priority_category_keys else list(DOMAIN_CONFIG.keys())
    for k in DOMAIN_CONFIG.keys():
        if k not in ordered_keys:
            ordered_keys.append(k)

    for category_key in ordered_keys:
        answers = state.get("qna", {}).get(category_key, [])
        for a in answers:
            milestone = str(a.get("milestone", "")).strip()
            norm_answer = str(a.get("norm_answer", a.get("raw_answer", ""))).strip()
            months = a.get("months", "")
            status = a.get("answer_status", "ok")
            subdomain = a.get("subdomain", "unspecified")
            followup = str(a.get("followup_label", "")).strip()
            scoring_norm = str(a.get("scoring_norm_answer", norm_answer)).strip()

            line = f"[{months}m | {subdomain}] {milestone} -> {norm_answer}"
            if followup:
                line += f" | why: {followup}"
            if scoring_norm and scoring_norm != norm_answer:
                line += f" | scored_as: {scoring_norm}"
            line += f" ({status})"
            full_lines.append(line)

    pdf_lines = full_lines[:max_lines_for_pdf]
    return full_lines, pdf_lines

def extract_focus_areas(summary_df: pd.DataFrame, max_items: int = 3):
    if summary_df is None or summary_df.empty:
        return []

    work = summary_df.copy()
    tier_col = "planning_tier" if "planning_tier" in work.columns else "support_tier"
    work = work[work[tier_col].isin(["needs_special_support", "monitor_and_enrich", "enrich_and_observe"])].copy()
    if work.empty:
        return []

    work["tier_rank"] = work[tier_col].map({
        "needs_special_support": 2,
        "monitor_and_enrich": 1,
        "enrich_and_observe": 1,
        "no_special_support": 0,
    })
    sort_col = "milestone_gap_months" if "milestone_gap_months" in work.columns else "estimated_delay_months"
    work = work.sort_values(["tier_rank", sort_col], ascending=[False, False])

    return work["category"].tolist()[:max_items]

def extract_activities_for_case(state: dict, max_activities: int = 5):
    activities = []
    weekly_schedule = state.get("weekly_schedule", {})
    days = weekly_schedule.get("days", {})

    if isinstance(days, dict):
        for day_label, day_info in days.items():
            day_items = day_info.get("items", [])
            for item in day_items:
                activities.append({
                    "day": day_label,
                    "category": item.get("category", item.get("category_display", "")),
                    "title": item.get("title", ""),
                    "description": item.get("instructions", item.get("description", "")),
                    "duration_min": item.get("duration_min", ""),
                    "goal": item.get("goal", item.get("level", item.get("category", ""))),
                })

    if not activities:
        for category_key, bank in state.get("activity_banks", {}).items():
            for item in bank.get("activities", [])[:2]:
                activities.append({
                    "day": "",
                    "category": DOMAIN_CONFIG.get(category_key, {}).get("display", category_key),
                    "title": item.get("title", ""),
                    "description": item.get("instructions", item.get("description", "")),
                    "duration_min": item.get("duration_min", ""),
                    "goal": item.get("goal", item.get("level", "current_or_next")),
                })

    return activities[:max_activities]

def build_case_payload(case: dict, state: dict, summary_df: pd.DataFrame):
    concerns_bullets = split_concerns_to_bullets(case.get("concerns", ""), max_items=4)
    ordered_category_keys = _ordered_category_keys_for_pdf(summary_df)
    parent_qa_full, parent_qa_pdf = extract_parent_qa_lines(
        state,
        priority_category_keys=ordered_category_keys,
        max_lines_for_pdf=16,
    )
    focus_areas = extract_focus_areas(summary_df, max_items=3)
    activities = extract_activities_for_case(state, max_activities=5)

    concern_profile = ensure_concern_profile(state)
    top_concern_subdomains = [x["subdomain"] for x in concern_profile.get("top_subdomains", [])[:4]]

    domain_rows = []
    if summary_df is not None and not summary_df.empty:
        for _, row in summary_df.iterrows():
            gap = row.get("milestone_gap_months", row.get("estimated_delay_months", None))
            domain_rows.append({
                "category": row.get("category", ""),
                "estimated_dev_age_months": row.get("estimated_dev_age_months", ""),
                "gap_months": gap,
                "support_tier": row.get("planning_tier", row.get("support_tier", "")),
                "severity": severity_from_gap(gap if gap is not None else 0),
                "needs_special_support": row.get("needs_special_support", True),
                "summary": row.get("activity_bank_summary", ""),
            })

    support_categories = [r["category"] for r in domain_rows if r.get("support_tier") in {"needs_special_support", "monitor_and_enrich", "enrich_and_observe"}]
    if support_categories:
        summary_text = (
            f"This plan prioritizes {', '.join(support_categories[:3])} for {case['child_name']}. "
            f"The activities are intended to be short, parent-friendly, and aligned with the developmental areas that appeared most delayed or most worth enriching in the interview."
        )
    else:
        family_floor = state.get("family_guidance_floor", {})
        if family_floor.get("enabled"):
            summary_text = family_floor.get("summary", "")
        else:
            summary_text = (
                f"This output did not identify a strong need for special support across the reviewed categories, "
                f"but the family can continue monitoring development and bring remaining questions to the clinician."
            )

    payload = {
        "case_id": case.get("case_id"),
        "child_name": case.get("child_name"),
        "age_label": case.get("age_label"),
        "age_months": case.get("age_months"),
        "diagnosis": case.get("diagnosis"),
        "concerns_bullets": concerns_bullets,
        "daily_time_min": state.get("child", {}).get("daily_time_min", ""),
        "parent_qa_full": parent_qa_full,
        "parent_qa_pdf": parent_qa_pdf,
        "top_concern_subdomains": top_concern_subdomains,
        "domain_rows": domain_rows,
        "focus_areas": focus_areas,
        "activities": activities,
        "summary_text": summary_text,
    }
    return payload

def print_case_screen_summary(payload: dict):
    print("\n" + "=" * 110)
    print(f"CASE {payload['case_id']} | {payload['child_name']} | {payload['age_label']} | {payload['diagnosis']}")
    print("=" * 110)

    print("\n1. CHILD PROFILE")
    print(f"- Name: {payload['child_name']}")
    print(f"- Age: {payload['age_label']} ({payload['age_months']} months)")
    print(f"- Diagnosis: {payload['diagnosis']}")
    print(f"- Daily time available: {payload['daily_time_min']} min/day")
    print("- Key concerns:")
    for c in payload["concerns_bullets"]:
        print(f"  • {c}")
    if payload.get("top_concern_subdomains"):
        print("- Routed concern subdomains:")
        for s in payload["top_concern_subdomains"]:
            print(f"  • {s}")

    print("\n2. PARENT INPUT (full transcript)")
    for line in payload["parent_qa_full"]:
        print(f"- {line}")

    print("\n3. GENEX OUTPUT")
    domain_df = pd.DataFrame(payload["domain_rows"])
    if not domain_df.empty:
        display(domain_df[["category", "estimated_dev_age_months", "gap_months", "support_tier", "severity"]])

    print("\nFocus areas:")
    for x in payload["focus_areas"]:
        print(f"  • {x}")

    print("\nDaily activities:")
    for i, a in enumerate(payload["activities"], start=1):
        print(f"{i}. {a['title']} | {a['category']} | {a['duration_min']} min")
        print(f"   Goal: {a['goal']}")
        print(f"   Description: {a['description']}")

    print("\n4. SUMMARY")
    print(payload["summary_text"])

    print("\n5. ADVISOR REVIEW SECTION")
    print("Rate 1–5:")
    print("- Clinical appropriateness")
    print("- Safety")
    print("- Practicality for parents")
    print("- Clarity")
    print("- Overall usefulness")
    print("Short feedback:")
    print("- What would you change?")
    print("- What is missing?")
    print("- Any concerns?")


In [50]:
# ------------------------------------------------------------
# PDF rendering + rating sheet
# ------------------------------------------------------------
def wrap_text_to_width(text, font_name, font_size, max_width):
    words = str(text).split()
    if not words:
        return [""]

    lines = []
    current = words[0]
    for word in words[1:]:
        trial = current + " " + word
        if stringWidth(trial, font_name, font_size) <= max_width:
            current = trial
        else:
            lines.append(current)
            current = word
    lines.append(current)
    return lines

def draw_wrapped_text(c, text, x, y, max_width, font_name="Helvetica", font_size=8.5, line_height=10, max_lines=None):
    lines = wrap_text_to_width(text, font_name, font_size, max_width)
    original_lines = lines[:]
    if max_lines is not None:
        lines = lines[:max_lines]
        if len(original_lines) > max_lines and lines:
            lines[-1] = lines[-1][:max(0, len(lines[-1]) - 3)] + "..."
    c.setFont(font_name, font_size)
    for line in lines:
        c.drawString(x, y, line)
        y -= line_height
    return y

def draw_bullets(c, items, x, y, max_width, font_name="Helvetica", font_size=8.5, line_height=10, max_items=None):
    items = list(items)
    if max_items is not None:
        items = items[:max_items]

    for item in items:
        bullet_lines = wrap_text_to_width(str(item), font_name, font_size, max_width - 12)
        if bullet_lines:
            c.drawString(x, y, u"\u2022")
            c.drawString(x + 10, y, bullet_lines[0])
            y -= line_height
            for extra in bullet_lines[1:]:
                c.drawString(x + 10, y, extra)
                y -= line_height
        else:
            y -= line_height
    return y

def draw_box(c, x, y_top, w, h, title):
    c.roundRect(x, y_top - h, w, h, 8, stroke=1, fill=0)
    c.setFont("Helvetica-Bold", 10)
    c.drawString(x + 8, y_top - 14, title)

def render_case_page(c, payload: dict):
    page_w, page_h = landscape(LETTER)
    margin = 0.35 * inch

    left_x = margin
    mid_x = page_w / 2 + 0.10 * inch
    col_w = (page_w / 2) - (1.5 * margin)

    c.setFont("Helvetica-Bold", 15)
    c.drawString(margin, page_h - 0.45 * inch, f"{payload['case_id']} — {payload['child_name']} ({payload['age_label']})")
    c.setFont("Helvetica", 10)
    c.drawString(margin, page_h - 0.68 * inch, f"Diagnosis: {payload['diagnosis']}")

    top_y = page_h - 0.90 * inch
    profile_h = 1.60 * inch
    input_h = 2.55 * inch
    output_h = 2.20 * inch
    activity_h = 2.10 * inch
    summary_h = 0.85 * inch
    rating_h = 1.20 * inch

    draw_box(c, left_x, top_y, col_w, profile_h, "1. Child Profile")
    draw_box(c, left_x, top_y - profile_h - 0.10 * inch, col_w, input_h, "2. Parent Input (questions + answers)")
    draw_box(c, mid_x, top_y, col_w, output_h, "3. Genex Output")
    draw_box(c, mid_x, top_y - output_h - 0.10 * inch, col_w, activity_h, "4. Daily Activities")

    bottom_top = page_h - 6.65 * inch
    full_w = page_w - 2 * margin
    draw_box(c, margin, bottom_top, full_w, summary_h, "5. Summary")
    draw_box(c, margin, bottom_top - summary_h - 0.10 * inch, full_w, rating_h, "6. Advisor Review")

    y = top_y - 28
    c.setFont("Helvetica", 9)
    c.drawString(left_x + 10, y, f"Name: {payload['child_name']}")
    y -= 12
    c.drawString(left_x + 10, y, f"Age: {payload['age_label']} ({payload['age_months']} months)")
    y -= 12
    c.drawString(left_x + 10, y, f"Diagnosis: {payload['diagnosis']}")
    y -= 12
    c.drawString(left_x + 10, y, f"Daily time: {payload['daily_time_min']} min/day")
    y -= 16
    c.setFont("Helvetica-Bold", 9)
    c.drawString(left_x + 10, y, "Key concerns:")
    y -= 11
    y = draw_bullets(c, payload["concerns_bullets"], left_x + 12, y, col_w - 20, max_items=4)

    y = top_y - profile_h - 0.10 * inch - 28
    y = draw_bullets(c, payload["parent_qa_pdf"], left_x + 12, y, col_w - 20, font_size=7.8, line_height=9, max_items=12)
    c.setFont("Helvetica-Oblique", 7.5)
    c.drawString(left_x + 10, top_y - profile_h - input_h - 0.10 * inch + 10, "Full transcript is preserved in JSON output.")

    y = top_y - 28
    domain_rows = payload["domain_rows"][:4]
    c.setFont("Helvetica-Bold", 8.5)
    c.drawString(mid_x + 10, y, "Domain")
    c.drawString(mid_x + 150, y, "Dev age")
    c.drawString(mid_x + 220, y, "Gap")
    c.drawString(mid_x + 270, y, "Tier")
    y -= 11
    c.setFont("Helvetica", 8.2)
    for row in domain_rows:
        c.drawString(mid_x + 10, y, str(row["category"])[:24])
        c.drawString(mid_x + 150, y, str(row["estimated_dev_age_months"]))
        c.drawString(mid_x + 220, y, str(row["gap_months"]))
        c.drawString(mid_x + 270, y, str(row["support_tier"])[:18])
        y -= 10

    y -= 8
    c.setFont("Helvetica-Bold", 9)
    c.drawString(mid_x + 10, y, "Focus areas:")
    y -= 11
    y = draw_bullets(c, payload["focus_areas"], mid_x + 12, y, col_w - 20, max_items=3)

    y = top_y - output_h - 0.10 * inch - 28
    for i, a in enumerate(payload["activities"][:5], start=1):
        line = f"{i}. {a['title']} ({a['duration_min']} min) — {a['goal']}"
        y = draw_wrapped_text(c, line, mid_x + 10, y, col_w - 20, font_name="Helvetica-Bold", font_size=8.2, line_height=9, max_lines=2)
        y = draw_wrapped_text(c, a["description"], mid_x + 16, y, col_w - 26, font_name="Helvetica", font_size=7.8, line_height=8.5, max_lines=2)
        y -= 4

    y = bottom_top - 28
    y = draw_wrapped_text(c, payload["summary_text"], margin + 10, y, full_w - 20, font_name="Helvetica", font_size=8.5, line_height=10, max_lines=4)

    y = bottom_top - summary_h - 0.10 * inch - 28
    rating_lines = [
        "Rate 1–5: Clinical appropriateness | Safety | Practicality for parents | Clarity | Overall usefulness",
        "Short feedback: What would you change? | What is missing? | Any concerns?"
    ]
    for line in rating_lines:
        y = draw_wrapped_text(c, line, margin + 10, y, full_w - 20, font_name="Helvetica", font_size=9, line_height=11, max_lines=2)

    c.showPage()

def build_advisor_rating_sheet(df_cases: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df_cases.iterrows():
        rows.append({
            "case_id": row["case_id"],
            "child_name": row["child_name"],
            "age_months": row["age_months"],
            "diagnosis": row["diagnosis"],
            "clinical_appropriateness_1to5": "",
            "safety_1to5": "",
            "practicality_for_parents_1to5": "",
            "clarity_1to5": "",
            "overall_usefulness_1to5": "",
            "what_would_you_change": "",
            "what_is_missing": "",
            "any_concerns": "",
        })
    return pd.DataFrame(rows)


In [51]:

# ------------------------------------------------------------
# Advisor-report export runners
# ------------------------------------------------------------
def run_cases_and_export(
    df_cases: pd.DataFrame,
    mode: str = "live",   # "live" or "gemini"
    output_dir: str = "outputs/advisor_case_pack_v6",
    default_daily_time_min: int = 10,
    pause_between_cases: bool = True,
    gemini_model: str = "gemini-2.5-flash",
):
    """Run all cases and export the advisor packet files."""
    mode = str(mode).strip().lower()
    if mode not in {"live", "gemini"}:
        raise ValueError("mode must be 'live' or 'gemini'")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_pdf_path = out_dir / f"genex_case_report_{mode}_{run_stamp}.pdf"
    rating_csv_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.csv"
    rating_xlsx_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.xlsx"
    json_path = out_dir / f"case_payloads_{mode}_{run_stamp}.json"

    c = canvas.Canvas(str(combined_pdf_path), pagesize=landscape(LETTER))
    all_payloads = []
    rating_df = build_advisor_rating_sheet(df_cases)

    for i, (_, row) in enumerate(df_cases.iterrows(), start=1):
        case = row.to_dict()

        if mode == "live":
            state, summary_df = run_case_live_from_prefilled_case(
                case=case,
                default_daily_time_min=default_daily_time_min,
                min_yes_confirm=2,
                yes_ratio_confirm=0.60,
            )
        else:
            state, summary_df = run_case_gemini_from_prefilled_case(
                case=case,
                daily_time_min=default_daily_time_min,
                min_yes_confirm=2,
                yes_ratio_confirm=0.60,
                gemini_model=gemini_model,
                delay_between_questions_sec=5,
            )

        payload = build_case_payload(case, state, summary_df)
        all_payloads.append(payload)

        print_case_screen_summary(payload)
        render_case_page(c, payload)

        per_case_summary_path = out_dir / f"{case['case_id']}_{case['child_name']}_summary.csv"
        summary_df.to_csv(per_case_summary_path, index=False)

        if pause_between_cases and i < len(df_cases):
            input("\nPress Enter to continue to the next case...")

    c.save()

    rating_df.to_csv(rating_csv_path, index=False)
    rating_df.to_excel(rating_xlsx_path, index=False)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_payloads, f, indent=2)

    print("\n" + "=" * 110)
    print("DONE")
    print("=" * 110)
    print("Mode:", mode)
    print("Combined PDF:", combined_pdf_path)
    print("Rating CSV :", rating_csv_path)
    print("Rating XLSX:", rating_xlsx_path)
    print("Payload JSON:", json_path)

    return all_payloads, rating_df, combined_pdf_path

def run_cases_and_export_in_batches(
    df_cases: pd.DataFrame,
    batch_size: int = 5,
    gap_between_batches_sec: int = 120,
    mode: str = "gemini",
    output_dir: str = "outputs/advisor_case_pack_v6",
    default_daily_time_min: int = 10,
    pause_between_cases: bool = False,
    gemini_model: str = "gemini-2.5-flash",
):
    """Run advisor export in batches, useful for Gemini quota stability."""
    mode = str(mode).strip().lower()
    if mode not in {"live", "gemini"}:
        raise ValueError("mode must be 'live' or 'gemini'")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_pdf_path = out_dir / f"genex_case_report_{mode}_{run_stamp}.pdf"
    rating_csv_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.csv"
    rating_xlsx_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.xlsx"
    json_path = out_dir / f"case_payloads_{mode}_{run_stamp}.json"

    c = canvas.Canvas(str(combined_pdf_path), pagesize=landscape(LETTER))
    all_payloads = []
    rating_df = build_advisor_rating_sheet(df_cases)
    total_cases = len(df_cases)

    for start_idx in range(0, total_cases, batch_size):
        end_idx = min(start_idx + batch_size, total_cases)
        batch_df = df_cases.iloc[start_idx:end_idx]

        print("\n" + "=" * 110)
        print(f"RUNNING BATCH {start_idx // batch_size + 1} | cases {start_idx + 1} to {end_idx} of {total_cases}")
        print("=" * 110)

        for _, row in batch_df.iterrows():
            case = row.to_dict()

            if mode == "live":
                state, summary_df = run_case_live_from_prefilled_case(
                    case=case,
                    default_daily_time_min=default_daily_time_min,
                    min_yes_confirm=2,
                    yes_ratio_confirm=0.60,
                )
            else:
                state, summary_df = run_case_gemini_from_prefilled_case(
                    case=case,
                    daily_time_min=default_daily_time_min,
                    min_yes_confirm=2,
                    yes_ratio_confirm=0.60,
                    gemini_model=gemini_model,
                    delay_between_questions_sec=5,
                )

            payload = build_case_payload(case, state, summary_df)
            all_payloads.append(payload)

            print_case_screen_summary(payload)
            render_case_page(c, payload)

            per_case_summary_path = out_dir / f"{case['case_id']}_{case['child_name']}_summary.csv"
            summary_df.to_csv(per_case_summary_path, index=False)

            if pause_between_cases:
                input("\nPress Enter to continue to the next case...")

        if end_idx < total_cases:
            print("\n" + "-" * 110)
            print(f"Batch complete. Waiting {gap_between_batches_sec} seconds before next batch...")
            print("-" * 110)
            time.sleep(gap_between_batches_sec)

    c.save()
    rating_df.to_csv(rating_csv_path, index=False)
    rating_df.to_excel(rating_xlsx_path, index=False)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_payloads, f, indent=2)

    print("\n" + "=" * 110)
    print("DONE")
    print("=" * 110)
    print("Mode:", mode)
    print("Combined PDF:", combined_pdf_path)
    print("Rating CSV :", rating_csv_path)
    print("Rating XLSX:", rating_xlsx_path)
    print("Payload JSON:", json_path)

    return all_payloads, rating_df, combined_pdf_path


## Internal legacy layer: v19 comprehensive patch

In [52]:
# ------------------------------------------------------------
# Legacy v19 bridge-chain + coverage patch used by v21
# ------------------------------------------------------------
# This cell is intentionally table-first and deterministic by default.
# The LLM is allowed only as an optional wording helper; milestone selection,
# bridge selection, themes, avoids, and schedule coverage are controlled by code
# and by Sara's all_with_bridge table.

from collections import Counter, defaultdict

V19_VERSION = "v19"
V19_MIN_MILESTONES_PER_DOMAIN = 3
V19_MAX_MILESTONES_PER_DOMAIN = 5
V19_CORE_ACTIVITIES_PER_MILESTONE = 2
V19_DEFAULT_CYCLE_DAYS = 14
V19_USE_LLM_FOR_ACTIVITIES = str(os.environ.get("GENEX_USE_LLM_ACTIVITIES", "false")).lower() in {"1", "true", "yes", "y"}

V19_ACTIVITY_FEEDBACK_OPTIONS = {
    "difficulty": ["too_hard", "just_right", "too_easy"],
    "performance": ["done_independently", "done_with_help", "couldnt_do_it"],
    "engagement": ["enjoyed_it", "resisted_it", "didnt_like_it"],
}

# ------------------------------------------------------------------
# 1) Explicit parent concern lock: if parent names up to 2 concerns,
#    assess both domains. Daily time controls activities/day, not domains.
# ------------------------------------------------------------------
V19_EXPLICIT_DOMAIN_PATTERNS = {
    "language_and_communication": [
        r"\bspeech\b", r"\blanguage\b", r"\btalk", r"\bword", r"\bpronounc", r"\bcommunicat",
        r"\bregress.*speech", r"\blate talk", r"\bnot talking", r"\bexpressive\b"
    ],
    "movement_and_physical": [
        r"\bfine motor\b", r"\bhand\b", r"\bfinger", r"\bpencil\b", r"\bcrayon\b", r"\bscissor",
        r"\bbutton", r"\bdraw", r"\bscribb", r"\bgrasp", r"\bfeed", r"\bdress", r"\bclothes",
        r"\bgross motor\b", r"\bwalk", r"\brun", r"\bjump", r"\bbalance", r"\bwobbl", r"\bunsteady", r"\bgait"
    ],
    "social_and_emotional": [
        r"\bsocial\b", r"\bemotion", r"\bpeer", r"\bplay with", r"\bturn taking", r"\bsharing", r"\bbehavior", r"\bmeltdown"
    ],
    "cognitive": [
        r"\blearning\b", r"\bcognitive\b", r"\bunderstand", r"\bfocus", r"\battention", r"\bproblem", r"\badaptive", r"\broutine"
    ],
}


def v19_explicit_concern_domains(state: Dict[str, Any], max_domains: int = 2) -> List[str]:
    child = state.get("child", {})
    text = " | ".join(str(child.get(k, "")) for k in ["diagnosis", "condition", "concern", "parent_concern"]).lower()
    profile = ensure_concern_profile(state)
    scores = []
    for category_key, patterns in V19_EXPLICIT_DOMAIN_PATTERNS.items():
        matched = []
        for pat in patterns:
            if re.search(pat, text):
                matched.append(pat)
        # Also count concern-router direct domain weights as an explicit signal only when the text matched.
        if matched:
            router_weight = float(profile.get("domain_weights", {}).get(category_key, 0.0))
            rank_score = len(matched) * 10.0 + router_weight
            scores.append((rank_score, category_key, matched))
    scores.sort(reverse=True)
    selected = [category_key for _, category_key, _ in scores[:max_domains]]
    state.setdefault("focus_domain_triage", {})["explicit_concern_domains"] = selected
    state.setdefault("focus_domain_triage", {})["explicit_concern_matches"] = {category_key: matches for _, category_key, matches in scores}
    return selected


def choose_default_focus_domains_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    """v19 override: explicit parent concerns are locked into onboarding, up to 2 domains."""
    explicit = v19_explicit_concern_domains(state, max_domains=max_domains)
    ranked = rank_focus_domains_v14(state)
    selected = list(explicit)
    for item in ranked:
        if len(selected) >= max_domains:
            break
        k = item["category_key"]
        if k not in selected:
            # Fill only if there was one or zero explicit domains.
            selected.append(k)
    if not selected:
        selected = [r["category_key"] for r in ranked[:max_domains]] if ranked else list(DOMAIN_CONFIG.keys())[:max_domains]
    return selected[:max_domains]


def select_focus_domains_live_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    ranked = rank_focus_domains_v14(state)
    suggested = choose_default_focus_domains_v14(state, max_domains=max_domains)
    explicit = v19_explicit_concern_domains(state, max_domains=max_domains)

    print("\n" + "=" * 100)
    print("FOCUS DOMAIN TRIAGE — v19")
    print("=" * 100)
    print("Genex keeps onboarding focused, but explicitly named parent concerns are included up to 2 domains.")
    print("Daily time controls the number of daily activities, not whether a concern is evaluated.")
    if explicit:
        print("Explicit concern domains detected:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in explicit))
    print("Suggested focus areas:")
    for idx, item in enumerate(ranked[:4], start=1):
        marker = " [suggested]" if item["category_key"] in suggested else ""
        explicit_marker = " [explicit concern]" if item["category_key"] in explicit else ""
        print(f"  {idx}. {item['display']} | triage_score={item['triage_score']}{marker}{explicit_marker}")

    raw = input("Press Enter to accept the suggested focus areas, or type up to two numbers separated by commas: ").strip()
    if not raw:
        selected = suggested
        source = "v19_live_confirm_default"
    else:
        picked = []
        parts = [p.strip() for p in raw.split(",") if p.strip()]
        for p in parts[:max_domains]:
            if p.isdigit():
                idx = int(p) - 1
                if 0 <= idx < len(ranked):
                    picked.append(ranked[idx]["category_key"])
        selected = picked or suggested
        source = "v19_live_manual_override"
    selected = apply_focus_domain_selection_v14(state, selected, source=source)
    print("Selected focus areas:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in selected))
    return selected


def select_focus_domains_gemini_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    selected = apply_focus_domain_selection_v14(
        state,
        choose_default_focus_domains_v14(state, max_domains=max_domains),
        source="v19_synthetic_auto",
    )
    print("\n" + "=" * 100)
    print("FOCUS DOMAIN TRIAGE — v19")
    print("=" * 100)
    explicit = v19_explicit_concern_domains(state, max_domains=max_domains)
    if explicit:
        print("Explicit concern domains included:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in explicit))
    print("Selected focus areas for this onboarding run:")
    for k in selected:
        print(f"- {DOMAIN_CONFIG[k]['display']}")
    return selected


# ------------------------------------------------------------------
# 2) Better language readiness: do not treat every "speech delay" as
#    pre-single-word when answers show 36–48m emerging language.
# ------------------------------------------------------------------
def _v19_language_answer_flags(state: Dict[str, Any]) -> Dict[str, Any]:
    answers = [a for a in state.get("qna", {}).get("language_and_communication", []) if a.get("answer_status", "ok") != "api_error"]
    flags = {
        "has_early_word_attempts": False,
        "early_words_absent": False,
        "has_word_combinations": False,
        "word_combinations_absent": False,
        "has_36m_or_higher_emerging": False,
        "has_sentence_level_emerging": False,
    }
    for a in answers:
        ans = _v18_answer_norm(a)
        month = int(a.get("months", a.get("age_months", 0)) or 0)
        text = _v18_norm(str(a.get("milestone", "")))
        positive = ans in {"yes", "sometimes", "with_help"}
        absent = ans in {"no", "not_sure"}
        if re.search(r"tries.*(two|2|three|3).*word|besides mama|besides dada|ba for ball|da for dog", text):
            if positive:
                flags["has_early_word_attempts"] = True
            if absent:
                flags["early_words_absent"] = True
        if re.search(r"two words together|more milk|50 words|fifty words", text):
            if positive:
                flags["has_word_combinations"] = True
            if absent:
                flags["word_combinations_absent"] = True
        if month >= 36 and positive:
            flags["has_36m_or_higher_emerging"] = True
        if re.search(r"sentence|four or more words|conversation|back and forth|story|what action|questions", text) and positive:
            flags["has_sentence_level_emerging"] = True
    return flags


def compute_language_activity_readiness(state: Dict[str, Any]) -> Dict[str, Any]:
    flags = _v19_language_answer_flags(state)
    child = state.get("child", {})
    concern_text = " | ".join(str(child.get(k, "")) for k in ("diagnosis", "condition", "concern", "parent_concern"))
    concern_limited = bool(LANGUAGE_LIMITED_EXPRESSIVE_PATTERN.search(concern_text)) if "LANGUAGE_LIMITED_EXPRESSIVE_PATTERN" in globals() else False

    # Higher observed answers override the generic concern label "speech delay".
    if flags["has_sentence_level_emerging"] or flags["has_36m_or_higher_emerging"]:
        level = "sentence_building"
    elif flags["has_word_combinations"]:
        level = "phrase_ready"
    elif flags["word_combinations_absent"] and flags["has_early_word_attempts"]:
        level = "early_phrase_modeling_ok"
    elif flags["has_early_word_attempts"]:
        level = "single_word_building"
    elif concern_limited or flags["early_words_absent"]:
        level = "pre_or_early_single_words"
    else:
        level = "single_word_building"
    result = dict(flags)
    result.update({"level": level, "concern_suggests_limited_words": concern_limited})
    return result


def apply_language_readiness_constraints_to_activities(state: Dict[str, Any], category_key: str, activities: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """v19: preserve table-anchored activity fields; only adjust wording/readiness, never blank out cards."""
    if category_key != "language_and_communication":
        return activities
    readiness = compute_language_activity_readiness(state)
    level = readiness.get("level", "single_word_building")
    corrected = []
    for activity in activities:
        a = dict(activity)
        instructions = str(a.get("instructions", "")).strip()
        success = str(a.get("success_criteria", a.get("success", ""))).strip()
        avoid = str(a.get("what_to_avoid", "")).strip()
        cdc = str(a.get("cdc_goal", ""))
        active = str(a.get("activity_focus_step", a.get("bridge_step", "")))
        # For very early language, do not require phrases/sentences. Preserve all metadata.
        if level in {"pre_or_early_single_words", "single_word_building"}:
            if not re.search(r"point|gesture|sound|one word|approximation", instructions.lower()):
                instructions = instructions.rstrip(".") + ". Child success can be pointing, gesturing, copying a sound, or trying one word."
            if re.search(r"sentence|two-word|two word|phrase|four or more words", cdc.lower() + " " + active.lower()):
                success = f"The child participates at the bridge level: {active}. Do not require the final phrase or sentence yet."
                avoid = (avoid + " Do not require two-word phrases or sentences until single words are stable.").strip()
        elif level == "early_phrase_modeling_ok":
            if "model" not in instructions.lower():
                instructions = instructions.rstrip(".") + ". Parent may model a short phrase, but one word, gesture, or sound still counts as participation."
            if not success:
                success = f"The child tries the bridge step with a sound, word, gesture, or short phrase: {active}."
        else:
            # Sentence-building children can work on phrases/sentences, but still at the active bridge step.
            if "active bridge" not in instructions.lower() and active:
                instructions = instructions.rstrip(".") + f". Keep the focus on the active bridge step: {active}."
            if not success:
                success = f"The child attempts the active bridge step: {active}."
        a["instructions"] = instructions
        a["success_criteria"] = success
        a["success"] = success
        a["what_to_avoid"] = avoid or _v19_activity_specific_avoid(a)
        corrected.append(a)
    state.setdefault("planner_debug", {})["language_activity_readiness"] = readiness
    return corrected


# ------------------------------------------------------------------
# 3) Bridge-first activity targeting.
# ------------------------------------------------------------------
def _v19_text_similarity(a: str, b: str) -> float:
    ta = set(re.findall(r"[a-z0-9]+", _v18_norm(a)))
    tb = set(re.findall(r"[a-z0-9]+", _v18_norm(b)))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / max(1, min(len(ta), len(tb)))


def _v19_is_same_or_final_step(step: str, cdc_goal: str) -> bool:
    ns = _v18_norm(step)
    nc = _v18_norm(cdc_goal)
    if not ns or not nc:
        return False
    if ns == nc:
        return True
    if len(ns) > 20 and (ns in nc or nc in ns):
        return True
    return _v19_text_similarity(ns, nc) >= 0.82


def _v19_bridge_status_from_qna(state: Dict[str, Any], category_key: str, step_text: str) -> str:
    step_norm = _v18_norm(step_text)
    if not step_norm:
        return "not_required"
    best = (0.0, "unknown")
    for a in state.get("qna", {}).get(category_key, []):
        milestone_text = str(a.get("milestone", ""))
        sim = _v19_text_similarity(step_norm, milestone_text)
        if sim > best[0]:
            ans = _v18_answer_norm(a)
            if ans == "yes":
                status = "confirmed"
            elif ans in {"sometimes", "with_help"}:
                status = "emerging"
            elif ans in {"no", "not_sure"}:
                status = "not_confirmed"
            else:
                status = "unknown"
            best = (sim, status)
    if best[0] >= 0.55:
        return best[1]
    return "unknown"


def _v19_resolve_activity_focus(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any]) -> Dict[str, str]:
    cdc_goal = str(bridge.get("cdc_milestone", bridge.get("target_milestone", ""))).strip()
    bridge_step = str(bridge.get("bridge_step", "")).strip()
    prev = str(bridge.get("previous_bridge_step", "")).strip()
    prev_status = _v19_bridge_status_from_qna(state, category_key, prev)
    progress_status = _v18_progress_status_for_bridge(state, _v18_norm(prev)) if prev else "not_required"
    if progress_status != "unknown":
        prev_status = progress_status

    focus = bridge_step
    rule = "practice_active_bridge_step"
    # If the bridge row is actually the final CDC milestone, do not practice the final goal first.
    if prev and _v19_is_same_or_final_step(bridge_step, cdc_goal):
        focus = prev
        rule = "active_bridge_equals_cdc_goal__practice_previous_bridge_first"
    elif prev and prev_status in {"not_confirmed"}:
        focus = prev
        rule = "previous_bridge_not_confirmed__practice_previous_bridge_first"
    elif prev and prev_status == "unknown":
        # Unknown prerequisites should be checked/warmed up first, but the activity can still gently approach the bridge.
        focus = bridge_step
        rule = "previous_bridge_unknown__warm_up_then_active_bridge"
    if not focus:
        focus = bridge_step or prev or "Bridge step missing from table — review all_with_bridge."
        rule = "missing_bridge_data__needs_table_review"
    return {"activity_focus_step": focus, "bridge_focus_rule": rule, "previous_bridge_status": prev_status}


def _v19_select_active_bridge(state: Dict[str, Any], category_key: str, target: Dict[str, Any]) -> Dict[str, Any]:
    rows = get_bridge_rows_for_target(category_key, int(target.get("months", 0)), target.get("milestone", ""), target.get("subdomain", "unspecified"))
    if not rows:
        row = {
            "target_milestone": target.get("milestone", ""),
            "cdc_milestone": target.get("milestone", ""),
            "months": int(target.get("months", 0) or 0),
            "subdomain": target.get("subdomain", "unspecified"),
            "bridge_step_number": 1,
            "bridge_step": "Bridge step missing from table — review all_with_bridge.",
            "previous_bridge_step": "",
            "previous_anchor_age": "",
            "allowed_themes": infer_allowed_themes(category_key, target.get("subdomain", ""), target.get("milestone", ""), ""),
            "what_to_avoid": infer_what_to_avoid(category_key, target.get("subdomain", ""), target.get("milestone", ""), ""),
            "source": "missing_bridge_table_row",
            "answer_norm": target.get("answer_norm", "not_sure"),
            "selection_reason": target.get("selection_reason", ""),
        }
    else:
        # v19 deliberately starts with the first bridge step for each target milestone.
        # This fixes the old drift toward practicing the final CDC milestone too directly.
        row = dict(rows[0])
        row["target_milestone"] = target.get("milestone", "")
        row["cdc_milestone"] = target.get("milestone", "")
        row["months"] = int(target.get("months", row.get("months", 0)) or 0)
        row["answer_norm"] = str(target.get("answer_norm", "not_sure")).lower()
        row["selection_reason"] = target.get("selection_reason", "")
        row["allowed_themes"] = _v18_allowed_themes_for_row(row)
        row["what_to_avoid"] = _v18_avoid_for_row(row)
        row["source"] = "all_with_bridge"
    row.update(_v19_resolve_activity_focus(state, category_key, row))
    return row


def build_bridge_plan_for_category(state: Dict[str, Any], category_key: str, target_milestones: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    targets = target_milestones or select_next_milestones(state, category_key).get("milestones", [])
    active_bridges = [_v19_select_active_bridge(state, category_key, target) for target in targets]
    plan = {
        "status": "ok" if active_bridges else "no_targets",
        "version": V19_VERSION,
        "category_key": category_key,
        "target_milestones": targets,
        "active_bridge_steps": active_bridges,
        "logic": "v19 chooses several target milestones per domain, then practices the earliest bridge/prerequisite step for each milestone before the final CDC goal.",
    }
    state.setdefault("bridge_plans", {})[category_key] = plan
    return plan


# ------------------------------------------------------------------
# 4) Allowed themes + activity-specific avoid rules.
# ------------------------------------------------------------------
def _v19_allowed_themes_for_bridge(category_key: str, bridge: Dict[str, Any]) -> List[str]:
    raw = bridge.get("allowed_themes", [])
    themes = _v18_parse_list(raw) if isinstance(raw, str) else list(raw or [])
    if themes:
        return _v18_unique([str(t).strip() for t in themes if str(t).strip()])
    text = " ".join(str(bridge.get(k, "")) for k in ["cdc_milestone", "bridge_step", "activity_focus_step", "subdomain"]).lower()
    if re.search(r"ball|catch|throw|toss|roll", text):
        return ["soft ball game", "basket toss", "parent-child roll game", "sibling ball turn", "beanbag target"]
    if re.search(r"action|picture|book|what.*happen|running|eating|playing", text):
        return ["picture action game", "book game", "puppet action game", "family acting game", "action card game"]
    if re.search(r"question|what is.*for|why|where|who", text):
        return ["object function game", "mystery bag", "kitchen helper", "picture question game", "pretend shopping"]
    if re.search(r"sentence|words together|phrase|conversation", text):
        return ["puppet phrase game", "photo talk", "story builder", "pretend play", "family conversation turn"]
    if re.search(r"song|rhyme|story|nursery", text):
        return ["song game", "rhyme pause game", "story echo", "puppet song", "family singing turn"]
    if re.search(r"crayon|scribble|draw|pencil|marker", text):
        return ["family art turn", "crayon road", "sticker and scribble", "vertical easel", "mail-the-picture game"]
    if re.search(r"button|zip|cloth|dress|jacket|pants|shirt", text):
        return ["dress-up game", "doll dressing", "laundry helper", "morning routine", "sibling helper"]
    if re.search(r"finger|feed|food|spoon|cup|pour", text):
        return ["snack routine", "pretend restaurant", "kitchen helper", "doll feeding", "picnic game"]
    if re.search(r"walk|run|jump|climb|step|stairs|balance|stand|bend|knee|bounce", text):
        return ["safe stepping game", "floor sticker path", "toy reach", "animal movement game", "music movement"]
    if category_key == "language_and_communication":
        return ["book game", "puppet play", "family photo game", "toy choice game", "song game"]
    if category_key == "movement_and_physical":
        return ["fine-motor treasure game", "floor sticker game", "block game", "pretend play", "family helper game"]
    return ["pretend play", "turn-taking game", "helper role game", "toy rescue", "family game"]


def _v19_condition_safety_text(state: Dict[str, Any], category_key: str) -> str:
    child = state.get("child", {})
    text = " ".join(str(child.get(k, "")) for k in ["diagnosis", "condition", "concern", "parent_concern"]).lower()
    if category_key == "movement_and_physical" and re.search(r"dravet|seizure|epilep|wobbl|unstable|unsteady|ataxia|gait|fall", text):
        return " For seizure, wobbly walking, or fall-risk profiles, use close adult support and avoid racing, obstacles, climbing onto high furniture, fast movement, jumping from surfaces, unsupported balance, or fatigue."
    return ""


def _v19_activity_specific_avoid(activity_or_bridge: Dict[str, Any], state: Optional[Dict[str, Any]] = None, category_key: Optional[str] = None) -> str:
    text = " ".join(str(activity_or_bridge.get(k, "")) for k in ["cdc_goal", "cdc_milestone", "target_milestone", "bridge_step", "activity_focus_step", "title", "subdomain"]).lower()
    avoid_parts = []
    if re.search(r"ball|catch|throw|toss|roll", text):
        avoid_parts.append("Avoid hard balls, fast throws, throwing at the face, long-distance throws, or forcing repeated catches.")
    elif re.search(r"jump|bounce|hop", text):
        avoid_parts.append("Avoid jumping from furniture, stairs, high surfaces, fast races, or continuing if the child is unsteady.")
    elif re.search(r"climb|couch|chair|stairs|step", text):
        avoid_parts.append("Avoid high furniture, unstable surfaces, rushing, or letting the child climb without close support.")
    elif re.search(r"crayon|scribble|draw|pencil|marker", text):
        avoid_parts.append("Avoid forcing finger position, long writing drills, sharp tools, or correcting every mark.")
    elif re.search(r"button|zip|cloth|dress|shirt|pants|jacket", text):
        avoid_parts.append("Avoid tight clothing, rushing, repeated correction, or making dressing feel like a test.")
    elif re.search(r"food|feed|snack|spoon|cup|drink|pour", text):
        avoid_parts.append("Avoid choking-risk foods, hot liquids, spills near electronics, or unsupervised eating/drinking practice.")
    elif re.search(r"sentence|phrase|two words|four or more words|question|song|story|word|sound|name|action", text):
        avoid_parts.append("Avoid pressure to perform, repeated correction, or requiring longer speech than the active bridge step.")
    else:
        avoid_parts.append("Avoid pressure, testing, repeated correction, or continuing when the child is frustrated.")
    if state is not None:
        avoid_parts.append(_v19_condition_safety_text(state, category_key or str(activity_or_bridge.get("category_key", ""))).strip())
    return " ".join(p for p in avoid_parts if p).strip()


# ------------------------------------------------------------------
# 5) Deterministic bridge-centered activity templates.
# ------------------------------------------------------------------
def _v19_group_play_line(bridge: Dict[str, Any], theme: str) -> str:
    cdc = str(bridge.get("cdc_milestone", bridge.get("target_milestone", ""))).lower()
    focus = str(bridge.get("activity_focus_step", bridge.get("bridge_step", ""))).lower()
    text = cdc + " " + focus
    if re.search(r"ball|catch|throw|toss|roll", text):
        return "2+ people: one adult rolls or gently tosses, another models a turn, and the child joins with support."
    if re.search(r"action|picture|book|happening", text):
        return "2+ people: one person acts out or shows the picture, another gives the cue, and the child points, gestures, or says the action."
    if re.search(r"question|what is.*for|where|why|who", text):
        return "2+ people: one person holds the object or picture, another asks the simple question, and the child answers by pointing, gesture, or words."
    if re.search(r"song|rhyme|story", text):
        return "2+ people: one person sings or reads, another pauses for the child’s turn, and everyone celebrates any sound, word, or gesture."
    if re.search(r"sentence|phrase|words together|conversation", text):
        return "2+ people: one person models the short phrase, another acts it out, and the child tries the bridge-level response."
    if re.search(r"crayon|scribble|draw|pencil|marker", text):
        return "2+ people: one person makes a simple mark, another takes a turn, and the child adds their own mark without pressure."
    if re.search(r"button|dress|cloth|shirt|pants|jacket", text):
        return "2+ people: one person models on a doll or clothing item, another gives one cue, and the child tries one small dressing step."
    if re.search(r"feed|food|spoon|cup|pour", text):
        return "2+ people: one person models the small movement, another offers the item safely, and the child tries one supported turn."
    if re.search(r"walk|run|jump|climb|step|balance|stand|bend|knee|bounce", text):
        return "2+ people: one person models one slow safe turn while another stays close to support the child."
    return "2+ people: one person models the bridge step, another gives a simple cue, and the child joins at their own level."


def _v19_materials_for_activity(theme: str, bridge: Dict[str, Any]) -> str:
    text = " ".join([theme, str(bridge.get("cdc_milestone", "")), str(bridge.get("activity_focus_step", ""))]).lower()
    if re.search(r"ball|catch|throw|toss|roll", text):
        return "soft ball or beanbag, laundry basket or box, clear floor space"
    if re.search(r"action|picture|book", text):
        return "picture book, action cards, family photos, or simple drawings"
    if re.search(r"question|object|function|kitchen", text):
        return "familiar household objects such as coat, cup, spoon, crayon, toy car"
    if re.search(r"song|rhyme|story", text):
        return "favorite song, nursery rhyme, storybook, puppet or stuffed animal"
    if re.search(r"crayon|scribble|draw|pencil|marker", text):
        return "chunky crayons or markers, paper, stickers, clipboard or vertical surface"
    if re.search(r"button|dress|cloth|shirt|pants|jacket", text):
        return "loose clothing item, doll/teddy, large buttons or zipper board"
    if re.search(r"feed|food|spoon|cup|pour", text):
        return "safe snack pieces or pretend food, spoon/cup, tray, towel for cleanup"
    if re.search(r"walk|run|jump|climb|step|balance|stand|bend|knee|bounce", text):
        return "floor tape or stickers, favorite toy, stable support, clear flat floor"
    return "common household toys or materials related to the bridge step"


def _v19_core_title_and_instructions(bridge: Dict[str, Any], theme: str, variant: int) -> Tuple[str, str]:
    cdc = str(bridge.get("cdc_milestone", "the CDC milestone"))
    focus = str(bridge.get("activity_focus_step", bridge.get("bridge_step", "the bridge step")))
    prev = str(bridge.get("previous_bridge_step", "")).strip()
    text = (cdc + " " + focus + " " + theme).lower()
    warmup = f"Start by checking the previous step: {prev}. " if prev else ""
    # Two distinct core activities for each milestone/bridge.
    if re.search(r"ball|catch|throw|toss|roll", text):
        if variant == 1:
            return "Soft Ball Get-and-Give", f"{warmup}Sit or stand close together and gently roll or toss a soft ball. The focus is {focus}, not the final milestone. Let the child reach, trap, drop, or toss at their current level."
        return "Basket Ball Drop/Toss", f"{warmup}Place a laundry basket or box very close. Practice {focus} by dropping, rolling, or softly tossing the ball into the target. Keep turns short and playful."
    if re.search(r"action|picture|book|happening", text):
        if variant == 1:
            return "Picture Action Point-and-Try", f"{warmup}Show one clear action picture. Ask what is happening. The focus is {focus}; pointing, acting it out, one action word, or an approximation can count."
        return "Family Action Guess", f"{warmup}A parent or sibling acts out one simple action like eating, sleeping, or jumping. The child guesses at the bridge level: {focus}."
    if re.search(r"question|what is.*for|what.*for|object function", text):
        if variant == 1:
            return "What Is It For? Basket", f"{warmup}Put 2–3 familiar objects in a basket. Pick one and ask a simple function question. Keep the focus on {focus}; pointing, gesture, or a short answer can count."
        return "Kitchen Helper Question", f"{warmup}During a real routine, hold up one familiar item and ask what it is for. Practice {focus} with modeling and choices."
    if re.search(r"sentence|four or more words", text):
        if variant == 1:
            return "Puppet Action Phrase Builder", f"{warmup}Use a puppet to do one action. Practice the bridge step {focus}; do not demand the final 4-word sentence until the bridge is easy."
        return "Photo Talk Builder", f"{warmup}Use a family photo or picture. Model a short bridge-level phrase and let the child copy, complete, or gesture toward {focus}."
    if re.search(r"song|rhyme|story|nursery", text):
        if variant == 1:
            return "Pause-and-Fill Song", f"{warmup}Sing a very familiar song and pause before one favorite word. The focus is {focus}; any sound, word attempt, or gesture can count if appropriate."
        return "Story Word Echo", f"{warmup}Read one short repeated line from a favorite story. Invite the child to echo or fill in one bridge-level word or phrase: {focus}."
    if re.search(r"word|sound|mama|dada|besides", text):
        if variant == 1:
            return "Toy Name Sound Hunt", f"{warmup}Choose 2–3 favorite toys. Model one simple sound or word, then wait. The focus is {focus}; accept approximations."
        return "Puppet First Words", f"{warmup}The puppet asks for or names one favorite item. Practice {focus} with playful imitation and no pressure."
    if re.search(r"direction|give it to me|one step", text):
        if variant == 1:
            return "Give-Me Toy Game", f"{warmup}Use one favorite toy and give one clear direction. Practice {focus}; add gesture only if the previous step is still needed."
        return "Clean-Up One-Step", f"{warmup}During cleanup, give one simple direction and celebrate any attempt at {focus}. Keep it short and predictable."
    if re.search(r"crayon|scribble|draw|pencil|marker", text):
        if variant == 1:
            return "Chunky Crayon Road", f"{warmup}Draw a simple road and invite the child to make marks with a chunky crayon. The focus is {focus}, not perfect drawing."
        return "Sticker-and-Scribble Turn", f"{warmup}Place a sticker on paper and invite one mark near it. Practice {focus} in short turns."
    if re.search(r"button|dress|cloth|shirt|pants|jacket", text):
        if variant == 1:
            return "Doll Dress-Up Step", f"{warmup}Use a doll, teddy, or loose clothing item. Practice one small dressing movement focused on {focus}."
        return "Laundry Helper Step", f"{warmup}During laundry or morning routine, invite one small clothing step. Keep the target at the bridge level: {focus}."
    if re.search(r"feed|food|finger|spoon|cup|pour", text):
        if variant == 1:
            return "Snack Pickup Practice", f"{warmup}Use safe snack pieces or pretend food. Practice {focus} with one or two playful turns."
        return "Pretend Restaurant Turn", f"{warmup}Set up a pretend restaurant with a doll or plate. Practice the bridge step {focus} using safe, supervised movements."
    if re.search(r"jump|bend|knee|bounce|hop", text):
        if variant == 1:
            return "Frog Knees Game", f"{warmup}Pretend to be frogs by bending knees and standing tall. The focus is {focus}; do not ask for full jumping until the bridge is mastered."
        return "Low Step-Down Treasure", f"{warmup}Use a very low stable surface or floor marker. Practice the bridge step {focus} slowly with support."
    if re.search(r"climb|couch|chair|step|balance|stand|walk|run", text):
        if variant == 1:
            return "Supported Toy Reach", f"{warmup}Place a favorite toy nearby and practice the bridge step {focus} slowly with close adult support."
        return "Safe Floor Sticker Path", f"{warmup}Use floor stickers or tape to guide one small safe movement. The target is {focus}, not speed or distance."
    if variant == 1:
        return f"{focus[:36]} Practice", f"{warmup}Use the {theme} format to practice the active bridge step: {focus}. Keep it playful and stop before frustration."
    return f"{focus[:36]} Family Turn", f"{warmup}Take short family turns practicing the active bridge step: {focus}. Keep success easy and low-pressure."


def _v19_make_activity(category_key: str, bridge: Dict[str, Any], activity_type: str, variant: int, state: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    allowed = _v19_allowed_themes_for_bridge(category_key, bridge)
    # Avoid repeating the same theme for the two core variants when possible.
    theme = allowed[(variant - 1) % len(allowed)] if allowed else "playful home routine"
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key.replace("_", " ").title())
    cdc_goal = str(bridge.get("cdc_milestone", bridge.get("target_milestone", ""))).strip()
    active_bridge = str(bridge.get("activity_focus_step", bridge.get("bridge_step", ""))).strip()
    table_bridge = str(bridge.get("bridge_step", "")).strip()
    prev = str(bridge.get("previous_bridge_step", "")).strip()
    avoid = _v19_activity_specific_avoid({**bridge, "activity_focus_step": active_bridge, "cdc_goal": cdc_goal}, state, category_key)
    title, instructions = _v19_core_title_and_instructions({**bridge, "activity_focus_step": active_bridge}, theme, variant)
    if activity_type == "easier_backup":
        title = "Easy " + title
        instructions = (f"Make the activity easier by practicing the previous bridge first: {prev}. " if prev else "Make the activity easier with more modeling and fewer choices. ") + instructions
        success = f"The child participates in an easier version of the active bridge step: {active_bridge}."
        easier = "Use more modeling, shorter turns, fewer choices, or hands-on support."
        harder = f"Return to the core activity when this feels comfortable: {active_bridge}."
    elif activity_type == "harder_stretch":
        title = "Stretch " + title
        instructions = instructions + " Only use this stretch if the core version is easy and enjoyable."
        success = f"The child does the active bridge step with less help or more independence: {active_bridge}."
        easier = f"Go back to the core version or previous bridge: {prev or active_bridge}."
        harder = f"Gently connect toward the CDC goal only after the bridge is easy: {cdc_goal}."
    else:
        success = f"The child attempts the active bridge step with support: {active_bridge}."
        easier = f"Warm up with the previous bridge: {prev}." if prev else "Make the turn shorter, slower, or more modeled."
        harder = "Reduce adult help slightly or add one extra turn only if the child enjoys it."
    return {
        "activity_id": f"{category_key}_{_v18_norm(cdc_goal)[:22]}_b{bridge.get('bridge_step_number', 1)}_{activity_type}_{variant}",
        "title": title,
        "activity_type": activity_type,
        "theme": theme,
        "play_theme": theme,
        "category": category_display,
        "category_key": category_key,
        "duration_min": 5,
        "cdc_goal": cdc_goal,
        "target_milestone": cdc_goal,
        "bridge_step": active_bridge,
        "table_bridge_step": table_bridge,
        "activity_focus_step": active_bridge,
        "previous_bridge_step": prev,
        "previous_bridge_status": bridge.get("previous_bridge_status", "unknown"),
        "bridge_focus_rule": bridge.get("bridge_focus_rule", "practice_active_bridge_step"),
        "allowed_themes": allowed,
        "what_to_avoid": avoid,
        "why_it_helps": f"This practices the active bridge step — {active_bridge} — which builds toward the CDC goal: {cdc_goal}.",
        "instructions": instructions,
        "success_criteria": success,
        "success": success,
        "make_easier": easier,
        "make_harder": harder,
        "group_play_line": _v19_group_play_line({**bridge, "activity_focus_step": active_bridge}, theme),
        "group_play": _v19_group_play_line({**bridge, "activity_focus_step": active_bridge}, theme),
        "materials": _v19_materials_for_activity(theme, {**bridge, "activity_focus_step": active_bridge}),
        "level": "bridge_step",
        "goal": "bridge_practice",
        "is_extended_activity": False,
        "extended_reason": "",
        "milestone_key": _v18_norm(cdc_goal),
        "bridge_step_number": bridge.get("bridge_step_number", 1),
        "subdomain": bridge.get("subdomain", "unspecified"),
    }


def _v19_validate_activity(activity: Dict[str, Any], bridge: Dict[str, Any], category_key: str, state: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    a = dict(activity)
    cdc_goal = str(bridge.get("cdc_milestone", bridge.get("target_milestone", a.get("cdc_goal", "")))).strip()
    focus = str(bridge.get("activity_focus_step", bridge.get("bridge_step", a.get("bridge_step", "")))).strip()
    prev = str(bridge.get("previous_bridge_step", a.get("previous_bridge_step", ""))).strip()
    # Force table/bridge anchoring.
    a["cdc_goal"] = cdc_goal
    a["target_milestone"] = cdc_goal
    a["bridge_step"] = focus
    a["activity_focus_step"] = focus
    a["previous_bridge_step"] = prev
    a["table_bridge_step"] = bridge.get("bridge_step", a.get("table_bridge_step", focus))
    a["allowed_themes"] = _v19_allowed_themes_for_bridge(category_key, bridge)
    if _v18_norm(a.get("theme", "")) not in {_v18_norm(t) for t in a["allowed_themes"]}:
        a["theme"] = a["allowed_themes"][0] if a["allowed_themes"] else "playful home routine"
        a["play_theme"] = a["theme"]
    required_defaults = {
        "title": f"{focus[:40]} Practice",
        "why_it_helps": f"This practices the active bridge step — {focus} — which builds toward the CDC goal: {cdc_goal}.",
        "instructions": f"Practice the active bridge step: {focus}. Use the previous bridge as a warm-up if needed: {prev}.",
        "success_criteria": f"The child attempts the active bridge step with support: {focus}.",
        "make_easier": f"Warm up with the previous bridge: {prev}." if prev else "Make the turn shorter, slower, or more modeled.",
        "make_harder": "Reduce adult help slightly only if the child is successful and enjoying it.",
        "group_play_line": _v19_group_play_line({**bridge, "activity_focus_step": focus}, a.get("theme", "")),
        "what_to_avoid": _v19_activity_specific_avoid({**bridge, "activity_focus_step": focus, "cdc_goal": cdc_goal}, state, category_key),
        "materials": _v19_materials_for_activity(a.get("theme", ""), {**bridge, "activity_focus_step": focus}),
    }
    for key, default in required_defaults.items():
        if not str(a.get(key, "")).strip():
            a[key] = default
    a["success"] = a.get("success_criteria", a.get("success", ""))
    if not str(a.get("activity_id", "")).strip():
        a["activity_id"] = f"{category_key}_{_v18_norm(cdc_goal)[:22]}_{_v18_norm(focus)[:18]}_{a.get('activity_type','core')}"
    return a


# Optional LLM prompt, disabled by default unless GENEX_USE_LLM_ACTIVITIES=true.
def _v19_make_activity_prompt(state: Dict[str, Any], category_key: str, active_bridges: List[Dict[str, Any]], planning_tier: str) -> str:
    contexts = []
    for idx, b in enumerate(active_bridges, start=1):
        contexts.append({
            "context_id": f"ctx_{idx}",
            "cdc_goal": b.get("cdc_milestone", b.get("target_milestone", "")),
            "previous_bridge_step": b.get("previous_bridge_step", ""),
            "table_bridge_step": b.get("bridge_step", ""),
            "activity_focus_step": b.get("activity_focus_step", b.get("bridge_step", "")),
            "bridge_focus_rule": b.get("bridge_focus_rule", ""),
            "allowed_themes": _v19_allowed_themes_for_bridge(category_key, b),
            "what_to_avoid": _v19_activity_specific_avoid(b, state, category_key),
        })
    child = state.get("child", {})
    return json.dumps({
        "task": "Create Genex v19 bridge-centered activities. Use ONLY the provided CDC goal, activity_focus_step, allowed themes, and what_to_avoid. Do not practice the final CDC goal directly unless activity_focus_step is the final approved step. Create 2 distinct core activities, 1 easier backup, and 1 harder stretch per context. Group play must involve 2+ people with specific roles.",
        "child": {
            "name": child.get("name"),
            "age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "concern": child.get("concern"),
            "daily_time_min": child.get("daily_time_min"),
        },
        "category": DOMAIN_CONFIG.get(category_key, {}).get("display", category_key),
        "contexts": contexts,
        "required_json_shape": {
            "activities": [{
                "context_id": "ctx_1",
                "activity_type": "core | core | easier_backup | harder_stretch",
                "title": "specific parent-friendly activity title",
                "theme": "one allowed theme",
                "instructions": "must directly practice activity_focus_step",
                "success_criteria": "success at activity_focus_step, not final CDC goal unless appropriate",
                "make_easier": "...",
                "make_harder": "...",
                "group_play_line": "2+ people: specific roles...",
                "materials": "...",
                "duration_min": 5
            }]
        }
    }, ensure_ascii=False, indent=2)


# ------------------------------------------------------------------
# 6) Activity bank generation: 2 core activities per milestone + 1 easy + 1 stretch.
# ------------------------------------------------------------------
def generate_category_activity_bank(state: Dict[str, Any], category_key: str, model: str = "gpt-4o-mini", activities_per_category: Optional[int] = None) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key.replace("_", " ").title())
    support_tier = get_support_tier(state, category_key)
    planning_tier = "enrich_and_observe" if is_family_guidance_category(state, category_key) else support_tier
    next_steps = select_next_milestones(state, category_key, max_milestones=V19_MAX_MILESTONES_PER_DOMAIN, min_milestones=V19_MIN_MILESTONES_PER_DOMAIN)
    if next_steps.get("status") == "no_special_support":
        bank = {"status": "no_special_support", "version": V19_VERSION, "support_tier": support_tier, "planning_tier": planning_tier, "summary": next_steps.get("message", ""), "activities": []}
        state.setdefault("activity_banks", {})[category_key] = bank
        return bank

    targets = next_steps.get("milestones", [])
    bridge_plan = build_bridge_plan_for_category(state, category_key, targets)
    active_bridges = bridge_plan.get("active_bridge_steps", [])
    activities: List[Dict[str, Any]] = []
    used_llm = False

    # Deterministic table-first activities are the default because this system must be stable.
    for idx, bridge in enumerate(active_bridges, start=1):
        activities.append(_v19_make_activity(category_key, bridge, "core", 1, state))
        activities.append(_v19_make_activity(category_key, bridge, "core", 2, state))
        activities.append(_v19_make_activity(category_key, bridge, "easier_backup", 1, state))
        activities.append(_v19_make_activity(category_key, bridge, "harder_stretch", 2, state))

    # Optional LLM wording helper. We only use it if explicitly enabled and only after validation.
    if V19_USE_LLM_FOR_ACTIVITIES and openai_client is not None and active_bridges:
        try:
            prompt = _v19_make_activity_prompt(state, category_key, active_bridges, planning_tier)
            resp = openai_client.chat.completions.create(
                model=model,
                temperature=0.05,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "You return strict JSON only and must obey table-provided CDC goals, activity_focus_step, allowed themes, and avoid rules."},
                    {"role": "user", "content": prompt},
                ],
            )
            parsed = json.loads(resp.choices[0].message.content)
            by_context = defaultdict(list)
            for raw in parsed.get("activities", []):
                by_context[str(raw.get("context_id", ""))].append(raw)
            llm_activities = []
            for i, bridge in enumerate(active_bridges, start=1):
                context_id = f"ctx_{i}"
                raws = by_context.get(context_id, [])
                # Keep at most 2 core + 1 easier + 1 harder.
                ordered = []
                cores = [r for r in raws if r.get("activity_type") == "core"][:2]
                ordered.extend(cores)
                for typ in ["easier_backup", "harder_stretch"]:
                    match = next((r for r in raws if r.get("activity_type") == typ), None)
                    if match:
                        ordered.append(match)
                # Fill missing with deterministic.
                while len([r for r in ordered if r.get("activity_type") == "core"]) < 2:
                    ordered.append({"activity_type": "core"})
                if not any(r.get("activity_type") == "easier_backup" for r in ordered):
                    ordered.append({"activity_type": "easier_backup"})
                if not any(r.get("activity_type") == "harder_stretch" for r in ordered):
                    ordered.append({"activity_type": "harder_stretch"})
                core_count = 0
                for raw in ordered[:4]:
                    typ = raw.get("activity_type", "core")
                    if typ == "core":
                        core_count += 1
                        variant = core_count
                    elif typ == "easier_backup":
                        variant = 1
                    else:
                        variant = 2
                    base = _v19_make_activity(category_key, bridge, typ, variant, state)
                    # Use only safe LLM wording fields; structure remains table-driven.
                    for key in ["title", "instructions", "success_criteria", "make_easier", "make_harder", "materials", "group_play_line"]:
                        val = str(raw.get(key, "")).strip()
                        if val:
                            base[key] = val
                    theme = str(raw.get("theme", "")).strip()
                    allowed = _v19_allowed_themes_for_bridge(category_key, bridge)
                    if _v18_norm(theme) in {_v18_norm(t) for t in allowed}:
                        base["theme"] = theme
                        base["play_theme"] = theme
                    llm_activities.append(_v19_validate_activity(base, bridge, category_key, state))
            # Use LLM activities only if every target has at least 2 cores.
            expected_min = len(active_bridges) * 4
            if len(llm_activities) >= expected_min:
                activities = llm_activities
                used_llm = True
        except Exception as exc:
            print(f"v19 optional LLM activity helper skipped for {category_display}: {exc}")

    # Readiness/safety overlays must preserve table fields.
    activities = apply_language_readiness_constraints_to_activities(state, category_key, activities)
    activities = v19_apply_safety_constraints_to_activities(state, category_key, activities)

    # Final validation: no blank cards and no detached CDC/bridge fields.
    bridge_by_goal = {_v18_norm(b.get("cdc_milestone", b.get("target_milestone", ""))): b for b in active_bridges}
    validated = []
    for act in activities:
        bridge = bridge_by_goal.get(_v18_norm(act.get("cdc_goal", "")))
        if bridge is None:
            # Fallback to milestone_key if needed.
            bridge = bridge_by_goal.get(_v18_norm(act.get("target_milestone", "")))
        if bridge is None and active_bridges:
            bridge = active_bridges[0]
        if bridge is not None:
            act = _v19_validate_activity(act, bridge, category_key, state)
        validated.append(act)

    # Guarantee at least two core activities per active bridge.
    by_goal_core = defaultdict(list)
    for a in validated:
        if a.get("activity_type") == "core":
            by_goal_core[_v18_norm(a.get("cdc_goal", ""))].append(a)
    for bridge in active_bridges:
        g = _v18_norm(bridge.get("cdc_milestone", bridge.get("target_milestone", "")))
        while len(by_goal_core[g]) < 2:
            new_act = _v19_make_activity(category_key, bridge, "core", len(by_goal_core[g]) + 1, state)
            validated.append(new_act)
            by_goal_core[g].append(new_act)

    bank = {
        "status": "ok",
        "version": V19_VERSION,
        "support_tier": support_tier,
        "planning_tier": planning_tier,
        "category_key": category_key,
        "category": category_display,
        "target_milestones": targets,
        "active_bridge_steps": active_bridges,
        "activities": validated,
        "repeat_window_days": 14,
        "used_llm": used_llm,
        "activity_bank_rule": "2 core activities + 1 easier backup + 1 harder stretch per selected milestone/active bridge.",
        "summary": "v19 multi-milestone, bridge-first activity bank with explicit concern coverage, activity-specific avoids, and theme compatibility.",
    }
    state.setdefault("activity_banks", {})[category_key] = bank
    return bank


def v19_apply_safety_constraints_to_activities(state: Dict[str, Any], category_key: str, activities: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    profile = ensure_safety_profile(state)
    risk_scores = profile.get("risk_scores", {})
    high_fall = risk_scores.get("falls_balance_gait", 0.0) >= 0.35 or risk_scores.get("mobility_equipment_support", 0.0) >= 0.35
    seizure_text = _v19_condition_safety_text(state, category_key)
    result = []
    for act in activities:
        a = dict(act)
        avoid = _v19_activity_specific_avoid(a, state, category_key)
        if avoid and avoid not in str(a.get("what_to_avoid", "")):
            a["what_to_avoid"] = avoid
        if high_fall and category_key == "movement_and_physical":
            risky = re.search(r"race|obstacle|trampoline|jump from|high surface|fast|unsupported balance", (str(a.get("title", "")) + " " + str(a.get("instructions", ""))).lower())
            if risky:
                a["title"] = "Supported " + str(a.get("activity_focus_step", a.get("bridge_step", "Bridge Step")))[:36]
                a["instructions"] = f"Use a clear, flat floor and close adult support. Practice the active bridge step slowly: {a.get('activity_focus_step', a.get('bridge_step', ''))}. Do not add speed, obstacles, heights, or jumping from surfaces."
            elif seizure_text and seizure_text.strip() not in str(a.get("instructions", "")):
                a["instructions"] = str(a.get("instructions", "")).rstrip() + " " + seizure_text.strip()
        result.append(a)
    return result


# ------------------------------------------------------------------
# 7) Scheduler: cover all milestones, avoid duplicates, keep 7+ distinct activities.
# ------------------------------------------------------------------
def _v19_core_activities(bank: Dict[str, Any]) -> List[Dict[str, Any]]:
    core = [a for a in bank.get("activities", []) if a.get("activity_type") == "core"]
    return core or list(bank.get("activities", []))


def _v19_choose_activity(bank: Dict[str, Any], day_idx: int, slot_idx: int, used_titles: set, used_milestones: set, counts_by_milestone: Counter, counts_by_activity: Counter) -> Optional[Dict[str, Any]]:
    activities = _v19_core_activities(bank)
    if not activities:
        return None
    def key(a: Dict[str, Any]):
        title = _v18_norm(a.get("title", ""))
        goal = _v18_norm(a.get("cdc_goal", a.get("target_milestone", "")))
        aid = _v18_norm(a.get("activity_id", title))
        theme = _v18_norm(a.get("theme", ""))
        return (
            1 if title in used_titles else 0,
            counts_by_activity[aid],
            counts_by_milestone[goal],
            1 if goal in used_milestones else 0,
            (day_idx + slot_idx + abs(hash(theme))) % 11,
        )
    return sorted(activities, key=key)[0]


def build_weekly_schedule(state: Dict[str, Any], cycle_days: int = V19_DEFAULT_CYCLE_DAYS) -> Dict[str, Any]:
    categories = state.get("planned_categories") or state.get("selected_categories", [])
    daily_time = int(state.get("child", {}).get("daily_time_min", 10) or 10)
    if not categories:
        schedule = {"status": "no_categories", "version": V19_VERSION, "summary": "No categories selected for planning.", "days": {}}
        state["weekly_schedule"] = schedule
        return schedule
    for cat in categories:
        if cat not in state.get("activity_banks", {}):
            generate_category_activity_bank(state, cat)
    per_activity_min = 5
    max_items = max(1, daily_time // per_activity_min)
    days = {}
    counts_by_milestone = Counter()
    counts_by_activity = Counter()
    last_day_titles = set()
    # Category slots: if enough time, show one from each of up to two selected domains. If not enough, alternate domains by day.
    for d in range(1, cycle_days + 1):
        items = []
        used_titles = set()
        used_milestones = set()
        if len(categories) >= 2 and max_items >= 2:
            slot_categories = categories[:max_items]
        elif len(categories) >= 2 and max_items == 1:
            slot_categories = [categories[(d - 1) % len(categories)]]
        else:
            slot_categories = [categories[0] for _ in range(max_items)]
        for slot_idx, cat in enumerate(slot_categories):
            bank = state.get("activity_banks", {}).get(cat, {})
            avoid_titles = used_titles | (last_day_titles if len(_v19_core_activities(bank)) > 3 else set())
            act = _v19_choose_activity(bank, d, slot_idx, avoid_titles, used_milestones, counts_by_milestone, counts_by_activity)
            if not act:
                continue
            item = _schedule_item_from_activity(cat, act)
            title = _v18_norm(item.get("title", ""))
            goal = _v18_norm(item.get("cdc_goal", ""))
            aid = _v18_norm(item.get("activity_id", title))
            if title in used_titles:
                continue
            used_titles.add(title)
            used_milestones.add(goal)
            counts_by_milestone[goal] += 1
            counts_by_activity[aid] += 1
            items.append(item)
        last_day_titles = set(used_titles)
        days[f"day_{d}"] = {"total_minutes": sum(int(i.get("duration_min", 5)) for i in items), "items": items}
    first_week_titles = set()
    for d in range(1, min(7, cycle_days) + 1):
        for item in days.get(f"day_{d}", {}).get("items", []):
            first_week_titles.add(_v18_norm(item.get("title", "")))
    schedule = {
        "status": "ok",
        "version": V19_VERSION,
        "summary": "v19 bridge-chain weekly schedule. The 1–2 week bank covers several milestones per selected domain, gives at least two core activities per milestone, targets active bridge steps first, varies compatible themes, and adjusts with parent feedback.",
        "daily_time_min": daily_time,
        "bank_repeat_window_days": 14,
        "feedback_options": V19_ACTIVITY_FEEDBACK_OPTIONS,
        "first_week_distinct_activity_count": len(first_week_titles),
        "selection_rules": [
            "Explicit parent concerns are included up to 2 domains regardless of daily time.",
            "Daily time controls activities per day, not whether a concern is evaluated.",
            "Cover 3–5 milestones per selected domain when possible.",
            "Generate at least 2 different core activities per selected milestone.",
            "Activities target the active bridge/prerequisite step, not the final CDC goal unless the bridge chain is ready.",
            "If the bridge step equals the CDC goal, practice the previous bridge step first.",
            "Use compatible themes and activity-specific avoid rules.",
            "Avoid duplicate titles on the same day and aim for at least 7 distinct activities in week 1.",
        ],
        "days": days,
    }
    state["weekly_schedule"] = schedule
    return schedule


def display_weekly_schedule(state: Dict[str, Any]) -> None:
    schedule = state.get("weekly_schedule", {})
    print_banner("V19 BRIDGE-CHAIN WEEKLY SCHEDULE")
    if schedule.get("status") in {"no_special_support", "no_categories"}:
        print(schedule.get("summary", "No weekly schedule available."))
        return
    print("Summary:", schedule.get("summary", ""))
    print("Daily time budget:", schedule.get("daily_time_min"))
    print("Bank repeat window:", schedule.get("bank_repeat_window_days", 14), "days")
    print("Feedback options:", schedule.get("feedback_options", V19_ACTIVITY_FEEDBACK_OPTIONS))
    print("First-week distinct activities:", schedule.get("first_week_distinct_activity_count"))
    print("Source bridge table:", globals().get("V18_BRIDGE_SOURCE_PATH", "not loaded"))
    for cat, bank in state.get("activity_banks", {}).items():
        if bank.get("target_milestones"):
            print(f"\nCoverage set — {DOMAIN_CONFIG.get(cat, {}).get('display', cat)}:")
            for idx, t in enumerate(bank.get("target_milestones", []), start=1):
                print(f"  {idx}. {int(t.get('months', 0))}m | {t.get('subdomain', 'unspecified')} | {t.get('milestone')} [{t.get('answer_norm', 'unasked')}]")
        if bank.get("active_bridge_steps"):
            print(f"Bridge focus — {DOMAIN_CONFIG.get(cat, {}).get('display', cat)}:")
            for idx, b in enumerate(bank.get("active_bridge_steps", []), start=1):
                print(f"  {idx}. CDC: {b.get('cdc_milestone')} | active: {b.get('activity_focus_step')} | previous: {b.get('previous_bridge_step') or 'None'} | rule: {b.get('bridge_focus_rule')}")
    for day_name, day_info in schedule.get("days", {}).items():
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min")
        print("-" * 100)
        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue
        for item in items:
            print(f"- [{item.get('category')}] {item.get('title')} ({item.get('duration_min', 5)} min)")
            print(f"  Type: {item.get('activity_type')} | Theme: {item.get('theme')}")
            print(f"  CDC goal: {item.get('cdc_goal')}")
            print(f"  Previous bridge: {item.get('previous_bridge_step') or 'None / not needed'}")
            print(f"  Active bridge: {item.get('bridge_step')}")
            if item.get("table_bridge_step") and item.get("table_bridge_step") != item.get("bridge_step"):
                print(f"  Table bridge step: {item.get('table_bridge_step')}")
            print(f"  Why: {item.get('why_it_helps')}")
            print(f"  Instructions: {item.get('instructions')}")
            print(f"  Success: {item.get('success_criteria')}")
            print(f"  Easier: {item.get('make_easier')}")
            print(f"  Harder: {item.get('make_harder')}")
            print(f"  Group play: {item.get('group_play_line')}")
            print(f"  Avoid: {item.get('what_to_avoid')}")
            print(f"  Materials: {item.get('materials')}")


def run_all_categories_live():
    """Run one focused live onboarding with v19 planning."""
    state = new_state()
    intake_agent_live(state)
    print_child_reference(state)
    print_concern_profile(state)
    delay_agent_all_categories(state)
    selected_categories = select_focus_domains_live_v14(state)
    for category_key in selected_categories:
        qna_agent_live(state, category_key, min_yes_confirm=2, yes_ratio_confirm=0.60)
    dev_age_map = state.get("dev_age", {})
    selected_categories = [c for c in selected_categories if dev_age_map.get(c) is not None]
    summary_df = run_downstream_planning(state, display_schedule=True, selected_categories=selected_categories)
    return state, summary_df


def run_all_categories_gemini_parent(case: Dict[str, Any], daily_time_min: int = 10, gemini_model: str = "gemini-2.5-flash"):
    """Run one focused case in synthetic-parent mode using Gemini with v19 planning."""
    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_child_reference(state)
    print_concern_profile(state)
    delay_agent_all_categories(state)
    selected_categories = select_focus_domains_gemini_v14(state)
    for category_key in selected_categories:
        qna_agent_gemini_parent(
            state,
            category_key,
            ask_limit_per_band=V14_TARGET_QUESTIONS_PER_BAND,
            min_yes_confirm=2,
            yes_ratio_confirm=0.60,
            gemini_model=gemini_model,
            delay_between_questions_sec=5,
        )
    dev_age_map = state.get("dev_age", {})
    selected_categories = [c for c in selected_categories if dev_age_map.get(c) is not None]
    summary_df = run_downstream_planning(state, display_schedule=True, selected_categories=selected_categories)
    return state, summary_df

print("Loaded Genex Brain v19 patch: explicit-concern domain lock + bridge-first activities + 2 core activities per milestone + specific avoids + duplicate-resistant scheduler.")


Loaded Genex Brain v19 patch: explicit-concern domain lock + bridge-first activities + 2 core activities per milestone + specific avoids + duplicate-resistant scheduler.


## Internal legacy layer: v20 patch

In [53]:
# ------------------------------------------------------------
# Genex Brain v20 — Parent-ready activity writer + 7-day bank / week-2 repeat loop
# ------------------------------------------------------------
# v20 is intentionally conservative and deterministic. The LLM is no longer trusted
# to decide milestone coverage, bridge focus, themes, avoids, or scheduling.
# The LLM can be re-enabled later as a wording helper only after validation.

V20_VERSION = "v20"
V20_DEFAULT_CYCLE_DAYS = 14
V20_WEEK1_DAYS = 7
V20_PER_ACTIVITY_MIN = 5
V20_MIN_DISTINCT_WEEK1 = 7
V20_USE_LLM_FOR_ACTIVITIES = False

# Expanded concern mapping. This replaces v19's behavior of filling a second
# domain just because a model-estimated delay exists. Explicit parent concerns
# determine onboarding domains; daily time only controls daily activity slots.
V20_EXPLICIT_DOMAIN_PATTERNS = {
    "language_and_communication": [
        r"\bspeech\b", r"\blanguage\b", r"\btalk", r"\bword", r"\bpronounc", r"\bcommunicat",
        r"\bregress.*speech", r"\blate talk", r"\bnot talking", r"\bexpressive\b", r"\barticulation\b"
    ],
    "movement_and_physical": [
        r"\bfine motor\b", r"\bhand\b", r"\bfinger", r"\bpencil\b", r"\bcrayon\b", r"\bscissor",
        r"\bbutton", r"\bdraw", r"\bscribb", r"\bgrasp", r"\bfeed", r"\bdress", r"\bclothes",
        r"\bself[- ]?care\b", r"\bgross motor\b", r"\bwalk", r"\brun", r"\bjump", r"\bbalance", r"\bwobbl", r"\bunsteady", r"\bgait", r"\bfall", r"\bstair", r"\bwalker\b",
        r"\bcerebral palsy\b", r"\bcp\b"
    ],
    "social_and_emotional": [
        r"\bsocial\b", r"\bemotion", r"\bpeer", r"\bplay with", r"\bturn taking", r"\bsharing",
        r"\bbehavior", r"\bmeltdown", r"\btantrum", r"\bregulation", r"\bimpuls", r"\bhyperactive", r"\bhyper active", r"\bself[- ]?regulat"
    ],
    "cognitive": [
        r"\blearning\b", r"\bcognitive delay\b", r"\bunderstand", r"\bfocus", r"\black of focus", r"\battention", r"\bproblem", r"\badaptive", r"\broutine", r"\bfollowing direction", r"\bpreschool readiness"
    ],
}

V20_COGNITION_STRENGTH_PATTERNS = [
    r"\bgood cognition\b", r"\bcognition (is )?good\b", r"\bcognitively (strong|typical|fine|good)\b",
    r"\bgood understanding\b", r"\bunderstands well\b", r"\bvery bright\b", r"\bsmart\b"
]

V20_TRUE_COGNITIVE_CONCERN_PATTERNS = [
    r"\blearning delay\b", r"\bcognitive delay\b", r"\black of focus\b", r"\bfocus\b", r"\battention\b",
    r"\bfollowing direction", r"\bproblem solving\b", r"\broutine", r"\badaptive"
]


def _v20_child_text(state: Dict[str, Any]) -> str:
    child = state.get("child", {})
    return " | ".join(str(child.get(k, "")) for k in ["diagnosis", "condition", "concern", "parent_concern"]).lower()


def _v20_cognitive_blocked_by_strength(text: str) -> bool:
    if any(re.search(p, text) for p in V20_COGNITION_STRENGTH_PATTERNS):
        return not any(re.search(p, text) for p in V20_TRUE_COGNITIVE_CONCERN_PATTERNS)
    return False


def v19_explicit_concern_domains(state: Dict[str, Any], max_domains: int = 2) -> List[str]:
    """v20 override: lock explicitly named concerns; do not auto-fill with unrelated domains.

    Examples:
    - "speech delay, fine motor delay" -> Language + Movement
    - "hyperactive and lack of focus" -> Cognitive + Social/Emotional
    - "frequent falls, self-care delays, good cognition" -> Movement only
    """
    text = _v20_child_text(state)
    profile = ensure_concern_profile(state)
    scores = []
    cognitive_blocked = _v20_cognitive_blocked_by_strength(text)
    for category_key, patterns in V20_EXPLICIT_DOMAIN_PATTERNS.items():
        if category_key == "cognitive" and cognitive_blocked:
            continue
        matched = [pat for pat in patterns if re.search(pat, text)]
        if matched:
            router_weight = float(profile.get("domain_weights", {}).get(category_key, 0.0))
            # rank by number of explicit matches + router weight, but keep deterministic tie-break.
            rank_score = len(matched) * 10.0 + router_weight
            scores.append((rank_score, category_key, matched))
    scores.sort(reverse=True)
    selected = [category_key for _, category_key, _ in scores[:max_domains]]
    state.setdefault("focus_domain_triage", {})["explicit_concern_domains"] = selected
    state.setdefault("focus_domain_triage", {})["explicit_concern_matches"] = {category_key: matches for _, category_key, matches in scores}
    state.setdefault("focus_domain_triage", {})["cognitive_blocked_by_strength_note"] = cognitive_blocked
    return selected


def choose_default_focus_domains_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    """v20: explicit parent concerns drive focused onboarding.

    If there are explicit concerns, select only those domains up to max_domains.
    If there are no explicit domains, fall back to the ranked triage logic.
    """
    explicit = v19_explicit_concern_domains(state, max_domains=max_domains)
    if explicit:
        return explicit[:max_domains]
    ranked = rank_focus_domains_v14(state)
    blocked_cognitive = _v20_cognitive_blocked_by_strength(_v20_child_text(state))
    selected = []
    for item in ranked:
        k = item["category_key"]
        if blocked_cognitive and k == "cognitive":
            continue
        selected.append(k)
        if len(selected) >= max_domains:
            break
    return selected or [r["category_key"] for r in ranked[:max_domains]] or list(DOMAIN_CONFIG.keys())[:max_domains]


def select_focus_domains_live_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    ranked = rank_focus_domains_v14(state)
    suggested = choose_default_focus_domains_v14(state, max_domains=max_domains)
    explicit = v19_explicit_concern_domains(state, max_domains=max_domains)
    print("\n" + "=" * 100)
    print("FOCUS DOMAIN TRIAGE — v20")
    print("=" * 100)
    print("Genex includes explicitly named parent concerns up to 2 domains. Daily time only controls daily activity count.")
    if state.setdefault("focus_domain_triage", {}).get("cognitive_blocked_by_strength_note"):
        print("Note: Cognitive/Adaptive was not selected because the parent described cognition as a strength and did not name a cognitive concern.")
    if explicit:
        print("Explicit concern domains detected:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in explicit))
    print("Suggested focus areas:")
    for idx, item in enumerate(ranked[:4], start=1):
        marker = " [suggested]" if item["category_key"] in suggested else ""
        explicit_marker = " [explicit concern]" if item["category_key"] in explicit else ""
        print(f"  {idx}. {item['display']} | triage_score={item['triage_score']}{marker}{explicit_marker}")
    raw = input("Press Enter to accept the suggested focus areas, or type up to two numbers separated by commas: ").strip()
    if not raw:
        selected = suggested
        source = "v20_live_confirm_default"
    else:
        picked = []
        parts = [p.strip() for p in raw.split(",") if p.strip()]
        for p in parts[:max_domains]:
            if p.isdigit():
                idx = int(p) - 1
                if 0 <= idx < len(ranked):
                    picked.append(ranked[idx]["category_key"])
        selected = picked or suggested
        source = "v20_live_manual_override"
    selected = apply_focus_domain_selection_v14(state, selected, source=source)
    print("Selected focus areas:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in selected))
    return selected


def select_focus_domains_gemini_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    selected = apply_focus_domain_selection_v14(
        state,
        choose_default_focus_domains_v14(state, max_domains=max_domains),
        source="v20_synthetic_auto",
    )
    print("\n" + "=" * 100)
    print("FOCUS DOMAIN TRIAGE — v20")
    print("=" * 100)
    explicit = v19_explicit_concern_domains(state, max_domains=max_domains)
    if explicit:
        print("Explicit concern domains included:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in explicit))
    if state.setdefault("focus_domain_triage", {}).get("cognitive_blocked_by_strength_note"):
        print("Cognitive/Adaptive not selected because cognition was described as a strength without a direct cognitive concern.")
    print("Selected focus areas for this onboarding run:")
    for k in selected:
        print(f"- {DOMAIN_CONFIG[k]['display']}")
    return selected


# ---------------------------
# Activity writing helpers
# ---------------------------

def _v20_has_fall_or_medical_movement_risk(state: Dict[str, Any]) -> bool:
    text = _v20_child_text(state)
    flags = state.get("safety_profile", {}).get("flags", []) or []
    return bool(
        any(f in flags for f in ["falls_balance_gait", "seizure_or_medical_monitoring", "postural_low_tone_fatigue"]) or
        re.search(r"cerebral palsy|\bcp\b|walker|fall|falls|wobbly|unsteady|seizure|dravet|ataxia|difficulty with stairs", text)
    )


def _v20_skill_type(category_key: str, bridge: Dict[str, Any], theme: str = "") -> str:
    text = " ".join(str(bridge.get(k, "")) for k in [
        "cdc_milestone", "target_milestone", "bridge_step", "activity_focus_step", "previous_bridge_step", "subdomain"
    ]).lower() + " " + str(theme).lower()
    if re.search(r"ball|catch|throw|toss|roll", text):
        return "ball"
    if re.search(r"jump|hop|bounce|bend|knee|frog", text):
        return "jump_prep"
    if re.search(r"stair|step down|step up|climb|couch|chair", text):
        return "step_transition"
    if re.search(r"walk|run|balance|stand|steady|weight shift|walker", text):
        return "supported_balance"
    if re.search(r"fork|spoon|feed|food|snack|cup|drink|pour", text):
        return "feeding"
    if re.search(r"cloth|clothes|dress|shirt|pants|jacket|button|zip", text):
        return "dressing"
    if re.search(r"bead|macaroni|string|peg|stick|slot|coin|posting|thread", text):
        return "beading_peg"
    if re.search(r"crayon|marker|pencil|scribble|draw|circle|person", text):
        return "crayon_art"
    if re.search(r"action|picture|book|happening|running|eating|playing", text):
        return "language_action"
    if re.search(r"what is.*for|what.*for|question|coat|crayon for|answer", text):
        return "language_question"
    if re.search(r"sentence|four or more words|phrase", text):
        return "language_sentence"
    if re.search(r"song|rhyme|story|nursery", text):
        return "language_song"
    if re.search(r"word|sound|mama|dada|name|pronounc", text):
        return "language_word"
    if re.search(r"direction|give it to me|one step|two step|follow", text):
        return "directions"
    if re.search(r"turn|wait|share|social|emotion|pretend", text):
        return "social_play"
    return "general"


def _v20_clean_sentence(text: str) -> str:
    text = str(text or "").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def _v20_actionable_focus(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any]) -> Tuple[str, str]:
    """Return a parent-actionable current practice step and a rule explaining why.

    v20 is stricter than v19:
    - if the table bridge equals the CDC goal, use the previous bridge first
    - if previous bridge is unknown or not confirmed, practice/check previous bridge first
    - for fall-risk gross motor, replace broad/risky phrases like "runs" with safer actionable steps
    """
    cdc_goal = _v20_clean_sentence(bridge.get("cdc_milestone") or bridge.get("target_milestone") or "")
    table_bridge = _v20_clean_sentence(bridge.get("bridge_step") or "")
    prev = _v20_clean_sentence(bridge.get("previous_bridge_step") or "")
    prev_status = bridge.get("previous_bridge_status") or _v19_bridge_status_from_qna(state, category_key, prev)

    focus = table_bridge or prev or cdc_goal or "Practice one small prerequisite step."
    rule = "practice_table_bridge_step"
    if prev and (_v19_is_same_or_final_step(table_bridge, cdc_goal) or _v18_norm(table_bridge) == _v18_norm(cdc_goal)):
        focus = prev
        rule = "table_bridge_equals_cdc_goal__practice_previous_bridge_first"
    elif prev and prev_status in {"not_confirmed", "unknown"}:
        focus = prev
        rule = f"previous_bridge_{prev_status}__practice_previous_bridge_first"

    # Runtime safety/actionability override for fall-risk gross motor. This does not edit the table;
    # it converts broad bridge language into an activity-ready focus.
    combined = " ".join([cdc_goal, table_bridge, prev, focus]).lower()
    if category_key == "movement_and_physical" and _v20_has_fall_or_medical_movement_risk(state):
        if re.search(r"jump|hop|bounce", combined):
            if re.search(r"run", focus.lower()) or _v19_is_same_or_final_step(focus, cdc_goal):
                focus = "bend knees and stand tall with hand support on a clear, flat floor"
                rule = "fall_risk_jump_goal__use_safe_knee_bend_or_supported_standing_prep"
        elif re.search(r"run", combined):
            focus = "take slow supported steps to a floor marker and return to steady standing"
            rule = "fall_risk_running_goal__use_supported_step_to_marker"
        elif re.search(r"climb|couch|chair|stair|step", combined):
            focus = "shift weight and practice one low step or surface transition with close adult support"
            rule = "fall_risk_climb_step_goal__use_low_supported_transition"
    return focus, rule


def _v20_allowed_themes_for_bridge(category_key: str, bridge: Dict[str, Any]) -> List[str]:
    raw = bridge.get("allowed_themes", "")
    if isinstance(raw, list):
        themes = [str(x).strip() for x in raw if str(x).strip()]
    else:
        themes = [x.strip() for x in str(raw or "").split(",") if x.strip()]
    if not themes:
        themes = _v19_allowed_themes_for_bridge(category_key, bridge)
    # Filter incompatible generic themes for common skill types.
    skill = _v20_skill_type(category_key, bridge)
    skill_defaults = {
        "dressing": ["dress-up game", "doll/teddy dressing", "morning routine", "laundry helper", "mirror game"],
        "feeding": ["snack routine", "pretend restaurant", "kitchen helper", "doll feeding", "family meal turn"],
        "beading_peg": ["fine-motor treasure game", "peg game", "ring stacker", "playdough candles", "family turn"],
        "crayon_art": ["family art turn", "sticker-and-draw", "crayon road", "vertical easel", "shape art"],
        "jump_prep": ["floor sticker game", "frog knees", "toy reach", "music freeze", "supported movement game"],
        "step_transition": ["safe stepping game", "toy reach", "floor sticker path", "low-step treasure", "caregiver helper game"],
        "supported_balance": ["floor sticker path", "toy reach", "supported movement game", "music freeze", "treasure walk"],
        "ball": ["soft ball game", "laundry basket toss", "beanbag game", "parent catch game", "sibling roll game"],
        "language_action": ["picture action game", "family action guess", "puppet action", "book game", "action card game"],
        "language_question": ["object function game", "kitchen helper", "toy basket question", "picture question game", "pretend shopping"],
        "language_sentence": ["puppet story", "photo talk", "story builder", "family action sentence", "toy scene"],
        "language_song": ["song game", "rhyme game", "story repeat", "puppet song", "family singing turn"],
        "language_word": ["toy naming", "puppet play", "sound imitation", "family photo game", "book naming"],
        "directions": ["puppet direction", "clean-up game", "give-me toy game", "snack helper", "toy rescue"],
    }
    if skill in skill_defaults:
        # Prefer specific defaults, but keep table themes that overlap semantically.
        return skill_defaults[skill]
    return themes[:5] if themes else ["playful home routine"]


def _v20_materials_for_activity(skill: str, theme: str, bridge: Dict[str, Any]) -> str:
    materials = {
        "dressing": "loose jacket or elastic-waist pants, doll/teddy, child-safe mirror, laundry basket",
        "feeding": "child fork/spoon, soft safe snack pieces or pretend food, plate/tray, napkin or towel",
        "beading_peg": "ring stacker or large pegs, chunky beads, pipe cleaner or taped shoelace, paper towel roll, large pasta only with supervision",
        "crayon_art": "chunky crayons or markers, paper, stickers, clipboard or vertical surface",
        "jump_prep": "floor tape or stickers, favorite toy, clear flat floor, stable caregiver hand support",
        "step_transition": "floor tape, very low stable step or cushion only if safe, favorite toy, caregiver hand support",
        "supported_balance": "floor tape or stickers, favorite toy, stable chair/wall/caregiver hand support, clear flat floor",
        "ball": "soft ball, beanbag, rolled socks, laundry basket or box, clear floor space",
        "language_action": "picture book, action cards, family photos, puppet or stuffed animal",
        "language_question": "2–3 familiar objects such as coat, crayon, spoon, cup, toy car, small basket",
        "language_sentence": "puppet, family photo, toy figures, simple picture scene",
        "language_song": "favorite song/rhyme, storybook, puppet or stuffed animal",
        "language_word": "2–3 favorite toys, family photos, animal pictures, puppet",
        "directions": "favorite toy, basket/box, puppet, familiar household items",
        "social_play": "favorite toy, turn-taking object, simple game pieces, family member or sibling",
    }
    return materials.get(skill, "simple household toys or materials that match the bridge step")


def _v20_activity_specific_avoid(skill: str, state: Dict[str, Any], category_key: str) -> str:
    # One short, activity-specific avoid line. Avoid pasting medical warnings everywhere.
    if skill == "ball":
        return "Avoid hard balls, fast throws, throwing at the face, long distances, or repeated catches after frustration."
    if skill in {"jump_prep", "step_transition", "supported_balance"}:
        return "Use a clear flat floor and close support; avoid racing, high furniture, stairs without help, jumping from surfaces, or fatigue."
    if skill == "feeding":
        return "Use soft safe foods and close supervision; avoid choking-risk foods, hot liquids, rushing, or unsupervised eating practice."
    if skill == "dressing":
        return "Use loose clothing and allow sitting if balance is hard; avoid tight clothes, rushing, pulling, or repeated correction."
    if skill == "beading_peg":
        return "Use large items only and supervise closely; avoid small choking hazards, sharp ends, or forcing finger movements."
    if skill == "crayon_art":
        return "Avoid forcing finger position, long writing drills, sharp tools, or correcting every mark."
    if skill.startswith("language") or skill == "directions":
        return "Avoid testing, pressure to perform, repeated correction, or asking for longer speech than the child is ready for."
    return "Avoid pressure, testing, repeated correction, or continuing when the child is tired or frustrated."


def _v20_group_play_line(skill: str, theme: str, focus: str) -> str:
    if skill in {"language_action", "language_sentence"}:
        return "2+ people: one person acts or shows a picture, another models a short answer, and the child points, gestures, or tries the word/phrase."
    if skill in {"language_song", "language_word"}:
        return "2+ people: one person sings or names the item, another pauses expectantly, and the child joins with a sound, gesture, or word attempt."
    if skill == "language_question":
        return "2+ people: one person holds the object, another asks the question, and the child answers by pointing, gesturing, or saying a word."
    if skill == "directions":
        return "2+ people: one person gives the simple cue, another models once, and the child completes one small turn."
    if skill in {"jump_prep", "step_transition", "supported_balance"}:
        return "2+ people: one person models one slow safe turn while another stays close to support the child."
    if skill == "dressing":
        return "2+ people: one person dresses the doll first, another holds the clothing open, and the child tries one small clothing step."
    if skill == "feeding":
        return "2+ people: one person models the small feeding movement, another offers the item safely, and the child tries one supported turn."
    if skill in {"beading_peg", "crayon_art"}:
        return "2+ people: one person models the hand movement, another gives one simple cue, and the child takes a short turn."
    if skill == "ball":
        return "2+ people: one person rolls or places the ball, another cheers or holds the target, and the child tries one easy turn."
    return "2+ people: one person models, another gives a simple cue, and the child joins at their own level."


def _v20_title_and_instruction(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any], theme: str, variant: int) -> Tuple[str, str]:
    child_name = state.get("child", {}).get("name") or "your child"
    focus = _v20_clean_sentence(bridge.get("activity_focus_step") or bridge.get("bridge_step") or "one small step")
    cdc = _v20_clean_sentence(bridge.get("cdc_milestone") or bridge.get("target_milestone") or "the next skill")
    skill = _v20_skill_type(category_key, bridge, theme)

    # Parent-facing instructions must not expose internal bridge/CDC jargon.
    if skill == "dressing":
        if variant == 1:
            title = "Doll Dress-Up Step"
            instructions = f"Put a loose jacket or pants on a teddy bear first. Say one simple cue, like “arm in” or “pants down.” Then hold the clothing open for {child_name} and invite one small push, pull, or clothing movement. Help as much as needed and stop after 2–3 easy turns."
        else:
            title = "Laundry Helper Step"
            instructions = f"During laundry or morning routine, choose one loose clothing item. Model the movement once, then invite {child_name} to try just one part, such as pushing an arm through, pulling fabric, or moving the clothing down. A small assisted try counts."
    elif skill == "feeding":
        if variant == 1:
            title = "Snack Pickup Practice"
            instructions = f"Sit together at a table with soft safe snack pieces or pretend food. Model one small fork, spoon, or finger-food movement, then let {child_name} try one turn with help. Keep it to a few calm tries and celebrate effort."
        else:
            title = "Pretend Restaurant Turn"
            instructions = f"Set up a pretend restaurant with a plate and doll. Model one feeding movement, then invite {child_name} to feed the doll or take one supported bite/scoop. Keep the game slow and supervised."
    elif skill == "beading_peg":
        if variant == 1:
            title = "Peg Treasure Stack"
            instructions = f"Place a large peg, ring stacker, or paper towel roll in front of {child_name}. Model putting one large item onto it. Then offer one item and help as needed. The goal is one calm placement, not speed or a finished necklace."
        else:
            title = "Big Bead Helper"
            instructions = f"Use a pipe cleaner or taped shoelace with one large bead or pasta piece. Hold the end steady and invite {child_name} to push the large item on. Switch to a peg or ring stacker if threading is too hard."
    elif skill == "crayon_art":
        if variant == 1:
            title = "Chunky Crayon Road"
            instructions = f"Draw a simple road or sticker target on paper. Give {child_name} a chunky crayon and model one short mark. Invite one small mark near the target. Any mark or grasp attempt counts."
        else:
            title = "Sticker-and-Scribble Turn"
            instructions = f"Put one sticker on paper. Take turns making a tiny mark near it. Keep each turn short so {child_name} can practice hand control without a long drawing drill."
    elif skill in {"jump_prep", "step_transition", "supported_balance"}:
        if skill == "jump_prep":
            if variant == 1:
                title = "Frog Knees Game"
                instructions = f"On a clear flat floor, hold {child_name}'s hands or stand very close. Say “frog knees,” bend your knees, then stand tall. Invite {child_name} to bend a little and stand back up. Do not ask for a jump; a small bend or weight shift counts."
            else:
                title = "Sticker Stand-and-Reach"
                instructions = f"Put a floor sticker or toy close by. Help {child_name} stand steady, bend slightly, or reach toward the target with support. Keep turns slow and stop before tiredness or wobbling increases."
        elif skill == "step_transition":
            title = "Low Step Toy Reach" if variant == 1 else "Supported Step-Down Treasure"
            instructions = f"Use a very low stable surface only if it is safe, or use a floor marker instead. Stay close and help {child_name} shift weight, place one foot, or step toward the toy. Keep it slow; no climbing challenge or race."
        else:
            title = "Supported Sticker Path" if variant == 1 else "Toy Reach Balance Game"
            instructions = f"Place two floor stickers very close together. With a hand, wall, or walker support as needed, invite {child_name} to step or reach toward one sticker, then pause in a steady position. One slow supported step counts."
    elif skill == "ball":
        if variant == 1:
            title = "Soft Ball Get-and-Give"
            instructions = f"Sit or stand close together. Gently roll or place a soft ball toward {child_name}. Let them reach, trap, pick up, hand back, or softly toss at their current level. Keep it slow and close."
        else:
            title = "Basket Ball Drop"
            instructions = f"Put a laundry basket or box very close. Model dropping or softly placing the ball into the target. Invite {child_name} to try one easy turn; aim and distance do not matter."
    elif skill == "language_action":
        if variant == 1:
            title = "Picture Action Point-and-Try"
            instructions = f"Show one clear picture of an action, like eating or sleeping. Ask, “What is happening?” {child_name} can point, act it out, make a sound, or try one action word. Model the word once and move on."
        else:
            title = "Family Action Guess"
            instructions = f"A parent or sibling acts out one simple action. Ask {child_name} to show or say what the person is doing. Accept pointing, gesture, one word, or a short phrase depending on readiness."
    elif skill == "language_question":
        if variant == 1:
            title = "What Is It For? Basket"
            instructions = f"Put 2–3 familiar objects in a basket. Pick one object and ask a simple question like “What is a coat for?” Offer two choices or model the answer if needed. A gesture or short answer can count."
        else:
            title = "Kitchen Helper Question"
            instructions = f"During a real routine, hold up one familiar item and ask what it is for. Give {child_name} time to answer by pointing, gesturing, or saying a word. Keep it to 2–3 objects."
    elif skill == "language_sentence":
        if variant == 1:
            title = "Puppet Phrase Builder"
            instructions = f"Use a puppet to do one action. Model a short phrase at {child_name}'s level, such as “puppet eats” or “puppet is eating.” Invite a word, phrase, or gesture; do not demand a long sentence."
        else:
            title = "Photo Talk Builder"
            instructions = f"Look at a family photo or simple picture. Model a short sentence and pause for {child_name} to add one word or phrase. Keep the conversation playful and brief."
    elif skill == "language_song":
        if variant == 1:
            title = "Pause-and-Fill Song"
            instructions = f"Sing a very familiar song and pause before one favorite word. Wait for {child_name} to join with a sound, gesture, or word attempt. Fill it in warmly if they do not respond."
        else:
            title = "Story Word Echo"
            instructions = f"Read one repeated line from a favorite story or rhyme. Emphasize one word, then invite {child_name} to echo or fill it in. One sound or approximation counts."
    elif skill == "language_word":
        if variant == 1:
            title = "Toy Name Sound Hunt"
            instructions = f"Choose 2–3 favorite toys. Model one simple sound or word, then wait. Let {child_name} point, gesture, copy the sound, or try the word. Repeat only while it feels fun."
        else:
            title = "Puppet First Words"
            instructions = f"Have a puppet ask for or name one favorite item. Model the word clearly, then pause for {child_name}. Accept approximations and do not correct repeatedly."
    elif skill == "directions":
        if variant == 1:
            title = "Give-Me Toy Game"
            instructions = f"Use one favorite toy. Give one clear direction, such as “give me the car.” Add a gesture only if needed. Celebrate any attempt and keep the turn short."
        else:
            title = "Clean-Up One-Step"
            instructions = f"During cleanup, give one simple direction with a familiar object. Model once if needed, then let {child_name} try. Stop after a few successful turns."
    else:
        title = f"{focus[:34]} Game" if variant == 1 else f"{focus[:34]} Family Turn"
        instructions = f"Set up one simple playful turn. Model the small step once, then invite {child_name} to try it with help. A partial attempt counts; repeat 2–3 times only if it stays enjoyable."
    return title, instructions


def _v20_make_activity(category_key: str, bridge: Dict[str, Any], activity_type: str, variant: int, state: Dict[str, Any]) -> Dict[str, Any]:
    # Copy bridge so we never mutate the source plan.
    b = dict(bridge)
    focus, rule = _v20_actionable_focus(state, category_key, b)
    b["activity_focus_step"] = focus
    b["bridge_focus_rule"] = rule
    allowed = _v20_allowed_themes_for_bridge(category_key, b)
    theme = allowed[(variant - 1) % len(allowed)] if allowed else "playful home routine"
    skill = _v20_skill_type(category_key, b, theme)
    cdc_goal = _v20_clean_sentence(b.get("cdc_milestone") or b.get("target_milestone") or "")
    table_bridge = _v20_clean_sentence(b.get("bridge_step") or "")
    prev = _v20_clean_sentence(b.get("previous_bridge_step") or "")
    title, instructions = _v20_title_and_instruction(state, category_key, b, theme, variant)

    if activity_type == "easier_backup":
        title = "Easy " + title
        instructions = instructions + " Make it easier by modeling first, reducing the number of turns, or giving more physical/visual support."
    elif activity_type == "harder_stretch":
        title = "Stretch " + title
        instructions = instructions + " Only use this stretch if the regular version is easy and your child is enjoying it."

    avoid = _v20_activity_specific_avoid(skill, state, category_key)
    materials = _v20_materials_for_activity(skill, theme, b)
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key.replace("_", " ").title())
    why = f"This builds {focus.lower()} as a small step toward “{cdc_goal}.”" if cdc_goal else f"This builds {focus.lower()} through playful practice."
    success = f"Success is any calm attempt at this small step: {focus.lower()}."
    easier = "Model first, use fewer choices, shorten the turn, or give more support."
    harder = "Only if easy and enjoyable: reduce help slightly, add one extra turn, or move a little closer to the next step."
    group = _v20_group_play_line(skill, theme, focus)

    return {
        "activity_id": f"v20_{category_key}_{_v18_norm(cdc_goal)[:24]}_b{b.get('bridge_step_number', 1)}_{activity_type}_{variant}",
        "title": title,
        "activity_type": activity_type,
        "theme": theme,
        "play_theme": theme,
        "category": category_display,
        "category_key": category_key,
        "duration_min": 5,
        "cdc_goal": cdc_goal,
        "target_milestone": cdc_goal,
        "bridge_step": focus,
        "table_bridge_step": table_bridge,
        "activity_focus_step": focus,
        "previous_bridge_step": prev,
        "previous_bridge_status": b.get("previous_bridge_status", "unknown"),
        "bridge_focus_rule": rule,
        "allowed_themes": allowed,
        "what_to_avoid": avoid,
        "why_it_helps": why,
        "instructions": instructions,
        "success_criteria": success,
        "success": success,
        "make_easier": easier,
        "make_harder": harder,
        "group_play_line": group,
        "group_play": group,
        "materials": materials,
        "level": "bridge_step",
        "goal": "bridge_practice",
        "is_extended_activity": False,
        "extended_reason": "",
        "milestone_key": _v18_norm(cdc_goal),
        "bridge_step_number": b.get("bridge_step_number", 1),
        "subdomain": b.get("subdomain", "unspecified"),
        "skill_type": skill,
    }


def _v20_sanitize_activity(activity: Dict[str, Any], category_key: str, state: Dict[str, Any]) -> Dict[str, Any]:
    # Ensure required fields are never blank and remove internal wording from instructions.
    a = dict(activity)
    cdc_goal = _v20_clean_sentence(a.get("cdc_goal") or a.get("target_milestone") or "the selected milestone")
    bridge = _v20_clean_sentence(a.get("bridge_step") or a.get("activity_focus_step") or "one small step")
    a["cdc_goal"] = cdc_goal
    a["target_milestone"] = cdc_goal
    a["bridge_step"] = bridge
    a["activity_focus_step"] = bridge
    a["previous_bridge_step"] = _v20_clean_sentence(a.get("previous_bridge_step") or "None needed")
    # Strip internal jargon from parent-facing instructions.
    instr = _v20_clean_sentence(a.get("instructions") or "Model the small step once, invite one short turn, accept partial success, and stop before frustration.")
    banned_replacements = {
        r"(?i)active bridge step": "small step",
        r"(?i)current bridge step": "small step",
        r"(?i)previous bridge": "earlier step",
        r"(?i)bridge step": "small step",
        r"(?i)cdc goal": "longer-term goal",
        r"(?i)final milestone": "later skill",
        r"(?i)target at the bridge level": "activity small and achievable",
    }
    for pat, repl in banned_replacements.items():
        instr = re.sub(pat, repl, instr)
    a["instructions"] = instr
    skill = a.get("skill_type") or _v20_skill_type(category_key, a, a.get("theme", ""))
    a["skill_type"] = skill
    a["what_to_avoid"] = _v20_clean_sentence(a.get("what_to_avoid")) or _v20_activity_specific_avoid(skill, state, category_key)
    a["materials"] = _v20_clean_sentence(a.get("materials")) or _v20_materials_for_activity(skill, a.get("theme", ""), a)
    a["why_it_helps"] = _v20_clean_sentence(a.get("why_it_helps")) or f"This builds {bridge.lower()} as a small step toward “{cdc_goal}.”"
    a["success_criteria"] = _v20_clean_sentence(a.get("success_criteria") or a.get("success")) or f"Any calm attempt at {bridge.lower()} counts."
    a["make_easier"] = _v20_clean_sentence(a.get("make_easier") or "Model first, shorten the turn, or give more support.")
    a["make_harder"] = _v20_clean_sentence(a.get("make_harder") or "Only if easy and enjoyable, reduce help slightly or add one extra turn.")
    a["group_play_line"] = _v20_clean_sentence(a.get("group_play_line") or a.get("group_play")) or _v20_group_play_line(skill, a.get("theme", ""), bridge)
    return a


def generate_category_activity_bank(state: Dict[str, Any], category_key: str, model: str = "gpt-4o-mini", activities_per_category: Optional[int] = None) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key.replace("_", " ").title())
    support_tier = get_support_tier(state, category_key)
    planning_tier = "enrich_and_observe" if is_family_guidance_category(state, category_key) else support_tier
    next_steps = select_next_milestones(state, category_key, max_milestones=V19_MAX_MILESTONES_PER_DOMAIN, min_milestones=V19_MIN_MILESTONES_PER_DOMAIN)
    if next_steps.get("status") == "no_special_support":
        bank = {"status": "no_special_support", "version": V20_VERSION, "support_tier": support_tier, "planning_tier": planning_tier, "summary": next_steps.get("message", ""), "activities": []}
        state.setdefault("activity_banks", {})[category_key] = bank
        return bank

    targets = next_steps.get("milestones", [])
    bridge_plan = build_bridge_plan_for_category(state, category_key, targets)
    active_bridges = bridge_plan.get("active_bridge_steps", [])
    activities: List[Dict[str, Any]] = []

    # Generate two core activities per selected milestone. Add a third core if needed
    # to guarantee enough week-1 variety when a domain has only three targets.
    core_variants = 2
    if len(active_bridges) * 2 < V20_MIN_DISTINCT_WEEK1:
        core_variants = 3
    for bridge in active_bridges:
        for variant in range(1, core_variants + 1):
            activities.append(_v20_make_activity(category_key, bridge, "core", variant, state))
        activities.append(_v20_make_activity(category_key, bridge, "easier_backup", 1, state))
        activities.append(_v20_make_activity(category_key, bridge, "harder_stretch", 2, state))

    activities = [_v20_sanitize_activity(a, category_key, state) for a in activities]
    activities = apply_language_readiness_constraints_to_activities(state, category_key, activities)
    activities = [_v20_sanitize_activity(a, category_key, state) for a in activities]
    # Keep v19 hard safety pass but avoid letting it paste fall-risk warnings into non-gross-motor items.
    activities = v19_apply_safety_constraints_to_activities(state, category_key, activities)
    activities = [_v20_sanitize_activity(a, category_key, state) for a in activities]

    # Validate no blank cards.
    required = ["title", "cdc_goal", "bridge_step", "why_it_helps", "instructions", "success_criteria", "make_easier", "make_harder", "group_play_line", "what_to_avoid", "materials"]
    for a in activities:
        for key in required:
            if not str(a.get(key, "")).strip():
                a[key] = "Needs table/content review."

    # Count unique core titles for planning diagnostics.
    core_titles = {_v18_norm(a.get("title", "")) for a in activities if a.get("activity_type") == "core"}
    bank = {
        "status": "ok",
        "version": V20_VERSION,
        "category_key": category_key,
        "category": category_display,
        "support_tier": support_tier,
        "planning_tier": planning_tier,
        "target_milestones": targets,
        "bridge_plan": bridge_plan,
        "activities": activities,
        "core_activity_count": len([a for a in activities if a.get("activity_type") == "core"]),
        "distinct_core_title_count": len(core_titles),
        "summary": f"v20 activity bank for {category_display}: {len(targets)} milestones, bridge-first activities, parent-facing instructions, activity-specific materials/safety.",
    }
    state.setdefault("activity_banks", {})[category_key] = bank
    return bank


# ---------------------------
# v20 scheduler: Week 1 varied bank, Week 2 repeat/adapt
# ---------------------------

def _v20_core_pool(bank: Dict[str, Any]) -> List[Dict[str, Any]]:
    core = [a for a in bank.get("activities", []) if a.get("activity_type") == "core"]
    return core or bank.get("activities", [])


def _v20_choose_from_bank(bank: Dict[str, Any], used_titles: set, used_goals: set, counts_by_goal: Counter, counts_by_title: Counter, allow_repeat: bool = False) -> Optional[Dict[str, Any]]:
    pool = _v20_core_pool(bank)
    if not pool:
        return None
    def score(a: Dict[str, Any]):
        title = _v18_norm(a.get("title", ""))
        goal = _v18_norm(a.get("cdc_goal", ""))
        return (
            1 if (title in used_titles and not allow_repeat) else 0,
            1 if goal in used_goals else 0,
            counts_by_title[title],
            counts_by_goal[goal],
            title,
        )
    candidates = sorted(pool, key=score)
    for a in candidates:
        t = _v18_norm(a.get("title", ""))
        if allow_repeat or t not in used_titles:
            return a
    return candidates[0]


def _v20_slot_categories(active_categories: List[str], max_items: int, day_idx: int) -> List[str]:
    if not active_categories or max_items <= 0:
        return []
    if len(active_categories) >= max_items:
        # Rotate order so the same domain is not always first.
        rot = (day_idx - 1) % len(active_categories)
        cats = active_categories[rot:] + active_categories[:rot]
        return cats[:max_items]
    # If only one domain has active goals, use the available time for two activities
    # from different milestones/subdomains in that same domain.
    cats = []
    while len(cats) < max_items:
        cats.append(active_categories[len(cats) % len(active_categories)])
    return cats


def _v20_make_repeat_item(item: Dict[str, Any], day_idx: int) -> Dict[str, Any]:
    repeat = dict(item)
    repeat["repeat_phase"] = "week_2_repeat_or_adapt"
    repeat["title"] = str(item.get("title", "Activity")) + " — repeat/adapt"
    base_instr = str(item.get("instructions", "")).strip()
    repeat["instructions"] = (
        "Repeat the familiar game from week 1. Keep the same skill, but make one small adjustment: "
        "use the easier version if it was too hard, the harder version if it was too easy, or a new theme/material if the child did not like it. "
        + base_instr
    )
    repeat["week2_rule"] = "same_skill_repeat_with_feedback_based_adjustment"
    return repeat


def build_weekly_schedule(state: Dict[str, Any], cycle_days: int = V20_DEFAULT_CYCLE_DAYS) -> Dict[str, Any]:
    categories = state.get("planned_categories") or state.get("selected_categories", [])
    daily_time = int(state.get("child", {}).get("daily_time_min", 10) or 10)
    if not categories:
        schedule = {"status": "no_categories", "version": V20_VERSION, "summary": "No categories selected for planning.", "days": {}}
        state["weekly_schedule"] = schedule
        return schedule

    for cat in categories:
        if cat not in state.get("activity_banks", {}):
            generate_category_activity_bank(state, cat)
    active_categories = [cat for cat in categories if _v20_core_pool(state.get("activity_banks", {}).get(cat, {}))]
    if not active_categories:
        schedule = {"status": "no_activities", "version": V20_VERSION, "summary": "No active activity banks were available.", "days": {}}
        state["weekly_schedule"] = schedule
        return schedule

    max_items = max(1, daily_time // V20_PER_ACTIVITY_MIN)
    max_items = min(max_items, 3)  # keep parent load reasonable
    days: Dict[str, Dict[str, Any]] = {}
    counts_by_goal = Counter()
    counts_by_title = Counter()
    week1_titles = set()

    # Week 1: varied introduction bank.
    for d in range(1, min(V20_WEEK1_DAYS, cycle_days) + 1):
        items = []
        used_titles = set()
        used_goals = set()
        slot_categories = _v20_slot_categories(active_categories, max_items, d)
        for cat in slot_categories:
            bank = state.get("activity_banks", {}).get(cat, {})
            act = _v20_choose_from_bank(bank, used_titles, used_goals, counts_by_goal, counts_by_title, allow_repeat=False)
            if not act:
                continue
            item = _schedule_item_from_activity(cat, act)
            title_key = _v18_norm(item.get("title", ""))
            goal_key = _v18_norm(item.get("cdc_goal", ""))
            if title_key in used_titles:
                # try one more time allowing same goal but different title
                act = _v20_choose_from_bank(bank, used_titles, set(), counts_by_goal, counts_by_title, allow_repeat=False) or act
                item = _schedule_item_from_activity(cat, act)
                title_key = _v18_norm(item.get("title", ""))
                goal_key = _v18_norm(item.get("cdc_goal", ""))
            used_titles.add(title_key)
            used_goals.add(goal_key)
            week1_titles.add(title_key)
            counts_by_goal[goal_key] += 1
            counts_by_title[title_key] += 1
            items.append(item)
        days[f"day_{d}"] = {"week": 1, "phase": "learn_the_bank", "total_minutes": sum(int(i.get("duration_min", 5)) for i in items), "items": items}

    # Week 2: repeat/adapt each corresponding day from week 1.
    for d in range(V20_WEEK1_DAYS + 1, cycle_days + 1):
        base_day = ((d - 1) % V20_WEEK1_DAYS) + 1
        base_items = days.get(f"day_{base_day}", {}).get("items", [])
        repeat_items = [_v20_make_repeat_item(item, d) for item in base_items]
        days[f"day_{d}"] = {"week": 2, "phase": "repeat_and_adapt", "repeat_of_day": base_day, "total_minutes": sum(int(i.get("duration_min", 5)) for i in repeat_items), "items": repeat_items}

    schedule = {
        "status": "ok",
        "version": V20_VERSION,
        "summary": "v20 bridge-chain schedule. Week 1 introduces a varied 7-day activity bank; week 2 repeats the same skills with feedback-based adaptations rather than 14 unrelated new activities.",
        "daily_time_min": daily_time,
        "bank_repeat_window_days": 14,
        "first_week_distinct_activity_count": len(week1_titles),
        "feedback_options": V19_ACTIVITY_FEEDBACK_OPTIONS,
        "selection_rules": [
            "Explicit parent concerns are selected up to 2 domains; daily time does not remove a named concern.",
            "If only one domain has active goals, use the available time for multiple activities from that domain.",
            "Week 1 introduces a varied bank across several milestones and bridge steps.",
            "Week 2 repeats/adapts week-1 activities until mastery instead of generating 14 new activities.",
            "Parent-facing instructions hide internal bridge/CDC jargon and focus on setup, model, child turn, success, and stop cues.",
            "Safety, materials, and avoid notes are activity-specific rather than pasted globally.",
        ],
        "days": days,
    }
    state["weekly_schedule"] = schedule
    return schedule


def display_weekly_schedule(state: Dict[str, Any], max_days: Optional[int] = None):
    sched = state.get("weekly_schedule") or build_weekly_schedule(state)
    print("\n" + "=" * 100)
    print("V20 BRIDGE-CHAIN WEEKLY SCHEDULE")
    print("=" * 100)
    print("Summary:", sched.get("summary", ""))
    print("Daily time budget:", sched.get("daily_time_min", ""))
    print("Bank repeat window:", sched.get("bank_repeat_window_days", ""), "days")
    print("First-week distinct activities:", sched.get("first_week_distinct_activity_count", ""))
    print("Feedback options:", sched.get("feedback_options", {}))
    source = globals().get("V18_BRIDGE_SOURCE_PATH", None) or globals().get("BRIDGE_SOURCE_PATH", None)
    if source:
        print("Source bridge table:", source)

    for cat, bank in state.get("activity_banks", {}).items():
        if bank.get("status") != "ok":
            continue
        print(f"\nCoverage set — {DOMAIN_CONFIG.get(cat, {}).get('display', cat)}:")
        for idx, m in enumerate(bank.get("target_milestones", []), start=1):
            print(f"  {idx}. {m.get('months')}m | {m.get('subdomain')} | {m.get('milestone')} [{m.get('answer_norm', 'target')}]")
        bp = bank.get("bridge_plan", {})
        if bp.get("active_bridge_steps"):
            print(f"Bridge focus — {DOMAIN_CONFIG.get(cat, {}).get('display', cat)}:")
            for idx, b in enumerate(bp.get("active_bridge_steps", []), start=1):
                focus, rule = _v20_actionable_focus(state, cat, b)
                print(f"  {idx}. CDC: {b.get('cdc_milestone')} | table: {b.get('bridge_step')} | practice: {focus} | previous: {b.get('previous_bridge_step') or 'None'} | rule: {rule}")

    days = sched.get("days", {})
    day_names = list(days.keys())
    if max_days:
        day_names = day_names[:max_days]
    for day_name in day_names:
        day_info = days.get(day_name, {})
        week = day_info.get("week")
        phase = day_info.get("phase", "")
        extra = f" | Week {week}: {phase.replace('_', ' ')}" if week else ""
        if day_info.get("repeat_of_day"):
            extra += f" | repeats Day {day_info.get('repeat_of_day')} with adaptation"
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min{extra}")
        print("-" * 100)
        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue
        for item in items:
            print(f"- [{item.get('category')}] {item.get('title')} ({item.get('duration_min', 5)} min)")
            print(f"  Type: {item.get('activity_type')} | Theme: {item.get('theme')}")
            print(f"  CDC goal: {item.get('cdc_goal')}")
            print(f"  Previous bridge: {item.get('previous_bridge_step') or 'None / not needed'}")
            print(f"  Active practice step: {item.get('bridge_step')}")
            if item.get("table_bridge_step") and item.get("table_bridge_step") != item.get("bridge_step"):
                print(f"  Table bridge step: {item.get('table_bridge_step')}")
            print(f"  Why: {item.get('why_it_helps')}")
            print(f"  Instructions: {item.get('instructions')}")
            print(f"  Success: {item.get('success_criteria')}")
            print(f"  Easier: {item.get('make_easier')}")
            print(f"  Harder: {item.get('make_harder')}")
            print(f"  Group play: {item.get('group_play_line')}")
            print(f"  Avoid: {item.get('what_to_avoid')}")
            print(f"  Materials: {item.get('materials')}")


print("Genex Brain v20 patch loaded: explicit concern domains, parent-facing activity writer, activity-specific safety/materials, 7-day bank + week-2 repeat/adapt scheduler.")


Genex Brain v20 patch loaded: explicit concern domains, parent-facing activity writer, activity-specific safety/materials, 7-day bank + week-2 repeat/adapt scheduler.


## Internal legacy layer: v20 activity-alignment correction patch

In [54]:
# ------------------------------------------------------------
# Genex Brain v20 — Activity-alignment correction patch
# ------------------------------------------------------------
# This patch keeps the v20 architecture but fixes the parent-facing activity layer:
# 1) Language activities are now typed from language milestones first, so "ba for ball" cannot become a ball game.
# 2) Expressive-language prerequisites are enforced before phrase/sentence goals.
# 3) Activity templates are matched to the active practice step and compatible themes.
# 4) Language-readiness corrections preserve CDC/bridge metadata instead of replacing cards with generic blank activities.
# 5) The v19 safety post-processor is bypassed for v20 because it could paste broad movement warnings or rewrite cards.

V20_ACTIVITY_ALIGNMENT_PATCH = "v20_activity_alignment_patch_2026_06_03"

# Preserve the pre-patch v20 helpers for non-language fallbacks.
try:
    _v20_title_and_instruction_base
except NameError:
    _v20_title_and_instruction_base = _v20_title_and_instruction
try:
    _v20_group_play_line_base
except NameError:
    _v20_group_play_line_base = _v20_group_play_line



def _v20_answer_records(state: Dict[str, Any], category_key: str) -> List[Dict[str, Any]]:
    return [
        a for a in state.get("qna", {}).get(category_key, [])
        if str(a.get("answer_status", "ok")) != "api_error"
    ]


def _v20_norm_answer_value(a: Dict[str, Any]) -> str:
    return str(a.get("scoring_norm_answer", a.get("norm_answer", a.get("answer", "")))).strip().lower()


def _v20_best_answer_for_patterns(state: Dict[str, Any], category_key: str, patterns: List[str]) -> str:
    """Return yes / partial / no / unknown for any answer whose milestone matches patterns."""
    matched = []
    for a in _v20_answer_records(state, category_key):
        text = " ".join(str(a.get(k, "")) for k in ["milestone", "question", "parent_explanation", "subdomain"]).lower()
        if any(re.search(p, text) for p in patterns):
            ans = _v20_norm_answer_value(a)
            if ans in {"yes"}:
                matched.append("yes")
            elif ans in {"sometimes", "with_help", "partial", "emerging"}:
                matched.append("partial")
            elif ans in {"no"}:
                matched.append("no")
            else:
                matched.append("unknown")
    if "yes" in matched:
        return "yes"
    if "partial" in matched:
        return "partial"
    if "no" in matched:
        return "no"
    return "unknown"


def _v20_language_foundation_state(state: Dict[str, Any]) -> Dict[str, str]:
    return {
        "tries_one_two_words": _v20_best_answer_for_patterns(
            state,
            "language_and_communication",
            [r"tries.*(one|two|2).*words?.*besides", r"two or more words besides", r"ba for ball|da for dog"],
        ),
        "tries_three_words": _v20_best_answer_for_patterns(
            state,
            "language_and_communication",
            [r"tries.*three|three or more words"],
        ),
        "two_word_combo": _v20_best_answer_for_patterns(
            state,
            "language_and_communication",
            [r"two words together|more milk|word combinations?"],
        ),
        "follows_with_gesture": _v20_best_answer_for_patterns(
            state,
            "language_and_communication",
            [r"directions.*gesture.*words|gesture and words|hold out your hand"],
        ),
        "follows_no_gesture": _v20_best_answer_for_patterns(
            state,
            "language_and_communication",
            [r"one step directions without.*gesture|without any gestures|give it to me"],
        ),
        "points_or_gestures": _v20_best_answer_for_patterns(
            state,
            "language_and_communication",
            [r"points to ask|uses more gestures|waving or pointing|body parts"],
        ),
    }


def _v20_skill_type(category_key: str, bridge: Dict[str, Any], theme: str = "") -> str:
    """v20 corrected: classify language before generic object words like ball.

    This prevents milestones such as "ba for ball" from being turned into ball games.
    """
    text = " ".join(str(bridge.get(k, "")) for k in [
        "cdc_milestone", "target_milestone", "bridge_step", "activity_focus_step", "previous_bridge_step", "subdomain"
    ]).lower() + " " + str(theme).lower()
    if category_key == "language_and_communication":
        if re.search(r"direction|give it to me|one step|two step|follow", text):
            return "directions"
        if re.search(r"song|rhyme|nursery|story word|words from a story|fill.*song", text):
            return "language_song"
        if re.search(r"what is.*for|what.*for|question|coat for|crayon for|answer", text):
            return "language_question"
        if re.search(r"what action|action is happening|happening in a picture|running or eating or playing", text):
            return "language_action"
        if re.search(r"sentence|four or more words|4 or more words", text):
            # If the actual focus is still words/vocabulary, keep it as word-building.
            focus = str(bridge.get("activity_focus_step", "")).lower()
            if re.search(r"one|two|three|3|50|word|sound|mama|dada|vocab", focus):
                return "language_word"
            return "language_sentence"
        if re.search(r"two words together|more milk|phrase|word combination", text):
            focus = str(bridge.get("activity_focus_step", "")).lower()
            if re.search(r"50 words|three|3|one|two.*besides|sound|mama|dada|vocab|single", focus):
                return "language_word"
            return "language_phrase"
        if re.search(r"50 words|word|sound|mama|dada|name|pronounc|vocab|ba for ball|da for dog", text):
            return "language_word"
        return "language_word"
    # Non-language classification.
    if re.search(r"ball|catch|throw|toss|roll", text):
        return "ball"
    if re.search(r"jump|hop|bounce|bend|knee|frog", text):
        return "jump_prep"
    if re.search(r"stair|step down|step up|climb|couch|chair", text):
        return "step_transition"
    if re.search(r"walk|run|balance|stand|steady|weight shift|walker", text):
        return "supported_balance"
    if re.search(r"fork|spoon|feed|food|snack|cup|drink|pour", text):
        return "feeding"
    if re.search(r"cloth|clothes|dress|shirt|pants|jacket|button|zip", text):
        return "dressing"
    if re.search(r"bead|macaroni|string|peg|stick|slot|coin|posting|thread", text):
        return "beading_peg"
    if re.search(r"crayon|marker|pencil|scribble|draw|circle|person", text):
        return "crayon_art"
    if re.search(r"turn|wait|share|social|emotion|pretend", text):
        return "social_play"
    return "general"


def _v20_actionable_focus(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any]) -> Tuple[str, str]:
    """Return the actual practice target for the activity.

    Corrected v20 rule: for language, do not practice a later word-combination/sentence
    step when earlier expressive prerequisites are not reliable. For movement, keep
    fall-risk children on safe prerequisite steps.
    """
    cdc_goal = _v20_clean_sentence(bridge.get("cdc_milestone") or bridge.get("target_milestone") or "")
    table_bridge = _v20_clean_sentence(bridge.get("bridge_step") or "")
    prev = _v20_clean_sentence(bridge.get("previous_bridge_step") or "")
    prev_status = bridge.get("previous_bridge_status") or _v19_bridge_status_from_qna(state, category_key, prev)

    focus = table_bridge or prev or cdc_goal or "Practice one small prerequisite step."
    rule = "practice_table_bridge_step"

    if prev and (_v19_is_same_or_final_step(table_bridge, cdc_goal) or _v18_norm(table_bridge) == _v18_norm(cdc_goal)):
        focus = prev
        rule = "table_bridge_equals_cdc_goal__practice_previous_bridge_first"
    elif prev and prev_status in {"not_confirmed", "unknown"}:
        focus = prev
        rule = f"previous_bridge_{prev_status}__practice_previous_bridge_first"

    combined = " ".join([cdc_goal, table_bridge, prev, focus]).lower()

    if category_key == "language_and_communication":
        f = _v20_language_foundation_state(state)
        # Later expressive goals must collapse back to the earliest unsteady prerequisite.
        if re.search(r"two words together|more milk|word combination|phrase|sentence|four or more words|50 words", combined):
            if f["tries_one_two_words"] not in {"yes", "partial"}:
                focus = "copy simple sounds or one word for favorite people, toys, or needs"
                rule = "language_prereq_first__sound_or_first_word_attempts"
            elif f["tries_three_words"] != "yes":
                focus = "try one to three meaningful single words for favorite people, toys, or needs"
                rule = "language_prereq_first__build_three_single_words"
            elif f["two_word_combo"] != "yes":
                focus = "build a larger single-word vocabulary for people, objects, actions, and needs"
                rule = "language_prereq_first__single_word_vocabulary_before_phrases"
        elif re.search(r"tries.*three|three or more words", combined):
            if f["tries_one_two_words"] not in {"yes", "partial"}:
                focus = "copy simple sounds or one word for favorite objects"
                rule = "language_prereq_first__one_or_two_words_before_three_words"
            else:
                focus = "try one to three meaningful single words for favorite people, toys, or needs"
                rule = "language_current_step__three_single_words"
        elif re.search(r"tries.*two|two or more words besides|ba for ball|da for dog", combined):
            focus = "copy simple sounds or one-word approximations for favorite objects"
            rule = "language_current_step__early_word_attempts"
        elif re.search(r"one step directions without.*gesture|without any gestures|give it to me", combined):
            if f["follows_with_gesture"] != "yes":
                focus = "follow one simple direction with both words and a gesture"
                rule = "receptive_prereq_first__gesture_plus_words"
            else:
                focus = "follow one familiar direction with words first, adding a gesture only if needed"
                rule = "receptive_current_step__fade_gesture"

    # Runtime safety/actionability override for fall-risk gross motor.
    combined = " ".join([cdc_goal, table_bridge, prev, focus]).lower()
    if category_key == "movement_and_physical" and _v20_has_fall_or_medical_movement_risk(state):
        if re.search(r"jump|hop|bounce", combined):
            if re.search(r"run", focus.lower()) or _v19_is_same_or_final_step(focus, cdc_goal):
                focus = "bend knees and stand tall with hand support on a clear, flat floor"
                rule = "fall_risk_jump_goal__use_safe_knee_bend_or_supported_standing_prep"
        elif re.search(r"run", combined):
            focus = "take slow supported steps to a floor marker and return to steady standing"
            rule = "fall_risk_running_goal__use_supported_step_to_marker"
        elif re.search(r"climb|couch|chair|stair|step", combined):
            focus = "shift weight and practice one low step or surface transition with close adult support"
            rule = "fall_risk_climb_step_goal__use_low_supported_transition"
    return focus, rule


def _v20_allowed_themes_for_bridge(category_key: str, bridge: Dict[str, Any]) -> List[str]:
    skill = _v20_skill_type(category_key, bridge)
    skill_defaults = {
        "dressing": ["dress-up game", "doll/teddy dressing", "morning routine", "laundry helper", "mirror game"],
        "feeding": ["snack routine", "pretend restaurant", "kitchen helper", "doll feeding", "family meal turn"],
        "beading_peg": ["fine-motor treasure game", "peg game", "ring stacker", "playdough candles", "family turn"],
        "crayon_art": ["family art turn", "sticker-and-draw", "crayon road", "vertical easel", "shape art"],
        "jump_prep": ["floor sticker game", "frog knees", "toy reach", "music freeze", "supported movement game"],
        "step_transition": ["safe stepping game", "toy reach", "floor sticker path", "low-step treasure", "caregiver helper game"],
        "supported_balance": ["floor sticker path", "toy reach", "supported movement game", "music freeze", "treasure walk"],
        "ball": ["soft ball game", "laundry basket toss", "beanbag game", "parent catch game", "sibling roll game"],
        "language_word": ["real object naming", "picture book naming", "family photo naming", "toy choice", "sound imitation"],
        "language_phrase": ["puppet phrase", "toy request", "snack request", "photo phrase", "parent modeling"],
        "language_action": ["picture action game", "family action guess", "puppet action", "book game", "action card game"],
        "language_question": ["object function game", "kitchen helper", "toy basket question", "picture question game", "pretend shopping"],
        "language_sentence": ["puppet story", "photo talk", "story builder", "family action sentence", "toy scene"],
        "language_song": ["song game", "rhyme game", "story repeat", "puppet song", "family singing turn"],
        "directions": ["puppet direction", "clean-up game", "give-me toy game", "snack helper", "toy rescue"],
        "social_play": ["turn-taking game", "sibling helper", "pretend play", "emotion faces", "family circle"],
    }
    if skill in skill_defaults:
        return skill_defaults[skill]
    raw = bridge.get("allowed_themes", "")
    themes = [x.strip() for x in str(raw or "").split(",") if x.strip()]
    return themes[:5] if themes else ["playful home routine"]


def _v20_materials_for_activity(skill: str, theme: str, bridge: Dict[str, Any]) -> str:
    materials = {
        "language_word": "2–3 favorite real objects, matching picture cards or picture book pages, family photos, small basket",
        "language_phrase": "favorite snack/toy choices, picture cards, puppet or stuffed animal, simple request items",
        "language_action": "clear action pictures, family photos, action cards, puppet or stuffed animal",
        "language_question": "2–3 familiar objects such as coat, crayon, spoon, cup, toy car, small basket",
        "language_sentence": "puppet, family photo, toy figures, simple picture scene",
        "language_song": "favorite song/rhyme, storybook, puppet or stuffed animal",
        "directions": "favorite toy, basket/box, puppet, familiar household items",
        "dressing": "loose jacket or elastic-waist pants, doll/teddy, child-safe mirror, laundry basket",
        "feeding": "child fork/spoon, soft safe snack pieces or pretend food, plate/tray, napkin or towel",
        "beading_peg": "ring stacker or large pegs, chunky beads, pipe cleaner or taped shoelace, paper towel roll, large pasta only with supervision",
        "crayon_art": "chunky crayons or markers, paper, stickers, clipboard or vertical surface",
        "jump_prep": "floor tape or stickers, favorite toy, clear flat floor, stable caregiver hand support",
        "step_transition": "floor tape, very low stable step or cushion only if safe, favorite toy, caregiver hand support",
        "supported_balance": "floor tape or stickers, favorite toy, stable chair/wall/caregiver hand support, clear flat floor",
        "ball": "soft ball, beanbag, rolled socks, laundry basket or box, clear floor space",
        "social_play": "favorite toy, turn-taking object, simple game pieces, family member or sibling",
    }
    return materials.get(skill, "simple household toys or materials that match the bridge step")


def _v20_activity_specific_avoid(skill: str, state: Dict[str, Any], category_key: str) -> str:
    if skill in {"language_word", "language_phrase", "language_sentence", "language_action", "language_question", "language_song", "directions"}:
        return "Avoid pressure to speak, repeated correction, or asking for longer speech than the active practice step. Accept gestures, sounds, approximations, or one word when appropriate."
    if skill == "ball":
        return "Avoid hard balls, fast throws, throwing at the face, long distances, or repeated catches after frustration."
    if skill in {"jump_prep", "step_transition", "supported_balance"}:
        return "Use a clear flat floor and close support; avoid racing, high furniture, stairs without help, jumping from surfaces, or fatigue."
    if skill == "feeding":
        return "Use soft safe foods and close supervision; avoid choking-risk foods, hot liquids, rushing, or unsupervised eating practice."
    if skill == "dressing":
        return "Use loose clothing and allow sitting if balance is hard; avoid tight clothes, rushing, pulling, or repeated correction."
    if skill == "beading_peg":
        return "Use large items only and supervise closely; avoid small choking hazards, sharp ends, or forcing finger movements."
    if skill == "crayon_art":
        return "Avoid forcing finger position, long writing drills, sharp tools, or correcting every mark."
    return "Avoid pressure, testing, repeated correction, or continuing when the child is tired or frustrated."


def _v20_group_play_line(skill: str, theme: str, focus: str) -> str:
    if skill == "language_word":
        return "2+ people: one person shows the object or picture, another models one word, and the child points, gestures, makes a sound, or tries the word."
    if skill == "language_phrase":
        return "2+ people: one person offers a choice, another models a short phrase, and the child joins with a gesture, one word, or phrase attempt."
    if skill in {"language_action", "language_sentence"}:
        return "2+ people: one person acts or shows a picture, another models a short answer, and the child points, gestures, or tries the word/phrase."
    if skill == "language_song":
        return "2+ people: one person sings and pauses, another gestures the next word, and the child joins with a sound, gesture, or word attempt."
    if skill == "language_question":
        return "2+ people: one person holds the object, another asks the question, and the child answers by pointing, gesturing, or saying a word."
    if skill == "directions":
        return "2+ people: one person gives the simple cue, another models once, and the child completes one small turn."
    return globals().get("_v20_group_play_line_base", lambda s,t,f: "2+ people: one person models, another gives a simple cue, and the child joins at their own level.")(skill, theme, focus)


def _v20_title_and_instruction(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any], theme: str, variant: int) -> Tuple[str, str]:
    child_name = state.get("child", {}).get("name") or "your child"
    focus = _v20_clean_sentence(bridge.get("activity_focus_step") or bridge.get("bridge_step") or "one small step")
    skill = _v20_skill_type(category_key, bridge, theme)

    if skill == "language_word":
        if variant == 1:
            title = "Favorite Object Word Try"
            instructions = f"Choose two favorite real objects, such as a ball and a cup. Hold up one item, say one clear word, then wait. {child_name} can point, reach, make a sound, or try the word. Repeat with only 2–3 words so it stays easy."
        elif variant == 2:
            title = "Picture Book Word Hunt"
            instructions = f"Open a familiar picture book or use simple cards. Ask {child_name} to find one item, then model the word clearly, such as “bear” or “cup.” Accept pointing, a sound, or any word attempt; do not ask for phrases."
        else:
            title = "Family Photo Naming"
            instructions = f"Show one family photo at a time. Say the person’s name or one easy word, then pause for {child_name}. A look, point, sound, sign, or word attempt counts. Keep it warm and brief."
        return title, instructions
    if skill == "language_phrase":
        if variant == 1:
            title = "One Word Plus Model"
            instructions = f"Offer a favorite item and model one word first, such as “milk” or “car.” If {child_name} uses a word or gesture, you can model a short phrase like “more milk,” but do not require two words yet."
        else:
            title = "Choice Word Builder"
            instructions = f"Offer two choices and name each one slowly. Let {child_name} point, gesture, or try one word. If one word is easy, model a two-word phrase once and move on without pressure."
        return title, instructions

    # Reuse the existing v20 templates for non-language and other language types by falling back to the saved base if available.
    base = globals().get("_v20_title_and_instruction_base")
    if base:
        return base(state, category_key, bridge, theme, variant)
    title = f"{focus[:34]} Game" if variant == 1 else f"{focus[:34]} Family Turn"
    instructions = f"Model one small step, invite {child_name} to try with help, accept partial success, and stop before frustration."
    return title, instructions


def _v20_make_activity(category_key: str, bridge: Dict[str, Any], activity_type: str, variant: int, state: Dict[str, Any]) -> Dict[str, Any]:
    b = dict(bridge)
    focus, rule = _v20_actionable_focus(state, category_key, b)
    b["activity_focus_step"] = focus
    b["bridge_focus_rule"] = rule
    skill_for_theme = _v20_skill_type(category_key, b)
    allowed = _v20_allowed_themes_for_bridge(category_key, b)
    theme = allowed[(variant - 1) % len(allowed)] if allowed else "playful home routine"
    skill = _v20_skill_type(category_key, b, theme)
    cdc_goal = _v20_clean_sentence(b.get("cdc_milestone") or b.get("target_milestone") or "")
    table_bridge = _v20_clean_sentence(b.get("bridge_step") or "")
    prev = _v20_clean_sentence(b.get("previous_bridge_step") or "")
    title, instructions = _v20_title_and_instruction(state, category_key, b, theme, variant)
    if activity_type == "easier_backup":
        title = "Easy " + title
        instructions += " Make it easier by using one item, modeling first, shortening the turn, or accepting a gesture/sound."
    elif activity_type == "harder_stretch":
        title = "Stretch " + title
        instructions += " Only use this if the regular version is easy and enjoyable; add one small step, not pressure."
    avoid = _v20_activity_specific_avoid(skill, state, category_key)
    materials = _v20_materials_for_activity(skill, theme, b)
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key.replace("_", " ").title())
    why = f"This practices {focus.lower()} as the next small step toward “{cdc_goal}.”" if cdc_goal else f"This practices {focus.lower()} through playful repetition."
    if skill == "language_word":
        success = "Success is any communication attempt: looking, pointing, reaching, signing, making a sound, or trying one word."
        easier = "Use one very favorite item, model the sound/word first, and accept pointing or a sound."
        harder = "Only if easy: add one new familiar word or let the child choose which word to try."
    elif skill == "language_phrase":
        success = "Success can still be one word, gesture, or sound; a phrase is optional, not required."
        easier = "Return to one-word modeling with a favorite object."
        harder = "Only if ready: model one simple two-word phrase once, such as “more milk,” without requiring repetition."
    else:
        success = f"Success is any calm attempt at this small step: {focus.lower()}."
        easier = "Model first, use fewer choices, shorten the turn, or give more support."
        harder = "Only if easy and enjoyable: reduce help slightly, add one extra turn, or move a little closer to the next step."
    group = _v20_group_play_line(skill, theme, focus)
    return {
        "activity_id": f"v20fix_{category_key}_{_v18_norm(cdc_goal)[:24]}_b{b.get('bridge_step_number', 1)}_{activity_type}_{variant}",
        "title": title,
        "activity_type": activity_type,
        "theme": theme,
        "play_theme": theme,
        "category": category_display,
        "category_key": category_key,
        "duration_min": 5,
        "cdc_goal": cdc_goal,
        "target_milestone": cdc_goal,
        "bridge_step": focus,
        "table_bridge_step": table_bridge,
        "activity_focus_step": focus,
        "previous_bridge_step": prev,
        "previous_bridge_status": b.get("previous_bridge_status", "unknown"),
        "bridge_focus_rule": rule,
        "allowed_themes": allowed,
        "what_to_avoid": avoid,
        "why_it_helps": why,
        "instructions": instructions,
        "success_criteria": success,
        "success": success,
        "make_easier": easier,
        "make_harder": harder,
        "group_play_line": group,
        "group_play": group,
        "materials": materials,
        "level": "bridge_step",
        "goal": "bridge_practice",
        "is_extended_activity": False,
        "extended_reason": "",
        "milestone_key": _v18_norm(cdc_goal),
        "bridge_step_number": b.get("bridge_step_number", 1),
        "subdomain": b.get("subdomain", "unspecified"),
        "skill_type": skill,
        "alignment_patch": V20_ACTIVITY_ALIGNMENT_PATCH,
    }


def apply_language_readiness_constraints_to_activities(
    state: Dict[str, Any],
    category_key: str,
    activities: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """v20 corrected: never replace cards with blank generic safe activities.

    If a card is too advanced, rewrite it while preserving CDC goal, previous bridge,
    active practice step, and schedule metadata.
    """
    if category_key != "language_and_communication":
        return activities
    readiness = compute_language_activity_readiness(state)
    level = readiness.get("level", "single_word_building")
    corrected = []
    replacements = 0
    for activity in activities:
        a = dict(activity)
        text = " ".join(str(a.get(k, "")) for k in ["title", "instructions", "success_criteria", "bridge_step", "cdc_goal"]).lower()
        too_advanced = level in {"pre_or_early_single_words", "single_word_building"} and re.search(
            r"require.*(two|2).*word|must.*phrase|full sentence|four or more words|required.*sentence|tell a story", text
        )
        if too_advanced:
            replacements += 1
            a["skill_type"] = "language_word"
            a["theme"] = "real object naming"
            a["play_theme"] = "real object naming"
            a["title"] = "Favorite Object Word Try"
            child_name = state.get("child", {}).get("name") or "your child"
            a["instructions"] = (
                f"Choose two favorite real objects. Model one clear word, then wait. {child_name} can point, reach, make a sound, sign, or try the word. Do not ask for phrases yet."
            )
            a["success_criteria"] = "Success is any communication attempt: looking, pointing, reaching, signing, making a sound, or trying one word."
            a["success"] = a["success_criteria"]
            a["materials"] = _v20_materials_for_activity("language_word", "real object naming", a)
            a["what_to_avoid"] = _v20_activity_specific_avoid("language_word", state, category_key)
            a["group_play_line"] = _v20_group_play_line("language_word", "real object naming", a.get("bridge_step", ""))
            a["group_play"] = a["group_play_line"]
            a["readiness_adjustment"] = "Rewritten in place because phrase/sentence demands were too advanced for current expressive-language readiness."
        # Add gentle reminder without changing metadata.
        if level in {"pre_or_early_single_words", "single_word_building"}:
            instr = str(a.get("instructions", ""))
            if "do not ask for phrases" not in instr.lower() and "do not require phrases" not in instr.lower():
                a["instructions"] = instr.rstrip() + " Do not require phrases; gestures, sounds, approximations, or one word can count."
        corrected.append(a)
    state.setdefault("planner_debug", {})["language_activity_readiness"] = {**readiness, "in_place_rewrites": replacements}
    return corrected


def generate_category_activity_bank(state: Dict[str, Any], category_key: str, model: str = "gpt-4o-mini", activities_per_category: Optional[int] = None) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key.replace("_", " ").title())
    support_tier = get_support_tier(state, category_key)
    planning_tier = "enrich_and_observe" if is_family_guidance_category(state, category_key) else support_tier
    next_steps = select_next_milestones(state, category_key, max_milestones=V19_MAX_MILESTONES_PER_DOMAIN, min_milestones=V19_MIN_MILESTONES_PER_DOMAIN)
    if next_steps.get("status") == "no_special_support":
        bank = {"status": "no_special_support", "version": V20_VERSION, "support_tier": support_tier, "planning_tier": planning_tier, "summary": next_steps.get("message", ""), "activities": []}
        state.setdefault("activity_banks", {})[category_key] = bank
        return bank
    targets = next_steps.get("milestones", [])
    bridge_plan = build_bridge_plan_for_category(state, category_key, targets)
    active_bridges = bridge_plan.get("active_bridge_steps", [])
    activities: List[Dict[str, Any]] = []
    core_variants = 2
    if len(active_bridges) * 2 < V20_MIN_DISTINCT_WEEK1:
        core_variants = 3
    for bridge in active_bridges:
        for variant in range(1, core_variants + 1):
            activities.append(_v20_make_activity(category_key, bridge, "core", variant, state))
        activities.append(_v20_make_activity(category_key, bridge, "easier_backup", 1, state))
        activities.append(_v20_make_activity(category_key, bridge, "harder_stretch", 2, state))
    activities = [_v20_sanitize_activity(a, category_key, state) for a in activities]
    activities = apply_language_readiness_constraints_to_activities(state, category_key, activities)
    activities = [_v20_sanitize_activity(a, category_key, state) for a in activities]
    required = ["title", "cdc_goal", "bridge_step", "why_it_helps", "instructions", "success_criteria", "make_easier", "make_harder", "group_play_line", "what_to_avoid", "materials"]
    for a in activities:
        # Recompute skill after any readiness adjustment to ensure theme/material alignment.
        skill = _v20_skill_type(category_key, a, a.get("theme", ""))
        a["skill_type"] = skill
        if category_key == "language_and_communication" and skill == "ball":
            # Absolute guard: language cards must never become ball/motor cards.
            a["skill_type"] = "language_word"
            a["theme"] = "real object naming"
            a["play_theme"] = "real object naming"
            a["title"] = "Favorite Object Word Try"
            a["materials"] = _v20_materials_for_activity("language_word", "real object naming", a)
            a["what_to_avoid"] = _v20_activity_specific_avoid("language_word", state, category_key)
        for key in required:
            if not str(a.get(key, "")).strip():
                a[key] = "Needs table/content review."
    core_titles = {_v18_norm(a.get("title", "")) for a in activities if a.get("activity_type") == "core"}
    bank = {
        "status": "ok",
        "version": V20_VERSION,
        "activity_alignment_patch": V20_ACTIVITY_ALIGNMENT_PATCH,
        "category_key": category_key,
        "category": category_display,
        "support_tier": support_tier,
        "planning_tier": planning_tier,
        "target_milestones": targets,
        "bridge_plan": bridge_plan,
        "activities": activities,
        "core_activity_count": len([a for a in activities if a.get("activity_type") == "core"]),
        "distinct_core_title_count": len(core_titles),
        "summary": f"v20 corrected activity bank for {category_display}: {len(targets)} milestones, bridge-first activities, language/motor-aligned themes, parent-facing instructions.",
    }
    state.setdefault("activity_banks", {})[category_key] = bank
    return bank

# Add a small visible marker in schedule display by overriding summary after schedule build.
try:
    _v20_build_weekly_schedule_base
except NameError:
    _v20_build_weekly_schedule_base = build_weekly_schedule

def build_weekly_schedule(state: Dict[str, Any], cycle_days: int = V20_DEFAULT_CYCLE_DAYS) -> Dict[str, Any]:
    sched = _v20_build_weekly_schedule_base(state, cycle_days=cycle_days)
    if sched.get("status") == "ok":
        sched["activity_alignment_patch"] = V20_ACTIVITY_ALIGNMENT_PATCH
        sched["summary"] = (
            "v20 corrected bridge-chain schedule. Week 1 introduces a varied 7-day bank; week 2 repeats/adapts the same skills. "
            "Language activities are locked to speech/language practice, not motor games, and activities target the active prerequisite step."
        )
    state["weekly_schedule"] = sched
    return sched

print("Loaded Genex Brain v20 activity-alignment correction patch:", V20_ACTIVITY_ALIGNMENT_PATCH)


Loaded Genex Brain v20 activity-alignment correction patch: v20_activity_alignment_patch_2026_06_03


## Internal base layer: v21 patch

First-bridge-first planning, no-clear-gap concern support, `activity_family` constrained activity writing, validator, and Week 1 unique / Week 2 repeat-adapt scheduler.

In [55]:
# ------------------------------------------------------------
# Genex Brain v21 — first-bridge-first + concern-support + activity_family constrained LLM writer
# ------------------------------------------------------------
# v21 implements the 4-part decision plan Sara approved:
# 1) Explicit concern routing: select named concern domains; include peer/social phrases.
# 2) Target selection: answered gaps first; if no clear gap, create a concern-support plan around closest age-appropriate concern milestones.
# 3) Bridge selection: initial plan uses bridge_step_number = 1 only. previous_bridge_step is NOT used in the first plan.
# 4) Activity generation/scheduling: use activity_family to constrain the LLM; validate alignment; Week 1 unique, Week 2 repeat/adapt.

import math
from collections import Counter, defaultdict

V21_VERSION = "v21_first_bridge_activity_family"
V21_DEFAULT_CYCLE_DAYS = 14
V21_WEEK1_DAYS = 7
V21_PER_ACTIVITY_MIN = 5
V21_MIN_MILESTONES_PER_DOMAIN = 3
V21_MAX_MILESTONES_PER_DOMAIN = 5
V21_MAX_DAILY_ACTIVITIES = 3
V21_USE_LLM_ACTIVITY_WRITER = str(os.environ.get("GENEX_V21_USE_LLM_ACTIVITY_WRITER", "true")).lower() in {"1", "true", "yes", "y"}
V21_ACTIVITY_MODEL = os.environ.get("GENEX_V21_ACTIVITY_MODEL", "gpt-4o")
V21_ACTIVITY_TEMPERATURE = float(os.environ.get("GENEX_V21_ACTIVITY_TEMPERATURE", "0.2"))

V21_ACTIVITY_FEEDBACK_OPTIONS = {
    "difficulty": ["too_hard", "just_right", "too_easy"],
    "performance": ["done_independently", "done_with_help", "couldnt_do_it"],
    "engagement": ["enjoyed_it", "resisted_it", "didnt_like_it"],
}


# Prefer Sara's final app-ready table first; older files are fallbacks only.
V21_PREFERRED_BRIDGE_FILENAMES = [
    "cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx",
    "cdc_milestones_with_bridges_family_cleaned.xlsx",
    "cdc_milestones_with_bridges_family.xlsx",
    "cdc_milestones_with_bridges_previous(1).xlsx",
    "cdc_milestones_with_bridges_previous.xlsx",
    "cdc_milestones_with_bridges.xlsx",
    "cdc_milestones_with_bridges.numbers",
]
try:
    if "V18_PREFERRED_BRIDGE_FILENAMES" in globals():
        V18_PREFERRED_BRIDGE_FILENAMES = V21_PREFERRED_BRIDGE_FILENAMES + [
            x for x in V18_PREFERRED_BRIDGE_FILENAMES if x not in V21_PREFERRED_BRIDGE_FILENAMES
        ]
    # If a preferred file exists and differs from the current loaded source, reload tables now.
    _v21_bridge_candidate = None
    for _root in _v18_search_roots() if "_v18_search_roots" in globals() else [Path.cwd(), Path("/mnt/data")]:
        for _name in V21_PREFERRED_BRIDGE_FILENAMES:
            for _p in [_root / _name, _root / "notebooks" / _name, _root / "mvp" / _name, _root / "mvp" / "notebooks" / _name]:
                if _p.exists():
                    _v21_bridge_candidate = _p
                    break
            if _v21_bridge_candidate:
                break
        if _v21_bridge_candidate:
            break
    if _v21_bridge_candidate and str(globals().get("V18_BRIDGE_SOURCE_PATH", "")) != str(_v21_bridge_candidate):
        _v18_refresh_tables(str(_v21_bridge_candidate))
except Exception as _v21_table_reload_error:
    print("v21 table reload note:", _v21_table_reload_error)

# ------------------------------------------------------------------
# 1) Explicit parent concern routing — v21
# ------------------------------------------------------------------
# Important change from v20: “problem around other kids” and similar peer/group phrases
# now explicitly select Social / Emotional. The word “problem” alone is no longer enough
# to force Cognitive / Adaptive.
V21_EXPLICIT_DOMAIN_PATTERNS = {
    "language_and_communication": [
        r"\bspeech\b", r"\blanguage\b", r"\btalk", r"\bword", r"\bpronounc", r"\bcommunicat",
        r"\bregress.*speech", r"\blate talk", r"\bnot talking", r"\bexpressive\b", r"\barticulation\b",
        r"\bapraxia\b", r"\blimited speech\b", r"\bfew words\b", r"\bunclear speech\b", r"\bgesture"
    ],
    "movement_and_physical": [
        r"\bfine motor\b", r"\bhand\b", r"\bfinger", r"\bpencil\b", r"\bcrayon\b", r"\bscissor",
        r"\bbutton", r"\bdraw", r"\bscribb", r"\bgrasp", r"\bfeed", r"\bfork\b", r"\bspoon\b",
        r"\bdress", r"\bclothes", r"\bself[- ]?care\b", r"\bgross motor\b", r"\bwalk", r"\brun",
        r"\bjump", r"\bbalance", r"\bwobbl", r"\bunsteady", r"\bgait", r"\bfall", r"\bstair",
        r"\bwalker\b", r"\bcerebral palsy\b", r"\bcp\b", r"\bcoordination\b", r"\bclumsy\b"
    ],
    "social_and_emotional": [
        r"\bsocial\b", r"\bemotion", r"\bpeer", r"\bpeers\b", r"\bclassmate", r"\bother kids\b",
        r"\baround (the )?(other )?kids\b", r"\bchildren\b", r"\bproblem around", r"\bgroup play\b",
        r"\bplay with", r"\bturn[- ]?taking\b", r"\btake turns\b", r"\bsharing\b", r"\bshare\b",
        r"\bbehavior\b", r"\bpreschool behavior\b", r"\bclassroom behavior\b", r"\bmeltdown",
        r"\btantrum", r"\bregulation\b", r"\bimpuls", r"\bhyperactive\b", r"\bhyper active\b",
        r"\binterrupt", r"\bpersonal space\b", r"\bconflict", r"\bhit\b", r"\bhitting\b", r"\bbiting\b",
        r"\bwait (his|her|their)? ?turn\b", r"\bself[- ]?regulat"
    ],
    "cognitive": [
        r"\blearning delay\b", r"\blearning\b", r"\bcognitive delay\b", r"\bunderstand", r"\black of focus\b",
        r"\bfocus\b", r"\battention\b", r"\bdistract", r"\btask", r"\bexecutive", r"\badaptive",
        r"\broutine", r"\bfollowing direction", r"\bfollow directions", r"\bproblem solving\b",
        r"\bpreschool readiness\b", r"\bmemory\b", r"\bcount", r"\bletter", r"\bnumber"
    ],
}

V21_COGNITION_STRENGTH_PATTERNS = [
    r"\bgood cognition\b", r"\bcognition (is )?good\b", r"\bcognitively (strong|typical|fine|good)\b",
    r"\bgood understanding\b", r"\bunderstands well\b", r"\bvery bright\b", r"\bsmart\b"
]

V21_TRUE_COGNITIVE_CONCERN_PATTERNS = V21_EXPLICIT_DOMAIN_PATTERNS["cognitive"]


def _v21_child_text(state: Dict[str, Any]) -> str:
    child = state.get("child", {})
    return " | ".join(str(child.get(k, "")) for k in ["diagnosis", "condition", "concern", "parent_concern"]).lower()


def _v21_cognitive_blocked_by_strength(text: str) -> bool:
    if any(re.search(p, text) for p in V21_COGNITION_STRENGTH_PATTERNS):
        return not any(re.search(p, text) for p in V21_TRUE_COGNITIVE_CONCERN_PATTERNS)
    return False


def v19_explicit_concern_domains(state: Dict[str, Any], max_domains: int = 2) -> List[str]:
    """v21 override: explicit parent concerns drive onboarding, up to max_domains.

    Key fixes:
    - “problem around other kids” maps to Social / Emotional.
    - “problem” by itself no longer maps to Cognitive / Adaptive.
    - “good cognition” blocks Cognitive / Adaptive unless there is a true cognitive concern.
    """
    text = _v21_child_text(state)
    profile = ensure_concern_profile(state)
    scores = []
    cognitive_blocked = _v21_cognitive_blocked_by_strength(text)
    for category_key, patterns in V21_EXPLICIT_DOMAIN_PATTERNS.items():
        if category_key == "cognitive" and cognitive_blocked:
            continue
        matched = [pat for pat in patterns if re.search(pat, text)]
        if matched:
            router_weight = float(profile.get("domain_weights", {}).get(category_key, 0.0))
            # Match count dominates; router weight breaks ties.
            scores.append((len(matched) * 10.0 + router_weight, category_key, matched))
    scores.sort(reverse=True)
    selected = [category_key for _, category_key, _ in scores[:max_domains]]
    triage = state.setdefault("focus_domain_triage", {})
    triage["explicit_concern_domains"] = selected
    triage["explicit_concern_matches"] = {category_key: matches for _, category_key, matches in scores}
    triage["cognitive_blocked_by_strength_note"] = cognitive_blocked
    triage["version"] = V21_VERSION
    return selected


def choose_default_focus_domains_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    explicit = v19_explicit_concern_domains(state, max_domains=max_domains)
    if explicit:
        return explicit[:max_domains]
    ranked = rank_focus_domains_v14(state)
    blocked_cognitive = _v21_cognitive_blocked_by_strength(_v21_child_text(state))
    selected = []
    for item in ranked:
        k = item["category_key"]
        if blocked_cognitive and k == "cognitive":
            continue
        selected.append(k)
        if len(selected) >= max_domains:
            break
    return selected or [r["category_key"] for r in ranked[:max_domains]] or list(DOMAIN_CONFIG.keys())[:max_domains]


def select_focus_domains_live_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    ranked = rank_focus_domains_v14(state)
    suggested = choose_default_focus_domains_v14(state, max_domains=max_domains)
    explicit = v19_explicit_concern_domains(state, max_domains=max_domains)
    print("\n" + "=" * 100)
    print("FOCUS DOMAIN TRIAGE — v21")
    print("=" * 100)
    print("Genex includes explicitly named parent concerns up to 2 domains. Daily time only controls daily activity count.")
    if state.setdefault("focus_domain_triage", {}).get("cognitive_blocked_by_strength_note"):
        print("Note: Cognitive/Adaptive was not selected because cognition was described as a strength without a direct cognitive concern.")
    if explicit:
        print("Explicit concern domains detected:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in explicit))
    print("Suggested focus areas:")
    for idx, item in enumerate(ranked[:4], start=1):
        marker = " [suggested]" if item["category_key"] in suggested else ""
        explicit_marker = " [explicit concern]" if item["category_key"] in explicit else ""
        print(f"  {idx}. {item['display']} | triage_score={item['triage_score']}{marker}{explicit_marker}")
    raw = input("Press Enter to accept the suggested focus areas, or type up to two numbers separated by commas: ").strip()
    if not raw:
        selected = suggested
        source = "v21_live_confirm_default"
    else:
        picked = []
        for p in [p.strip() for p in raw.split(",") if p.strip()][:max_domains]:
            if p.isdigit():
                idx = int(p) - 1
                if 0 <= idx < len(ranked):
                    picked.append(ranked[idx]["category_key"])
        selected = picked or suggested
        source = "v21_live_manual_override"
    selected = apply_focus_domain_selection_v14(state, selected, source=source)
    print("Selected focus areas:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in selected))
    return selected


def select_focus_domains_gemini_v14(state: Dict[str, Any], max_domains: int = V14_MAX_ONBOARDING_DOMAINS) -> List[str]:
    selected = apply_focus_domain_selection_v14(
        state,
        choose_default_focus_domains_v14(state, max_domains=max_domains),
        source="v21_synthetic_auto",
    )
    print("\n" + "=" * 100)
    print("FOCUS DOMAIN TRIAGE — v21")
    print("=" * 100)
    explicit = v19_explicit_concern_domains(state, max_domains=max_domains)
    if explicit:
        print("Explicit concern domains included:", ", ".join(DOMAIN_CONFIG[k]["display"] for k in explicit))
    if state.setdefault("focus_domain_triage", {}).get("cognitive_blocked_by_strength_note"):
        print("Cognitive/Adaptive not selected because cognition was described as a strength without a direct cognitive concern.")
    print("Selected focus areas for this onboarding run:")
    for k in selected:
        print(f"- {DOMAIN_CONFIG[k]['display']}")
    return selected


# ------------------------------------------------------------------
# 2) Target milestone selection — answered gaps first; no-gap => concern support.
# ------------------------------------------------------------------

def _v21_answer_index(state: Dict[str, Any], category_key: str) -> Dict[Tuple[int, str], Dict[str, Any]]:
    out = {}
    for a in state.get("qna", {}).get(category_key, []) or []:
        out[(int(a.get("months", 0) or 0), _v18_norm(a.get("milestone", "")))] = a
    return out


def _v21_answer_norm(answer: Optional[Dict[str, Any]]) -> str:
    if not answer:
        return "not_asked"
    return _v18_answer_norm(answer)


def _v21_parent_has_concern_for_category(state: Dict[str, Any], category_key: str) -> bool:
    explicit = state.setdefault("focus_domain_triage", {}).get("explicit_concern_domains") or v19_explicit_concern_domains(state)
    if category_key in explicit:
        return True
    text = _v21_child_text(state)
    return any(re.search(p, text) for p in V21_EXPLICIT_DOMAIN_PATTERNS.get(category_key, []))


def _v21_concern_support_score(state: Dict[str, Any], category_key: str, row: Dict[str, Any], age_center: int) -> float:
    month = int(row.get("months", age_center) or age_center)
    milestone = str(row.get("milestone", ""))
    subdomain = str(row.get("subdomain", ""))
    profile = ensure_concern_profile(state)
    score = max(0.0, 60.0 - abs(month - age_center) * 5.0)
    score += 35.0 * float(profile.get("subdomain_weights", {}).get(subdomain, 0.0))
    score += _concern_milestone_relevance_score(state, category_key, milestone, subdomain)
    # Very broad category-specific boosts for common parent-concern language.
    text = _v21_child_text(state)
    m = milestone.lower()
    s = subdomain.lower()
    if category_key == "social_and_emotional" and re.search(r"other kids|peer|classmate|group|share|turn|around.*kids|behavior", text):
        if re.search(r"turn|share|play|children|rule|wait|behavior|friend|pretend|emotion|comfort", m + " " + s):
            score += 35
    if category_key == "cognitive" and re.search(r"focus|attention|distract|learning|routine|direction", text):
        if re.search(r"attention|direction|routine|story|counts|letter|number|time|sort|match", m + " " + s):
            score += 35
    if category_key == "language_and_communication" and re.search(r"speech|word|talk|language|apraxia|gesture", text):
        if re.search(r"word|sound|gesture|point|say|talk|direction|communicat|vocal", m + " " + s):
            score += 35
    if category_key == "movement_and_physical" and re.search(r"fall|walk|stair|balance|self care|fine motor|fork|dress|hand", text):
        if re.search(r"walk|run|jump|stair|balance|climb|fork|spoon|clothes|button|page|bead|crayon", m + " " + s):
            score += 35
    return round(score, 3)


def _v21_select_diverse_candidates(candidates: List[Dict[str, Any]], max_milestones: int, min_milestones: int = 0) -> List[Dict[str, Any]]:
    selected = []
    used_keys = set()
    used_subdomains = Counter()
    for cand in sorted(candidates, key=lambda x: x.get("score", x.get("target_score", 0)), reverse=True):
        key = (int(cand.get("months", 0) or 0), _v18_norm(cand.get("milestone", "")))
        if key in used_keys:
            continue
        sub = str(cand.get("subdomain", "unspecified"))
        # Keep some subdomain diversity unless we still need to reach minimum.
        if used_subdomains[sub] >= 2 and len(selected) >= min_milestones:
            continue
        selected.append(cand)
        used_keys.add(key)
        used_subdomains[sub] += 1
        if len(selected) >= max_milestones:
            break
    return selected


def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = V21_MAX_MILESTONES_PER_DOMAIN, min_milestones: int = V21_MIN_MILESTONES_PER_DOMAIN) -> Dict[str, Any]:
    """v21 target selection.

    Rules:
    1) Answered gaps (no/sometimes/with_help/not_sure) are selected first.
    2) Answered yes milestones are not gap targets.
    3) If no clear gap is found but the parent named a concern, create a concern-support plan
       using closest age-appropriate milestones related to that concern.
    4) Do not return an empty/no-plan response just because the quick screen did not find a gap.
    """
    child = state.get("child", {})
    chrono = int(min(child.get("chronological_months", 60) or 60, 60))
    working_age = _v18_working_age_band(state, category_key)
    answer_idx = _v21_answer_index(state, category_key)
    domain_df = cdc_df[cdc_df["category_key"] == category_key].copy()
    if domain_df.empty:
        return {"status": "no_milestones", "milestones": [], "message": "No milestone rows found for this category."}

    # 1) Answered gaps from the focused interview.
    answered_candidates = []
    seen = set()
    for key, answer in answer_idx.items():
        ans = _v21_answer_norm(answer)
        if ans == "yes":
            continue
        month, mkey = key
        matches = domain_df[(domain_df["months"] == month) & (domain_df["milestone_key"] == mkey)]
        if matches.empty:
            matches = domain_df[domain_df["milestone_key"] == mkey]
        for _, row in matches.iterrows():
            if int(row.get("months", 0)) > chrono:
                continue
            d = row.to_dict()
            d["answer_norm"] = ans
            d["selection_reason"] = f"parent_answer_{ans}"
            d["score"] = _v18_milestone_candidate_score(state, category_key, d, working_age, answer) + 1000
            dk = (int(d.get("months", 0)), _v18_norm(d.get("milestone", "")))
            if dk not in seen:
                answered_candidates.append(d)
                seen.add(dk)
    selected = _v21_select_diverse_candidates(answered_candidates, max_milestones=max_milestones, min_milestones=min_milestones)
    if selected:
        return {
            "status": "needs_support",
            "mode": "evidence_based_answered_gaps",
            "working_age_band": working_age,
            "milestones": selected,
            "message": f"v21 selected {len(selected)} answered gap milestone(s) for bridge-step-1 planning.",
        }

    # 2) No clear gap, but parent concern exists. Build a light support/enrichment plan.
    if _v21_parent_has_concern_for_category(state, category_key):
        # Closest age-appropriate means centered on chronological age, not regressing far below the child.
        age_center = chrono
        min_m = max(2, age_center - 12)
        max_m = min(60, age_center + 6)
        pool = domain_df[(domain_df["months"] >= min_m) & (domain_df["months"] <= max_m)].copy()
        if pool.empty:
            pool = domain_df[domain_df["months"] <= chrono].copy()
        candidates = []
        for _, row in pool.iterrows():
            d = row.to_dict()
            key = (int(d.get("months", 0) or 0), _v18_norm(d.get("milestone", "")))
            ans = _v21_answer_norm(answer_idx.get(key))
            d["answer_norm"] = ans if ans != "not_asked" else "concern_support"
            d["selection_reason"] = "parent_concern_support_no_clear_gap"
            d["score"] = _v21_concern_support_score(state, category_key, d, age_center)
            # Do not over-select already-demonstrated yes rows unless they are strongly concern-relevant.
            if ans == "yes":
                d["score"] -= 10
            candidates.append(d)
        selected = _v21_select_diverse_candidates(candidates, max_milestones=max_milestones, min_milestones=min_milestones)
        return {
            "status": "concern_support",
            "mode": "parent_concern_support_no_clear_gap",
            "working_age_band": age_center,
            "milestones": selected,
            "message": (
                f"No clear gap was found in this quick screen for {DOMAIN_CONFIG.get(category_key, {}).get('display', category_key)}, "
                "but because the parent named this concern, v21 selected closest age-appropriate related milestones for a light support plan."
            ),
        }

    # 3) Fallback: no explicit concern and no gaps; return no plan, but with a parent-friendly status.
    return {
        "status": "no_clear_gap_no_named_concern",
        "mode": "no_plan",
        "milestones": [],
        "message": "No clear gap was found in this quick screen and no direct parent concern was mapped to this domain.",
    }


# ------------------------------------------------------------------
# 3) Bridge selection — first bridge only. Previous bridge is hidden fallback for later feedback.
# ------------------------------------------------------------------

def _v21_first_bridge_row(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not rows:
        return {}
    clean_rows = []
    for r in rows:
        d = dict(r)
        try:
            d["bridge_step_number"] = int(float(d.get("bridge_step_number", 1) or 1))
        except Exception:
            d["bridge_step_number"] = 1
        clean_rows.append(d)
    step1 = [r for r in clean_rows if int(r.get("bridge_step_number", 1)) == 1]
    return sorted(step1 or clean_rows, key=lambda r: (int(r.get("bridge_step_number", 1)), str(r.get("bridge_step", ""))))[0]


def _v21_actionable_focus(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any]) -> Tuple[str, str]:
    """Initial-plan rule: activities target bridge_step_number=1 only.

    previous_bridge_step is intentionally not used in the initial 1–2 week plan.
    It is saved only for future troubleshooting after repeated feedback.
    """
    step = _v20_clean_sentence(bridge.get("bridge_step") or bridge.get("activity_focus_step") or "")
    cdc_goal = _v20_clean_sentence(bridge.get("cdc_milestone") or bridge.get("target_milestone") or "")
    if step and not step.lower().startswith("nan"):
        rule = "v21_initial_plan_uses_bridge_step_1_only"
        if _v19_is_same_or_final_step(step, cdc_goal):
            rule = "v21_bridge_step_1_equals_or_nearly_equals_cdc_goal__table_review_recommended_but_do_not_use_previous_bridge"
        return step, rule
    return "Bridge step 1 missing from table — review all_with_bridge.", "v21_missing_bridge_step_1__table_review_needed"

# Override v20 helper name too, because display/debug functions may call it.
def _v20_actionable_focus(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any]) -> Tuple[str, str]:
    return _v21_actionable_focus(state, category_key, bridge)


def _v21_activity_family_for_bridge(category_key: str, bridge: Dict[str, Any]) -> str:
    raw = str(bridge.get("activity_family", "") or bridge.get("activity_family_suggested", "") or "").strip().lower()
    raw = re.sub(r"[^a-z0-9]+", "_", raw).strip("_")
    if raw and raw not in {"nan", "none", "missing", "not_available"}:
        return raw

    text = " ".join(str(bridge.get(k, "")) for k in ["cdc_milestone", "target_milestone", "bridge_step", "activity_focus_step", "subdomain"]).lower()
    sub = str(bridge.get("subdomain", "")).lower()

    if category_key == "language_and_communication":
        if re.search(r"conversation|back and forth|exchange", text): return "conversation_turns"
        if re.search(r"sentence|four or more words|4 or more words", text): return "expressive_sentence"
        if re.search(r"two words together|more milk|phrase|word combination", text): return "expressive_two_word_phrase"
        if re.search(r"action|running|eating|playing|happening", text): return "language_action_words"
        if re.search(r"what .* for|function|question|answer", text): return "language_function_questions"
        if re.search(r"song|story|rhyme|nursery", text): return "language_song_story"
        if re.search(r"direction|give it to me|one step|follow", text):
            return "receptive_direction_words_only" if re.search(r"without.*gesture|words alone|no gesture", text) else "receptive_direction_with_gesture"
        if re.search(r"body part|nose|eyes|hands|feet", text): return "receptive_body_parts"
        if re.search(r"point|gesture|lift.*arms|ask for|help|reach", text) or "gestural" in sub: return "gesture_requesting"
        if re.search(r"sound|coo|squeal|raspber|bababa|mamama|vocal", text) or "vocal" in sub or "babbling" in sub: return "sound_imitation"
        return "expressive_single_word"

    if category_key == "cognitive":
        if re.search(r"attention|focus|stays with|pays attention|activity for", text) or "attention" in sub: return "attention_sustained"
        if re.search(r"yesterday|tomorrow|morning|night|time|later|now|first.*then", text): return "time_words"
        if re.search(r"count|counts to", text): return "counting"
        if re.search(r"letter", text): return "letter_naming"
        if re.search(r"number", text): return "number_naming"
        if re.search(r"match|sort|same|color|shape", text): return "matching_sorting"
        if re.search(r"pretend|make believe", text): return "pretend_play"
        if re.search(r"routine|direction|first|then", text): return "routine_following"
        return "problem_solving"

    if category_key == "social_and_emotional":
        if re.search(r"turn|share|rule|wait", text): return "peer_turn_taking"
        if re.search(r"children|kids|peer|friend|play with", text): return "peer_play"
        if re.search(r"comfort|hurt|sad|cry", text): return "empathy_emotion"
        if re.search(r"emotion|calm|tantrum|upset|frustrat|regulat", text): return "emotion_regulation"
        if re.search(r"helper|help", text): return "helper_role"
        if re.search(r"place|library|worship|playground|behavior", text): return "context_behavior"
        if re.search(r"pretend|teacher|superhero|dog", text): return "pretend_play"
        return "peer_play"

    if category_key == "movement_and_physical":
        if re.search(r"book page|pages one at a time|turns book", text): return "book_page_turning"
        if re.search(r"fork|spoon|food|feed|snack|cup|drink|pour", text): return "fork_spoon_use"
        if re.search(r"clothes|dress|pants|jacket|shirt|button|zip", text):
            return "dressing_off" if re.search(r"take|off|remove|down", text) else "dressing_on"
        if re.search(r"bead|macaroni|string|thread|peg|stick|ring", text): return "beading_peg"
        if re.search(r"crayon|scribble|draw|pencil|circle|person", text): return "crayon_scribble"
        if re.search(r"jump|hop|bend|knee|frog|bounce", text): return "jump_prep"
        if re.search(r"stair|step|climb|couch|chair", text): return "safe_balance_transition"
        if re.search(r"walk|run|balance|stand|steady|walker|weight shift", text): return "safe_walking_balance"
        if re.search(r"catch", text): return "ball_catching"
        if re.search(r"throw|toss|ball", text): return "ball_throwing"
        if re.search(r"wash|hand washing", text): return "hand_washing"
        return "fine_motor_play"

    return "general_play"

# Override the old skill-type helper so existing debug/validators lean on v21 families.
def _v20_skill_type(category_key: str, bridge: Dict[str, Any], theme: str = "") -> str:
    b = dict(bridge)
    if theme:
        b["theme"] = theme
    return _v21_activity_family_for_bridge(category_key, b)


def _v21_select_active_bridge(state: Dict[str, Any], category_key: str, target: Dict[str, Any]) -> Dict[str, Any]:
    rows = get_bridge_rows_for_target(category_key, int(target.get("months", 0)), target.get("milestone", ""), target.get("subdomain", "unspecified"))
    if not rows:
        row = {
            "target_milestone": target.get("milestone", ""),
            "cdc_milestone": target.get("milestone", ""),
            "months": int(target.get("months", 0) or 0),
            "subdomain": target.get("subdomain", "unspecified"),
            "bridge_step_number": 1,
            "bridge_step": "Bridge step 1 missing from table — review all_with_bridge.",
            "previous_bridge_step": "",
            "previous_anchor_age": "",
            "allowed_themes": "",
            "what_to_avoid": "",
            "source": "missing_bridge_table_row",
            "answer_norm": target.get("answer_norm", "not_sure"),
            "selection_reason": target.get("selection_reason", ""),
            "table_review_warning": "Missing bridge table row for this target.",
        }
    else:
        row = dict(_v21_first_bridge_row(rows))
        row["target_milestone"] = target.get("milestone", "")
        row["cdc_milestone"] = target.get("milestone", "")
        row["months"] = int(target.get("months", row.get("months", 0)) or 0)
        row["subdomain"] = target.get("subdomain", row.get("subdomain", "unspecified"))
        row["answer_norm"] = str(target.get("answer_norm", "not_sure")).lower()
        row["selection_reason"] = target.get("selection_reason", "")
        row["source"] = "all_with_bridge"
    focus, rule = _v21_actionable_focus(state, category_key, row)
    row["activity_focus_step"] = focus
    row["bridge_focus_rule"] = rule
    row["previous_bridge_status"] = "not_used_initial_plan__feedback_fallback_only"
    row["previous_bridge_usage"] = "future_feedback_troubleshooting_only"
    row["activity_family"] = _v21_activity_family_for_bridge(category_key, row)
    if _v19_is_same_or_final_step(str(row.get("bridge_step", "")), str(row.get("cdc_milestone", ""))):
        row["table_review_warning"] = "bridge_step_1 appears to equal the CDC goal; clean table before production. v21 still does not use previous_bridge_step initially."
    return row


def build_bridge_plan_for_category(state: Dict[str, Any], category_key: str, target_milestones: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    targets = target_milestones or select_next_milestones(state, category_key).get("milestones", [])
    active_bridges = [_v21_select_active_bridge(state, category_key, target) for target in targets]
    plan = {
        "status": "ok" if active_bridges else "no_targets",
        "version": V21_VERSION,
        "category_key": category_key,
        "target_milestones": targets,
        "active_bridge_steps": active_bridges,
        "logic": "v21 initial plan uses bridge_step_number=1 only. previous_bridge_step is saved only for future feedback troubleshooting.",
    }
    state.setdefault("bridge_plans", {})[category_key] = plan
    return plan


# ------------------------------------------------------------------
# 4) Activity writer — activity_family constrained LLM with deterministic fallback + validation.
# ------------------------------------------------------------------

V21_FAMILY_CATEGORY = {
    # Language
    "expressive_first_words": "language_and_communication", "expressive_single_word": "language_and_communication",
    "expressive_two_word_phrase": "language_and_communication", "expressive_sentence": "language_and_communication",
    "sound_imitation": "language_and_communication", "gesture_requesting": "language_and_communication",
    "receptive_direction_with_gesture": "language_and_communication", "receptive_direction_words_only": "language_and_communication",
    "receptive_body_parts": "language_and_communication", "language_action_words": "language_and_communication",
    "language_function_questions": "language_and_communication", "language_song_story": "language_and_communication",
    "conversation_turns": "language_and_communication",
    # Cognitive
    "attention_sustained": "cognitive", "time_words": "cognitive", "counting": "cognitive",
    "letter_naming": "cognitive", "number_naming": "cognitive", "matching_sorting": "cognitive",
    "routine_following": "cognitive", "problem_solving": "cognitive",
    # Social
    "peer_turn_taking": "social_and_emotional", "peer_play": "social_and_emotional",
    "emotion_regulation": "social_and_emotional", "empathy_emotion": "social_and_emotional",
    "helper_role": "social_and_emotional", "context_behavior": "social_and_emotional", "pretend_play": "social_and_emotional",
    # Movement
    "dressing_on": "movement_and_physical", "dressing_off": "movement_and_physical",
    "fork_spoon_use": "movement_and_physical", "book_page_turning": "movement_and_physical",
    "crayon_scribble": "movement_and_physical", "beading_peg": "movement_and_physical",
    "jump_prep": "movement_and_physical", "safe_balance_transition": "movement_and_physical",
    "safe_walking_balance": "movement_and_physical", "ball_throwing": "movement_and_physical", "ball_catching": "movement_and_physical",
    "hand_washing": "movement_and_physical", "fine_motor_play": "movement_and_physical",
}


def _v21_clean_display_text(x: Any) -> str:
    s = str(x or "").strip()
    s = re.sub(r"^[\[\('\"]+|[\]\)'\"]+$", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace("activit ", "activity ").replace("activit", "activity")
    return s


def _v21_allowed_themes_for_family(activity_family: str) -> List[str]:
    themes = {
        "expressive_single_word": ["real object naming", "family photo naming", "picture book naming", "toy choice", "puppet naming"],
        "expressive_first_words": ["real object naming", "family photo naming", "toy choice", "sound imitation", "picture book naming"],
        "expressive_two_word_phrase": ["toy request", "snack request", "puppet phrase", "photo phrase", "choice phrase"],
        "expressive_sentence": ["photo talk", "toy scene", "puppet story", "family action sentence", "story builder"],
        "sound_imitation": ["silly sound mirror", "animal sounds", "song pause", "puppet sounds", "bath/car sounds"],
        "gesture_requesting": ["choice request", "help request", "reach and point", "more/all done", "favorite toy request"],
        "receptive_direction_with_gesture": ["give-me toy game", "cleanup with pointing", "puppet direction", "snack helper", "toy rescue"],
        "receptive_direction_words_only": ["words-first direction", "give-me toy game", "cleanup game", "hide and find", "snack helper"],
        "receptive_body_parts": ["body part song", "teddy body parts", "mirror body parts", "bath routine", "sticker body parts"],
        "language_action_words": ["picture action game", "family action guess", "puppet action", "book game", "action cards"],
        "language_function_questions": ["object function game", "kitchen helper", "toy basket question", "picture question", "pretend shopping"],
        "language_song_story": ["song pause", "rhyme game", "story repeat", "puppet song", "family singing turn"],
        "conversation_turns": ["two-turn chat", "photo conversation", "puppet conversation", "toy interview", "story chat"],
        "attention_sustained": ["two-minute timer game", "sticker card", "block build", "short craft", "book focus"],
        "time_words": ["routine picture sort", "morning/night game", "now/later choice", "first/then routine", "today/tomorrow talk"],
        "counting": ["counting blocks", "snack counting", "toy lineup", "stair-free count game", "count and drop"],
        "letter_naming": ["letter hunt", "name-letter cards", "magnet letter match", "book letter search", "letter basket"],
        "number_naming": ["number card match", "toy number hunt", "snack number choice", "number basket", "count and name"],
        "matching_sorting": ["color sort", "same/different match", "toy sorting", "sock match", "shape basket"],
        "routine_following": ["first/then routine", "morning steps", "cleanup sequence", "picture schedule", "bedtime steps"],
        "problem_solving": ["hide-and-find", "simple puzzle", "open-close box", "which one works", "toy rescue"],
        "peer_turn_taking": ["my turn/your turn", "rolling turn", "sharing basket", "two-person game", "waiting turn"],
        "peer_play": ["parallel play", "copy-me game", "toy picnic", "sibling helper", "pretend restaurant"],
        "emotion_regulation": ["feelings faces", "calm-down choice", "puppet feelings", "stop-and-breathe", "emotion book"],
        "empathy_emotion": ["teddy care", "sad puppet", "friend helper", "feelings book", "comfort card"],
        "helper_role": ["helper job", "cleanup helper", "snack helper", "doll helper", "family helper"],
        "context_behavior": ["library voice", "place rules", "quiet/loud sort", "where are we", "practice place cues"],
        "pretend_play": ["pretend school", "pretend doctor", "animal pretend", "superhero helper", "pretend restaurant"],
        "dressing_on": ["dress-up game", "doll dressing", "morning routine", "laundry helper", "mirror dressing"],
        "dressing_off": ["laundry helper", "doll undressing", "pants-down practice", "jacket-off game", "mirror routine"],
        "fork_spoon_use": ["pretend restaurant", "snack routine", "doll feeding", "kitchen helper", "family meal turn"],
        "book_page_turning": ["board book helper", "page peek", "favorite book turn", "one-page treasure", "teddy story"],
        "crayon_scribble": ["sticker-and-draw", "crayon road", "family art turn", "vertical easel", "shape art"],
        "beading_peg": ["peg game", "ring stacker", "fine-motor treasure", "playdough candles", "family turn"],
        "jump_prep": ["frog knees", "floor sticker game", "bend-and-stand", "music freeze", "toy reach"],
        "safe_balance_transition": ["low-step helper", "floor marker step", "toy reach", "supported transition", "caregiver helper"],
        "safe_walking_balance": ["floor marker walk", "treasure walk", "supported steps", "music freeze", "toy reach"],
        "ball_throwing": ["soft toss", "laundry basket toss", "rolled sock toss", "beanbag game", "parent target"],
        "ball_catching": ["slow roll catch", "trap ball to chest", "balloon catch", "basket catch", "parent helper catch"],
        "hand_washing": ["sink routine", "soap song", "teddy hand wash", "wash-and-dry", "mirror helper"],
    }
    return themes.get(activity_family, ["playful home routine", "family turn", "toy game", "short routine", "helper game"])


def _v21_materials_for_family(activity_family: str) -> str:
    materials = {
        "expressive_single_word": "2–3 favorite real objects, family photos, picture cards or picture book pages, small basket",
        "expressive_first_words": "2–3 favorite real objects, family photos, picture cards, puppet or stuffed animal",
        "expressive_two_word_phrase": "favorite snack/toy choices, picture cards, puppet, simple request items",
        "expressive_sentence": "family photos, toy figures, puppet, simple picture scene",
        "sound_imitation": "mirror, puppet, toy animals, favorite song, bath/car toys",
        "gesture_requesting": "two favorite choices, clear container, toy slightly out of reach, small basket",
        "receptive_direction_with_gesture": "favorite toy, basket/box, puppet, familiar household items",
        "receptive_direction_words_only": "favorite toy, basket/box, two familiar objects, simple cleanup item",
        "receptive_body_parts": "mirror, teddy/doll, stickers, favorite song",
        "language_action_words": "clear action pictures, family photos, action cards, puppet or stuffed animal",
        "language_function_questions": "2–3 familiar objects such as cup, spoon, coat, crayon, toy car",
        "language_song_story": "favorite song/rhyme, storybook, puppet or stuffed animal",
        "conversation_turns": "family photo, toy figure, puppet, familiar object",
        "attention_sustained": "timer, 3–5 simple pieces, stickers, blocks, short book, small tray",
        "time_words": "routine pictures/photos, morning/night cards, first/then card, small basket",
        "counting": "blocks, snacks, toy animals, cups, small basket",
        "letter_naming": "letter cards, magnetic letters, child’s name card, picture book",
        "number_naming": "number cards, blocks, toy numbers, small basket",
        "matching_sorting": "colored blocks, socks, shape toys, cups, small baskets",
        "routine_following": "2–3 routine pictures, first/then card, familiar routine items",
        "problem_solving": "simple puzzle, box/container, hidden toy, cause-effect toy",
        "peer_turn_taking": "two toys, turn-taking object, timer if helpful, sibling/parent partner",
        "peer_play": "two similar toys, pretend food, dolls/animals, sibling/parent partner",
        "emotion_regulation": "feelings cards, mirror, puppet, calm-down choices",
        "empathy_emotion": "teddy/doll, feelings cards, bandage sticker, storybook",
        "helper_role": "small cleanup items, laundry item, snack napkin, toy basket",
        "context_behavior": "picture cards for library/home/playground, quiet/loud cards",
        "pretend_play": "dolls/animals, pretend food, toy doctor kit, blocks",
        "dressing_on": "loose jacket or elastic-waist pants, doll/teddy, mirror, laundry basket",
        "dressing_off": "loose jacket or elastic-waist pants, doll/teddy, mirror, laundry basket",
        "fork_spoon_use": "child fork/spoon, soft safe snack pieces or pretend food, plate/tray, napkin",
        "book_page_turning": "sturdy board book, favorite picture book, page tabs/sticky notes if helpful",
        "crayon_scribble": "chunky crayons/markers, paper, stickers, clipboard or vertical surface",
        "beading_peg": "ring stacker, large pegs, chunky beads, pipe cleaner/taped shoelace, large pasta with supervision",
        "jump_prep": "floor tape/stickers, favorite toy, clear flat floor, caregiver hand support",
        "safe_balance_transition": "floor tape, low stable surface only if safe, favorite toy, caregiver hand support",
        "safe_walking_balance": "floor tape/stickers, favorite toy, stable chair/wall/caregiver hand support, clear floor",
        "ball_throwing": "soft ball, beanbag, rolled socks, laundry basket/box, clear floor space",
        "ball_catching": "soft large ball, balloon, rolled socks, basket/box, clear floor space",
        "hand_washing": "sink, soap, towel, picture cue, teddy/doll if helpful",
    }
    return materials.get(activity_family, "simple household toys or materials that match the bridge step")


def _v21_avoid_for_family(activity_family: str, state: Dict[str, Any], category_key: str) -> str:
    if activity_family.startswith("expressive") or activity_family in {"sound_imitation", "gesture_requesting", "receptive_direction_with_gesture", "receptive_direction_words_only", "receptive_body_parts", "language_action_words", "language_function_questions", "language_song_story", "conversation_turns"}:
        return "Avoid pressure to speak, repeated correction, testing, or requiring longer speech than the bridge step. Accept gestures, sounds, approximations, or one word when appropriate."
    if activity_family in {"jump_prep", "safe_balance_transition", "safe_walking_balance"}:
        return "Use a clear flat floor and close support; avoid racing, climbing, high furniture, unsupported stairs, jumping from surfaces, or fatigue."
    if activity_family in {"ball_throwing", "ball_catching"}:
        return "Avoid hard balls, fast throws, throwing at the face, long distances, or repeated attempts after frustration."
    if activity_family == "fork_spoon_use":
        return "Use soft safe foods and close supervision; avoid choking-risk foods, hot liquids, rushing, or unsupervised eating practice."
    if activity_family in {"beading_peg", "fine_motor_play"}:
        return "Use large items only and supervise closely; avoid small choking hazards, forced grasp, or continuing after frustration."
    if activity_family == "crayon_scribble":
        return "Avoid forcing finger position, long writing drills, sharp tools, or correcting every mark."
    if activity_family in {"dressing_on", "dressing_off"}:
        return "Use loose clothing and allow sitting if balance is hard; avoid tight clothes, rushing, pulling, or repeated correction."
    if activity_family == "book_page_turning":
        return "Use sturdy books and keep turns short; avoid thin pages if frustrating or correcting every page turn."
    return "Avoid pressure, testing, repeated correction, or continuing when the child is tired or frustrated."


def _v21_group_play_for_family(activity_family: str) -> str:
    if activity_family in {"peer_turn_taking", "peer_play"}:
        return "2+ people: one person models the social turn, another gently supports the child’s turn, and everyone keeps it short and positive."
    if activity_family.startswith("expressive") or activity_family in {"sound_imitation", "gesture_requesting", "language_action_words", "conversation_turns"}:
        return "2+ people: one person shows the object/picture or models one word, another waits warmly, and the child points, gestures, makes a sound, or tries the word."
    if activity_family in {"jump_prep", "safe_balance_transition", "safe_walking_balance"}:
        return "2+ people: one person models one slow safe turn while another stays close to support the child."
    if activity_family in {"dressing_on", "dressing_off"}:
        return "2+ people: one person models on a doll or clothing item, another holds the clothing open, and the child tries one small step."
    return "2+ people: one person models, another gives one simple cue, and the child joins at their own level."


def _v21_fallback_activity_text(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any], variant: int) -> Dict[str, str]:
    child_name = state.get("child", {}).get("name") or "your child"
    fam = bridge.get("activity_family") or _v21_activity_family_for_bridge(category_key, bridge)
    theme = _v21_allowed_themes_for_family(fam)[(variant - 1) % len(_v21_allowed_themes_for_family(fam))]

    # Concrete fallback wording. This is deliberately small and family-based; Claude/app can later replace it with a richer template library.
    if fam in {"expressive_single_word", "expressive_first_words"}:
        options = [
            ("Favorite Object Word Try", f"Choose two favorite real objects. Hold up one item, say one clear word, then wait. {child_name} can look, point, reach, make a sound, sign, or try the word. Repeat with only 2–3 words so it stays easy."),
            ("Family Photo Naming", f"Show one family photo at a time. Say the person’s name or one easy word, then pause for {child_name}. A look, point, sound, sign, or word attempt counts. Keep it warm and brief."),
            ("Picture Book Word Hunt", f"Open a familiar picture book. Point to one picture, model the word once, then pause. {child_name} can point, make a sound, or try the word. Stop after a few easy turns."),
            ("Toy Choice Word", f"Offer two favorite toys and name each one slowly. Let {child_name} choose by looking, reaching, pointing, or trying a sound/word. Model the word once more after the choice."),
            ("Puppet First Word", f"Use a puppet to hold one familiar object. The puppet says the word once, then waits. {child_name} can point, make a sound, sign, or try the word before the puppet celebrates."),
        ]
    elif fam == "sound_imitation":
        options = [
            ("Silly Sound Mirror", f"Sit face-to-face or near a mirror. Make one easy sound like “mmm,” “ba,” or “ah,” then pause. Copy any sound {child_name} makes back to them and keep it playful."),
            ("Animal Sound Turn", f"Pick two toy animals. Make one animal sound, then pause for {child_name}. Any look, smile, movement, sound, or attempt counts."),
            ("Song Pause Sound", f"Sing a familiar song and pause at one fun sound. Wait for {child_name} to look, move, vocalize, or smile, then continue the song."),
        ]
    elif fam == "gesture_requesting":
        options = [
            ("Choice Request Game", f"Hold up two favorite items just within view. Wait for {child_name} to look, reach, point, gesture, or make a sound. Give the chosen item right away and name it once."),
            ("Help Request Box", f"Put a favorite toy in a clear container that is easy for you to open. Wait for {child_name} to look, reach, point, hand it to you, or make a sound, then say “help” and open it."),
            ("Up or More Gesture", f"During a favorite routine, pause and wait for {child_name} to reach, lift arms, point, or make a sound to continue. Respond right away so the gesture feels powerful."),
        ]
    elif fam in {"receptive_direction_with_gesture", "receptive_direction_words_only"}:
        gesture_note = "Use a gesture if needed." if fam == "receptive_direction_with_gesture" else "Try words first, then add a gesture only if needed."
        options = [
            ("Give-Me Toy Game", f"Place one familiar toy near {child_name}. Say “give me the toy.” {gesture_note} Celebrate any attempt to hand, push, or move the toy toward you."),
            ("Clean-Up One Thing", f"Put one item beside a small basket. Say “put it in.” {gesture_note} Help as much as needed and stop after 2–3 easy turns."),
            ("Puppet Direction", f"Let a puppet ask for one simple action, like “give bear the cup.” {gesture_note} Keep the direction familiar and playful."),
        ]
    elif fam == "time_words":
        options = [
            ("Morning or Night Sort", f"Show two routine pictures, such as breakfast and bedtime. Say “morning” for breakfast and “night” for bedtime. Help {child_name} place each picture under morning or night."),
            ("Now or Later Choice", f"Use two simple cards: now and later. Show one thing happening now and one thing for later. Say the time word and let {child_name} point or place the card."),
            ("First Then Routine", f"Pick a real routine. Say “first shoes, then outside” or another simple pair. Let {child_name} move the picture or object for the first step."),
        ]
    elif fam == "attention_sustained":
        options = [
            ("Two-Minute Focus Basket", f"Put 3–4 simple pieces in a small basket. Set a short timer for 2 minutes. Model one step, then help {child_name} stay with the same activity until the basket is done or the timer ends."),
            ("Sticker Finish Card", f"Draw three small boxes on paper. After each tiny turn, {child_name} adds one sticker. The goal is finishing the three-box card, not perfect answers."),
            ("Short Build Together", f"Build a tiny tower or pattern with 3–5 pieces. Give {child_name} one clear role and praise staying with the same game until the pieces are used."),
        ]
    elif fam == "counting":
        options = [
            ("Count and Drop", f"Put a few blocks or snacks in a bowl. Count slowly as {child_name} drops each one in. Help point or touch each item so one number matches one object."),
            ("Toy Line Count", f"Line up a few toy animals or cars. Touch each one and count together. Stop before it becomes a test; joining, pointing, or listening counts."),
            ("Snack Count Helper", f"During snack, count a small number of pieces together. Let {child_name} move each piece as you count."),
        ]
    elif fam in {"letter_naming", "number_naming"}:
        kind = "letter" if fam == "letter_naming" else "number"
        options = [
            (f"{kind.title()} Hunt", f"Show 2–3 familiar {kind} cards. Name one, then let {child_name} point, match, or try to name it. Keep the set very small."),
            (f"Find My {kind.title()}", f"Hide one familiar {kind} card among two choices. Name it and help {child_name} find or point to it."),
            (f"{kind.title()} Basket", f"Put one {kind} card into a basket after naming it. {child_name} can point, place, or say the {kind}."),
        ]
    elif fam == "matching_sorting":
        options = [
            ("Same Basket Sort", f"Put two matching items and one different item on the table. Show “same,” then help {child_name} put the matching pair together."),
            ("Color Cup Sort", f"Use two cups or bowls and a few colored items. Model one sort, then invite {child_name} to place one item with help."),
            ("Sock Match Helper", f"Use two socks that match and one that does not. Help {child_name} find the same one and put the pair together."),
        ]
    elif fam in {"peer_turn_taking", "peer_play"}:
        options = [
            ("My Turn Your Turn", f"Use one favorite toy with a parent or sibling. Say “my turn,” take one short turn, then say “your turn” and hand it to {child_name}. Keep turns very short."),
            ("Copy-Me Play", f"One person does a simple action with a toy, then {child_name} gets a turn to copy or do their own version. Praise joining, waiting, or watching."),
            ("Toy Picnic Together", f"Set up two plates or toys. Each person gives one pretend snack to a toy. Help {child_name} share one item or wait for one short turn."),
        ]
    elif fam == "emotion_regulation":
        options = [
            ("Feelings Face Choice", f"Show two simple feeling faces. Name one feeling and let {child_name} point, copy the face, or choose a calm-down helper like squeeze, hug, or breath."),
            ("Puppet Gets Upset", f"Have a puppet feel upset for a simple reason. Model one calm choice, then let {child_name} hand the puppet a calm card or comfort item."),
            ("Stop and Breathe Game", f"During a calm moment, practice one playful breath with a stuffed animal. Keep it short and do not wait for a real meltdown to teach it first."),
        ]
    elif fam in {"dressing_on", "dressing_off"}:
        action = "push an arm through or pull fabric into place" if fam == "dressing_on" else "push loose fabric down or pull an open jacket off"
        options = [
            ("Doll Dress-Up Step", f"Put loose clothing on a teddy first. Say one simple cue, then hold the clothing for {child_name} and invite one small step: {action}. Help as much as needed."),
            ("Laundry Helper Step", f"During laundry or morning routine, choose one loose clothing item. Model one movement, then invite {child_name} to try just one small clothing step."),
            ("Mirror Clothing Turn", f"Sit or stand safely by a mirror. Show one clothing movement on yourself, then help {child_name} try one small part while watching in the mirror."),
        ]
    elif fam == "fork_spoon_use":
        options = [
            ("Pretend Restaurant Turn", f"Set up a pretend restaurant with a plate and doll. Model one spoon or fork movement, then invite {child_name} to feed the doll or take one supported bite/scoop."),
            ("Snack Pickup Practice", f"Sit at a table with soft safe snack pieces or pretend food. Model one small scoop/stab/pickup, then let {child_name} try one turn with help."),
            ("Doll Feeding Helper", f"Use a spoon or child fork to feed a teddy. {child_name} can hold, scoop, stab, or bring the utensil toward the teddy with support."),
        ]
    elif fam == "book_page_turning":
        options = [
            ("One Page Treasure", f"Use a sturdy board book. Hide a small sticky note or favorite picture on the next page. Help {child_name} lift and turn just one thick page to find it."),
            ("Page Peek Game", f"Hold the book steady and slightly separate one page. Invite {child_name} to touch, lift, or turn that page. Stop after a few successful turns."),
            ("Teddy Story Helper", f"Tell teddy needs help turning the page. Hold the book steady while {child_name} turns one page or pats the page edge."),
        ]
    elif fam == "crayon_scribble":
        options = [
            ("Sticker and Scribble", f"Put one sticker on paper and give {child_name} a chunky crayon. Model a quick mark near the sticker, then invite any mark, tap, or scribble."),
            ("Crayon Road", f"Draw a short road for a toy car. Help {child_name} make any line or mark on the road with a chunky crayon."),
            ("Family Art Turn", f"Each person makes one quick mark on the same paper. {child_name} can tap, scribble, or make any mark and then pass the crayon."),
        ]
    elif fam == "beading_peg":
        options = [
            ("Peg Treasure", f"Use a ring stacker, large peg, or paper towel roll. Hold the base steady and invite {child_name} to place one large item on it with help."),
            ("Playdough Candles", f"Put thick straws or large pegs into playdough. Invite {child_name} to push one item into place or pull it out."),
            ("Big Bead Helper", f"Use a pipe cleaner or taped shoelace with one large bead. Hold the end steady and invite {child_name} to push the bead on with help."),
        ]
    elif fam == "jump_prep":
        options = [
            ("Frog Knees Game", f"On a clear flat floor, hold {child_name}'s hands or stand very close. Say “frog knees,” bend your knees, then stand tall. A small bend or weight shift counts; do not ask for a jump."),
            ("Sticker Stand-and-Reach", f"Put a floor sticker or toy close by. Help {child_name} stand steady, bend slightly, or reach toward the target with support. Stop before tiredness or wobbling increases."),
            ("Music Freeze Bend", f"Play a few seconds of music, then freeze and bend knees slightly together. Help {child_name} bend and stand tall safely."),
        ]
    elif fam in {"safe_balance_transition", "safe_walking_balance"}:
        options = [
            ("Floor Marker Step", f"Place two floor markers close together. With hand, wall, or caregiver support as needed, invite {child_name} to take one slow step or shift weight toward the marker."),
            ("Toy Reach Balance", f"Place a favorite toy just within reach. Help {child_name} stay steady while reaching or shifting weight. One small supported movement counts."),
            ("Supported Treasure Walk", f"Put one toy a short distance away on a clear flat floor. Support {child_name} to take slow steps or move toward it, then stop and rest."),
        ]
    elif fam in {"ball_throwing", "ball_catching"}:
        if fam == "ball_catching":
            options = [
                ("Slow Roll Catch", f"Sit or stand close and slowly roll a soft ball toward {child_name}. Help them trap it with hands or body. Catching does not need to be clean."),
                ("Basket Catch", f"Hold a basket or box with {child_name}. Let a soft ball roll or drop slowly into it so they feel success with catching/trapping."),
                ("Balloon Catch", f"Use a balloon or very soft ball. Toss it slowly from close distance and help {child_name} touch, trap, or hug it."),
            ]
        else:
            options = [
                ("Soft Basket Toss", f"Put a laundry basket very close. Model placing or softly tossing a rolled sock into the basket. Invite {child_name} to try one easy turn; distance does not matter."),
                ("Parent Target Toss", f"Hold a box or basket close. Let {child_name} drop, place, or toss a soft item into it. Celebrate the release, not accuracy."),
                ("Beanbag Helper", f"Use a beanbag or rolled sock. Model one gentle forward toss, then help {child_name} release it toward a nearby target."),
            ]
    elif fam == "hand_washing":
        options = [
            ("Soap Song Step", f"At the sink, practice one small hand-washing step during a short song: hands under water, soap touch, rub, rinse, or towel. One step is enough."),
            ("Teddy Washes Hands", f"Pretend teddy needs clean hands. Model one step with teddy, then invite {child_name} to try the same step with help."),
            ("Wash and Dry Helper", f"Offer a simple cue like “rinse” or “dry.” Help {child_name} complete one small part of the routine and stop while it is still positive."),
        ]
    else:
        options = [
            ("Short Helper Game", f"Choose one small familiar routine. Model the bridge step once, invite {child_name} to try one small part with help, and stop after a few positive turns."),
            ("Family Turn Game", f"One person models the small step, then {child_name} tries at their own level. A partial attempt counts."),
            ("Toy Practice Game", f"Use a favorite toy to make the bridge step playful. Keep it short, concrete, and low-pressure."),
        ]
    title, instructions = options[(variant - 1) % len(options)]
    return {"title": title, "theme": theme, "instructions": instructions}


def _v21_activity_prompt(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any], variant: int) -> str:
    child = state.get("child", {})
    safety = format_safety_constraints_for_prompt(ensure_safety_profile(state)) if "ensure_safety_profile" in globals() else "Use common-sense child safety and caregiver supervision."
    return f"""
You are writing one parent-facing Genex home activity card.

Rules you must follow:
- Write the activity ONLY for the activity_family and bridge_step_1 below.
- Do NOT use previous_bridge_step. It is reserved for future troubleshooting and must not drive this first plan.
- Do NOT create a motor activity for language/cognitive/social goals.
- Do NOT create a language drill for movement goals.
- Instructions must be concrete: setup, model once, child turn, what counts as success, and when to stop.
- Avoid generic phrases like “set up one simple playful turn.”
- Keep it playful, low-pressure, and doable in 5 minutes.
- Use only safe household materials.
- Return JSON only with keys: title, theme, instructions, success_criteria, make_easier, make_harder, group_play_line, what_to_avoid, materials.

Child:
- name: {child.get('name', 'your child')}
- chronological age months: {child.get('chronological_months', '')}
- diagnosis/condition: {child.get('diagnosis', child.get('condition', ''))}
- parent concern: {child.get('concern', child.get('parent_concern', ''))}

Planning inputs:
- domain: {DOMAIN_CONFIG.get(category_key, {}).get('display', category_key)}
- subdomain: {bridge.get('subdomain', '')}
- CDC milestone: {bridge.get('cdc_milestone', bridge.get('target_milestone', ''))}
- bridge_step_1: {bridge.get('activity_focus_step') or bridge.get('bridge_step', '')}
- activity_family: {bridge.get('activity_family', '')}
- variant number: {variant}
- allowed themes: {', '.join(_v21_allowed_themes_for_family(bridge.get('activity_family', '')))}
- safety notes: {safety}
""".strip()


def _v21_call_llm_activity_writer(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any], variant: int) -> Optional[Dict[str, str]]:
    if not V21_USE_LLM_ACTIVITY_WRITER or globals().get("openai_client", None) is None:
        return None
    try:
        prompt = _v21_activity_prompt(state, category_key, bridge, variant)
        resp = openai_client.chat.completions.create(
            model=V21_ACTIVITY_MODEL,
            messages=[
                {"role": "system", "content": "You write safe, concrete, parent-facing developmental support activities as strict JSON."},
                {"role": "user", "content": prompt},
            ],
            temperature=V21_ACTIVITY_TEMPERATURE,
            response_format={"type": "json_object"},
        )
        content = resp.choices[0].message.content or "{}"
        data = json.loads(content)
        if isinstance(data, dict):
            return {k: _v21_clean_display_text(v) for k, v in data.items()}
    except Exception as e:
        state.setdefault("v21_llm_activity_errors", []).append(str(e))
    return None


def _v21_activity_has_alignment_problem(activity: Dict[str, Any], category_key: str) -> List[str]:
    problems = []
    fam = activity.get("activity_family", "")
    text = " ".join(str(activity.get(k, "")) for k in ["title", "theme", "instructions", "materials"]).lower()
    expected_cat = V21_FAMILY_CATEGORY.get(fam)
    if expected_cat and expected_cat != category_key:
        problems.append(f"activity_family_category_mismatch:{fam}->{expected_cat} not {category_key}")
    if category_key != "movement_and_physical" and re.search(r"\b(floor sticker|step|walk|jump|toss|throw|catch|balance|race|climb)\b", text):
        # Allow language/cognitive cards to mention object words like ball only when they are real-object naming, not motor play.
        if not (category_key == "language_and_communication" and re.search(r"word|name|picture|object|photo|sound", text)):
            problems.append("non_movement_card_contains_motor_activity")
    if category_key == "language_and_communication" and re.search(r"basketball|basket ball|laundry basket toss|throw|toss|catch|race|obstacle", text):
        problems.append("language_card_contains_motor_game")
    if fam == "time_words" and re.search(r"floor sticker|step|walk|balance|reach", text):
        problems.append("time_words_card_contains_motor_template")
    if re.search(r"set up one simple playful turn|model the small step once|materials that match the bridge step", text):
        problems.append("generic_placeholder_instruction")
    for key in ["title", "instructions", "success_criteria", "materials", "what_to_avoid"]:
        if not str(activity.get(key, "")).strip():
            problems.append(f"missing_{key}")
    return problems


def _v21_success_easier_harder(activity_family: str) -> Tuple[str, str, str]:
    if activity_family.startswith("expressive") or activity_family in {"sound_imitation", "gesture_requesting"}:
        return (
            "Success is any communication attempt: looking, pointing, reaching, signing, making a sound, or trying the target word/sound.",
            "Use one very favorite item, model first, and accept a look, point, gesture, or sound.",
            "Only if easy: add one new familiar word/sound or let the child choose the item."
        )
    if activity_family in {"attention_sustained"}:
        return (
            "Success is staying with the activity for the bridge-level amount of time with support, even with brief redirection.",
            "Shorten the activity, reduce pieces, or sit beside the child and guide the next step.",
            "Only if easy: add 30–60 seconds or one extra piece, not a harder task."
        )
    return (
        "Success is any calm attempt at the bridge step with the level of help the child needs.",
        "Model first, use fewer choices, shorten the turn, or give more support.",
        "Only if easy and enjoyable: reduce help slightly, add one extra turn, or move one small step closer to independence."
    )


def _v21_make_activity(category_key: str, bridge: Dict[str, Any], activity_type: str, variant: int, state: Dict[str, Any]) -> Dict[str, Any]:
    b = dict(bridge)
    focus, rule = _v21_actionable_focus(state, category_key, b)
    b["activity_focus_step"] = focus
    b["bridge_focus_rule"] = rule
    activity_family = _v21_activity_family_for_bridge(category_key, b)
    b["activity_family"] = activity_family

    llm_data = _v21_call_llm_activity_writer(state, category_key, b, variant)
    fallback = _v21_fallback_activity_text(state, category_key, b, variant)
    data = dict(fallback)
    if llm_data:
        # Use LLM only if it passes alignment validation below.
        data.update({k: v for k, v in llm_data.items() if str(v).strip()})

    title = _v21_clean_display_text(data.get("title") or fallback["title"])
    theme = _v21_clean_display_text(data.get("theme") or fallback["theme"])
    instructions = _v21_clean_display_text(data.get("instructions") or fallback["instructions"])
    cdc_goal = _v20_clean_sentence(b.get("cdc_milestone") or b.get("target_milestone") or "")
    table_bridge = _v20_clean_sentence(b.get("bridge_step") or "")
    prev = _v20_clean_sentence(b.get("previous_bridge_step") or "")
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key.replace("_", " ").title())
    success, easier, harder = _v21_success_easier_harder(activity_family)

    # Let validated LLM improve supporting fields, but keep fallbacks if missing.
    success = _v21_clean_display_text(data.get("success_criteria") or success)
    easier = _v21_clean_display_text(data.get("make_easier") or easier)
    harder = _v21_clean_display_text(data.get("make_harder") or harder)
    group = _v21_clean_display_text(data.get("group_play_line") or _v21_group_play_for_family(activity_family))
    avoid = _v21_clean_display_text(data.get("what_to_avoid") or _v21_avoid_for_family(activity_family, state, category_key))
    materials = _v21_clean_display_text(data.get("materials") or _v21_materials_for_family(activity_family))

    if activity_type == "easier_backup":
        title = "Easy " + title
        instructions += " Make it easier by using one item, modeling first, shortening the turn, or accepting a smaller response."
    elif activity_type == "harder_stretch":
        title = "Stretch " + title
        instructions += " Only use this if the regular version is easy and enjoyable; add one small step, not pressure."

    activity = {
        "activity_id": f"v21_{category_key}_{_v18_norm(cdc_goal)[:24]}_b1_{activity_type}_{variant}",
        "title": title,
        "activity_type": activity_type,
        "theme": theme,
        "play_theme": theme,
        "category": category_display,
        "category_key": category_key,
        "duration_min": V21_PER_ACTIVITY_MIN,
        "cdc_goal": cdc_goal,
        "target_milestone": cdc_goal,
        "bridge_step": focus,
        "table_bridge_step": table_bridge,
        "activity_focus_step": focus,
        "previous_bridge_step": prev,
        "previous_bridge_status": "not_used_initial_plan__feedback_fallback_only",
        "previous_bridge_usage": "future_feedback_troubleshooting_only",
        "bridge_focus_rule": rule,
        "activity_family": activity_family,
        "allowed_themes": _v21_allowed_themes_for_family(activity_family),
        "what_to_avoid": avoid,
        "why_it_helps": f"This works on the first bridge step — {focus.lower()} — as a small step toward “{cdc_goal}.”" if cdc_goal else f"This works on {focus.lower()} through playful repetition.",
        "instructions": instructions,
        "success_criteria": success,
        "success": success,
        "make_easier": easier,
        "make_harder": harder,
        "group_play_line": group,
        "group_play": group,
        "materials": materials,
        "level": "bridge_step_1",
        "goal": "first_bridge_practice",
        "is_extended_activity": False,
        "extended_reason": "",
        "milestone_key": _v18_norm(cdc_goal),
        "bridge_step_number": 1,
        "subdomain": b.get("subdomain", "unspecified"),
        "source": b.get("source", "all_with_bridge"),
        "alignment_patch": V21_VERSION,
        "llm_activity_writer_used": bool(llm_data),
        "table_review_warning": b.get("table_review_warning", ""),
    }
    problems = _v21_activity_has_alignment_problem(activity, category_key)
    if problems and llm_data:
        # Reject bad LLM output and rebuild from deterministic fallback.
        data = fallback
        activity["title"] = _v21_clean_display_text(data["title"])
        activity["theme"] = _v21_clean_display_text(data["theme"])
        activity["play_theme"] = activity["theme"]
        activity["instructions"] = _v21_clean_display_text(data["instructions"])
        activity["materials"] = _v21_materials_for_family(activity_family)
        activity["what_to_avoid"] = _v21_avoid_for_family(activity_family, state, category_key)
        activity["llm_activity_writer_used"] = False
        activity["llm_rejected_reasons"] = problems
        problems = _v21_activity_has_alignment_problem(activity, category_key)
    activity["validation_warnings"] = problems
    return activity


def _v21_uniquify_core_titles(activities: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = Counter()
    out = []
    for a in activities:
        b = dict(a)
        if b.get("activity_type") == "core":
            base = _v21_clean_display_text(b.get("title", "Activity")) or "Activity"
            key = _v18_norm(base)
            seen[key] += 1
            if seen[key] > 1:
                theme = _v21_clean_display_text(b.get("theme", ""))
                suffix = theme.title() if theme and _v18_norm(theme) not in key else f"Variation {seen[key]}"
                b["title"] = f"{base} — {suffix}"
        out.append(b)
    return out


def generate_category_activity_bank(state: Dict[str, Any], category_key: str, model: str = V21_ACTIVITY_MODEL, activities_per_category: Optional[int] = None) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key.replace("_", " ").title())
    support_tier = get_support_tier(state, category_key)
    next_steps = select_next_milestones(state, category_key, max_milestones=V21_MAX_MILESTONES_PER_DOMAIN, min_milestones=V21_MIN_MILESTONES_PER_DOMAIN)
    targets = next_steps.get("milestones", [])
    if not targets:
        bank = {
            "status": "no_targets",
            "version": V21_VERSION,
            "category_key": category_key,
            "category": category_display,
            "support_tier": support_tier,
            "planning_mode": next_steps.get("mode", "no_targets"),
            "summary": next_steps.get("message", "No target milestones were selected."),
            "activities": [],
            "target_milestones": [],
        }
        state.setdefault("activity_banks", {})[category_key] = bank
        return bank

    bridge_plan = build_bridge_plan_for_category(state, category_key, targets)
    active_bridges = bridge_plan.get("active_bridge_steps", [])
    daily_time = int(state.get("child", {}).get("daily_time_min", 10) or 10)
    daily_slots = min(max(1, daily_time // V21_PER_ACTIVITY_MIN), V21_MAX_DAILY_ACTIVITIES)
    desired_week1_slots = V21_WEEK1_DAYS * daily_slots
    core_variants = max(1, math.ceil(desired_week1_slots / max(1, len(active_bridges))))
    core_variants = min(max(core_variants, 2), 5)

    activities: List[Dict[str, Any]] = []
    for bridge in active_bridges:
        for variant in range(1, core_variants + 1):
            activities.append(_v21_make_activity(category_key, bridge, "core", variant, state))
        # Backups remain in the bank for future feedback; scheduler uses core cards for week 1.
        activities.append(_v21_make_activity(category_key, bridge, "easier_backup", 1, state))
        activities.append(_v21_make_activity(category_key, bridge, "harder_stretch", 2, state))

    activities = _v21_uniquify_core_titles(activities)
    required = ["title", "cdc_goal", "bridge_step", "why_it_helps", "instructions", "success_criteria", "make_easier", "make_harder", "group_play_line", "what_to_avoid", "materials", "activity_family"]
    warnings = []
    for a in activities:
        for key in required:
            if not str(a.get(key, "")).strip():
                a[key] = "Needs table/content review."
                warnings.append(f"missing_{key}:{a.get('cdc_goal')}")
        warnings.extend([f"{w}:{a.get('title')}" for w in a.get("validation_warnings", [])])
    core_titles = {_v18_norm(a.get("title", "")) for a in activities if a.get("activity_type") == "core"}
    if len(core_titles) < min(desired_week1_slots, len([a for a in activities if a.get("activity_type") == "core"])):
        warnings.append("core_title_uniqueness_below_desired_week1_slots")

    bank = {
        "status": "ok",
        "version": V21_VERSION,
        "category_key": category_key,
        "category": category_display,
        "support_tier": support_tier,
        "planning_mode": next_steps.get("mode", "unknown"),
        "target_milestones": targets,
        "bridge_plan": bridge_plan,
        "activities": activities,
        "core_activity_count": len([a for a in activities if a.get("activity_type") == "core"]),
        "distinct_core_title_count": len(core_titles),
        "desired_week1_slots": desired_week1_slots,
        "validation_warnings": sorted(set(warnings)),
        "summary": f"v21 activity bank for {category_display}: {len(targets)} milestones, bridge_step_1 targets, activity_family-constrained activity writing, planning mode={next_steps.get('mode', 'unknown')}.",
    }
    state.setdefault("activity_banks", {})[category_key] = bank
    return bank


def _v21_core_pool(bank: Dict[str, Any]) -> List[Dict[str, Any]]:
    return [a for a in bank.get("activities", []) if a.get("activity_type") == "core"] or bank.get("activities", [])


def _v21_choose_from_bank(bank: Dict[str, Any], used_week_titles: set, used_day_titles: set, used_day_goals: set, counts_by_goal: Counter, counts_by_family: Counter) -> Optional[Dict[str, Any]]:
    pool = _v21_core_pool(bank)
    if not pool:
        return None
    def score(a: Dict[str, Any]):
        title = _v18_norm(a.get("title", ""))
        goal = _v18_norm(a.get("cdc_goal", ""))
        fam = a.get("activity_family", "")
        return (
            1000 if title in used_week_titles else 0,
            500 if title in used_day_titles else 0,
            100 if goal in used_day_goals else 0,
            counts_by_goal[goal],
            counts_by_family[fam],
            title,
        )
    candidates = sorted(pool, key=score)
    for a in candidates:
        if _v18_norm(a.get("title", "")) not in used_week_titles:
            return a
    return None


def _v21_slot_categories(active_categories: List[str], max_items: int, day_idx: int) -> List[str]:
    if not active_categories:
        return []
    cats = []
    start = (day_idx - 1) % len(active_categories)
    for i in range(max_items):
        cats.append(active_categories[(start + i) % len(active_categories)])
    return cats


def _v21_make_repeat_item(item: Dict[str, Any], day_idx: int) -> Dict[str, Any]:
    repeat = dict(item)
    repeat["repeat_phase"] = "week_2_repeat_or_adapt"
    repeat["title"] = str(item.get("title", "Activity")) + " — repeat/adapt"
    base_instr = str(item.get("instructions", "")).strip()
    repeat["instructions"] = (
        "Repeat the familiar Week 1 activity. Keep the same bridge step and activity family. "
        "Use the easier version if it was too hard, the harder version if it was too easy, or a new material/theme if the child disliked it. "
        + base_instr
    )
    repeat["week2_rule"] = "same_bridge_step_repeat_with_feedback_based_adjustment"
    return repeat


def build_weekly_schedule(state: Dict[str, Any], cycle_days: int = V21_DEFAULT_CYCLE_DAYS) -> Dict[str, Any]:
    categories = state.get("planned_categories") or state.get("selected_categories", [])
    daily_time = int(state.get("child", {}).get("daily_time_min", 10) or 10)
    if not categories:
        schedule = {"status": "no_categories", "version": V21_VERSION, "summary": "No categories selected for planning.", "days": {}}
        state["weekly_schedule"] = schedule
        return schedule

    for cat in categories:
        if cat not in state.get("activity_banks", {}):
            generate_category_activity_bank(state, cat)
    active_categories = [cat for cat in categories if _v21_core_pool(state.get("activity_banks", {}).get(cat, {}))]
    if not active_categories:
        schedule = {
            "status": "no_activities",
            "version": V21_VERSION,
            "summary": "No activity cards could be built. Check that the selected concern domains have milestones and bridge_step_1 rows in all_with_bridge.",
            "days": {},
        }
        state["weekly_schedule"] = schedule
        return schedule

    max_items = min(max(1, daily_time // V21_PER_ACTIVITY_MIN), V21_MAX_DAILY_ACTIVITIES)
    days: Dict[str, Dict[str, Any]] = {}
    counts_by_goal = Counter()
    counts_by_family = Counter()
    week1_titles = set()
    warnings = []

    # Week 1: no duplicate titles when enough validated activities exist.
    for d in range(1, min(V21_WEEK1_DAYS, cycle_days) + 1):
        items = []
        used_day_titles = set()
        used_day_goals = set()
        for cat in _v21_slot_categories(active_categories, max_items, d):
            bank = state.get("activity_banks", {}).get(cat, {})
            act = _v21_choose_from_bank(bank, week1_titles, used_day_titles, used_day_goals, counts_by_goal, counts_by_family)
            if not act:
                warnings.append(f"day_{d}:not_enough_unique_week1_activities_for_{cat}")
                continue
            item = _schedule_item_from_activity(cat, act)
            # Preserve v21 fields not included by older schedule helper.
            for extra in ["activity_family", "table_bridge_step", "bridge_focus_rule", "previous_bridge_usage", "validation_warnings", "table_review_warning", "llm_activity_writer_used"]:
                item[extra] = act.get(extra, "")
            title_key = _v18_norm(item.get("title", ""))
            goal_key = _v18_norm(item.get("cdc_goal", ""))
            fam = item.get("activity_family", "")
            week1_titles.add(title_key)
            used_day_titles.add(title_key)
            used_day_goals.add(goal_key)
            counts_by_goal[goal_key] += 1
            counts_by_family[fam] += 1
            items.append(item)
        days[f"day_{d}"] = {"week": 1, "phase": "learn_unique_bank", "total_minutes": sum(int(i.get("duration_min", 5)) for i in items), "items": items}

    # Week 2 repeats/adapts corresponding Week 1 day.
    for d in range(V21_WEEK1_DAYS + 1, cycle_days + 1):
        base_day = ((d - 1) % V21_WEEK1_DAYS) + 1
        base_items = days.get(f"day_{base_day}", {}).get("items", [])
        repeat_items = [_v21_make_repeat_item(item, d) for item in base_items]
        days[f"day_{d}"] = {"week": 2, "phase": "repeat_and_adapt", "repeat_of_day": base_day, "total_minutes": sum(int(i.get("duration_min", 5)) for i in repeat_items), "items": repeat_items}

    expected_week1_slots = V21_WEEK1_DAYS * max_items
    if len(week1_titles) < expected_week1_slots:
        warnings.append(f"week1_unique_titles_{len(week1_titles)}_below_expected_{expected_week1_slots}")

    schedule = {
        "status": "ok",
        "version": V21_VERSION,
        "summary": "v21 schedule: initial plan targets bridge_step_1 only, uses activity_family-constrained activity writing, Week 1 introduces unique activities, and Week 2 repeats/adapts based on parent feedback.",
        "daily_time_min": daily_time,
        "bank_repeat_window_days": 14,
        "first_week_distinct_activity_count": len(week1_titles),
        "expected_week1_activity_slots": expected_week1_slots,
        "feedback_options": V21_ACTIVITY_FEEDBACK_OPTIONS,
        "validation_warnings": sorted(set(warnings + sum([state.get("activity_banks", {}).get(cat, {}).get("validation_warnings", []) for cat in categories], []))),
        "selection_rules": [
            "Explicit parent concerns are selected up to 2 domains; peer/other-kids phrases map to Social / Emotional.",
            "If no clear gap is found, v21 still creates a concern-support plan from closest age-appropriate milestones related to the parent concern.",
            "Initial activities target bridge_step_number=1 only.",
            "previous_bridge_step is not used in the first plan; it is saved only for future feedback troubleshooting.",
            "Activity writing is constrained by activity_family to prevent speech-goal/motor-activity and cognitive-goal/motor-activity mismatches.",
            "Week 1 should have no duplicate activity titles when enough validated cards exist; Week 2 repeats/adapts Week 1.",
        ],
        "days": days,
    }
    state["weekly_schedule"] = schedule
    return schedule


def display_weekly_schedule(state: Dict[str, Any], max_days: Optional[int] = None):
    sched = state.get("weekly_schedule") or build_weekly_schedule(state)
    print("\n" + "=" * 100)
    print("V21 FIRST-BRIDGE ACTIVITY-FAMILY WEEKLY SCHEDULE")
    print("=" * 100)
    print("Summary:", sched.get("summary", ""))
    print("Daily time budget:", sched.get("daily_time_min", ""))
    print("Bank repeat window:", sched.get("bank_repeat_window_days", ""), "days")
    print("First-week distinct activities:", sched.get("first_week_distinct_activity_count", ""), "/ expected", sched.get("expected_week1_activity_slots", ""))
    print("Feedback options:", sched.get("feedback_options", {}))
    if sched.get("validation_warnings"):
        print("Validation warnings:")
        for w in sched.get("validation_warnings", [])[:20]:
            print("-", w)
    source = globals().get("V18_BRIDGE_SOURCE_PATH", None) or globals().get("BRIDGE_SOURCE_PATH", None)
    if source:
        print("Source bridge table:", source)

    for cat, bank in state.get("activity_banks", {}).items():
        if bank.get("status") != "ok":
            continue
        print(f"\nCoverage set — {DOMAIN_CONFIG.get(cat, {}).get('display', cat)}:")
        print("  Planning mode:", bank.get("planning_mode", ""))
        for idx, m in enumerate(bank.get("target_milestones", []), start=1):
            print(f"  {idx}. {m.get('months')}m | {m.get('subdomain')} | {m.get('milestone')} [{m.get('answer_norm', 'target')}] | {m.get('selection_reason', '')}")
        bp = bank.get("bridge_plan", {})
        if bp.get("active_bridge_steps"):
            print(f"Bridge focus — {DOMAIN_CONFIG.get(cat, {}).get('display', cat)}:")
            for idx, b in enumerate(bp.get("active_bridge_steps", []), start=1):
                focus, rule = _v21_actionable_focus(state, cat, b)
                warning = f" | WARNING: {b.get('table_review_warning')}" if b.get("table_review_warning") else ""
                print(f"  {idx}. CDC: {b.get('cdc_milestone')} | bridge_step_1: {focus} | activity_family: {b.get('activity_family')} | previous bridge: feedback fallback only | rule: {rule}{warning}")

    days = sched.get("days", {})
    day_names = list(days.keys())
    if max_days:
        day_names = day_names[:max_days]
    for day_name in day_names:
        day_info = days.get(day_name, {})
        week = day_info.get("week")
        phase = day_info.get("phase", "")
        extra = f" | Week {week}: {phase.replace('_', ' ')}" if week else ""
        if day_info.get("repeat_of_day"):
            extra += f" | repeats Day {day_info.get('repeat_of_day')} with adaptation"
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min{extra}")
        print("-" * 100)
        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue
        for item in items:
            print(f"- [{item.get('category')}] {item.get('title')} ({item.get('duration_min', 5)} min)")
            print(f"  Type: {item.get('activity_type')} | Family: {item.get('activity_family')} | Theme: {item.get('theme')}")
            print(f"  CDC goal: {item.get('cdc_goal')}")
            print(f"  Bridge step 1: {item.get('bridge_step')}")
            print(f"  Previous bridge: not used in initial plan; feedback fallback only")
            if item.get("table_review_warning"):
                print(f"  Table warning: {item.get('table_review_warning')}")
            print(f"  Why: {item.get('why_it_helps')}")
            print(f"  Instructions: {item.get('instructions')}")
            print(f"  Success: {item.get('success_criteria')}")
            print(f"  Easier: {item.get('make_easier')}")
            print(f"  Harder: {item.get('make_harder')}")
            print(f"  Group play: {item.get('group_play_line')}")
            print(f"  Avoid: {item.get('what_to_avoid')}")
            print(f"  Materials: {item.get('materials')}")

print("Loaded Genex Brain v21 patch:", V21_VERSION)
print("- Initial plan uses bridge_step_number=1 only; previous_bridge_step is feedback fallback only.")
print("- No-clear-gap cases create parent-concern support plans instead of empty schedules.")
print("- activity_family constrains LLM/fallback activity writing and validation.")
print("- Week 1 enforces unique titles when possible; Week 2 repeats/adapts.")


Loaded Genex Brain v21 patch: v21_first_bridge_activity_family
- Initial plan uses bridge_step_number=1 only; previous_bridge_step is feedback fallback only.
- No-clear-gap cases create parent-concern support plans instead of empty schedules.
- activity_family constrains LLM/fallback activity writing and validation.
- Week 1 enforces unique titles when possible; Week 2 repeats/adapts.


## Internal base layer: v21 table-ready patch

This final patch aligns v21 with the cleaned app-ready bridge table: `all_with_bridge_family`, the `activity_family` column, and the `activity_family_legend` sheet. It reloads the final table if present, builds dynamic activity-family/category mappings from the table, gives the LLM the legend description, and tightens validation without over-flagging words like “step” in non-motor activities.

In [56]:

# ------------------------------------------------------------
# Genex Brain v21 — table-ready patch for cleaned all_with_bridge_family table
# ------------------------------------------------------------
# This patch should be run AFTER the original v21 patch cell.
# It makes v21 compatible with the final app-ready table:
#   cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx
# Sheet:
#   all_with_bridge_family
# Legend:
#   activity_family_legend

from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter
import re

try:
    import pandas as pd
except Exception:
    pd = globals().get("pd")

V21_VERSION = "v21_table_ready_first_bridge_activity_family_legend"

# Prefer Sara's final table. Keep older filenames as fallbacks so the notebook still runs
# if the file is renamed in the repo.
V21_PREFERRED_BRIDGE_FILENAMES = [
    "cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx",
    "cdc_milestones_with_bridges_family_cleaned.xlsx",
    "cdc_milestones_with_bridges_family.xlsx",
    "cdc_milestones_with_bridges_previous(1).xlsx",
    "cdc_milestones_with_bridges_previous.xlsx",
    "cdc_milestones_with_bridges.xlsx",
    "cdc_milestones_with_bridges.numbers",
]

# v18 table loader is still used by v21 internally. Point it at the family sheet first.
try:
    V18_BRIDGE_SHEET_NAME = "all_with_bridge_family"
    if "V18_PREFERRED_BRIDGE_FILENAMES" in globals():
        V18_PREFERRED_BRIDGE_FILENAMES = V21_PREFERRED_BRIDGE_FILENAMES + [
            x for x in V18_PREFERRED_BRIDGE_FILENAMES if x not in V21_PREFERRED_BRIDGE_FILENAMES
        ]
except Exception:
    pass


def _v21_search_roots_table_ready() -> List[Path]:
    roots = []
    if "_v18_search_roots" in globals():
        try:
            roots.extend(_v18_search_roots())
        except Exception:
            pass
    roots.extend([Path.cwd(), Path("/mnt/data")])
    # De-duplicate while preserving order.
    out = []
    seen = set()
    for r in roots:
        rp = Path(r).expanduser()
        if str(rp) not in seen:
            out.append(rp)
            seen.add(str(rp))
    return out


def _v21_find_bridge_file_table_ready(path: Optional[str] = None) -> Optional[Path]:
    if path:
        p = Path(path).expanduser()
        if p.exists():
            return p
    for root in _v21_search_roots_table_ready():
        for name in V21_PREFERRED_BRIDGE_FILENAMES:
            candidates = [
                root / name,
                root / "notebooks" / name,
                root / "mvp" / name,
                root / "notebooks" / "mvp" / name,
                root / "mvp" / "notebooks" / name,
            ]
            for p in candidates:
                if p.exists():
                    return p
    return None


def _v21_clean_activity_family_value(value: Any) -> str:
    raw = str(value or "").strip().lower()
    raw = re.sub(r"[^a-z0-9]+", "_", raw).strip("_")
    return raw


def _v21_normalize_activity_family_column() -> None:
    """Normalize activity_family values in the loaded bridge table."""
    global BRIDGE_MILESTONE_DF
    df = globals().get("BRIDGE_MILESTONE_DF", None)
    if df is None or getattr(df, "empty", True):
        return
    if "activity_family" not in df.columns:
        # Infer if older table is loaded, but final table should always include it.
        inferred = []
        for _, row in df.iterrows():
            cat = str(row.get("category_key", ""))
            try:
                inferred.append(_v21_activity_family_for_bridge(cat, row.to_dict()))
            except Exception:
                inferred.append("general_play")
        df["activity_family"] = inferred
    else:
        df["activity_family"] = df["activity_family"].map(_v21_clean_activity_family_value)
    # Keep global pointer updated.
    BRIDGE_MILESTONE_DF = df


def _v21_read_sheet_with_header(path: Path, preferred_sheets: List[str], required_header: str) -> Tuple[Optional["pd.DataFrame"], Optional[str]]:
    if pd is None:
        return None, None
    try:
        xls = pd.ExcelFile(path)
    except Exception:
        return None, None
    lower = {s.lower(): s for s in xls.sheet_names}
    ordered = []
    for s in preferred_sheets:
        if s.lower() in lower:
            ordered.append(lower[s.lower()])
    ordered += [s for s in xls.sheet_names if s not in ordered]
    for sheet in ordered:
        try:
            raw = pd.read_excel(path, sheet_name=sheet, header=None)
        except Exception:
            continue
        if raw.empty:
            continue
        header_row = None
        for idx in range(min(len(raw), 10)):
            vals = [_v21_clean_activity_family_value(v) for v in raw.iloc[idx].tolist()]
            if _v21_clean_activity_family_value(required_header) in vals:
                header_row = idx
                break
        if header_row is None:
            continue
        headers = [_v21_clean_activity_family_value(v) or f"col_{i}" for i, v in enumerate(raw.iloc[header_row].tolist())]
        df = raw.iloc[header_row + 1:].copy().reset_index(drop=True)
        df.columns = headers
        return df, sheet
    return None, None


V21_ACTIVITY_FAMILY_LEGEND: Dict[str, str] = {}
V21_ACTIVITY_FAMILY_CATEGORY_FROM_TABLE: Dict[str, str] = {}


def _v21_load_activity_family_legend(path: Optional[Path] = None) -> Dict[str, str]:
    """Load the optional activity_family legend sheet from the final workbook."""
    global V21_ACTIVITY_FAMILY_LEGEND
    p = Path(path) if path else _v21_find_bridge_file_table_ready()
    legend: Dict[str, str] = {}
    if p is None or pd is None or p.suffix.lower() != ".xlsx":
        V21_ACTIVITY_FAMILY_LEGEND = legend
        return legend
    df, sheet = _v21_read_sheet_with_header(
        p,
        preferred_sheets=["activity_family_legend", "activity_family_legend_v2"],
        required_header="activity_family",
    )
    if df is None or df.empty:
        V21_ACTIVITY_FAMILY_LEGEND = legend
        return legend
    fam_col = "activity_family"
    desc_cols = [c for c in df.columns if "description" in c or "allowed" in c or c == "description"]
    desc_col = desc_cols[0] if desc_cols else (df.columns[1] if len(df.columns) > 1 else fam_col)
    for _, row in df.iterrows():
        fam = _v21_clean_activity_family_value(row.get(fam_col, ""))
        desc = str(row.get(desc_col, "") or "").strip()
        if fam:
            legend[fam] = desc or "Use activities that directly practice this activity family; do not switch domains."
    V21_ACTIVITY_FAMILY_LEGEND = legend
    return legend


def _v21_update_family_category_from_loaded_table() -> Dict[str, str]:
    """Build a dynamic activity_family -> category_key map from the actual bridge table."""
    global V21_ACTIVITY_FAMILY_CATEGORY_FROM_TABLE, V21_FAMILY_CATEGORY
    df = globals().get("BRIDGE_MILESTONE_DF", None)
    mapping: Dict[str, str] = {}
    if df is not None and not getattr(df, "empty", True) and "activity_family" in df.columns and "category_key" in df.columns:
        for fam, group in df.groupby("activity_family"):
            fam = _v21_clean_activity_family_value(fam)
            if not fam:
                continue
            counts = Counter(str(x) for x in group["category_key"].tolist() if str(x).strip())
            if counts:
                mapping[fam] = counts.most_common(1)[0][0]
    V21_ACTIVITY_FAMILY_CATEGORY_FROM_TABLE = mapping
    if "V21_FAMILY_CATEGORY" in globals():
        # Keep hardcoded definitions, but add all final-table families too.
        V21_FAMILY_CATEGORY.update(mapping)
    return mapping


def _v21_reload_final_bridge_table(path: Optional[str] = None) -> Optional[Path]:
    """Reload the final all_with_bridge_family table if available and refresh family supports."""
    p = _v21_find_bridge_file_table_ready(path)
    if p is None:
        print("v21 table-ready patch: no bridge table found; using whatever table is already loaded.")
        _v21_normalize_activity_family_column()
        _v21_update_family_category_from_loaded_table()
        return None
    if "_v18_refresh_tables" in globals():
        _v18_refresh_tables(str(p))
    _v21_normalize_activity_family_column()
    _v21_load_activity_family_legend(p)
    _v21_update_family_category_from_loaded_table()
    print("v21 table-ready patch loaded bridge table:", p)
    print("Activity families in table:", len(V21_ACTIVITY_FAMILY_CATEGORY_FROM_TABLE))
    print("Activity family legend entries loaded:", len(V21_ACTIVITY_FAMILY_LEGEND))
    return p


# Save original helpers before overriding them.
_V21_ORIGINAL_ALLOWED_THEMES = globals().get("_v21_allowed_themes_for_family")
_V21_ORIGINAL_MATERIALS = globals().get("_v21_materials_for_family")
_V21_ORIGINAL_AVOID = globals().get("_v21_avoid_for_family")
_V21_ORIGINAL_GROUP = globals().get("_v21_group_play_for_family")
_V21_ORIGINAL_FALLBACK_TEXT = globals().get("_v21_fallback_activity_text")


def _v21_family_category(activity_family: str) -> Optional[str]:
    fam = _v21_clean_activity_family_value(activity_family)
    if fam in V21_ACTIVITY_FAMILY_CATEGORY_FROM_TABLE:
        return V21_ACTIVITY_FAMILY_CATEGORY_FROM_TABLE[fam]
    return globals().get("V21_FAMILY_CATEGORY", {}).get(fam)


def _v21_family_description(activity_family: str) -> str:
    fam = _v21_clean_activity_family_value(activity_family)
    return V21_ACTIVITY_FAMILY_LEGEND.get(fam, "Use activities that directly practice this activity family; do not switch domains.")


def _v21_allowed_themes_for_family(activity_family: str) -> List[str]:
    fam = _v21_clean_activity_family_value(activity_family)
    if callable(_V21_ORIGINAL_ALLOWED_THEMES):
        old = _V21_ORIGINAL_ALLOWED_THEMES(fam)
        if old and old != ["playful home routine", "family turn", "toy game", "short routine", "helper game"]:
            return old
    cat = _v21_family_category(fam)
    if cat == "language_and_communication":
        return ["real object play", "picture book game", "family photo game", "puppet turn", "choice game"]
    if cat == "social_and_emotional":
        return ["caregiver social game", "peekaboo/copy-me game", "turn-taking game", "pretend family play", "feelings helper"]
    if cat == "cognitive":
        return ["matching/sorting game", "routine picture game", "short focus game", "helper routine", "toy problem-solving"]
    if cat == "movement_and_physical":
        return ["floor or table play", "daily routine helper", "fine-motor game", "safe movement game", "self-care practice"]
    return ["playful home routine", "family turn", "toy game", "short routine", "helper game"]


def _v21_materials_for_family(activity_family: str) -> str:
    fam = _v21_clean_activity_family_value(activity_family)
    if callable(_V21_ORIGINAL_MATERIALS):
        old = _V21_ORIGINAL_MATERIALS(fam)
        if old and old != "simple household toys or materials that match the bridge step":
            return old
    cat = _v21_family_category(fam)
    if cat == "language_and_communication":
        return "2–3 favorite real objects, picture book or picture cards, family photos, puppet or stuffed animal"
    if cat == "social_and_emotional":
        return "favorite toy, puppet/stuffed animal, family member or sibling, simple picture/feeling cards if useful"
    if cat == "cognitive":
        return "small set of toys/cards, routine pictures, basket/cups, blocks or matching items"
    if cat == "movement_and_physical":
        return "safe household items matched to the skill, clear space or stable seating as needed, caregiver support"
    return "simple household toys or materials that match the bridge step"


def _v21_avoid_for_family(activity_family: str, state: Dict[str, Any], category_key: str) -> str:
    fam = _v21_clean_activity_family_value(activity_family)
    if callable(_V21_ORIGINAL_AVOID):
        old = _V21_ORIGINAL_AVOID(fam, state, category_key)
        # Keep specific old avoid text when available.
        if old and old != "Avoid pressure, testing, repeated correction, or continuing when the child is tired or frustrated.":
            return old
    cat = category_key or _v21_family_category(fam)
    if cat == "language_and_communication":
        return "Avoid pressure to speak, repeated correction, testing, or requiring longer speech than the bridge step. Accept gestures, sounds, approximations, or one word when appropriate."
    if cat == "social_and_emotional":
        return "Avoid forcing interaction, long turns, shaming, or continuing after distress. Keep social play short, warm, and easy to join."
    if cat == "movement_and_physical":
        return "Use close supervision and safe positioning; avoid rushing, unsafe surfaces, choking hazards, fatigue, or forcing movement."
    return "Avoid pressure, testing, repeated correction, or continuing when the child is tired or frustrated."


def _v21_group_play_for_family(activity_family: str) -> str:
    fam = _v21_clean_activity_family_value(activity_family)
    if callable(_V21_ORIGINAL_GROUP):
        old = _V21_ORIGINAL_GROUP(fam)
        if old and old != "2+ people: one person models, another gives one simple cue, and the child joins at their own level.":
            return old
    cat = _v21_family_category(fam)
    if cat == "language_and_communication":
        return "2+ people: one person shows the object/picture or models one sound/word, another waits warmly, and the child points, gestures, makes a sound, or tries the word."
    if cat == "social_and_emotional":
        return "2+ people: one person models the social turn, another supports the child’s easy turn, and everyone keeps it short and positive."
    if cat == "movement_and_physical":
        return "2+ people: one person models the movement or routine step while another stays close to support safety and success."
    return "2+ people: one person models, another gives one simple cue, and the child joins at their own level."


def _v21_activity_prompt(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any], variant: int) -> str:
    child = state.get("child", {})
    safety = format_safety_constraints_for_prompt(ensure_safety_profile(state)) if "ensure_safety_profile" in globals() else "Use common-sense child safety and caregiver supervision."
    fam = _v21_clean_activity_family_value(bridge.get("activity_family", ""))
    family_desc = _v21_family_description(fam)
    return f"""
You are writing one parent-facing Genex home activity card.

Rules you must follow:
- Write the activity ONLY for the activity_family and bridge_step_1 below.
- Use bridge_step_1 as the real activity target. Do not use previous_bridge_step in the initial plan.
- Do NOT create a motor activity for language/cognitive/social goals.
- Do NOT create a language drill for movement goals.
- Do NOT change the skill family, developmental target, or domain.
- Instructions must be concrete: setup, adult model, child turn, what counts as success, and when to stop.
- Avoid generic phrases like “set up one simple playful turn,” “model the small step once,” or “materials that match the bridge step.”
- Keep it playful, low-pressure, and doable in 5 minutes.
- Use only safe household materials.
- Return JSON only with keys: title, theme, instructions, success_criteria, make_easier, make_harder, group_play_line, what_to_avoid, materials.

Child:
- name: {child.get('name', 'your child')}
- chronological age months: {child.get('chronological_months', '')}
- diagnosis/condition: {child.get('diagnosis', child.get('condition', ''))}
- parent concern: {child.get('concern', child.get('parent_concern', ''))}

Planning inputs:
- domain: {DOMAIN_CONFIG.get(category_key, {}).get('display', category_key)}
- subdomain: {bridge.get('subdomain', '')}
- CDC milestone: {bridge.get('cdc_milestone', bridge.get('target_milestone', ''))}
- bridge_step_1: {bridge.get('activity_focus_step') or bridge.get('bridge_step', '')}
- activity_family: {fam}
- activity_family_description: {family_desc}
- variant number: {variant}
- allowed themes: {', '.join(_v21_allowed_themes_for_family(fam))}
- suggested materials: {_v21_materials_for_family(fam)}
- safety notes: {safety}
""".strip()


def _v21_activity_has_alignment_problem(activity: Dict[str, Any], category_key: str) -> List[str]:
    problems = []
    fam = _v21_clean_activity_family_value(activity.get("activity_family", ""))
    text = " ".join(str(activity.get(k, "")) for k in ["title", "theme", "instructions", "materials"]).lower()
    expected_cat = _v21_family_category(fam)
    if expected_cat and expected_cat != category_key:
        problems.append(f"activity_family_category_mismatch:{fam}->{expected_cat} not {category_key}")

    # Do not flag the generic word “step” because language/cognitive cards often contain
    # “one-step direction,” “first/then step,” or “routine step.”
    motor_terms = r"\b(floor sticker|walk|jump|hop|toss|throw|catch|balance beam|race|obstacle|climb|stairs|step up|step down)\b"
    if category_key != "movement_and_physical" and re.search(motor_terms, text):
        # Allow language cards to name objects like ball when the activity is word/picture/object practice.
        if not (category_key == "language_and_communication" and re.search(r"word|name|picture|object|photo|sound|gesture", text)):
            problems.append("non_movement_card_contains_motor_activity")
    if category_key == "language_and_communication" and re.search(r"basketball|basket ball|laundry basket toss|throw|toss|catch|race|obstacle", text):
        problems.append("language_card_contains_motor_game")
    if fam in {"time_words", "daily_recall_narrative"} and re.search(r"floor sticker|walk|balance|reach|jump|climb", text):
        problems.append("time_words_card_contains_motor_template")
    if re.search(r"set up one simple playful turn|model the small step once|materials that match the bridge step", text):
        problems.append("generic_placeholder_instruction")
    for key in ["title", "instructions", "success_criteria", "materials", "what_to_avoid"]:
        if not str(activity.get(key, "")).strip():
            problems.append(f"missing_{key}")
    return problems


def _v21_fallback_activity_text(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any], variant: int) -> Dict[str, str]:
    """Less-generic fallback for final-table activity families.

    Use the original v21 fallback when it has a specific template. If it falls back to
    the generic Short Helper/Toy Practice pattern, rewrite it using bridge_step_1,
    activity_family, and legend description.
    """
    fam = _v21_clean_activity_family_value(bridge.get("activity_family", ""))
    if callable(_V21_ORIGINAL_FALLBACK_TEXT):
        old = _V21_ORIGINAL_FALLBACK_TEXT(state, category_key, bridge, variant)
        old_title = str(old.get("title", "")).strip().lower()
        if old_title not in {"short helper game", "family turn game", "toy practice game"}:
            return old
    child_name = state.get("child", {}).get("name", "your child")
    focus = _v20_clean_sentence(bridge.get("activity_focus_step") or bridge.get("bridge_step") or "the bridge step")
    cdc = _v20_clean_sentence(bridge.get("cdc_milestone") or bridge.get("target_milestone") or "the milestone")
    family_words = fam.replace("_", " ").strip() or "skill"
    theme = _v21_allowed_themes_for_family(fam)[(variant - 1) % len(_v21_allowed_themes_for_family(fam))]
    materials = _v21_materials_for_family(fam)
    desc = _v21_family_description(fam)
    title = f"{family_words.title()} Practice"
    cat = category_key or _v21_family_category(fam)
    if cat == "language_and_communication":
        instructions = f"Use {materials}. Show or model one very small example of {focus}. Pause and let {child_name} respond with a look, point, gesture, sound, approximation, or word. Do 2–3 warm turns and stop before it feels like a test."
    elif cat == "social_and_emotional":
        instructions = f"Use {materials}. Start a short playful interaction for {focus}. Model one easy social turn, then invite {child_name} to join at their own level. Keep turns brief, warm, and successful."
    elif cat == "movement_and_physical":
        instructions = f"Use {materials}. Model one safe, small movement or routine part for {focus}. Support {child_name} as needed and count any partial attempt. Stop before fatigue or frustration."
    elif cat == "cognitive":
        instructions = f"Use {materials}. Make a short game around {focus}. Show one example, then invite {child_name} to try one small turn with help. Keep the set small and praise staying with the task."
    else:
        instructions = f"Use {materials}. Make a short playful routine around {focus}. Show one example, invite one small turn, and stop after a few successful tries."
    return {"title": title, "theme": theme, "instructions": instructions, "family_description": desc, "cdc_goal": cdc}


# Reload the final table and prepare family support now.
_V21_TABLE_READY_SOURCE_PATH = _v21_reload_final_bridge_table()

print("Loaded Genex Brain v21 table-ready patch:", V21_VERSION)
print("- Looks for all_with_bridge_family / final app-ready table first.")
print("- Loads activity_family_legend and includes descriptions in the LLM prompt.")
print("- Dynamically maps every activity_family in the table to its category for validation.")
print("- Validator no longer treats the word 'step' alone as a motor activity.")
print("- Initial activities still target bridge_step_number=1 only; previous_bridge_step remains feedback fallback only.")


Loaded Genex Brain v18 table-first bridge source: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx
Loaded bridge sheet/table: all_with_bridge_family
Bridge rows: 369
Deduped CDC milestones: 163
Rows with parent_explanation: 369
Rows with previous_bridge_step: 359
Rows with explicit allowed_themes: 0
Rows with explicit what_to_avoid: 0
v21 table-ready patch loaded bridge table: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx
Activity families in table: 128
Activity family legend entries loaded: 131
Loaded Genex Brain v21 table-ready patch: v21_table_ready_first_bridge_activity_family_legend
- Looks for all_with_bridge_family / final app-ready table first.
- Loads activity_family_legend and includes descriptions in the LLM prompt.
- Dynamically maps every activity_family in the table to its category for validation.
- Validator no longer treats the word 'step' alone as a motor 

## Genex Brain v22 final patch

This is the final planner path for testing and Claude handoff. It overrides legacy v17/v20/v21 planner functions.

In [57]:

# ------------------------------------------------------------
# Genex Brain v22 — Claude-ready final patch
# ------------------------------------------------------------
# This cell must run after the legacy base layers and after the v21 table-ready patch.
#
# v22 fixes the six production blockers found in v21 sample runs:
# 1) Final output/display says V22 only; old v17/v20 labels are not used by the final planner.
# 2) Initial plans always target bridge_step_number = 1. previous_bridge_step is never used for first plans.
# 3) No-clear-gap cases with parent concerns create a parent-concern support plan instead of an empty schedule.
# 4) Week 1 enforces unique activity cards when enough valid cards exist; Week 2 repeats/adapts.
# 5) activity_family is used as a hard guardrail for LLM/fallback activity writing and validation.
# 6) Target selection avoids regressing to much younger not_sure milestones when stronger current evidence exists.

from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple
import math, os, re, json

V22_VERSION = "v22_claude_ready_first_bridge_concern_support"
V21_VERSION = V22_VERSION

# Keep the final app-ready table first everywhere.
V22_PREFERRED_BRIDGE_FILENAMES = [
    "cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx",
    "cdc_milestones_with_bridges_family_cleaned.xlsx",
    "cdc_milestones_with_bridges_family.xlsx",
    "cdc_milestones_with_bridges_previous.xlsx",
    "cdc_milestones_with_bridges.xlsx",
    "cdc_milestones_with_bridges.numbers",
]
try:
    V21_PREFERRED_BRIDGE_FILENAMES = V22_PREFERRED_BRIDGE_FILENAMES
    V18_PREFERRED_BRIDGE_FILENAMES = V22_PREFERRED_BRIDGE_FILENAMES + [
        x for x in globals().get("V18_PREFERRED_BRIDGE_FILENAMES", []) if x not in V22_PREFERRED_BRIDGE_FILENAMES
    ]
    V18_BRIDGE_SHEET_NAME = "all_with_bridge_family"
except Exception:
    pass

# Reload final table after all legacy cells have run.
try:
    if "_v21_reload_final_bridge_table" in globals():
        _V22_TABLE_READY_SOURCE_PATH = _v21_reload_final_bridge_table()
except Exception as e:
    print("v22 table reload note:", e)


def _v22_norm(value: Any) -> str:
    try:
        return _v18_norm(value)
    except Exception:
        return re.sub(r"\s+", " ", str(value or "").strip().lower())


def _v22_clean(value: Any) -> str:
    try:
        return _v21_clean_display_text(value)
    except Exception:
        return re.sub(r"\s+", " ", str(value or "").strip())


def _v22_child_text(state: Dict[str, Any]) -> str:
    child = state.get("child", {})
    return " ".join(str(child.get(k, "")) for k in ["name", "diagnosis", "condition", "concern", "parent_concern", "concerns"]).lower()


def _v22_has_regression_concern(state: Dict[str, Any], category_key: str) -> bool:
    txt = _v22_child_text(state)
    if category_key == "language_and_communication":
        return bool(re.search(r"\b(regress|lost words|lost speech|stopped talking|language loss|speech loss)\b", txt))
    return bool(re.search(r"\b(regress|lost skill|lost ability)\b", txt))


def _v22_answer_index(state: Dict[str, Any], category_key: str) -> Dict[Tuple[int, str], str]:
    if "_v21_answer_index" in globals():
        try:
            return _v21_answer_index(state, category_key)
        except Exception:
            pass
    out = {}
    for a in state.get("qna", {}).get(category_key, []):
        out[(int(a.get("months", 0) or 0), _v22_norm(a.get("milestone", "")))] = str(
            a.get("scoring_norm_answer") or a.get("norm_answer") or a.get("skill_ability") or "not_sure"
        ).lower()
    return out


def _v22_answer_norm(value: Any) -> str:
    if "_v21_answer_norm" in globals():
        try:
            return _v21_answer_norm(value)
        except Exception:
            pass
    v = str(value or "not_asked").strip().lower()
    if v in {"yes", "sometimes", "with_help", "with help", "no", "not_sure", "not sure"}:
        return v.replace(" ", "_")
    return "not_asked" if v in {"", "none", "nan"} else v


def _v22_confirmed_floor_from_answers(state: Dict[str, Any], category_key: str) -> Optional[int]:
    """Highest age band with meaningful yes evidence. Used only to prevent regression to much younger not_sure targets."""
    by_month: Dict[int, Dict[str, int]] = defaultdict(lambda: {"total": 0, "yes": 0, "partial": 0})
    for a in state.get("qna", {}).get(category_key, []):
        if str(a.get("answer_status", "ok")) == "api_error":
            continue
        m = int(a.get("months", 0) or 0)
        ans = _v22_answer_norm(a.get("scoring_norm_answer") or a.get("norm_answer") or a.get("skill_ability"))
        if m <= 0:
            continue
        by_month[m]["total"] += 1
        if ans == "yes":
            by_month[m]["yes"] += 1
        elif ans in {"sometimes", "with_help"}:
            by_month[m]["partial"] += 1
    confirmed = []
    for m, info in by_month.items():
        total = max(1, info["total"])
        yes_ratio = info["yes"] / total
        # A band is considered "confirmed enough" for preventing lower not_sure regression.
        if info["yes"] >= 2 or yes_ratio >= 0.60 or (info["yes"] >= 1 and info["partial"] >= 1):
            confirmed.append(m)
    if confirmed:
        return max(confirmed)
    dev_age = state.get("dev_age", {}).get(category_key)
    if dev_age:
        try:
            return int(dev_age)
        except Exception:
            return None
    return None


def _v22_min_reasonable_target_month(state: Dict[str, Any], category_key: str, chrono: int, working_age: int) -> int:
    floor = _v22_confirmed_floor_from_answers(state, category_key)
    if floor:
        return max(2, int(floor) - 6)
    # Without confirmed evidence, stay near estimated working age but do not go too far back.
    return max(2, int(working_age or chrono) - 12)


def _v22_parent_has_concern_for_category(state: Dict[str, Any], category_key: str) -> bool:
    if "_v21_parent_has_concern_for_category" in globals():
        try:
            return bool(_v21_parent_has_concern_for_category(state, category_key))
        except Exception:
            pass
    profile = ensure_concern_profile(state) if "ensure_concern_profile" in globals() else {}
    return category_key in [d.get("category_key") for d in profile.get("explicit_domains", [])]


# ------------------------------------------------------------------
# v22 target selection — answered gaps first, no-gap concern support, no younger not_sure regression.
# ------------------------------------------------------------------

def select_next_milestones(
    state: Dict[str, Any],
    category_key: str,
    max_milestones: int = V21_MAX_MILESTONES_PER_DOMAIN,
    min_milestones: int = V21_MIN_MILESTONES_PER_DOMAIN,
) -> Dict[str, Any]:
    child = state.get("child", {})
    chrono = int(min(child.get("chronological_months", 60) or 60, 60))
    working_age = _v18_working_age_band(state, category_key) if "_v18_working_age_band" in globals() else chrono
    min_reasonable = _v22_min_reasonable_target_month(state, category_key, chrono, working_age)
    regression_ok = _v22_has_regression_concern(state, category_key)
    answer_idx = _v22_answer_index(state, category_key)

    domain_df = cdc_df[cdc_df["category_key"] == category_key].copy()
    if domain_df.empty:
        return {"status": "no_milestones", "mode": "no_table_rows", "milestones": [], "message": "No milestone rows found for this category."}

    answered_candidates = []
    seen = set()
    skipped_too_young = []
    for key, answer in answer_idx.items():
        ans = _v22_answer_norm(answer)
        if ans == "yes":
            continue
        month, mkey = key
        # v22: not_sure at a much younger age is not a target if the child has stronger later evidence.
        if ans == "not_sure" and not regression_ok and int(month) < min_reasonable:
            skipped_too_young.append((month, mkey))
            continue
        matches = domain_df[(domain_df["months"] == int(month)) & (domain_df["milestone_key"] == mkey)]
        if matches.empty:
            matches = domain_df[domain_df["milestone_key"] == mkey]
        for _, row in matches.iterrows():
            if int(row.get("months", 0) or 0) > chrono + 6:
                continue
            if ans == "not_sure" and not regression_ok and int(row.get("months", 0) or 0) < min_reasonable:
                continue
            d = row.to_dict()
            d["answer_norm"] = ans
            d["selection_reason"] = f"parent_answer_{ans}"
            base_score = _v18_milestone_candidate_score(state, category_key, d, working_age, answer) if "_v18_milestone_candidate_score" in globals() else 0
            # Strong answers outrank not_sure.
            ans_bonus = {"no": 1200, "with_help": 1150, "sometimes": 1100, "not_sure": 750}.get(ans, 500)
            d["score"] = base_score + ans_bonus
            dk = (int(d.get("months", 0) or 0), _v22_norm(d.get("milestone", "")))
            if dk not in seen:
                answered_candidates.append(d)
                seen.add(dk)

    selected = _v21_select_diverse_candidates(answered_candidates, max_milestones=max_milestones, min_milestones=min_milestones) if "_v21_select_diverse_candidates" in globals() else answered_candidates[:max_milestones]
    if selected:
        return {
            "status": "needs_support",
            "mode": "evidence_based_answered_gaps",
            "working_age_band": working_age,
            "milestones": selected,
            "message": f"v22 selected {len(selected)} answered gap milestone(s) for bridge_step_1 planning.",
            "skipped_too_young_not_sure": skipped_too_young[:10],
            "min_reasonable_target_month": min_reasonable,
        }

    # No clear answered gap, but parent concern exists: make a light support plan.
    if _v22_parent_has_concern_for_category(state, category_key):
        age_center = chrono
        min_m = max(2, min_reasonable)
        max_m = min(60, age_center + 6)
        pool = domain_df[(domain_df["months"] >= min_m) & (domain_df["months"] <= max_m)].copy()
        if pool.empty:
            pool = domain_df[domain_df["months"] <= min(60, chrono + 6)].copy()
        candidates = []
        for _, row in pool.iterrows():
            d = row.to_dict()
            key = (int(d.get("months", 0) or 0), _v22_norm(d.get("milestone", "")))
            ans = _v22_answer_norm(answer_idx.get(key))
            if ans == "yes":
                # Keep yes rows only if strongly relevant to concern; lower score.
                pass
            d["answer_norm"] = ans if ans != "not_asked" else "concern_support"
            d["selection_reason"] = "parent_concern_support_no_clear_gap"
            if "_v21_concern_support_score" in globals():
                d["score"] = _v21_concern_support_score(state, category_key, d, age_center)
            else:
                d["score"] = -abs(int(d.get("months", 0) or 0) - age_center)
            if ans == "yes":
                d["score"] -= 15
            candidates.append(d)
        selected = _v21_select_diverse_candidates(candidates, max_milestones=max_milestones, min_milestones=min_milestones) if "_v21_select_diverse_candidates" in globals() else sorted(candidates, key=lambda x: x.get("score", 0), reverse=True)[:max_milestones]
        return {
            "status": "concern_support",
            "mode": "parent_concern_support_no_clear_gap",
            "working_age_band": age_center,
            "milestones": selected,
            "message": (
                f"No clear gap was found in this quick screen for {DOMAIN_CONFIG.get(category_key, {}).get('display', category_key)}, "
                "but because the parent named this concern, v22 selected closest age-appropriate related milestones for a light support plan."
            ),
            "min_reasonable_target_month": min_reasonable,
        }

    return {
        "status": "no_clear_gap_no_named_concern",
        "mode": "no_plan",
        "milestones": [],
        "message": "No clear gap was found in this quick screen and no direct parent concern was mapped to this domain.",
        "min_reasonable_target_month": min_reasonable,
    }


# ------------------------------------------------------------------
# v22 bridge selection — bridge_step_1 only. No previous bridge in initial plan.
# ------------------------------------------------------------------

def _v21_actionable_focus(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any]) -> Tuple[str, str]:
    step = _v20_clean_sentence(bridge.get("bridge_step") or bridge.get("activity_focus_step") or "")
    cdc_goal = _v20_clean_sentence(bridge.get("cdc_milestone") or bridge.get("target_milestone") or "")
    if step and not step.lower().startswith("nan"):
        rule = "v22_initial_plan_uses_bridge_step_1_only"
        if "_v19_is_same_or_final_step" in globals() and _v19_is_same_or_final_step(step, cdc_goal):
            rule = "v22_bridge_step_1_may_equal_cdc_goal__table_review_recommended_but_previous_bridge_not_used"
        return step, rule
    return "Bridge step 1 missing from table — review all_with_bridge_family.", "v22_missing_bridge_step_1__table_review_needed"


def _v20_actionable_focus(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any]) -> Tuple[str, str]:
    return _v21_actionable_focus(state, category_key, bridge)


def _v21_select_active_bridge(state: Dict[str, Any], category_key: str, target: Dict[str, Any]) -> Dict[str, Any]:
    rows = get_bridge_rows_for_target(
        category_key,
        int(target.get("months", 0) or 0),
        target.get("milestone", ""),
        target.get("subdomain", "unspecified"),
    )
    if rows:
        row = dict(_v21_first_bridge_row(rows)) if "_v21_first_bridge_row" in globals() else dict(rows[0])
        row["source"] = "all_with_bridge_family"
    else:
        row = {
            "target_milestone": target.get("milestone", ""),
            "cdc_milestone": target.get("milestone", ""),
            "months": int(target.get("months", 0) or 0),
            "subdomain": target.get("subdomain", "unspecified"),
            "bridge_step_number": 1,
            "bridge_step": "Bridge step 1 missing from table — review all_with_bridge_family.",
            "previous_bridge_step": "",
            "source": "missing_bridge_table_row",
            "table_review_warning": "Missing bridge table row for this target.",
        }
    row["target_milestone"] = target.get("milestone", row.get("target_milestone", ""))
    row["cdc_milestone"] = target.get("milestone", row.get("cdc_milestone", row.get("target_milestone", "")))
    row["months"] = int(target.get("months", row.get("months", 0)) or 0)
    row["subdomain"] = target.get("subdomain", row.get("subdomain", "unspecified"))
    row["answer_norm"] = str(target.get("answer_norm", "not_sure")).lower()
    row["selection_reason"] = target.get("selection_reason", "")
    focus, rule = _v21_actionable_focus(state, category_key, row)
    row["activity_focus_step"] = focus
    row["bridge_focus_rule"] = rule
    row["previous_bridge_status"] = "not_used_initial_plan__feedback_fallback_only"
    row["previous_bridge_usage"] = "future_feedback_troubleshooting_only"
    row["activity_family"] = _v21_activity_family_for_bridge(category_key, row)
    return row


def build_bridge_plan_for_category(state: Dict[str, Any], category_key: str, target_milestones: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    targets = target_milestones or select_next_milestones(state, category_key).get("milestones", [])
    active_bridges = [_v21_select_active_bridge(state, category_key, target) for target in targets]
    plan = {
        "status": "ok" if active_bridges else "no_targets",
        "version": V22_VERSION,
        "category_key": category_key,
        "target_milestones": targets,
        "active_bridge_steps": active_bridges,
        "logic": "v22 initial plan uses bridge_step_number=1 only; previous_bridge_step is hidden for future feedback troubleshooting.",
    }
    state.setdefault("bridge_plans", {})[category_key] = plan
    return plan


# ------------------------------------------------------------------
# v22 family guardrails and concrete fallback writer.
# ------------------------------------------------------------------

def _v22_family_bucket(fam: str, category_key: str = "") -> str:
    fam = _v21_clean_activity_family_value(fam) if "_v21_clean_activity_family_value" in globals() else _v22_norm(fam)
    patterns = [
        ("book_page", r"book_page|page_turn"),
        ("fork_spoon", r"fork|spoon|utensil|feeding"),
        ("dressing_on", r"dressing_on|clothes_on|put.*cloth"),
        ("dressing_off", r"dressing_off|clothes_off|take.*cloth"),
        ("buttoning", r"button|fastener|zipper"),
        ("beading", r"bead|thread|peg|pincer|grasp|prewriting|scribble|crayon|draw|stack|block|fine_motor"),
        ("catch_ball", r"catch_ball|ball"),
        ("jump_prep", r"jump|hop|squat|balance|walk|stair|step_up|gross_motor|safe_"),
        ("expressive_word", r"expressive|vocabulary|first_word|single_word|three_words|book_object_naming|object_naming"),
        ("sound", r"sound|vocal|babbl|raspberr|squeal|coo"),
        ("gesture", r"gesture|request|attention_getting|arms_up|social_caregiver"),
        ("receptive_direction", r"receptive|direction|body_part|book_picture"),
        ("action_label", r"action_picture|action_words"),
        ("function_question", r"function_question"),
        ("sentence", r"sentence|phrase|two_word"),
        ("conversation", r"conversation"),
        ("time_words", r"time_words"),
        ("counting", r"count|number"),
        ("letters", r"letter"),
        ("attention", r"attention"),
        ("matching", r"match|sort|color|shape"),
        ("routine", r"routine|cleanup"),
        ("social_turn", r"peer|turn_taking|sharing|imitation|peekaboo|laughter|social|referencing|emotion|pretend|helper|context|affection|face"),
    ]
    for bucket, pat in patterns:
        if re.search(pat, fam):
            return bucket
    if category_key == "language_and_communication":
        return "expressive_word"
    if category_key == "social_and_emotional":
        return "social_turn"
    if category_key == "movement_and_physical":
        return "beading"
    if category_key == "cognitive":
        return "attention"
    return "general"


def _v22_variant(options: List[Tuple[str, str, str, str]], variant: int) -> Tuple[str, str, str, str]:
    return options[(max(1, int(variant or 1)) - 1) % len(options)]


def _v21_fallback_activity_text(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any], variant: int) -> Dict[str, str]:
    child_name = state.get("child", {}).get("name") or "your child"
    fam = bridge.get("activity_family") or _v21_activity_family_for_bridge(category_key, bridge)
    bucket = _v22_family_bucket(fam, category_key)
    focus = _v22_clean(bridge.get("activity_focus_step") or bridge.get("bridge_step") or "the next small step")
    cdc = _v22_clean(bridge.get("cdc_milestone") or bridge.get("target_milestone") or "the milestone")
    desc = _v21_family_description(fam) if "_v21_family_description" in globals() else ""

    # title, theme, materials, instructions
    if bucket == "book_page":
        title, theme, materials, instr = _v22_variant([
            ("Board Book Page Helper", "board book page turn", "board book or thick-page book, small sticker bookmark", f"Sit with {child_name} and a thick-page book. Help hold the book steady, lift one page edge, and invite {child_name} to pull or push one page over. One assisted page turn counts."),
            ("One Page Peek", "peek under one page", "board book, favorite picture, small bookmark", f"Place a favorite picture under the next thick page. Lift the edge slightly and invite {child_name} to find it by turning one page with help. Stop after 2–3 successful turns."),
            ("Page Turn Choice", "choose and turn", "two board books or one thick-page book", f"Let {child_name} choose a book. Open to one thick page, model turning exactly one page, then invite one small page turn with as much help as needed."),
        ], variant)
    elif bucket == "fork_spoon":
        title, theme, materials, instr = _v22_variant([
            ("Supported Fork Bite", "fork helper", "child fork, soft safe food pieces or pretend food, plate", f"Sit at a table with soft safe food or pretend food. Model one slow fork movement, then help {child_name} stab or scoop one piece and bring it toward the mouth or doll. Keep it supervised and brief."),
            ("Pretend Restaurant Bite", "pretend restaurant", "child fork/spoon, doll or teddy, pretend food", f"Set up a pretend restaurant. Feed the doll one bite first, then invite {child_name} to try one fork or spoon movement with help. Celebrate effort, not neatness."),
            ("Snack Scoop Turn", "snack routine", "child spoon/fork, soft snack, tray", f"Give {child_name} one small supported scoop or stab turn during snack. Help hand position as needed and stop after a few calm attempts."),
        ], variant)
    elif bucket == "dressing_on":
        title, theme, materials, instr = _v22_variant([
            ("Jacket Arm Helper", "dress-up game", "loose jacket, doll/teddy, mirror", f"Put the jacket on a teddy first. Hold a loose sleeve open for {child_name} and invite one arm push. Help as much as needed; one small push counts."),
            ("Laundry Dress-Up Step", "laundry helper", "loose clothing item, laundry basket", f"During laundry or morning routine, choose one loose clothing item. Model one movement, then invite {child_name} to push, pull, or place one part with support."),
            ("Mirror Clothes Step", "mirror routine", "loose pants or jacket, mirror", f"Stand or sit by a mirror. Say one simple cue like “arm in” and help {child_name} complete one small clothing step."),
        ], variant)
    elif bucket == "dressing_off":
        title, theme, materials, instr = _v22_variant([
            ("Pants Down Helper", "dressing-off routine", "elastic-waist pants or loose jacket", f"Use loose clothing. Start the movement for {child_name}, then invite one small pull down, pull off, or arm-out movement. Sitting is fine if balance is hard."),
            ("Teddy Takes It Off", "doll/teddy dressing", "teddy/doll, loose doll clothes", f"Take clothing off a teddy first, then offer {child_name} one easy turn to pull fabric or help remove one part."),
            ("Laundry Pull Game", "laundry helper", "loose socks, pants, or jacket", f"Put a loose item partly on/off and invite {child_name} to pull one piece into the laundry basket. Help as needed and keep it playful."),
        ], variant)
    elif bucket == "buttoning":
        title, theme, materials, instr = _v22_variant([
            ("Big Button Pull-Apart", "button board", "large button/fastener board or loose shirt with big buttons", f"Show one large button or simple closure. Help {child_name} pull apart, push through, or line up one closure step. One partial movement counts."),
            ("Dress-Up Fastener Turn", "dress-up fasteners", "large buttons, zipper board, doll clothing", f"Use dress-up clothes or a board. Model one fastener movement, then let {child_name} try one small part with help."),
            ("Button Treasure", "button treasure", "large safe buttons/fasteners, cloth", f"Hide a picture under a flap with a large closure. Help {child_name} open one closure step to find the treasure."),
        ], variant)
    elif bucket == "beading":
        title, theme, materials, instr = _v22_variant([
            ("Peg Treasure Helper", "peg/ring game", "large pegs or ring stacker, chunky pieces", f"Put one large peg or ring in front of {child_name}. Model placing it on, then invite one supported turn. Use large pieces only and stop before frustration."),
            ("Crayon Mark Try", "short art game", "thick crayon/marker, paper, stickers", f"Offer a thick crayon and paper. Model one short mark, then let {child_name} make any mark or grasp attempt. Do not correct grip repeatedly."),
            ("Big Bead Push", "large bead game", "chunky beads or large pasta, pipe cleaner/taped shoelace", f"Hold the pipe cleaner steady and invite {child_name} to push one large bead or pasta piece on. Switch to pegs if threading is too hard."),
            ("Block Stack Turn", "block build", "large blocks or cups", f"Model placing one block or cup, then invite {child_name} to place one piece. Keep the tower low and easy."),
        ], variant)
    elif bucket == "catch_ball":
        title, theme, materials, instr = _v22_variant([
            ("Slow Ball Look-and-Hands", "slow ball game", "soft ball or rolled socks", f"Sit or stand close. Roll or gently pass a soft ball slowly toward {child_name}. Encourage looking at it and bringing hands/body toward it; catching is not required."),
            ("Laundry Basket Ball Watch", "basket target", "soft ball, laundry basket", f"Place a basket close. Model watching the ball and moving hands toward it, then let {child_name} try with a soft ball. Keep the distance very short."),
            ("Ball Trap Hug", "ball hug", "large soft ball", f"Gently roll a large soft ball to {child_name}'s hands or body and help them trap or hug it. One slow supported trap counts."),
        ], variant)
    elif bucket == "jump_prep":
        title, theme, materials, instr = _v22_variant([
            ("Frog Knees", "frog knees", "clear flat floor, caregiver hand support", f"On a clear flat floor, model bending knees and standing tall. Hold {child_name}'s hands or stay close. Do not ask for a jump; a bend, weight shift, or stand-up counts."),
            ("Sticker Stand-and-Reach", "floor sticker reach", "floor tape/sticker, stable support", f"Place a sticker close by. Help {child_name} stand safely, bend a little, or reach toward it with support. Stop before wobbling or fatigue increases."),
            ("Low Squat Toy Pick-Up", "supported squat", "favorite toy, clear floor", f"Put a toy near knee height or on the floor. Help {child_name} bend slightly and stand back up with support. Keep it slow and safe."),
        ], variant)
    elif bucket == "sound":
        title, theme, materials, instr = _v22_variant([
            ("Silly Sound Mirror", "sound mirror", "mirror, puppet", f"Sit face-to-face or near a mirror. Make one easy sound, then pause. Copy any sound, smile, movement, or attempt {child_name} makes."),
            ("Animal Sound Pause", "animal sounds", "toy animals or picture cards", f"Show one animal and make one simple sound. Pause for {child_name}; any look, movement, vocalization, or sound attempt counts."),
            ("Song Pause Sound", "song pause", "favorite song or rhyme", f"Sing a familiar song and pause before a fun sound. Wait for {child_name} to look, move, vocalize, or smile, then continue."),
        ], variant)
    elif bucket == "gesture":
        title, theme, materials, instr = _v22_variant([
            ("Choice Request", "choice request", "two favorite objects", f"Hold up two favorite items. Wait for {child_name} to look, reach, point, gesture, or make a sound. Give the chosen item right away and name it once."),
            ("Help Box Request", "help request", "clear container, favorite toy", f"Put a favorite toy in a clear container. Pause and wait for {child_name} to look, reach, gesture, or vocalize for help, then open it."),
            ("Up or More Pause", "routine pause", "favorite routine item", f"Pause during a favorite routine and wait for {child_name} to use body movement, arms, eye contact, gesture, or sound to continue."),
        ], variant)
    elif bucket == "receptive_direction":
        title, theme, materials, instr = _v22_variant([
            ("Give-Me Toy Game", "one-step direction", "favorite toy, basket", f"Use one familiar toy. Say one clear direction like “give me the car.” Add a gesture only if needed. Celebrate any attempt."),
            ("Clean-Up One Step", "cleanup direction", "toy, basket/box", f"During cleanup, give one simple direction with a familiar object. Model once if needed, then let {child_name} try one short turn."),
            ("Body Part Show", "body part game", "mirror or teddy", f"Ask for one familiar body part using words first. Add a gesture only if needed. {child_name} can point, touch, or look toward it."),
        ], variant)
    elif bucket == "action_label":
        title, theme, materials, instr = _v22_variant([
            ("Action Picture Point", "action picture", "action cards or picture book", f"Show two clear action pictures. Say one action word, like “eating,” and help {child_name} point or choose. Model the word once; saying it is optional."),
            ("Family Action Guess", "family action", "family member or puppet", f"Act out one familiar action. Ask {child_name} to show, point, gesture, or say what is happening. Accept one word or gesture."),
            ("Puppet Action Word", "puppet action", "puppet, toy object", f"Make the puppet do one action, name it once, and invite {child_name} to point, copy, or try the action word."),
        ], variant)
    elif bucket == "function_question":
        title, theme, materials, instr = _v22_variant([
            ("What Is It For Choice", "object function", "cup, spoon, coat, crayon", f"Show two familiar objects. Ask a simple function question like “What do we drink with?” Help {child_name} point, choose, gesture, or answer."),
            ("Object Job Basket", "function basket", "2–3 household objects", f"Put two objects in a basket. Name one use, then let {child_name} choose the matching object. Model the short answer."),
            ("Pretend Shopping Function", "pretend shopping", "toy basket, familiar objects", f"Pretend to shop for something needed, like “something to draw with.” Let {child_name} choose or say the object."),
        ], variant)
    elif bucket == "sentence":
        title, theme, materials, instr = _v22_variant([
            ("Add-One-Word Phrase", "phrase expansion", "family photo or toy scene", f"Model a short phrase and add one word, like “big car” or “mommy eating.” Invite {child_name} to copy any part or make their own short phrase."),
            ("Toy Scene Talk", "toy scene", "two toy figures or picture scene", f"Make a simple toy scene. Model a 2–4 word sentence, then pause for {child_name}. Any shorter attempt is okay."),
            ("Photo Sentence Builder", "photo sentence", "family photo", f"Look at one photo. Model a short sentence about it and invite {child_name} to add one word, gesture, or phrase."),
        ], variant)
    elif bucket == "conversation":
        title, theme, materials, instr = _v22_variant([
            ("Two-Turn Chat", "short conversation", "favorite toy or photo", f"Ask one easy question about a favorite item, respond warmly, then offer one more related turn. Keep it to two back-and-forth turns."),
            ("Puppet Chat", "puppet conversation", "puppet, toy", f"Let a puppet ask or comment once. Help {child_name} respond with a word, gesture, or phrase, then the puppet replies."),
            ("Photo Conversation", "photo talk", "family photo", f"Look at one photo together. Make a comment, wait for any response, then make one related follow-up."),
        ], variant)
    elif bucket == "time_words":
        title, theme, materials, instr = _v22_variant([
            ("Morning or Night Sort", "routine picture sort", "morning/night routine pictures", f"Show two routine pictures, like breakfast and bedtime. Say “morning” or “night” and help {child_name} place each picture in the right spot."),
            ("Now or Later Choice", "now/later choice", "two routine objects or pictures", f"Use two routine pictures. Say “now” for what is happening and “later” for what comes next. Let {child_name} point or place one card."),
            ("First Then Routine", "first/then routine", "first/then card, routine pictures", f"Show two routine steps. Say “first ___, then ___.” Help {child_name} point, place, or say one time word."),
        ], variant)
    elif bucket == "counting":
        title, theme, materials, instr = _v22_variant([
            ("Count Three Blocks", "counting blocks", "3–5 blocks or toys", f"Place three objects in a row. Count slowly and invite {child_name} to point, tap, or say any number with you."),
            ("Snack Count", "snack counting", "small safe snack pieces", f"Count 1–3 snack pieces together. {child_name} can point, move pieces, or say a number."),
            ("Toy Line Count", "toy lineup", "toy animals or cars", f"Line up a few toys. Count together and stop while it is still fun."),
        ], variant)
    elif bucket == "letters":
        title, theme, materials, instr = _v22_variant([
            ("Name Letter Hunt", "letter hunt", "letter cards or magnetic letters", f"Show one or two letters, especially from {child_name}'s name. Name one letter and help {child_name} point, match, or say it."),
            ("Book Letter Search", "book letter search", "picture book, letter card", f"Pick one letter to find in a book. Model pointing to it, then invite {child_name} to find one."),
            ("Letter Basket", "letter basket", "2–3 letter cards", f"Put a few letter cards in a basket. Ask for one and help {child_name} choose or name it."),
        ], variant)
    elif bucket == "matching":
        title, theme, materials, instr = _v22_variant([
            ("Same Match", "same/different match", "two matching objects or cards", f"Show one item and two choices. Help {child_name} find the same one. One match with help counts."),
            ("Color Sort Basket", "color sort", "colored blocks or cups", f"Sort two colors into baskets. Model one, then let {child_name} place one piece."),
            ("Sock Match", "matching routine", "pairs of socks or household items", f"During laundry, show two socks and help {child_name} find the matching one."),
        ], variant)
    elif bucket == "routine":
        title, theme, materials, instr = _v22_variant([
            ("First-Then Cleanup", "routine cleanup", "toy basket, two routine pictures", f"Show “first cleanup, then play.” Help {child_name} put one item away, then celebrate with the next routine step."),
            ("Two-Step Picture Routine", "picture routine", "two routine pictures", f"Show two routine pictures. Point to the first, do it together, then point to the second."),
            ("Helper Job", "helper routine", "small household item", f"Give {child_name} one small helper job, like putting a napkin on the table or a toy in a basket. Keep it short and successful."),
        ], variant)
    elif bucket == "attention":
        title, theme, materials, instr = _v22_variant([
            ("Two-Minute Finish Game", "short focus game", "timer, 3–5 simple pieces", f"Choose a tiny task with 3–5 pieces. Set a short timer or count down. Help {child_name} stay until the small task is done, then stop."),
            ("Sticker Card Focus", "sticker card", "stickers, paper", f"Put 3 stickers on a card one at a time. Invite {child_name} to stay with the card until all three are placed."),
            ("Block Build Finish", "block build", "3–5 blocks", f"Build a tiny tower with a clear finish: three blocks. Praise staying with it until the last block."),
        ], variant)
    elif bucket == "social_turn":
        title, theme, materials, instr = _v22_variant([
            ("My Turn Your Turn", "turn-taking game", "one favorite toy", f"Use one toy. Say “my turn,” take a short turn, then “your turn” and hand it to {child_name}. Keep turns very short and predictable."),
            ("Peekaboo Pause", "peekaboo social game", "blanket or hands", f"Play peekaboo slowly. Pause before “boo” and watch for eye contact, smile, body excitement, sound, or laughter. Repeat 2–3 times."),
            ("Copy Me Social Game", "copy-me game", "no special materials", f"Do one simple action like clap or tap. Invite {child_name} to copy or respond in any way, then copy them back."),
            ("Look to My Face", "social referencing", "new toy/object", f"Show a mildly new or interesting object. Smile and label it calmly, then pause for {child_name} to look between you and the object."),
        ], variant)
    else:
        title, theme, materials, instr = _v22_variant([
            ("Bridge Step Play", "short home routine", "simple household toys or routine items", f"Choose one simple material and make a short playful routine for: {focus}. Model once, invite one small turn, and stop after a few successful tries."),
            ("Small Step Helper", "helper game", "simple household item", f"Show one small example of {focus}. Let {child_name} try with as much help as needed. Any partial attempt counts."),
            ("Family Practice Turn", "family turn", "favorite toy or household item", f"Take turns practicing one small piece of {focus}. Keep it brief, warm, and low pressure."),
        ], variant)

    return {
        "title": _v22_clean(title),
        "theme": _v22_clean(theme),
        "instructions": _v22_clean(instr),
        "success_criteria": f"Success is any calm attempt at bridge step 1: {focus}.",
        "make_easier": "Use fewer materials, model first, shorten the turn, or accept a smaller response.",
        "make_harder": "Only if easy and enjoyable: add one small turn, reduce help slightly, or move a little closer to the next bridge step.",
        "group_play_line": _v21_group_play_for_family(fam) if "_v21_group_play_for_family" in globals() else "2+ people: one person models, another supports, and the child takes one short turn.",
        "what_to_avoid": _v21_avoid_for_family(fam, state, category_key) if "_v21_avoid_for_family" in globals() else "Avoid pressure, repeated correction, or continuing after fatigue/frustration.",
        "materials": _v22_clean(materials),
        "family_description": desc,
        "cdc_goal": cdc,
    }


def _v21_activity_prompt(state: Dict[str, Any], category_key: str, bridge: Dict[str, Any], variant: int) -> str:
    child = state.get("child", {})
    fam = bridge.get("activity_family", "")
    desc = _v21_family_description(fam) if "_v21_family_description" in globals() else ""
    safety = format_safety_constraints_for_prompt(ensure_safety_profile(state)) if "ensure_safety_profile" in globals() else "Use caregiver supervision and safe household materials."
    focus = bridge.get("activity_focus_step") or bridge.get("bridge_step", "")
    return f"""
You are writing one parent-facing Genex home activity card.

Hard rules:
- Write ONLY for bridge_step_1 and activity_family.
- Do NOT use previous_bridge_step. It is hidden future troubleshooting metadata.
- Do NOT regress to earlier prerequisites.
- Do NOT create a motor game unless the domain is Movement / Physical or activity_family is a movement family.
- Do NOT create a generic placeholder activity.
- Instructions must say exactly what the parent does, what the child does, what counts as success, and when to stop.
- Keep it playful, low-pressure, and doable in 5 minutes.
- Use safe household materials.
- Return JSON only with keys: title, theme, instructions, success_criteria, make_easier, make_harder, group_play_line, what_to_avoid, materials.

Child:
- name: {child.get('name', 'your child')}
- chronological age months: {child.get('chronological_months', '')}
- diagnosis/condition: {child.get('diagnosis', child.get('condition', ''))}
- parent concern: {child.get('concern', child.get('parent_concern', child.get('concerns', '')))}

Planning inputs:
- domain: {DOMAIN_CONFIG.get(category_key, {}).get('display', category_key)}
- subdomain: {bridge.get('subdomain', '')}
- CDC milestone: {bridge.get('cdc_milestone', bridge.get('target_milestone', ''))}
- bridge_step_1: {focus}
- activity_family: {fam}
- activity_family_description: {desc}
- allowed themes: {', '.join(_v21_allowed_themes_for_family(fam)) if "_v21_allowed_themes_for_family" in globals() else ''}
- variant number: {variant}
- safety notes: {safety}
""".strip()


def _v21_activity_has_alignment_problem(activity: Dict[str, Any], category_key: str) -> List[str]:
    problems = []
    fam = _v21_clean_activity_family_value(activity.get("activity_family", "")) if "_v21_clean_activity_family_value" in globals() else _v22_norm(activity.get("activity_family", ""))
    bucket = _v22_family_bucket(fam, category_key)
    text = " ".join(str(activity.get(k, "")) for k in ["title", "theme", "instructions", "materials", "cdc_goal", "bridge_step"]).lower()
    expected_cat = V21_FAMILY_CATEGORY.get(fam) if "V21_FAMILY_CATEGORY" in globals() else None
    if expected_cat and expected_cat != category_key:
        problems.append(f"activity_family_category_mismatch:{fam}->{expected_cat} not {category_key}")

    # Domain mismatch checks. Do not flag "one-step directions" as motor just because of the word step.
    motor_pat = r"\b(floor sticker|jump|hop|toss|throw|catch|race|obstacle|climb|balance beam|stair|trampoline)\b"
    if category_key != "movement_and_physical" and re.search(motor_pat, text):
        problems.append("non_movement_card_contains_motor_activity")
    if category_key == "language_and_communication" and re.search(r"\b(laundry basket toss|basket ball|basketball|throw|toss|catch|race|obstacle)\b", text):
        problems.append("language_card_contains_motor_game")

    # Family-specific hard guardrails.
    if bucket == "book_page" and not re.search(r"\b(book|page|turn|lift|separate)\b", text):
        problems.append("book_page_family_missing_book_page_mechanics")
    if bucket == "fork_spoon" and not re.search(r"\b(fork|spoon|utensil|bite|scoop|stab|food|snack|plate)\b", text):
        problems.append("fork_spoon_family_missing_utensil_mechanics")
    if bucket == "time_words" and not re.search(r"\b(morning|night|now|later|first|then|today|tomorrow|routine)\b", text):
        problems.append("time_words_family_missing_time_routine_language")
    if bucket == "receptive_direction" and not re.search(r"\b(direction|give|put|show|point|find|cleanup|clean-up|body part)\b", text):
        problems.append("receptive_direction_family_missing_direction_action")
    if bucket == "action_label" and not re.search(r"\b(action|picture|doing|running|eating|playing|point|show|act)\b", text):
        problems.append("action_label_family_missing_action_picture_work")
    if bucket == "expressive_word" and not re.search(r"\b(word|sound|name|object|picture|photo|toy|say|try)\b", text):
        problems.append("expressive_family_missing_word_or_object_practice")

    if re.search(r"set up one simple playful turn|model the small step once|materials that match the bridge step", text):
        problems.append("generic_placeholder_instruction")
    for key in ["title", "instructions", "success_criteria", "materials", "what_to_avoid"]:
        if not str(activity.get(key, "")).strip():
            problems.append(f"missing_{key}")
    return problems


# ------------------------------------------------------------------
# v22 activity bank / scheduler — unique Week 1, Week 2 repeats/adapts.
# ------------------------------------------------------------------

def _v22_title_key(title: str) -> str:
    return _v22_norm(title)


def _v21_uniquify_core_titles(activities: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = Counter()
    out = []
    for a in activities:
        b = dict(a)
        if b.get("activity_type") == "core":
            base = _v22_clean(b.get("title", "Activity")) or "Activity"
            key = _v22_title_key(base)
            seen[key] += 1
            if seen[key] > 1:
                fam = str(b.get("activity_family", "")).replace("_", " ").title()
                bridge_hint = _v22_clean(b.get("bridge_step", ""))[:36]
                suffix = f"{fam} {seen[key]}"
                if bridge_hint and _v22_norm(bridge_hint) not in key:
                    suffix = f"{bridge_hint} {seen[key]}"
                b["title"] = f"{base} — {suffix}"
        out.append(b)
    return out


def generate_category_activity_bank(
    state: Dict[str, Any],
    category_key: str,
    model: str = V21_ACTIVITY_MODEL,
    activities_per_category: Optional[int] = None,
) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG.get(category_key, {}).get("display", category_key.replace("_", " ").title())
    support_tier = get_support_tier(state, category_key) if "get_support_tier" in globals() else "support"
    next_steps = select_next_milestones(state, category_key, max_milestones=V21_MAX_MILESTONES_PER_DOMAIN, min_milestones=V21_MIN_MILESTONES_PER_DOMAIN)
    targets = next_steps.get("milestones", [])
    if not targets:
        bank = {
            "status": "no_targets",
            "version": V22_VERSION,
            "category_key": category_key,
            "category": category_display,
            "support_tier": support_tier,
            "planning_mode": next_steps.get("mode", "no_targets"),
            "summary": next_steps.get("message", "No target milestones were selected."),
            "activities": [],
            "target_milestones": [],
            "validation_warnings": [next_steps.get("message", "no_targets")],
        }
        state.setdefault("activity_banks", {})[category_key] = bank
        return bank

    bridge_plan = build_bridge_plan_for_category(state, category_key, targets)
    active_bridges = bridge_plan.get("active_bridge_steps", [])
    daily_time = int(state.get("child", {}).get("daily_time_min", 10) or 10)
    daily_slots = min(max(1, daily_time // V21_PER_ACTIVITY_MIN), V21_MAX_DAILY_ACTIVITIES)
    desired_week1_slots = V21_WEEK1_DAYS * daily_slots
    # More variants for single-domain/single-skill plans; Week 1 should have enough distinct cards.
    core_variants = max(2, math.ceil(desired_week1_slots / max(1, len(active_bridges))))
    core_variants = min(max(core_variants, 2), 7)

    activities: List[Dict[str, Any]] = []
    for bridge in active_bridges:
        for variant in range(1, core_variants + 1):
            activities.append(_v21_make_activity(category_key, bridge, "core", variant, state))
        activities.append(_v21_make_activity(category_key, bridge, "easier_backup", 1, state))
        activities.append(_v21_make_activity(category_key, bridge, "harder_stretch", 2, state))

    activities = _v21_uniquify_core_titles(activities)
    required = ["title", "cdc_goal", "bridge_step", "why_it_helps", "instructions", "success_criteria", "make_easier", "make_harder", "group_play_line", "what_to_avoid", "materials", "activity_family"]
    warnings = []
    for a in activities:
        for key in required:
            if not str(a.get(key, "")).strip():
                warnings.append(f"missing_{key}:{a.get('title', '')}")
        problems = _v21_activity_has_alignment_problem(a, category_key)
        if problems:
            a["validation_warnings"] = problems
            warnings.extend([f"{a.get('title','activity')}:{p}" for p in problems])

    core_titles = {_v22_title_key(a.get("title", "")) for a in activities if a.get("activity_type") == "core"}
    if len(core_titles) < min(desired_week1_slots, len([a for a in activities if a.get("activity_type") == "core"])):
        warnings.append("week1_unique_title_pool_below_expected_after_generation")

    bank = {
        "status": "ok",
        "version": V22_VERSION,
        "category_key": category_key,
        "category": category_display,
        "support_tier": support_tier,
        "planning_mode": next_steps.get("mode", "evidence_based_answered_gaps"),
        "target_selection_message": next_steps.get("message", ""),
        "summary": f"v22 activity bank for {category_display}: bridge_step_1 only, previous bridge saved for future feedback, activity_family constrained.",
        "bridge_plan": bridge_plan,
        "target_milestones": targets,
        "activities": activities,
        "core_activity_count": len([a for a in activities if a.get("activity_type") == "core"]),
        "unique_core_title_count": len(core_titles),
        "validation_warnings": warnings[:100],
        "min_reasonable_target_month": next_steps.get("min_reasonable_target_month"),
        "skipped_too_young_not_sure": next_steps.get("skipped_too_young_not_sure", []),
    }
    state.setdefault("activity_banks", {})[category_key] = bank
    return bank


def _v22_core_pool(bank: Dict[str, Any]) -> List[Dict[str, Any]]:
    # Prefer valid core cards; keep warned cards as backup if nothing else exists.
    core = [a for a in bank.get("activities", []) if a.get("activity_type") == "core"]
    valid = [a for a in core if not a.get("validation_warnings")]
    return valid or core or bank.get("activities", [])


def _v21_core_pool(bank: Dict[str, Any]) -> List[Dict[str, Any]]:
    return _v22_core_pool(bank)


def _v21_choose_from_bank(bank: Dict[str, Any], used_week_titles: set, used_day_titles: set, used_day_goals: set, counts_by_goal: Counter, counts_by_family: Counter) -> Optional[Dict[str, Any]]:
    pool = _v22_core_pool(bank)
    if not pool:
        return None

    def score(a: Dict[str, Any]):
        title = _v22_title_key(a.get("title", ""))
        goal = _v22_norm(a.get("cdc_goal", ""))
        fam = a.get("activity_family", "")
        return (
            10000 if title in used_week_titles else 0,
            2000 if title in used_day_titles else 0,
            200 if goal in used_day_goals else 0,
            counts_by_goal[goal],
            counts_by_family[fam],
            len(a.get("validation_warnings", [])),
            title,
        )
    candidates = sorted(pool, key=score)
    for a in candidates:
        if _v22_title_key(a.get("title", "")) not in used_week_titles:
            return a
    return None


def build_weekly_schedule(state: Dict[str, Any], cycle_days: int = V21_DEFAULT_CYCLE_DAYS) -> Dict[str, Any]:
    categories = state.get("planned_categories") or state.get("selected_categories", [])
    daily_time = int(state.get("child", {}).get("daily_time_min", 10) or 10)
    if not categories:
        schedule = {"status": "no_categories", "version": V22_VERSION, "summary": "No categories selected for planning.", "days": {}}
        state["weekly_schedule"] = schedule
        return schedule

    for cat in categories:
        if cat not in state.get("activity_banks", {}):
            generate_category_activity_bank(state, cat)

    active_categories = [cat for cat in categories if _v22_core_pool(state.get("activity_banks", {}).get(cat, {}))]
    if not active_categories:
        concern_cats = [cat for cat in categories if _v22_parent_has_concern_for_category(state, cat)]
        msg = "No activity cards could be built."
        if concern_cats:
            msg = "No clear gap was found, but parent concern-support mode should have generated activities. Check table rows/activity_family for the selected concern domains."
        schedule = {
            "status": "no_activities",
            "version": V22_VERSION,
            "summary": msg,
            "daily_time_min": daily_time,
            "bank_repeat_window_days": cycle_days,
            "feedback_options": V21_ACTIVITY_FEEDBACK_OPTIONS,
            "days": {},
            "validation_warnings": [msg],
        }
        state["weekly_schedule"] = schedule
        return schedule

    max_items = min(max(1, daily_time // V21_PER_ACTIVITY_MIN), V21_MAX_DAILY_ACTIVITIES)
    days: Dict[str, Dict[str, Any]] = {}
    counts_by_goal = Counter()
    counts_by_family = Counter()
    week1_titles = set()
    warnings = []

    for d in range(1, min(V21_WEEK1_DAYS, cycle_days) + 1):
        items = []
        used_day_titles = set()
        used_day_goals = set()
        for cat in _v21_slot_categories(active_categories, max_items, d):
            bank = state.get("activity_banks", {}).get(cat, {})
            act = _v21_choose_from_bank(bank, week1_titles, used_day_titles, used_day_goals, counts_by_goal, counts_by_family)
            if not act:
                warnings.append(f"day_{d}:not_enough_unique_week1_activities_for_{cat}")
                continue
            item = _schedule_item_from_activity(cat, act)
            for extra in ["activity_family", "table_bridge_step", "bridge_focus_rule", "previous_bridge_usage", "validation_warnings", "table_review_warning", "llm_activity_writer_used"]:
                item[extra] = act.get(extra, "")
            title_key = _v22_title_key(item.get("title", ""))
            goal_key = _v22_norm(item.get("cdc_goal", ""))
            fam = item.get("activity_family", "")
            week1_titles.add(title_key)
            used_day_titles.add(title_key)
            used_day_goals.add(goal_key)
            counts_by_goal[goal_key] += 1
            counts_by_family[fam] += 1
            items.append(item)
        days[f"day_{d}"] = {"week": 1, "phase": "learn_the_bank_unique_cards", "total_minutes": sum(i.get("duration_min", 0) for i in items), "items": items}

    # Week 2 repeats/adapts Week 1.
    for d in range(V21_WEEK1_DAYS + 1, cycle_days + 1):
        source_day = f"day_{((d - 1) % V21_WEEK1_DAYS) + 1}"
        source_items = days.get(source_day, {}).get("items", [])
        repeated = []
        for item in source_items:
            b = dict(item)
            b["title"] = f"{item.get('title', 'Activity')} — repeat/adapt"
            b["instructions"] = (
                "Repeat the familiar Week 1 activity. Keep the same bridge step, but adjust one thing based on feedback: "
                "use the easier version if it was too hard, the harder version if it was too easy, or a new material/theme if the child disliked it. "
                + str(item.get("instructions", ""))
            )
            repeated.append(b)
        days[f"day_{d}"] = {"week": 2, "phase": "repeat_and_adapt", "repeat_of_day": ((d - 1) % V21_WEEK1_DAYS) + 1, "total_minutes": sum(i.get("duration_min", 0) for i in repeated), "items": repeated}

    expected_week1 = min(V21_WEEK1_DAYS, cycle_days) * max_items
    actual_week1 = len(week1_titles)
    if actual_week1 < expected_week1:
        warnings.append(f"week1_unique_titles_below_expected:{actual_week1}/{expected_week1}")

    bank_warnings = []
    for bank in state.get("activity_banks", {}).values():
        bank_warnings.extend(bank.get("validation_warnings", [])[:20])

    schedule = {
        "status": "ok",
        "version": V22_VERSION,
        "summary": "v22 bridge-step-1 schedule. Week 1 introduces unique activity cards when possible; Week 2 repeats/adapts the same skills based on parent feedback.",
        "daily_time_min": daily_time,
        "bank_repeat_window_days": cycle_days,
        "first_week_distinct_activity_count": actual_week1,
        "expected_week1_activity_slots": expected_week1,
        "feedback_options": V21_ACTIVITY_FEEDBACK_OPTIONS,
        "days": days,
        "validation_warnings": (warnings + bank_warnings)[:100],
    }
    state["weekly_schedule"] = schedule
    return schedule


def display_weekly_schedule(state: Dict[str, Any], max_days: Optional[int] = None):
    sched = state.get("weekly_schedule") or build_weekly_schedule(state)
    print("\n" + "=" * 100)
    print("V22 BRIDGE-STEP-1 WEEKLY SCHEDULE")
    print("=" * 100)
    print("Summary:", sched.get("summary", ""))
    print("Daily time budget:", sched.get("daily_time_min", ""))
    print("Bank repeat window:", sched.get("bank_repeat_window_days", ""), "days")
    print("First-week distinct activities:", sched.get("first_week_distinct_activity_count", ""), "/ expected", sched.get("expected_week1_activity_slots", ""))
    print("Feedback options:", sched.get("feedback_options", {}))
    if sched.get("validation_warnings"):
        print("Validation warnings:")
        for w in sched.get("validation_warnings", [])[:25]:
            print("-", w)
    source = globals().get("V18_BRIDGE_SOURCE_PATH", None) or globals().get("BRIDGE_SOURCE_PATH", None)
    if source:
        print("Source bridge table:", source)

    for cat, bank in state.get("activity_banks", {}).items():
        if bank.get("status") != "ok":
            continue
        print(f"\nCoverage set — {DOMAIN_CONFIG.get(cat, {}).get('display', cat)}:")
        print("  Planning mode:", bank.get("planning_mode", ""))
        if bank.get("target_selection_message"):
            print("  Note:", bank.get("target_selection_message"))
        if bank.get("skipped_too_young_not_sure"):
            print("  Skipped too-young not_sure targets:", bank.get("skipped_too_young_not_sure"))
        for idx, m in enumerate(bank.get("target_milestones", []), start=1):
            print(f"  {idx}. {m.get('months')}m | {m.get('subdomain')} | {m.get('milestone')} [{m.get('answer_norm', 'target')}] | {m.get('selection_reason', '')}")
        bp = bank.get("bridge_plan", {})
        if bp.get("active_bridge_steps"):
            print(f"Bridge focus — {DOMAIN_CONFIG.get(cat, {}).get('display', cat)}:")
            for idx, b in enumerate(bp.get("active_bridge_steps", []), start=1):
                focus, rule = _v21_actionable_focus(state, cat, b)
                print(f"  {idx}. CDC: {b.get('cdc_milestone')} | bridge_step_1: {focus} | activity_family: {b.get('activity_family')} | previous bridge: feedback fallback only | rule: {rule}")

    days = sched.get("days", {})
    day_names = list(days.keys())
    if max_days:
        day_names = day_names[:max_days]
    for day_name in day_names:
        day_info = days.get(day_name, {})
        week = day_info.get("week")
        phase = day_info.get("phase", "")
        extra = f" | Week {week}: {phase.replace('_', ' ')}" if week else ""
        if day_info.get("repeat_of_day"):
            extra += f" | repeats Day {day_info.get('repeat_of_day')} with adaptation"
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min{extra}")
        print("-" * 100)
        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue
        for item in items:
            print(f"- [{item.get('category')}] {item.get('title')} ({item.get('duration_min', 5)} min)")
            print(f"  Type: {item.get('activity_type')} | Family: {item.get('activity_family')} | Theme: {item.get('theme')}")
            print(f"  CDC goal: {item.get('cdc_goal')}")
            print(f"  Bridge step 1: {item.get('bridge_step')}")
            print("  Previous bridge: not used in initial plan; feedback fallback only")
            if item.get("validation_warnings"):
                print(f"  Validation: {item.get('validation_warnings')}")
            print(f"  Why: {item.get('why_it_helps')}")
            print(f"  Instructions: {item.get('instructions')}")
            print(f"  Success: {item.get('success_criteria')}")
            print(f"  Easier: {item.get('make_easier')}")
            print(f"  Harder: {item.get('make_harder')}")
            print(f"  Group play: {item.get('group_play_line')}")
            print(f"  Avoid: {item.get('what_to_avoid')}")
            print(f"  Materials: {item.get('materials')}")


# Patch run_downstream_planning so it does not block parent-concern support plans only because dev_age is None.
def run_downstream_planning(
    state: Dict[str, Any],
    display_schedule: bool = True,
    selected_categories: Optional[List[str]] = None,
) -> "pd.DataFrame":
    if "determine_family_guidance_floor" in globals():
        determine_family_guidance_floor(state)

    dev_age_map = state.get("dev_age", {})
    categories_to_plan = []
    for category_key in list(selected_categories or []):
        if dev_age_map.get(category_key) is not None or _v22_parent_has_concern_for_category(state, category_key):
            categories_to_plan.append(category_key)
        else:
            print(
                f"Skipping {DOMAIN_CONFIG[category_key]['display']} in planning "
                f"because no developmental age was estimated and no direct parent concern was mapped."
            )

    floor_category = state.get("family_guidance_floor", {}).get("category_key")
    if floor_category and floor_category not in categories_to_plan:
        if dev_age_map.get(floor_category) is not None or _v22_parent_has_concern_for_category(state, floor_category):
            categories_to_plan.append(floor_category)

    if not categories_to_plan:
        for category_key in DOMAIN_CONFIG:
            if dev_age_map.get(category_key) is not None or _v22_parent_has_concern_for_category(state, category_key):
                categories_to_plan.append(category_key)

    state["planned_categories"] = categories_to_plan

    for category_key in categories_to_plan:
        generate_category_activity_bank(state, category_key)

    # Legacy allocation can run, but v22 build_weekly_schedule is the final source of truth.
    try:
        allocate_weekly_slots(state)
    except Exception:
        pass
    build_weekly_schedule(state)

    summary_df = summarizer_agent(state) if "summarizer_agent" in globals() else pd.DataFrame()
    if display_schedule:
        display_weekly_schedule(state)
    return summary_df


print("Loaded Genex Brain v22 patch:", V22_VERSION)
print("- Final schedule/display is V22 only.")
print("- Initial activities target bridge_step_number=1 only; previous_bridge_step is future feedback fallback only.")
print("- No-clear-gap + parent concern creates concern-support activities.")
print("- Week 1 enforces unique activity cards when possible; Week 2 repeats/adapts.")
print("- activity_family is a hard guardrail for activity writing and validation.")
print("- Target selection avoids much-younger not_sure regression unless regression/loss is explicitly mentioned.")


Loaded Genex Brain v18 table-first bridge source: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx
Loaded bridge sheet/table: all_with_bridge_family
Bridge rows: 369
Deduped CDC milestones: 163
Rows with parent_explanation: 369
Rows with previous_bridge_step: 359
Rows with explicit allowed_themes: 0
Rows with explicit what_to_avoid: 0
v21 table-ready patch loaded bridge table: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx
Activity families in table: 128
Activity family legend entries loaded: 131
Loaded Genex Brain v22 patch: v22_claude_ready_first_bridge_concern_support
- Final schedule/display is V22 only.
- Initial activities target bridge_step_number=1 only; previous_bridge_step is future feedback fallback only.
- No-clear-gap + parent concern creates concern-support activities.
- Week 1 enforces unique activity cards when possible; Week 2 repeats/adapts.
- activity_fam

## v22 quick checks

In [58]:

# v17 quick smoke check — uses one small synthetic state and deterministic fallback if OpenAI is not available.
# Uncomment to run manually.

# smoke_state = new_state()
# smoke_state["child"] = {
#     "name": "Test Child",
#     "chronological_months": 36,
#     "diagnosis": "No formal diagnosis",
#     "concern": "speech delay and difficulty jumping with both feet",
#     "daily_time_min": 15,
# }
# smoke_state["delay_estimates"] = {
#     k: {"delay_months": 6 if k in {"language_and_communication", "movement_and_physical"} else 0, "reason": "smoke test"}
#     for k in DOMAIN_CONFIG
# }
# smoke_state["selected_categories"] = ["language_and_communication", "movement_and_physical"]
# smoke_state["dev_age"] = {"language_and_communication": 24, "movement_and_physical": 24}
# smoke_state["qna"] = {
#     "language_and_communication": [
#         {"months": 24, "milestone": "says about 50 words", "subdomain": "expressive_language", "norm_answer": "no", "scoring_norm_answer": "no", "answer_status": "ok", "score": 0},
#         {"months": 24, "milestone": "says two words together like more milk", "subdomain": "expressive_language", "norm_answer": "no", "scoring_norm_answer": "no", "answer_status": "ok", "score": 0},
#     ],
#     "movement_and_physical": [],
# }
# determine_family_guidance_floor(smoke_state)
# smoke_state["planned_categories"] = smoke_state["selected_categories"]
# for cat in smoke_state["planned_categories"]:
#     generate_category_activity_bank(smoke_state, cat)
# allocate_weekly_slots(smoke_state)
# build_weekly_schedule(smoke_state)
# display_weekly_schedule(smoke_state)
# display(summarizer_agent(smoke_state))
# print(generate_parent_progress_summary(smoke_state))


## Example run cells

Use **only one** of the cells below at a time.


In [59]:
# Example 1 — one fully live run
# state, summary_df = run_all_categories_live()
# display(summary_df)


In [60]:
# # Example 2 — one synthetic case using Gemini as the parent
# case = df_cases.iloc[0].to_dict()
# state, summary_df = run_all_categories_gemini_parent(
#     case,
#     daily_time_min=10,
#     gemini_model="gemini-2.5-flash"
# )
# display(summary_df)


In [61]:
# # V9 test case 1 — Emma (speech/apraxia stress test)
# case = next(c for c in cases if c["case_id"] == "C08")

# state, summary_df = run_all_categories_gemini_parent(
#     case,
#     daily_time_min=10,
#     gemini_model="gemini-2.5-flash",
# )

# display(summary_df)
# display_weekly_schedule(state)


In [ ]:
# Example batch run disabled by default in v22.
# Uncomment manually after setting API keys.
case_ids = ["C05", "C08", "C11", "C14"]

results = {}

for case_id in case_ids:
    print(f"\n### Running case {case_id}")

    case = next(c for c in cases if c["case_id"] == case_id)

    state, summary_df = run_all_categories_gemini_parent(
        case,
        daily_time_min=10,
        gemini_model="gemini-2.5-flash",
    )

    results[case_id] = {
        "state": state,
        "summary_df": summary_df
    }

    display(summary_df)
    display_weekly_schedule(state)


### Running case C05

CHILD REFERENCE CHECK
Name: Ethan
Chronological age: 54 months
Diagnosis / condition: Cerebral palsy
Parent concern: Trouble keeping up in preschool, Self-care delays, frequent falls, difficulty with stairs, uses walker sometimes, good cognition
Daily time available: 10 minutes

CONCERN ROUTER PROFILE
Domain weights:
- Movement / Physical: 1.0
Top subdomains:
- gross_motor_mobility_and_coordination: 1.0
- postural_control_and_transitions: 0.4
- self_help_motor_skills: 0.35

DELAY ESTIMATOR AGENT
Movement / Physical: 18 month starting delay | Cerebral palsy and associated mobility challenges likely impact physical development significantly.
Social / Emotional: 6 month starting delay | Social interactions may be impacted by mobility challenges and self-care delays.
Language / Communication: 6 month starting delay | Cerebral palsy may impact speech clarity and expressive language skills.
Cognitive / Adaptive: 12 month starting delay | Cerebral palsy may impact cogni

,child_name,chronological_age_months,diagnosis,concern,daily_time_min,category,estimated_delay_months,delay_reason,raw_dev_age_months,estimated_dev_age_months,...,weekly_target_minutes_for_category,activity_bank_status,activity_bank_summary,v17_cdc_targets,v17_active_bridge_steps,v17_bridge_table_path,selected_for_onboarding,assessed_in_this_run,family_guidance_floor_active,family_guidance_summary
0,Ethan,54,Cerebral palsy,"Trouble keeping up in preschool, Self-care del...",10,Movement / Physical,18,Cerebral palsy and associated mobility challen...,36.0,36.0,...,0,ok,v22 activity bank for Movement / Physical: bri...,,,,True,True,False,
1,Ethan,54,Cerebral palsy,"Trouble keeping up in preschool, Self-care del...",10,Social / Emotional,6,Social interactions may be impacted by mobilit...,NaN,NaN,...,0,missing,,,,,False,False,False,
2,Ethan,54,Cerebral palsy,"Trouble keeping up in preschool, Self-care del...",10,Language / Communication,6,Cerebral palsy may impact speech clarity and e...,NaN,NaN,...,0,missing,,,,,False,False,False,
3,Ethan,54,Cerebral palsy,"Trouble keeping up in preschool, Self-care del...",10,Cognitive / Adaptive,12,Cerebral palsy may impact cognitive/adaptive s...,NaN,NaN,...,0,missing,,,,,False,False,False,



V22 BRIDGE-STEP-1 WEEKLY SCHEDULE
Summary: v22 bridge-step-1 schedule. Week 1 introduces unique activity cards when possible; Week 2 repeats/adapts the same skills based on parent feedback.
Daily time budget: 10
Bank repeat window: 14 days
First-week distinct activities: 14 / expected 14
Feedback options: {'difficulty': ['too_hard', 'just_right', 'too_easy'], 'performance': ['done_independently', 'done_with_help', 'couldnt_do_it'], 'engagement': ['enjoyed_it', 'resisted_it', 'didnt_like_it']}
Source bridge table: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx

Coverage set — Movement / Physical:
  Planning mode: evidence_based_answered_gaps
  Note: v22 selected 5 answered gap milestone(s) for bridge_step_1 planning.
  1. 30m | gross_motor_mobility_and_coordination | jumps off the ground with both feet [no] | parent_answer_no
  2. 36m | self_help_motor_skills | puts on some clothes by himself like loose pants or a jacket [with_h

,child_name,chronological_age_months,diagnosis,concern,daily_time_min,category,estimated_delay_months,delay_reason,raw_dev_age_months,estimated_dev_age_months,...,weekly_target_minutes_for_category,activity_bank_status,activity_bank_summary,v17_cdc_targets,v17_active_bridge_steps,v17_bridge_table_path,selected_for_onboarding,assessed_in_this_run,family_guidance_floor_active,family_guidance_summary
0,Emma,32,Suspected childhood apraxia of speech,"Very limited speech output, Uses gestures, get...",10,Movement / Physical,2,Suspected childhood apraxia of speech may lead...,NaN,NaN,...,0,missing,,,,,False,False,False,
1,Emma,32,Suspected childhood apraxia of speech,"Very limited speech output, Uses gestures, get...",10,Social / Emotional,6,Limited speech output and frustration may hind...,NaN,NaN,...,0,missing,,,,,False,False,False,
2,Emma,32,Suspected childhood apraxia of speech,"Very limited speech output, Uses gestures, get...",10,Language / Communication,12,Suspected childhood apraxia of speech likely i...,24.0,15.0,...,0,ok,v22 activity bank for Language / Communication...,,,,True,True,False,
3,Emma,32,Suspected childhood apraxia of speech,"Very limited speech output, Uses gestures, get...",10,Cognitive / Adaptive,6,Limited speech output and frustration may indi...,NaN,NaN,...,0,missing,,,,,False,False,False,



V22 BRIDGE-STEP-1 WEEKLY SCHEDULE
Summary: v22 bridge-step-1 schedule. Week 1 introduces unique activity cards when possible; Week 2 repeats/adapts the same skills based on parent feedback.
Daily time budget: 10
Bank repeat window: 14 days
First-week distinct activities: 14 / expected 14
Feedback options: {'difficulty': ['too_hard', 'just_right', 'too_easy'], 'performance': ['done_independently', 'done_with_help', 'couldnt_do_it'], 'engagement': ['enjoyed_it', 'resisted_it', 'didnt_like_it']}
Source bridge table: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx

Coverage set — Language / Communication:
  Planning mode: evidence_based_answered_gaps
  Note: v22 selected 3 answered gap milestone(s) for bridge_step_1 planning.
  1. 18m | expressive_language | tries to say three or more words besides mama or dada [no] | parent_answer_no
  2. 24m | expressive_language | says at least two words together like more milk [no] | parent_answ

,child_name,chronological_age_months,diagnosis,concern,daily_time_min,category,estimated_delay_months,delay_reason,raw_dev_age_months,estimated_dev_age_months,...,weekly_target_minutes_for_category,activity_bank_status,activity_bank_summary,v17_cdc_targets,v17_active_bridge_steps,v17_bridge_table_path,selected_for_onboarding,assessed_in_this_run,family_guidance_floor_active,family_guidance_summary
0,Aria,20,No diagnosis,"Only ~10 words, very active, good eye contact,...",10,Movement / Physical,0,The child is very active and demonstrates good...,NaN,NaN,...,0,missing,,,,,False,False,False,
1,Aria,20,No diagnosis,"Only ~10 words, very active, good eye contact,...",10,Social / Emotional,0,The child shows good social engagement and und...,NaN,NaN,...,0,missing,,,,,False,False,False,
2,Aria,20,No diagnosis,"Only ~10 words, very active, good eye contact,...",10,Language / Communication,2,Child shows good understanding but may have sl...,18.0,17.0,...,0,ok,v22 activity bank for Language / Communication...,,,,True,True,False,
3,Aria,20,No diagnosis,"Only ~10 words, very active, good eye contact,...",10,Cognitive / Adaptive,0,"Child shows good understanding and engagement,...",24.0,24.0,...,0,ok,v22 activity bank for Cognitive / Adaptive: br...,,,,True,True,False,



V22 BRIDGE-STEP-1 WEEKLY SCHEDULE
Summary: v22 bridge-step-1 schedule. Week 1 introduces unique activity cards when possible; Week 2 repeats/adapts the same skills based on parent feedback.
Daily time budget: 10
Bank repeat window: 14 days
First-week distinct activities: 14 / expected 14
Feedback options: {'difficulty': ['too_hard', 'just_right', 'too_easy'], 'performance': ['done_independently', 'done_with_help', 'couldnt_do_it'], 'engagement': ['enjoyed_it', 'resisted_it', 'didnt_like_it']}
Source bridge table: /Users/sara/Projects/Genex/notebooks/mvp/cdc_milestones_with_bridges_family_cleaned_final_app_ready.xlsx

Coverage set — Language / Communication:
  Planning mode: parent_concern_support_no_clear_gap
  Note: No clear gap was found in this quick screen for Language / Communication, but because the parent named this concern, v22 selected closest age-appropriate related milestones for a light support plan.
  1. 24m | expressive_language | says at least two words together like mo

In [63]:
# Example 3A — export advisor packet in live mode
# all_payloads, advisor_rating_df, pdf_path = run_cases_and_export(
#     df_cases=df_cases,
#     mode="live",
#     output_dir="outputs/advisor_case_pack_v22",
#     default_daily_time_min=10,
#     pause_between_cases=True,
# )
# display(advisor_rating_df.head())
# pdf_path


In [64]:
# Example 3B — export advisor packet in Gemini synthetic-parent mode, in batches of 5 with 2 min gap
# all_payloads, advisor_rating_df, pdf_path = run_cases_and_export_in_batches(
#     df_cases=df_cases,
#     batch_size=5,
#     gap_between_batches_sec=120,
#     mode="gemini",
#     output_dir="outputs/advisor_case_pack_v22",
#     default_daily_time_min=10,
#     pause_between_cases=False,
#     gemini_model="gemini-2.5-flash",
# )
# display(advisor_rating_df.head())
# pdf_path


In [ ]:
adhd_cases = [
    {
        "case_id": "A01",
        "child_name": "Mason",
        "age_months": 36,
        "age_label": "3y 0m",
        "diagnosis": "No diagnosis",
        "concerns": "Very active, hard to sit for meals or stories, big tantrums when stopped"
    },
    {
        "case_id": "A02",
        "child_name": "Chloe",
        "age_months": 42,
        "age_label": "3y 6m",
        "diagnosis": "No formal diagnosis",
        "concerns": "Difficulty with transitions, runs off in public, interrupts constantly, emotional outbursts when routines change"
    },
    {
        "case_id": "A03",
        "child_name": "Jayden",
        "age_months": 48,
        "age_label": "4y 0m",
        "diagnosis": "Attention and self-regulation concerns",
        "concerns": "Easily distracted, does not finish tasks, seems not to listen unless one-on-one, forgets simple instructions"
    },
    {
        "case_id": "A04",
        "child_name": "Sienna",
        "age_months": 54,
        "age_label": "4y 6m",
        "diagnosis": "Possible ADHD",
        "concerns": "Cannot wait her turn, blurts out answers, very talkative, struggles during group activities in preschool"
    },
    {
        "case_id": "A05",
        "child_name": "Owen",
        "age_months": 60,
        "age_label": "5y 0m",
        "diagnosis": "ADHD",
        "concerns": "Trouble following multi-step directions, frustrated by fixed schedules and rigid invironment, intrupts the class so much, imaginative and verbally strong"
    }
]

In [ ]:
df_adhd_cases = pd.DataFrame(adhd_cases)
df_adhd_cases

In [ ]:
# ADHD batch run disabled by default in v22.
# Uncomment manually after setting API keys.
# # Run one by one like a partial case 
# adhd_case_ids = ["A01", "A02", "A03", "A04", "A05"]
#
# adhd_results = {}
#
# for case_id in adhd_case_ids:
#     print(f"\n### Running case {case_id}")
#
#     case = next(c for c in adhd_cases if c["case_id"] == case_id)
#
#     state, summary_df = run_all_categories_gemini_parent(
#         case,
#         daily_time_min=10,
#         gemini_model="gemini-2.5-flash",
#     )
#
#     adhd_results[case_id] = {
#         "state": state,
#         "summary_df": summary_df
#     }
#
#     display(summary_df)
#     display_weekly_schedule(state)

# How We Designed This Notebook and How the Current Algorithm Works (v22)

We built this notebook as a structured prototype of the Genex developmental interview and home-support engine for children ages 0–5. Version 17 keeps the shorter focused-onboarding workflow from v14, while making the first-pass estimates a little more robust.

## What changed in v22

The main changes are:

- **focused onboarding**
  - the system ranks likely affected domains from the child profile and concern text
  - onboarding evaluates only the top **1–2 domains**
  - this keeps the first experience shorter and more parent-friendly

- **no follow-up questions after milestone answers**
  - milestone answers stay simple: **yes / sometimes / with_help / no / not_sure**
  - v22 avoids adding extra clarification prompts that lengthen the interview

- **slightly deeper question budget + smarter early stopping**
  - v22 asks a few more milestone questions within the selected domains
  - the interview does not stop after only confirming lower bands
  - it tries to confirm a likely level and also test the next higher band before stopping early

- **first-plan transparency**
  - the output makes it clearer that onboarding focused only on selected domains
  - other areas can be reviewed later

## Core pipeline

The shared pipeline is:

1. collect or load the child profile  
2. route diagnosis/concern text into weighted **subdomain** signals  
3. build a broad **safety/practical constraint profile** from the same intake text  
4. estimate a starting delay by developmental domain  
5. triage likely focus domains and select the top **1–2** for onboarding  
6. select milestone questions within those domains using developmental-age proximity plus concern/subdomain relevance  
7. ask a smaller set of milestone questions with no extra follow-up prompts  
8. stop once the estimate is stable enough for a useful first plan  
9. derive a developmental age per reviewed domain from the highest sufficiently supported band  
10. assign support tiers and build a first activity plan for the reviewed domains  
11. apply the family-guidance floor when no reviewed category needs scheduled support

## Product philosophy

Version 17 treats onboarding as a **focused first-plan workflow**, not a full developmental census on day one.

That means the goal is:
- identify the most relevant areas to start with
- gather enough milestone signal to make a useful first plan
- keep the process short enough that parents will actually complete onboarding

The goal is **not** to fully profile all domains in the first session.


# Genex Advisor Review Handout  

## Developmental Interview + Home Support Prototype (Ages 0–5, v22)

Thank you for reviewing this Genex prototype.

This packet contains sample child cases produced by our current developmental interview and home-support planning system for children ages 0–5. At this stage, our goal is to evaluate whether the logic, developmental estimates, concern-routing behavior, support scoring, and suggested home activities feel clinically reasonable, practical, and safe.

### What is new in v22
Version 17 keeps the v11 family-guidance floor and playdate bonus rule, while using a shorter onboarding flow than the earlier full-domain versions. The main changes are:
- onboarding focuses on the top **1–2 domains** instead of all 4
- milestone answers stay simple: **yes / sometimes / with_help / no / not_sure**
- the first-pass interview is shorter, but asks enough to avoid stopping after only lower-band confirmation
- the weekly plan makes it clearer that this is a focused first onboarding pass, and that other domains can be reviewed later

### What this prototype is trying to do
This prototype is designed to:
- take a child profile and parent concern
- ask structured milestone questions in the most relevant domains first
- estimate developmental level across the reviewed domains
- assign one of three core support tiers
- still give families a practical home plan even when no category meets the threshold for scheduled support
- keep the core weekly schedule focused on short, repeatable, at-home activities

It is **not diagnostic** and does **not replace clinical judgment**.


# Short Cover Note

**Dear Advisor,**

Thank you for reviewing this early Genex prototype.

This packet contains sample child cases generated through our current developmental interview and home-support planning workflow for ages 0-5. Version 17 keeps the stable milestone backbone and home-planning logic, but changes onboarding to focus on only the top 1–2 likely domains, removes follow-up questions after milestone answers, and reduces question count through a smaller question budget plus early stopping.

This is still a prototype and is not intended to diagnose or replace clinical judgment. We are using this review process to understand where the system is working well, where it is too aggressive or too conservative, and what should be revised before further development.

Your feedback on milestone selection, focus-domain triage, developmental-age estimates, support tiers, activity quality, and whether the shorter onboarding still captures the right signal would be incredibly valuable.

**Best,**  
**Sara / Genex**
